# 7팀 통합 분석 탐색 기록

이 파일은 제공받은 노트북의 **공개용 복사본**을 보관한 탐색 기록입니다. 기존 표현·잠정 결론·가설 탐색 시도와 중간 변수가 포함되어 있습니다. 원본 다운로드 파일은 수정하지 않았습니다.

현재 확인된 집계와 해석 기준은 [검증된 집계 결과 요약](../00_key_results.ipynb)과 [README](../../README.md)를 우선합니다. 수치·분모·관찰창의 차이는 [검토 기록](../../docs/review-notes.md)에 정리했습니다.

개인별 행이 포함될 수 있는 출력, 실행 번호, 계정 연결은 제거했습니다. 원본 분석 전체를 위에서 아래로 실행하는 것은 검증하지 않았으며, 입력 마트·월별 이벤트·콘텐츠 자료와 실행 환경이 별도로 필요합니다. [보관본 실행 범위](README.md)를 먼저 확인하세요.

원본 셀 번호는 각 셀의 `source_cell_index` 메타데이터로 확인할 수 있습니다. 전체 코드의 개인 단독 저작을 의미하지 않습니다.

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'
try:
    import plotly.io as pio
    pio.renderers.default = 'plotly_mimetype'
except Exception:
    pass


# 교육 구독 서비스 무료체험 폐지 전/후 전환·리텐션 통합 분석

본 노트북은 무료체험 폐지 전/후의 가입·첫 결제 성과와 결제 이후 리텐션 변화를 연결해 분석하는 최종 보고서형 노트북입니다.


## 프로젝트 개요

### 분석 배경

무료체험 폐지 이후 가입자 수는 증가했지만, 가입 후 30일 내 첫 결제율은 하락했습니다.  
따라서 정책 성과를 단순 가입자 수나 첫 결제율만으로 판단하기 어렵고, 결제 이후 유저 품질과 정기구독 유지까지 함께 확인할 필요가 있습니다.

### 핵심 질문

1. 무료체험 폐지 전/후 가입·결제 구조는 어떻게 달라졌는가?
2. 첫 결제 전환에 영향을 주는 가입 초기 학습 행동은 무엇인가?
3. 무료체험 폐지 후 첫 결제자의 구독 품질은 개선되었는가?
4. 결제 이후 정기구독 유지에 영향을 주는 행동 신호는 무엇인가?
5. 최종적으로 어떤 유저에게 어떤 액션을 우선 적용해야 하는가?

### 분석 기준

| 항목 | 내용 |
|---|---|
| 분석 주제 | 무료체험 폐지 전/후 첫 결제 전환 및 정기구독 리텐션 분석 |
| 분석 단위 | 가입 유저 단위, 첫 결제자 단위, 콘텐츠 단위 |
| 분석 기간 | 폐지 전: 2022.05 - 2022.11 / 폐지 후: 2023.05 - 2023.11 |
| 첫 결제 전환 기준 | 가입 후 30일 내 첫 결제 여부 |
| 초기 행동 기준 | 가입 후 7일 내 콘텐츠·레슨 행동 |
| 리텐션 기준 | 첫 결제 후 7일/30일/60일 내 학습·취소·정기결제 행동 |
| 핵심 전환 신호 | 가입 후 7일 내 레슨 7개 이상 완료 |
| 핵심 리텐션 신호 | 결제 후 30일 내 6일 이상 학습 활동 |
| 데이터 한계 | 취소 버튼 클릭은 실제 해지가 아닌 해지 의도를 추정하는 액션이며, 일부 결제수단/콘텐츠 메타데이터는 결측 또는 보조 지표로만 활용 |


## 1. Google Colab 실행 환경 세팅

첫 결제 전환 분석용 마트와 리텐션 분석용 마트를 함께 로드합니다.  
최종 노트북은 Google Drive에 CSV 파일을 둔 상태에서 위에서 아래로 순서대로 실행되는 것을 전제로 합니다.


In [ ]:
# 저장소 정리: Google Drive 자동 연결을 제거했습니다. 로컬 입력 파일을 사용하세요.


In [ ]:
# 분석 라이브러리 로드
# 데이터 처리, 시각화, 파일 경로 탐색, 결과 표시를 위한 기본 환경을 설정합니다.
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

# 비율 라벨 생성 함수
def add_percent_label(df, col='비율(%)'):
    result = df.copy()
    result['비율 라벨'] = result[col].round(1).astype(str) + '%'
    return result

# 그래프 공통 레이아웃 함수
def apply_common_layout(fig, height=560, y_max=None, xangle=-15):
    fig.update_traces(textposition='outside')
    fig.update_layout(
        height=height,
        xaxis_tickangle=xangle,
        legend_title_text='',
        margin=dict(l=40, r=40, t=80, b=80)
    )
    if y_max is not None:
        fig.update_yaxes(range=[0, y_max])
    return fig


In [ ]:
# 데이터 파일 위치 탐색 및 통합 로드
# 첫 결제 전환 분석용 데이터와 리텐션 분석용 데이터를 함께 로드합니다.
possible_dirs = [
    Path('data/private'),
    Path('data/private'),
    Path('data/private'),
    Path('.'),
    Path('./data'),
    Path('data/private')  # 로컬/샌드박스 실행 검증용 보조 경로
]

def find_data_file(file_candidates, required=True):
    for directory in possible_dirs:
        for file_name in file_candidates:
            file_path = directory / file_name
            if file_path.exists():
                return file_path

    if required:
        raise FileNotFoundError(
            f"{file_candidates} 파일을 찾을 수 없습니다. "
            "Google Drive의 data/private"
        )
    return None

activation_path = find_data_file(['dm_user_activation_payment.csv'] + [f'dm_user_activation_payment({i}).csv' for i in range(1, 21)])
content_path = find_data_file(['dm_content_7d_performance.csv'] + [f'dm_content_7d_performance({i}).csv' for i in range(1, 21)])
retention_path = find_data_file(['dm_user_subscription_retention_core.csv'] + [f'dm_user_subscription_retention_core({i}).csv' for i in range(1, 21)])

data_dir = activation_path.parent

print(f'첫 결제 전환 마트 경로: {activation_path}')
print(f'콘텐츠 성과 마트 경로: {content_path}')
print(f'리텐션 마트 경로: {retention_path}')

# 데이터 로드
user = pd.read_csv(activation_path, low_memory=False)
content = pd.read_csv(content_path, low_memory=False)
retention = pd.read_csv(retention_path, low_memory=False)

# 첫 결제 전환 분석용 데이터 날짜형 변환
for col in ['signup_time', 'first_payment_time', 'cohort_month']:
    if col in user.columns:
        user[col] = pd.to_datetime(user[col], errors='coerce')

if 'cohort_month' in content.columns:
    content['cohort_month'] = pd.to_datetime(content['cohort_month'], errors='coerce')

# 첫 결제 전환 분석용 숫자형 컬럼 변환
activation_numeric_cols = [
    'paid_30d', 'first_plan_price', 'first_paid_amount', 'first_discount_amount',
    'content_view_7d', 'content_start_7d', 'lesson_enter_7d', 'lesson_complete_7d',
    'content_end_7d', 'review_click_7d', 'question_click_7d', 'payment_page_enter_7d',
    'lesson_complete_prepay_7d', 'content_end_prepay_7d', 'unique_content_prepay_7d'
]

for col in activation_numeric_cols:
    if col in user.columns:
        user[col] = pd.to_numeric(user[col], errors='coerce')

# 행동 횟수 결측값 0 처리
activation_behavior_cols = [
    'content_view_7d', 'content_start_7d', 'lesson_enter_7d', 'lesson_complete_7d',
    'content_end_7d', 'review_click_7d', 'question_click_7d', 'payment_page_enter_7d',
    'lesson_complete_prepay_7d', 'content_end_prepay_7d', 'unique_content_prepay_7d'
]

for col in activation_behavior_cols:
    if col in user.columns:
        user[col] = user[col].fillna(0)

if 'paid_30d' in user.columns:
    user['paid_30d'] = user['paid_30d'].fillna(0).astype(int)

# 첫 결제 전환 분석 코드와 호환되는 기간 라벨
period_label = {
    'pre_2022_05_11': '폐지 전',
    'post_2023_05_11': '폐지 후'
}

user['기간'] = user['period_group'].map(period_label)
content['기간'] = content['period_group'].map(period_label)

# 리텐션 분석용 데이터 날짜형 변환
retention_date_cols = [
    'signup_time', 'first_payment_time', 'first_payment_month',
    'first_renew_time', 'first_cancel_time', 'first_resubscribe_time'
]

for col in retention_date_cols:
    if col in retention.columns:
        retention[col] = pd.to_datetime(retention[col], errors='coerce')

# 리텐션 분석용 숫자형 컬럼 변환
retention_numeric_cols = [
    'is_pre_retention_comparable', 'is_post_retention_comparable', 'is_post_removed',
    'can_observe_30d', 'can_observe_60d', 'can_observe_90d', 'can_observe_180d',
    'first_plan_price', 'first_paid_amount', 'first_discount_amount', 'first_discount_rate',
    'complete_subscription_count', 'has_multiple_complete_subscription',
    'lesson_complete_postpay_7d', 'lesson_complete_postpay_30d',
    'content_end_postpay_7d', 'content_end_postpay_30d',
    'active_learning_days_postpay_30d', 'days_to_first_learning_after_payment',
    'days_to_first_renew', 'renew_count_after_payment',
    'renewed_7d', 'renewed_30d', 'renewed_60d', 'renewed_90d',
    'days_to_first_cancel', 'cancel_count_after_payment',
    'cancel_7d', 'cancel_30d', 'cancel_60d', 'cancel_90d', 'cancel_before_first_renew',
    'days_to_first_resubscribe', 'resubscribe_count_after_payment',
    'resubscribed_90d', 'resubscribed_180d'
]

for col in retention_numeric_cols:
    if col in retention.columns:
        retention[col] = pd.to_numeric(retention[col], errors='coerce')

# 리텐션 flag 결측 처리
retention_flag_cols = [
    'is_pre_retention_comparable', 'is_post_retention_comparable', 'is_post_removed',
    'can_observe_30d', 'can_observe_60d', 'can_observe_90d', 'can_observe_180d',
    'has_multiple_complete_subscription',
    'renewed_7d', 'renewed_30d', 'renewed_60d', 'renewed_90d',
    'cancel_7d', 'cancel_30d', 'cancel_60d', 'cancel_90d', 'cancel_before_first_renew',
    'resubscribed_90d', 'resubscribed_180d'
]

for col in retention_flag_cols:
    if col in retention.columns:
        retention[col] = retention[col].fillna(0).astype(int)

# 리텐션 기간 및 생애주기 세그먼트 라벨
retention_period_label = {
    'pre_free_trial_removed': '폐지 전',
    'post_free_trial_removed': '폐지 후'
}

segment_label = {
    'retention_core': '리텐션 우수 그룹',
    'renew_then_cancel': '중도 이탈 그룹',
    'early_cancel_before_renew': '초기 이탈 위험 그룹',
    'cancel_then_resubscribe': '취소 후 재구독 그룹',
    'silent_no_followup': '후속 행동 없음 그룹',
    'other': '기타'
}

segment_order = [
    'retention_core',
    'renew_then_cancel',
    'early_cancel_before_renew',
    'cancel_then_resubscribe',
    'silent_no_followup',
    'other'
]

retention['기간'] = retention['period_group'].map(retention_period_label).fillna(retention['period_group'])
retention['세그먼트'] = retention['subscription_lifecycle_segment'].map(segment_label).fillna(retention['subscription_lifecycle_segment'])

# 폐지 후 전체 리텐션 분석군 생성
post_retention = retention[retention['is_post_removed'] == 1].copy()

# 결제 후 학습 여부 및 학습 활동일 구간 생성
post_retention['결제후_7일내_학습여부'] = np.where(
    (post_retention['lesson_complete_postpay_7d'] > 0) |
    (post_retention['content_end_postpay_7d'] > 0),
    '7일 내 학습 O',
    '7일 내 학습 X'
)

post_retention['결제후_30일내_학습활동일구간'] = pd.cut(
    post_retention['active_learning_days_postpay_30d'].fillna(0),
    bins=[-1, 0, 2, 5, 10, 20, np.inf],
    labels=['0일', '1~2일', '3~5일', '6~10일', '11~20일', '21일 이상']
)

# 공통 리텐션 요약 함수
def make_retention_summary(df, group_col):
    summary_rows = []
    total_users = df['user_id'].nunique()

    for group_value, temp in df.groupby(group_col, dropna=False, observed=False):
        temp_60d = temp[temp['can_observe_60d'] == 1]
        user_count = temp['user_id'].nunique()

        summary_rows.append({
            group_col: group_value,
            '유저 수': user_count,
            '유저 비중(%)': user_count / total_users * 100 if total_users > 0 else np.nan,
            '60일 관측 가능 유저 수': temp_60d['user_id'].nunique(),
            '60일 정기결제율(%)': temp_60d['renewed_60d'].mean() * 100 if len(temp_60d) > 0 else np.nan,
            '30일 내 취소 버튼 클릭률(%)': temp['cancel_30d'].mean() * 100 if len(temp) > 0 else np.nan,
            '정기결제 전 취소 버튼 클릭률(%)': temp['cancel_before_first_renew'].mean() * 100 if len(temp) > 0 else np.nan,
            '리텐션 우수 그룹 비중(%)': (temp['subscription_lifecycle_segment'] == 'retention_core').mean() * 100 if len(temp) > 0 else np.nan,
            '초기 이탈 위험 그룹 비중(%)': (temp['subscription_lifecycle_segment'] == 'early_cancel_before_renew').mean() * 100 if len(temp) > 0 else np.nan,
            '후속 행동 없음 그룹 비중(%)': (temp['subscription_lifecycle_segment'] == 'silent_no_followup').mean() * 100 if len(temp) > 0 else np.nan,
            '30일 내 레슨 완료 수 중앙값': temp['lesson_complete_postpay_30d'].median(),
            '30일 내 학습 활동일 수 중앙값': temp['active_learning_days_postpay_30d'].median(),
            '첫 결제금액 중앙값': temp['first_paid_amount'].median()
        })

    return pd.DataFrame(summary_rows)

print('데이터 로드 및 통합 분석 변수 변환 완료')
print(f'유저 단위 첫 결제 전환 마트: {user.shape[0]:,}행 × {user.shape[1]:,}컬럼')
print(f'콘텐츠 성과 마트: {content.shape[0]:,}행 × {content.shape[1]:,}컬럼')
print(f'리텐션 마트: {retention.shape[0]:,}행 × {retention.shape[1]:,}컬럼')


## 2. 데이터 구성 및 정합성 확인

본 분석은 2가지 종류의 데이터 마트를 함께 사용합니다.

- **첫 결제 전환 분석용 데이터**: 가입 유저 단위로 가입 후 7일 내 행동과 가입 후 30일 내 첫 결제 여부를 확인
- **리텐션 분석용 데이터**: 첫 결제자 단위로 결제 후 학습 행동, 취소 버튼 클릭, 정기결제 여부를 확인

먼저 2가지 데이터의 분석 단위, 중복, 관측 기간, 주요 컬럼 결측 여부를 점검합니다.


### 2-1. 첫 결제 전환 분석용 데이터 점검


In [ ]:
# 첫 결제 전환 분석용 데이터 구성 및 분석 기준 확인
activation_check = pd.DataFrame({
    '확인 항목': [
        '유저 단위 마트 행 수',
        '고유 유저 수',
        '중복 유저 행 수',
        '콘텐츠 성과 마트 행 수',
        '분석 시작월',
        '분석 종료월',
        '폐지 전 가입 유저 수',
        '폐지 후 가입 유저 수',
        '첫 결제자 수',
        '결제자 중 결제시간 결측 수',
        '결제자 중 결제금액 결측 수'
    ],
    '값': [
        f'{len(user):,}',
        f'{user["user_id"].nunique():,}',
        f'{len(user) - user["user_id"].nunique():,}',
        f'{len(content):,}',
        user['cohort_month'].min().strftime('%Y-%m'),
        user['cohort_month'].max().strftime('%Y-%m'),
        f'{user.loc[user["period_group"] == "pre_2022_05_11", "user_id"].nunique():,}',
        f'{user.loc[user["period_group"] == "post_2023_05_11", "user_id"].nunique():,}',
        f'{user["paid_30d"].sum():,}',
        f'{user.loc[user["paid_30d"] == 1, "first_payment_time"].isna().sum():,}',
        f'{user.loc[user["paid_30d"] == 1, "first_paid_amount"].isna().sum():,}'
    ]
})

display(activation_check)

activation_key_null = (
    user[
        [
            'paid_30d',
            'content_view_7d',
            'content_start_7d',
            'lesson_enter_7d',
            'lesson_complete_7d',
            'content_end_7d',
            'lesson_complete_prepay_7d',
            'content_end_prepay_7d'
        ]
    ]
    .isna()
    .sum()
    .reset_index()
)
activation_key_null.columns = ['주요 컬럼', '결측 수']
display(activation_key_null)


### 첫 결제 전환 데이터 점검 해석

- 현상
    - 유저 단위 마트는 가입 유저를 기준으로 구성되며, 가입 후 7일 내 행동과 30일 내 첫 결제 여부를 함께 포함
    - 콘텐츠 성과 마트는 콘텐츠별 조회·시작·완료·결제 연결성을 확인하기 위한 보조 데이터로 활용
    - 폐지 전/후 비교는 동일한 5~11월 가입월 코호트를 기준으로 구성

- 인사이트
    - 첫 결제 전환 분석에서는 가입 후 초기 학습 행동과 첫 결제 여부의 관계를 확인 가능
    - 다만 첫 결제 이후 정기구독 유지 여부는 별도 리텐션 마트를 통해 확인 필요

- 정리
    - 이후 첫 결제 전환 분석에서는 가입 후 7일 내 레슨 완료 수와 콘텐츠 완료 경험을 핵심 행동 신호로 사용


### 2-2. 리텐션 분석용 데이터 점검


In [ ]:
# 분석 대상 및 관측 가능 기간 점검
basic_check = pd.DataFrame({
    '점검 항목': [
        '전체 유저 수',
        'user_id 중복 수',
        '첫 결제일 최소값',
        '첫 결제일 최대값',
        '폐지 전 비교군 유저 수',
        '폐지 후 비교군 유저 수',
        '폐지 후 전체 분석군 유저 수'
    ],
    '값': [
        f"{retention['user_id'].nunique():,}명",
        f"{retention.duplicated('user_id').sum():,}건",
        retention['first_payment_time'].min(),
        retention['first_payment_time'].max(),
        f"{int(retention['is_pre_retention_comparable'].sum()):,}명",
        f"{int(retention['is_post_retention_comparable'].sum()):,}명",
        f"{int(retention['is_post_removed'].sum()):,}명"
    ]
})

display(basic_check)

observe_summary = pd.DataFrame({
    '관측 가능 기간': ['30일', '60일', '90일', '180일'],
    '유저 수': [
        int(retention['can_observe_30d'].sum()),
        int(retention['can_observe_60d'].sum()),
        int(retention['can_observe_90d'].sum()),
        int(retention['can_observe_180d'].sum())
    ]
})

observe_summary['비율(%)'] = observe_summary['유저 수'] / retention['user_id'].nunique() * 100

display(observe_summary.round(2))

segment_dist = (
    retention
    .groupby(['subscription_lifecycle_segment', '세그먼트'], observed=False)
    .agg(유저_수=('user_id', 'nunique'))
    .reset_index()
)

segment_dist['비율(%)'] = segment_dist['유저_수'] / retention['user_id'].nunique() * 100
segment_dist['subscription_lifecycle_segment'] = pd.Categorical(
    segment_dist['subscription_lifecycle_segment'],
    categories=segment_order,
    ordered=True
)
segment_dist = segment_dist.sort_values('subscription_lifecycle_segment')

display(segment_dist.round(2))

segment_plot = add_percent_label(segment_dist, '비율(%)')

fig = px.bar(
    segment_plot,
    x='세그먼트',
    y='유저_수',
    text='비율 라벨',
    hover_data=['비율(%)'],
    title='구독 생애주기 세그먼트 분포',
    labels={'세그먼트': '구독 생애주기 세그먼트', '유저_수': '유저 수'}
)

fig = apply_common_layout(
    fig,
    height=540,
    y_max=segment_plot['유저_수'].max() * 1.2,
    xangle=0
)
fig.show()


- 정의

  - 리텐션 우수 그룹 : 정기결제 O / 취소 X / 재구독 X
  - 중도 이탈 그룹 : 정기결제 O / 취소 O / 재구독 X
  - 초기 이탈 위험 그룹 : 정기결제 X / 취소 O / 재구독 X
  - 취소 후 재구독 그룹 : 정기결제 무관 / 취소 O / 재구독 O
  - 후속 행동 없음 그룹 : 정기결제 X / 취소 X / 재구독 X
  - 기타 : 정기결제 X / 취소 X / 재구독 O

- 현상

  - 리텐션 마트 기준 첫 결제 유저 5,201명, `user_id` 중복 0건 확인
  - 폐지 전 비교군 1,333명, 폐지 후 비교군 1,299명, 폐지 후 전체 분석군 3,868명 구성
  - 60일 관측 가능 유저 4,656명, 90일 관측 가능 유저 4,299명, 180일 관측 가능 유저 2,456명 구성
  - 구독 생애주기 세그먼트는 중도 이탈 그룹, 초기 이탈 위험 그룹, 리텐션 우수 그룹, 후속 행동 없음 그룹 중심 구성

- 인사이트

  - 60일·90일·180일 지표 계산 시 관측 가능 flag 적용 필요
  - `click_cancel_plan_button`은 실제 해지 완료가 아닌 구독 취소 버튼 클릭 기반 해지 추정 액션으로 해석 필요
  - 정기결제 후 취소 그룹과 정기결제 전 취소 그룹을 분리한 리텐션 해석 필요


> # **Main Analysis. 무료체험 폐지 전/후 첫 결제 전환 및 리텐션 통합 분석**

### Main Analysis 구성

1. 무료체험 폐지 전/후 성과 판단 기준 재정립
2. 가입 후 7일 내 초기 학습 행동과 첫 결제 전환
3. 첫 결제 전환 아하 모먼트: 레슨 7개 이상 완료
4. 콘텐츠 완료 경험과 초기 행동 기반 유저 세그먼트
5. 무료체험 폐지 후 첫 결제자 품질 변화
6. 결제 후 학습 행동과 정기구독 유지
7. 첫 결제 전/후 학습 조합 분석
8. 종합 결론 및 실행 제안


## 1. 무료체험 폐지 전/후 성과 판단 기준 재정립

해당 분석은 무료체험 폐지 이후 가입 규모와 첫 결제 전환 효율이 어떻게 달라졌는지 확인하기 위한 분석입니다.  
가입자 수 증가만으로 정책 성과를 판단하기 어렵기 때문에, 첫 결제율과 첫 결제자 수, 결제금액, 월별 추이를 함께 확인합니다.


### 1-1. 폐지 전/후 핵심 KPI 비교


In [ ]:
# 폐지 전/후 핵심 KPI 산출
kpi = user.groupby('period_group').agg(
    가입_유저_수=('user_id', 'nunique'),
    첫_결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean'),
    첫_결제금액_합계=('first_paid_amount', 'sum'),
    평균_첫_결제금액=('first_paid_amount', 'mean'),
    중앙값_첫_결제금액=('first_paid_amount', 'median'),
    평균_할인금액=('first_discount_amount', 'mean'),
    콘텐츠_시작률=('content_start_7d', lambda x: (x > 0).mean()),
    레슨_완료율=('lesson_complete_7d', lambda x: (x > 0).mean()),
    콘텐츠_완료율=('content_end_7d', lambda x: (x > 0).mean())
).reset_index()

kpi['기간'] = kpi['period_group'].map(period_label)

# 출력용 비율 및 금액 포맷 적용
kpi_display = kpi.copy()
for col in ['첫_결제율', '콘텐츠_시작률', '레슨_완료율', '콘텐츠_완료율']:
    kpi_display[col] = (kpi_display[col] * 100).round(2)

for col in ['첫_결제금액_합계', '평균_첫_결제금액', '중앙값_첫_결제금액', '평균_할인금액']:
    kpi_display[col] = kpi_display[col].round(0).astype('Int64')

kpi_display = kpi_display[[
    '기간', '가입_유저_수', '첫_결제자_수', '첫_결제율',
    '첫_결제금액_합계', '평균_첫_결제금액', '중앙값_첫_결제금액', '평균_할인금액',
    '콘텐츠_시작률', '레슨_완료율', '콘텐츠_완료율'
]]

kpi_display.columns = [
    '기간', '가입 유저 수', '첫 결제자 수', '첫 결제율(%)',
    '첫 결제금액 합계', '평균 첫 결제금액', '중앙값 첫 결제금액', '평균 할인금액',
    '콘텐츠 시작률(%)', '레슨 완료율(%)', '콘텐츠 완료율(%)'
]

display(kpi_display)


### 핵심 KPI 해석

- 현상
    - 무료체험 폐지 후 가입 유저 수는 증가했지만, 가입 후 30일 내 첫 결제율은 하락
    - 첫 결제자 수는 가입자 증가의 영향을 함께 받기 때문에 전환율과 별도로 해석 필요
    - 콘텐츠 시작률, 레슨 완료율, 콘텐츠 완료율은 가입 초기 행동 변화의 보조 지표로 활용 가능

- 인사이트
    - 유입 규모는 확대되었으나, 가입자가 결제로 전환되는 효율은 약화된 구조로 해석
    - 단순 첫 결제율만 보면 정책 효과를 부정적으로 볼 수 있으나, 결제 이후 유지 품질까지 확인해야 최종 판단 가능
    - 따라서 본 분석은 첫 결제 전환 요인과 첫 결제 이후 리텐션 요인을 순차적으로 분리해 확인

- 액션
    - 첫 결제율 하락의 원인을 가입 초기 학습 경험, 결제 확신 형성, 결제 이후 학습 지속성 관점에서 나누어 점검 필요


### 1-2. 가입 및 결제 규모 비교


In [ ]:
# 폐지 전후 가입자 수 및 첫 결제자 수 비교
volume_data = kpi[['기간', '가입_유저_수', '첫_결제자_수']].melt(
    id_vars='기간', var_name='지표', value_name='유저_수'
)
volume_data['지표'] = volume_data['지표'].replace({
    '가입_유저_수': '가입 유저 수',
    '첫_결제자_수': '첫 결제자 수'
})

fig = px.bar(
    volume_data,
    x='기간', y='유저_수', color='지표', barmode='group',
    text='유저_수',
    labels={'기간': '기간', '유저_수': '유저 수', '지표': '지표'},
    title='폐지 전후 가입 유저 수 및 첫 결제자 수'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=480, yaxis_title='유저 수', legend_title_text='지표', uniformtext_minsize=10)
fig.show()


### 1-3. 결제 효율 및 첫 결제금액 비교


In [ ]:
# 폐지 전후 첫 결제율 및 첫 결제금액 비교
rate_amount = kpi[['기간', '첫_결제율', '첫_결제금액_합계', '평균_첫_결제금액', '평균_할인금액']].copy()
rate_amount['첫_결제율'] = rate_amount['첫_결제율'] * 100

fig = go.Figure()
fig.add_trace(go.Bar(
    x=rate_amount['기간'], y=rate_amount['첫_결제율'],
    name='첫 결제율(%)', text=rate_amount['첫_결제율'].round(2), textposition='outside'
))
fig.add_trace(go.Scatter(
    x=rate_amount['기간'], y=rate_amount['평균_첫_결제금액'],
    name='평균 첫 결제금액', mode='lines+markers+text', text=rate_amount['평균_첫_결제금액'].round(0), textposition='top center', yaxis='y2'
))
fig.update_layout(
    title='폐지 전후 첫 결제율 및 평균 첫 결제금액',
    xaxis_title='기간',
    yaxis=dict(title='첫 결제율(%)'),
    yaxis2=dict(title='평균 첫 결제금액', overlaying='y', side='right'),
    height=500,
    legend_title_text='지표'
)
fig.show()


### 1-4. 월별 가입·결제 추세


In [ ]:
# 월별 첫 결제율 추세 확인
monthly = user.groupby(['period_group', 'cohort_month']).agg(
    가입_유저_수=('user_id', 'nunique'),
    첫_결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reset_index()

monthly['기간'] = monthly['period_group'].map(period_label)
monthly['월'] = monthly['cohort_month'].dt.month
monthly['첫_결제율_pct'] = monthly['첫_결제율'] * 100

monthly_display = monthly[['기간', '월', '가입_유저_수', '첫_결제자_수', '첫_결제율_pct']].copy()
monthly_display.columns = ['기간', '가입월', '가입 유저 수', '첫 결제자 수', '첫 결제율(%)']
monthly_display['첫 결제율(%)'] = monthly_display['첫 결제율(%)'].round(2)

display(monthly_display)

fig = px.line(
    monthly,
    x='월', y='첫_결제율_pct', color='기간', markers=True,
    labels={'월': '가입월', '첫_결제율_pct': '첫 결제율(%)', '기간': '기간'},
    title='월별 30일 내 첫 결제율 추세',
    hover_data={'가입_유저_수': ':,', '첫_결제자_수': ':,'}
)
fig.update_xaxes(tickmode='array', tickvals=list(range(5, 12)), ticktext=[f'{m}월' for m in range(5, 12)])
fig.update_layout(height=480, legend_title_text='기간')
fig.show()


> **핵심 결론:** 무료체험 폐지 이후 가입자 규모는 확대되었지만 30일 내 첫 결제율은 하락했으므로, 정책 성과는 첫 결제율만이 아니라 결제자 품질과 정기구독 유지까지 함께 판단이 필요함.


## 2. 가입 후 7일 내 초기 학습 행동과 첫 결제 전환

해당 분석은 가입 직후 어떤 행동이 첫 결제 전환과 연결되는지 확인하기 위한 단계입니다.  
단순 조회·탐색보다 실제 학습 진입과 완료 행동이 결제 전환에 더 강한 신호인지 확인합니다.


### 2-1. 주요 행동 도달률 비교


In [ ]:
# 폐지 전후 7일 내 주요 행동 도달률 비교
behavior_cols = {
    'content_view_7d': '콘텐츠 조회',
    'content_start_7d': '콘텐츠 시작',
    'lesson_enter_7d': '레슨 진입',
    'lesson_complete_7d': '레슨 완료',
    'content_end_7d': '콘텐츠 완료',
    'payment_page_enter_7d': '결제 페이지 진입'
}

behavior_summary = []
for period, temp in user.groupby('period_group'):
    for col, label in behavior_cols.items():
        behavior_summary.append({
            '기간': period_label[period],
            '행동': label,
            '도달률': (temp[col] > 0).mean() * 100
        })

behavior_summary = pd.DataFrame(behavior_summary)

fig = px.bar(
    behavior_summary,
    x='행동', y='도달률', color='기간', barmode='group',
    text=behavior_summary['도달률'].round(1),
    labels={'행동': '가입 후 7일 내 행동', '도달률': '도달률(%)', '기간': '기간'},
    title='폐지 전후 가입 후 7일 내 주요 행동 도달률'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=520, legend_title_text='기간', xaxis_tickangle=-20)
fig.show()


### 초기 학습 행동 해석

- 현상
    - 무료체험 폐지 후 가입 후 7일 내 주요 학습 행동 도달률이 전반적으로 상승
    - 레슨 진입과 레슨 완료율은 가입 초기 학습 몰입을 보여주는 핵심 지표로 활용 가능
    - 그러나 초기 행동 도달률 상승에도 첫 결제율은 하락했으므로, 학습 행동이 결제 확신으로 연결되는 과정에 병목이 있을 가능성이 존재

- 인사이트
    - 가입 직후 학습 행동 자체는 확대되었지만, 학습 이후 결제 CTA나 유료 가치 설득 구조가 충분하지 않을 수 있음
    - 따라서 단순 행동 도달률이 아니라 어느 수준의 학습 경험이 첫 결제 전환율을 끌어올리는지 확인 필요

- 액션
    - 다음 단계에서는 레슨 완료 수와 콘텐츠 완료 경험을 기준으로 첫 결제 전환 변곡점을 탐색


> **핵심 결론:** 가입 후 7일 내 학습 행동은 첫 결제 전환을 설명하는 핵심 후보이지만, 행동 도달률만으로는 충분하지 않으므로 레슨 완료 수와 완료 경험 기준의 변곡점 분석이 필요함.


## 3. 첫 결제 전환 아하 모먼트: 레슨 7개 이상 완료

해당 분석은 가입 후 7일 내 레슨 완료 수가 첫 결제율과 어떻게 연결되는지 확인합니다.  
특히 정확히 N개 완료 기준과 N개 이상 최소 달성 기준을 구분해, 전환 변곡점과 운영 목표를 각각 도출합니다.


### 3-1. 레슨 완료 수 지점별 첫 결제율


In [ ]:
# 폐지 후 레슨 완료 수별 첫 결제율 산출
post = user[user['period_group'] == 'post_2023_05_11'].copy()
post_overall_paid_rate = post['paid_30d'].mean() * 100
post_total_paid_users = post['paid_30d'].sum()

# 레슨 완료 수는 0~30개까지 개별 단위로 확인
lesson_exact = post[post['lesson_complete_prepay_7d'].between(0, 30)].groupby('lesson_complete_prepay_7d').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reset_index()

lesson_exact.columns = ['레슨 완료 수', '유저 수', '결제자 수', '첫 결제율']
lesson_exact['첫 결제율(%)'] = lesson_exact['첫 결제율'] * 100
lesson_exact['이전 대비 증가폭(%p)'] = lesson_exact['첫 결제율(%)'].diff()
lesson_exact['결제 기여도(%)'] = lesson_exact['결제자 수'] / post_total_paid_users * 100

# 아하 모먼트 후보 판단 기준 산출
# 0개 완료는 미학습/혼합군 규모가 크기 때문에 중앙값 기준선 산출에서는 제외
lesson_exact_benchmark = lesson_exact[lesson_exact['레슨 완료 수'].between(1, 30)].copy()
positive_increase = lesson_exact_benchmark.loc[
    lesson_exact_benchmark['이전 대비 증가폭(%p)'] > 0,
    '이전 대비 증가폭(%p)'
]

lesson_user_median = lesson_exact_benchmark['유저 수'].median()
lesson_paid_median = lesson_exact_benchmark['결제자 수'].median()
lesson_increase_median = positive_increase.median()

lesson_exact['증가폭 기준 충족'] = lesson_exact['이전 대비 증가폭(%p)'] >= lesson_increase_median
lesson_exact['유저 수 기준 충족'] = lesson_exact['유저 수'] >= lesson_user_median
lesson_exact['결제자 수 기준 충족'] = lesson_exact['결제자 수'] >= lesson_paid_median
lesson_exact['아하 핵심 후보'] = (
    (lesson_exact['레슨 완료 수'] != 0) &
    lesson_exact['증가폭 기준 충족'] &
    lesson_exact['유저 수 기준 충족'] &
    lesson_exact['결제자 수 기준 충족']
)

def interpret_aha_candidate(lesson_count):
    if lesson_count == 7:
        return '1차 아하 모먼트 후보'
    if lesson_count == 12:
        return '보조 상승 후보'
    if lesson_count == 15:
        return '반복 학습 추가 상승 후보'
    if lesson_count == 19:
        return '반복 학습 이후 추가 상승 후보'
    return '중앙값 조건 충족 후보'

# 핵심 후보 표: 중앙값 조건 3가지를 모두 만족하는 지점만 별도 표시
aha_candidate_table = lesson_exact[lesson_exact['아하 핵심 후보']].copy()
aha_candidate_table['후보 여부'] = '핵심 후보'
aha_candidate_table['해석'] = aha_candidate_table['레슨 완료 수'].apply(interpret_aha_candidate)

# 25개 완료 지점은 결제율 급등은 크지만 모수 기준 미충족 가능성이 있어 참고 지점으로 별도 확인
reference_points = [7, 12, 15, 19, 25]
reference_table = lesson_exact[lesson_exact['레슨 완료 수'].isin(reference_points)].copy()
def interpret_reference_point(row):
    lesson_count = row['레슨 완료 수']
    if row['아하 핵심 후보']:
        return interpret_aha_candidate(lesson_count)
    if lesson_count == 25:
        return '소수 고몰입 참고 지점'
    return '참고 지점'
reference_table['해석'] = reference_table.apply(interpret_reference_point, axis=1)

print('='*100)
print(f'폐지 후 전체 평균 첫 결제율: {post_overall_paid_rate:.2f}%')
print('레슨 완료 수 지점별 30일 내 첫 결제율 분석 목적')
print('- 정확히 N개 완료 기준: "가입 후 7일 내 레슨을 정확히 N개 완료한 유저의 30일 내 첫 결제율"이 특정 지점에서 급증하는지 확인')
print('- 아하 모먼트 후보 판단: 전 단계 대비 증가폭, 유저 수, 결제자 수를 함께 고려')
print('')
print('아하 모먼트 후보 판단 기준')
print('- 0개 완료 구간은 모수가 지나치게 커 중앙값 기준선을 왜곡할 수 있으므로 유저 수·결제자 수 중앙값 산출에서 제외')
print(f'- 증가폭 기준: 전 단계 대비 양수 증가폭의 중앙값 이상 ({lesson_increase_median:.2f}%p 이상)')
print(f'- 유저 수 기준: 1~30개 완료 구간의 유저 수 중앙값 이상 ({lesson_user_median:.1f}명 이상)')
print(f'- 결제자 수 기준: 1~30개 완료 구간의 결제자 수 중앙값 이상 ({lesson_paid_median:.1f}명 이상)')
print('- 중앙값 기준은 특정 고정값보다 데이터 변화에 대응하기 쉽고, 과도한 임의 기준을 줄이기 위한 목적으로 선정')
print('')
print('결제 기여도 계산식')
print('- 결제 기여도 = 해당 완료 수의 결제자 수 / 폐지 후 전체 30일 내 첫 결제자 수 × 100')
print('- 결제 기여도는 결제금액 기준이 아니라 결제자 수 기준')
print('='*100)

lesson_exact_display = lesson_exact[[
    '레슨 완료 수', '유저 수', '결제자 수', '첫 결제율(%)', '이전 대비 증가폭(%p)', '결제 기여도(%)'
]].copy()
for col in ['첫 결제율(%)', '이전 대비 증가폭(%p)', '결제 기여도(%)']:
    lesson_exact_display[col] = lesson_exact_display[col].round(2)
print('레슨 완료 수 지점별 30일 내 첫 결제율 현황 표')
display(lesson_exact_display)

aha_candidate_display = aha_candidate_table[[
    '레슨 완료 수', '유저 수', '결제자 수', '첫 결제율(%)', '이전 대비 증가폭(%p)', '후보 여부', '해석'
]].copy()
for col in ['첫 결제율(%)', '이전 대비 증가폭(%p)']:
    aha_candidate_display[col] = aha_candidate_display[col].round(2)
print('='*100)
print('중앙값 기준 핵심 아하 모먼트 후보 표')
display(aha_candidate_display)

reference_display = reference_table[[
    '레슨 완료 수', '유저 수', '결제자 수', '첫 결제율(%)', '이전 대비 증가폭(%p)', '결제 기여도(%)', '해석'
]].copy()
for col in ['첫 결제율(%)', '이전 대비 증가폭(%p)', '결제 기여도(%)']:
    reference_display[col] = reference_display[col].round(2)
print('='*100)
print('주요 해석 지점 및 참고 지점 요약')
display(reference_display)

fig = px.line(
    lesson_exact,
    x='레슨 완료 수', y='첫 결제율(%)', markers=True,
    hover_data={'유저 수': ':,', '결제자 수': ':,', '이전 대비 증가폭(%p)': ':.2f', '결제 기여도(%)': ':.2f'},
    labels={'레슨 완료 수': '가입 후 7일 내 레슨 완료 수', '첫 결제율(%)': '30일 내 첫 결제율(%)'},
    title='레슨 완료 수 지점별 30일 내 첫 결제율: 정확히 N개 완료 기준'
)
fig.add_hline(y=post_overall_paid_rate, line_dash='dash', line_color='gray', annotation_text=f'전체 평균 {post_overall_paid_rate:.2f}%')
fig.add_vline(x=7, line_dash='dot', line_color='red', annotation_text='아하 모먼트 후보 7개')
for point in [12, 15, 19]:
    fig.add_vline(x=point, line_dash='dot', line_color='purple', annotation_text=f'추가 상승 {point}개')
fig.add_vline(x=25, line_dash='dot', line_color='gray', annotation_text='소수 고몰입 25개')
fig.update_xaxes(dtick=1)
fig.update_layout(height=560)
fig.show()


### 레슨 완료 수 지점별 해석

- 현상
    - 레슨 완료 수가 증가할수록 첫 결제율이 전반적으로 상승하는 흐름을 확인
    - 특히 6개에서 7개로 넘어가는 구간에서 첫 결제율 상승폭이 두드러지는 편
    - 7개 완료는 유저 수와 결제자 수 기준도 함께 충족하는 핵심 후보 구간으로 해석 가능

- 인사이트
    - 레슨 7개 완료는 단순 학습량이 아니라, 유저가 서비스의 학습 가치를 체감하기 시작하는 변곡점 후보로 판단
    - 다만 관찰 데이터 기반 분석이므로 인과관계가 아니라 첫 결제 전환 신호로 해석 필요

- 액션
    - 가입 후 7일 내 레슨 7개 완료를 1차 온보딩 목표로 설정하고, 6개 완료 유저에게 7번째 완료를 유도하는 메시지와 추천 흐름 설계 필요


### 3-2. 최소달성 레슨 기준별 첫 결제율


In [ ]:
# 레슨 N개 이상 완료 기준별 첫 결제율 산출
# 특정 완료 수를 운영 목표로 잡았을 때 전환율, 유저 규모, 결제 기여도를 함께 확인
thresholds = list(range(0, 31))
lesson_threshold_rows = []

total_post_users = post['user_id'].nunique()
total_post_paid_users = post['paid_30d'].sum()

for threshold in thresholds:
    temp = post[post['lesson_complete_prepay_7d'] >= threshold]
    lesson_threshold_rows.append({
        '기준': f'{threshold}개 이상',
        '레슨 완료 기준': threshold,
        '유저 수': temp['user_id'].nunique(),
        '결제자 수': temp['paid_30d'].sum(),
        '첫 결제율(%)': temp['paid_30d'].mean() * 100 if len(temp) > 0 else np.nan,
        '유저 비중(%)': temp['user_id'].nunique() / total_post_users * 100,
        '결제 기여도(%)': temp['paid_30d'].sum() / total_post_paid_users * 100 if total_post_paid_users > 0 else np.nan
    })

lesson_threshold = pd.DataFrame(lesson_threshold_rows)
# 기존 변수명 호환용
lesson_cumulative = lesson_threshold.copy()

# 운영 목표 후보 판단 기준 산출
# 0개 이상은 전체 유저를 의미하므로 중앙값 기준선 산출에서는 제외
threshold_benchmark = lesson_threshold[lesson_threshold['레슨 완료 기준'].between(1, 30)].copy()
threshold_user_share_median = threshold_benchmark['유저 비중(%)'].median()
threshold_contribution_median = threshold_benchmark['결제 기여도(%)'].median()

lesson_threshold['첫 결제율 기준 충족'] = lesson_threshold['첫 결제율(%)'] >= post_overall_paid_rate
lesson_threshold['유저 비중 기준 충족'] = lesson_threshold['유저 비중(%)'] >= threshold_user_share_median
lesson_threshold['결제 기여도 기준 충족'] = lesson_threshold['결제 기여도(%)'] >= threshold_contribution_median
lesson_threshold['운영 목표 후보'] = (
    (lesson_threshold['레슨 완료 기준'] != 0) &
    lesson_threshold['첫 결제율 기준 충족'] &
    lesson_threshold['유저 비중 기준 충족'] &
    lesson_threshold['결제 기여도 기준 충족']
)

def interpret_threshold_candidate(threshold):
    if threshold < 7:
        return '초기 학습 예비 기준'
    if threshold == 7:
        return '1차 운영 목표 후보'
    if 8 <= threshold <= 10:
        return '초기 고관여 유지 기준'
    if 11 <= threshold <= 14:
        return '반복 학습 진입 기준'
    if threshold == 15:
        return '고관여 결제 CTA 기준'
    return '고몰입 소수 관리 기준'

operating_candidate_table = lesson_threshold[lesson_threshold['운영 목표 후보']].copy()
operating_candidate_table['해석'] = operating_candidate_table['레슨 완료 기준'].apply(interpret_threshold_candidate)

print('='*100)
print('레슨 최소 달성 기준별 30일 내 첫 결제율 분석 목적')
print('- 특정 레슨 완료 수를 운영 목표로 잡았을 때 "30일 내 첫 결제율", "유저 규모", "결제자 기여도"가 함께 유지되는지 확인하는 분석')
print('- [레슨 완료 수 지점별 30일 내 첫 결제율] 분석이 아하 모먼트 탐색용이라면, [레슨 최소 달성 기준별 30일 내 첫 결제율] 분석은 운영 목표의 현실성을 검토하기 위함')
print('- 최소 달성 기준 N이 높아질수록 더 많이 학습한 유저만 남기 때문에 유저 비중과 결제 기여도는 감소할 수 있음')
print('- 0개 이상 기준은 폐지 후 전체 가입 유저를 모두 포함하므로, 첫 결제율은 전체 평균 결제율과 동일')
print('')
print('운영 목표 후보 판단 기준')
print('- 0개 이상은 전체 유저 기준이므로 유저 비중과 결제 기여도 중앙값 산출에서는 제외')
print(f'- 첫 결제율 기준: 폐지 후 전체 평균 첫 결제율 이상 ({post_overall_paid_rate:.2f}% 이상)')
print(f'- 유저 비중 기준: 1~30개 이상 기준별 유저 비중 중앙값 이상 ({threshold_user_share_median:.2f}% 이상)')
print(f'- 결제 기여도 기준: 1~30개 이상 기준별 결제 기여도 중앙값 이상 ({threshold_contribution_median:.2f}% 이상)')
print('- 첫 결제율은 전체 평균 이상인지 확인하고, 유저 비중과 결제 기여도는 중앙값 이상인지 확인하여 운영 목표로 삼을 수 있는 현실성을 판단')
print('')
print('결제 기여도 계산식')
print('- 결제 기여도 = 해당 기준 이상 유저의 결제자 수 / 폐지 후 전체 30일 내 첫 결제자 수 × 100')
print('- 결제 기여도는 결제금액 기준이 아니라 결제자 수 기준')
print('='*100)

threshold_display = lesson_threshold[[
    '기준', '유저 수', '결제자 수', '첫 결제율(%)', '유저 비중(%)', '결제 기여도(%)'
]].copy()
for col in ['첫 결제율(%)', '유저 비중(%)', '결제 기여도(%)']:
    threshold_display[col] = threshold_display[col].round(2)
print('레슨 최소 달성 기준별 30일 내 첫 결제율 현황 표')
display(threshold_display)

candidate_display = operating_candidate_table[[
    '기준', '유저 수', '결제자 수', '첫 결제율(%)', '유저 비중(%)', '결제 기여도(%)', '해석'
]].copy()
candidate_display.columns = ['N개 이상 기준', '유저 수', '결제자 수', '첫 결제율(%)', '유저 비중(%)', '결제 기여도(%)', '해석']
for col in ['첫 결제율(%)', '유저 비중(%)', '결제 기여도(%)']:
    candidate_display[col] = candidate_display[col].round(2)
print('='*100)
print('운영 목표 핵심 후보 표')
display(candidate_display)

# 15개 이상 이후 약화 구간 확인용 요약
weakening_points = lesson_threshold[lesson_threshold['레슨 완료 기준'].isin([15, 16])][[
    '기준', '유저 수', '결제자 수', '첫 결제율(%)', '유저 비중(%)', '결제 기여도(%)'
]].copy()
for col in ['첫 결제율(%)', '유저 비중(%)', '결제 기여도(%)']:
    weakening_points[col] = weakening_points[col].round(2)
print('='*100)
print('15개 이상과 16개 이상 비교')
display(weakening_points)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lesson_threshold['레슨 완료 기준'],
    y=lesson_threshold['첫 결제율(%)'],
    mode='lines+markers',
    name='첫 결제율(%)',
    hovertemplate='기준: %{x}개 이상<br>첫 결제율: %{y:.2f}%<extra></extra>'
))
fig.add_trace(go.Bar(
    x=lesson_threshold['레슨 완료 기준'],
    y=lesson_threshold['유저 비중(%)'],
    name='유저 비중(%)',
    opacity=0.35,
    yaxis='y2',
    hovertemplate='기준: %{x}개 이상<br>유저 비중: %{y:.2f}%<extra></extra>'
))
fig.add_shape(
    type='line', xref='x', yref='y2',
    x0=0, x1=30, y0=threshold_user_share_median, y1=threshold_user_share_median,
    line=dict(color='gray', dash='dash')
)
fig.add_annotation(
    x=30, y=threshold_user_share_median, yref='y2', showarrow=False,
    text=f'유저 비중 중앙값 {threshold_user_share_median:.2f}%',
    xanchor='right', yanchor='bottom'
)
fig.add_vline(x=7, line_dash='dot', line_color='red', annotation_text='1차 운영 목표 7개')
fig.add_vline(x=15, line_dash='dot', line_color='purple', annotation_text='고관여 전환 기준 15개')
fig.update_layout(
    title='레슨 최소 달성 기준별 30일 내 첫 결제율 및 유저 비중: N개 이상 완료 기준',
    xaxis_title='가입 후 7일 내 최소 레슨 완료 수 기준',
    yaxis=dict(title='30일 내 첫 결제율(%)'),
    yaxis2=dict(title='유저 비중(%)', overlaying='y', side='right'),
    height=560,
    legend_title_text='지표'
)
fig.show()


### 최소달성 레슨 기준 해석

- 현상
    - N개 이상 완료 기준으로 보면 7개 이상 완료 구간은 첫 결제율과 유저 규모가 함께 유지되는 현실적인 운영 목표로 확인
    - 11개 이상, 15개 이상 구간은 고관여 CTA 또는 반복 학습 진입 기준으로 참고 가능
    - 기준을 지나치게 높이면 결제율은 높아질 수 있으나 대상 유저 규모와 결제 기여도가 감소할 수 있음

- 인사이트
    - 정확히 N개 기준은 전환율이 튀는 변곡점을 찾는 분석이고, N개 이상 기준은 실제 운영 목표로 삼을 수 있는지를 확인하는 분석
    - 따라서 첫 결제 전환의 1차 목표는 7개 이상 완료, 고관여 유저 대상 CTA 기준은 11~15개 이상으로 나누는 것이 적절하다고 판단

- 액션
    - 온보딩 목표는 7개 이상 완료로 설정하고, 11개 이상 반복 학습 유저에게는 유료 플랜 CTA와 심화 로드맵을 강화


### 3-3. 레슨 완료 구간별 결제 기여도


In [ ]:
# 레슨 완료 구간별 규모, 첫 결제율, 결제 기여도 산출
lesson_band_order = ['0개', '1~3개', '4~6개', '7~10개', '11~20개', '21~50개', '51~100개', '101~200개', '200개 이상']

def make_lesson_band(x):
    if x == 0:
        return '0개'
    elif x <= 3:
        return '1~3개'
    elif x <= 6:
        return '4~6개'
    elif x <= 10:
        return '7~10개'
    elif x <= 20:
        return '11~20개'
    elif x <= 50:
        return '21~50개'
    elif x <= 100:
        return '51~100개'
    elif x <= 200:
        return '101~200개'
    else:
        return '200개 이상'

post['레슨 완료 구간'] = post['lesson_complete_prepay_7d'].apply(make_lesson_band)

lesson_band = post.groupby('레슨 완료 구간').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reindex(lesson_band_order).reset_index()

lesson_band['유저 비중(%)'] = lesson_band['유저_수'] / len(post) * 100
lesson_band['결제 기여도(%)'] = lesson_band['결제자_수'] / post['paid_30d'].sum() * 100
lesson_band['첫 결제율(%)'] = lesson_band['첫_결제율'] * 100
lesson_band_user_share_median = lesson_band['유저 비중(%)'].median()

print('레슨 완료 구간 분석 목적')
print('- 레슨 완료 수 지점별 30일 내 첫 결제율(정확히 N개 완료 분석)은 아하 모먼트 탐색용')
print('- 레슨 최소 달성 기준별 30일 내 첫 결제율(N개 이상 완료 분석)은 운영 목표의 현실성 검토용')
print('- 구간별 분석은 어느 학습 구간이 실제 결제자 형성에 크게 기여하는지 확인하는 분석')
print('')
print('결제 기여도 계산식')
print('- 결제 기여도 = 해당 구간 결제자 수 / 폐지 후 전체 30일 내 첫 결제자 수 × 100')
print('- 결제 기여도는 결제금액 기준이 아니라 결제자 수 기준')

lesson_band_display = lesson_band[['레슨 완료 구간', '유저_수', '유저 비중(%)', '결제자_수', '첫 결제율(%)', '결제 기여도(%)']].copy()
lesson_band_display.columns = ['레슨 완료 구간', '유저 수', '유저 비중(%)', '결제자 수', '첫 결제율(%)', '결제 기여도(%)']
for col in ['유저 비중(%)', '첫 결제율(%)', '결제 기여도(%)']:
    lesson_band_display[col] = lesson_band_display[col].round(2)

display(lesson_band_display)


> **핵심 결론:** 가입 후 7일 내 레슨 7개 이상 완료는 첫 결제 전환을 설명하는 1차 아하 모먼트 후보이며, 이후 11~20개 반복 학습 구간은 결제 CTA를 강화할 수 있는 고관여 구간으로 해석 가능함.


## 4. 콘텐츠 완료 경험과 초기 행동 기반 유저 세그먼트

해당 분석은 첫 결제율을 높이기 위해 어떤 유저를 어떤 단계로 이동시켜야 하는지 정의하는 단계입니다.  
콘텐츠 완료 경험과 레슨 완료 수를 함께 사용해, 무경험 → 탐색 → 초기 학습 → 핵심 학습 흐름으로 유저를 구조화합니다.


### 4-1. 콘텐츠 완료 경험별 첫 결제율


In [ ]:
# 콘텐츠 완료 개수별 첫 결제율 및 기여도
post['콘텐츠 완료 구간'] = pd.cut(
    post['content_end_prepay_7d'],
    bins=[-1, 0, 1, 999999],
    labels=['0개', '1개', '2개 이상']
)

content_end_summary = post.groupby('콘텐츠 완료 구간', observed=False).agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reset_index()

content_end_summary['유저 비중(%)'] = content_end_summary['유저_수'] / len(post) * 100
content_end_summary['결제 기여도(%)'] = content_end_summary['결제자_수'] / post['paid_30d'].sum() * 100
content_end_summary['첫 결제율(%)'] = content_end_summary['첫_결제율'] * 100

content_end_display = content_end_summary[['콘텐츠 완료 구간', '유저_수', '유저 비중(%)', '결제자_수', '첫 결제율(%)', '결제 기여도(%)']].copy()
content_end_display.columns = ['콘텐츠 완료 구간', '유저 수', '유저 비중(%)', '결제자 수', '첫 결제율(%)', '결제 기여도(%)']
for col in ['유저 비중(%)', '첫 결제율(%)', '결제 기여도(%)']:
    content_end_display[col] = content_end_display[col].round(2)

display(content_end_display)

fig = px.bar(
    content_end_summary,
    x='콘텐츠 완료 구간', y='첫 결제율(%)',
    text=content_end_summary['첫 결제율(%)'].round(1),
    hover_data={'유저_수': ':,', '결제자_수': ':,', '결제 기여도(%)': ':.2f'},
    labels={'콘텐츠 완료 구간': '콘텐츠 완료 구간', '첫 결제율(%)': '첫 결제율(%)'},
    title='콘텐츠 완료 개수별 첫 결제율'
)
fig.add_hline(y=post_overall_paid_rate, line_dash='dash', line_color='gray', annotation_text=f'전체 평균 {post_overall_paid_rate:.2f}%')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=480)
fig.show()


### 콘텐츠 완료 경험 해석

- 현상
    - 콘텐츠 완료 경험이 있는 유저는 완료 경험이 없는 유저보다 첫 결제율이 높게 확인됨
    - 단순 진입이나 조회보다 실제 완주 경험이 전환 가능성을 높이는 신호로 해석
    - 콘텐츠 1개 완료는 레슨 7개 완료와 함께 초기 학습 경험의 질을 판단하는 보조 기준으로 확인

- 인사이트
    - 첫 결제 전환에는 단순 탐색보다 “완료했다”는 성취 경험이 중요
    - 이는 유저가 서비스의 학습 구조를 이해하고, 결제 후에도 계속 학습할 수 있다는 확신을 형성했을 가능성을 시사

- 액션
    - 첫 콘텐츠 완주 후 다음 콘텐츠 추천, 학습 성취 메시지, 유료 플랜 CTA 연결 필요


### 4-2. 초기 행동 기반 최종 유저 세그먼트


In [ ]:
# 최종 유저 세그먼트 정의 및 성과 산출
# 최종 세그먼트는 레슨 완료 수 중심의 학습 여정 단계로 정의
final_definition = pd.DataFrame({
    '세그먼트': ['무행동형', '탐색형', '약한 학습형', '임계 직전형', '핵심 학습형', '수익 핵심 학습형', '고몰입·헤비 학습형'],
    '정의': [
        '콘텐츠 경험 없음, 레슨 완료 0개',
        '콘텐츠 경험 있음, 레슨 완료 0개',
        '레슨 1~5개 완료',
        '레슨 6개 완료',
        '레슨 7~10개 완료',
        '레슨 11~20개 완료',
        '레슨 21개 이상 완료'
    ],
    '비즈니스 해석': [
        '가입 후 주요 행동이 없는 초기 이탈 가능 집단',
        '관심은 있으나 첫 학습 완료까지 도달하지 못한 집단',
        '맛보기 학습 후 반복 학습 전환이 필요한 집단',
        '아하 모먼트 직전의 운영 목표 집단',
        '전환 가능성이 높아지는 초기 고관여 집단',
        '결제 기여도가 커지는 반복 학습 집단',
        '장기 학습 또는 특수 행동 가능성이 있는 헤비 유저 집단'
    ],
    '주요 액션': [
        '첫 콘텐츠 추천',
        '첫 레슨 완료 유도',
        '반복 학습 유도',
        '7번째 레슨 완료 유도',
        '결제 CTA 시작',
        '유료 플랜 및 심화 로드맵 제안',
        '프리미엄·장기 학습 관리'
    ]
})

display(final_definition)

print('='*100)
print('최종 유저 세그먼트 분류 기준')
print('- 최종 세그먼트는 학습 여정 단계별 액션 설계를 위해 레슨 완료 수 중심으로 정의')
print('- 콘텐츠 완료, 질문 클릭 등은 앞선 보조 분석에서 확인했으므로 최종 세그먼트에서는 제외')
print('- 세그먼트 우선순위에 따라 레슨 21개 이상 유저가 다른 세그먼트로 빠지지 않도록 상호배타 분류')
print('- 이를 통해 고몰입·헤비 학습형 결제자 수가 0으로 잘못 표시되는 문제를 방지')

# 최종 유저 세그먼트별 성과 산출
# 무행동형과 탐색형은 레슨 완료 0개 유저를 콘텐츠 경험 여부로 구분
# 이후 구간은 레슨 완료 수를 기준으로 상호배타 분류
def make_final_segment(row):
    lesson_count = row['lesson_complete_prepay_7d']
    unique_content = row['unique_content_prepay_7d']
    content_experience = unique_content > 0 or row['content_view_7d'] > 0 or row['content_start_7d'] > 0

    if lesson_count >= 21:
        return '고몰입·헤비 학습형'
    elif lesson_count >= 11:
        return '수익 핵심 학습형'
    elif lesson_count >= 7:
        return '핵심 학습형'
    elif lesson_count == 6:
        return '임계 직전형'
    elif 1 <= lesson_count <= 5:
        return '약한 학습형'
    elif content_experience:
        return '탐색형'
    else:
        return '무행동형'

final_order = ['무행동형', '탐색형', '약한 학습형', '임계 직전형', '핵심 학습형', '수익 핵심 학습형', '고몰입·헤비 학습형']
post['최종 유저 세그먼트'] = post.apply(make_final_segment, axis=1)

final_segment = post.groupby('최종 유저 세그먼트').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reindex(final_order).fillna(0).reset_index()

final_segment['유저 비중(%)'] = final_segment['유저_수'] / len(post) * 100
final_segment['결제 기여도(%)'] = final_segment['결제자_수'] / post['paid_30d'].sum() * 100
final_segment['첫 결제율(%)'] = final_segment['첫_결제율'] * 100

user_share_median = final_segment['유저 비중(%)'].median()
segment_count = len(final_segment)
median_position = int(np.ceil(segment_count / 2))
median_reference = final_segment[['최종 유저 세그먼트', '유저 비중(%)']].sort_values('유저 비중(%)').reset_index(drop=True)
median_segment_name = median_reference.iloc[median_position - 1]['최종 유저 세그먼트']

final_display = final_segment[['최종 유저 세그먼트', '유저_수', '유저 비중(%)', '결제자_수', '첫 결제율(%)', '결제 기여도(%)']].copy()
final_display.columns = ['최종 유저 세그먼트', '유저 수', '유저 비중(%)', '결제자 수', '첫 결제율(%)', '결제 기여도(%)']
for col in ['유저 비중(%)', '첫 결제율(%)', '결제 기여도(%)']:
    final_display[col] = final_display[col].round(2)

display(final_display)

segment_group_table = pd.DataFrame({
    '대분류': ['미경험·탐색', '초기 학습', '전환 후보', '고몰입'],
    '포함 세그먼트': [
        '무행동형, 탐색형',
        '약한 학습형, 임계 직전형',
        '핵심 학습형, 수익 핵심 학습형',
        '고몰입·헤비 학습형'
    ],
    '비즈니스 의미': [
        '가입 후 콘텐츠 경험 또는 첫 학습 완료가 부족한 집단',
        '맛보기 학습 이후 아하 모먼트 도달 전 단계',
        '첫 결제 가능성이 상승하고 반복 학습으로 연결해야 하는 집단',
        '장기 학습 또는 특수 행동 가능성이 있는 고관여 집단'
    ],
    '주요 액션': [
        '첫 콘텐츠 추천, 첫 레슨 완료 유도',
        '반복 학습 유도, 7번째 레슨 완료 유도',
        '결제 CTA 시작, 유료 플랜 및 심화 로드맵 제안',
        '프리미엄·장기 학습 관리, 후속 유지 분석'
    ]
})
print('='*100)
print('유저 세그먼트 대분류 4유형 요약')
display(segment_group_table)

print('='*100)
print('버블차트 해석 기준')
print('- X축: 전체 가입 유저 중 해당 세그먼트가 차지하는 유저 비중')
print('- Y축: 해당 세그먼트의 30일 내 첫 결제율')
print('- 버블 크기: 전체 결제자 중 해당 세그먼트의 결제 기여도')
print(f'- 세로 기준선: 세그먼트별 유저 비중 중앙값 {user_share_median:.2f}%')
print(f'- 가로 기준선: 폐지 후 전체 평균 첫 결제율 {post_overall_paid_rate:.2f}%')
print('')

segment_movement = pd.DataFrame({
    '현재 세그먼트': ['무행동형', '탐색형', '약한 학습형', '임계 직전형', '핵심 학습형', '수익 핵심 학습형', '고몰입·헤비 학습형'],
    '현재 문제': [
        '콘텐츠 경험 없음',
        '콘텐츠 경험은 있으나 첫 레슨 완료 없음',
        '맛보기 학습 이후 반복 학습 부족',
        '아하 모먼트 직전 정체',
        '전환 가능성 상승 시작 단계',
        '반복 학습과 결제 기여가 큰 구간',
        '장기 학습 또는 특수 행동 가능성'
    ],
    '이동 목표': [
        '첫 콘텐츠 조회/시작',
        '첫 레슨 완료',
        '6개 완료',
        '7개 완료',
        '11~20개 반복 학습',
        '첫 결제 및 유료 플랜 전환',
        '장기 학습·프리미엄 관리'
    ],
    '액션': [
        '관심사 기반 첫 콘텐츠 추천',
        '짧은 레슨, 이어보기, 첫 완료 보상',
        '학습 목표 제시, 다음 레슨 추천',
        '7번째 레슨 유도 메시지',
        '학습 로드맵, 다음 커리큘럼 추천',
        '유료 플랜 CTA, 심화 로드맵, 할인·혜택 제안',
        '프리미엄 혜택, 장기 학습 관리, 후속 유지 분석'
    ]
})
display(segment_movement)

fig = px.scatter(
    final_segment,
    x='유저 비중(%)', y='첫 결제율(%)', size='결제 기여도(%)', color='최종 유저 세그먼트',
    hover_name='최종 유저 세그먼트',
    hover_data={'유저_수': ':,', '결제자_수': ':,', '유저 비중(%)': ':.2f', '첫 결제율(%)': ':.2f', '결제 기여도(%)': ':.2f'},
    labels={'유저 비중(%)': '전체 유저 비중(%)', '첫 결제율(%)': '첫 결제율(%)'},
    title='최종 유저 세그먼트별 규모, 첫 결제율, 결제 기여도'
)
fig.add_vline(x=user_share_median, line_dash='dash', line_color='gray', annotation_text=f'유저 비중 중앙값 {user_share_median:.2f}%')
fig.add_hline(y=post_overall_paid_rate, line_dash='dash', line_color='gray', annotation_text=f'전체 평균 {post_overall_paid_rate:.2f}%')
fig.update_layout(height=640, legend_title_text='세그먼트')
fig.show()


### 초기 행동 세그먼트 해석

- 현상
    - 최종 세그먼트는 레슨 완료 수와 콘텐츠 경험 여부를 기준으로 7단계로 구성
    - 무행동형과 탐색형은 첫 학습 완료 이전의 병목을 보여줌
    - 임계 직전형과 핵심 학습형은 레슨 7개 완료 전후의 전환 가능성 차이를 보여주는 핵심 운영 타깃

- 인사이트
    - 세그먼트 분석은 단순 분류가 아니라, 각 유저를 다음 행동 단계로 이동시키기 위한 전략 기준
    - 첫 결제 전환 전략은 전체 유저에게 동일하게 적용하기보다 세그먼트별 이동 목표를 다르게 설정 필요

- 액션
    - 무행동형은 첫 콘텐츠 경험, 탐색형은 첫 레슨 완료, 임계 직전형은 7개 완료, 핵심 학습형은 결제 CTA로 연결


### 4-3. 발표용 4개 유저 세그먼트 보조 검증

앞선 본문에서는 레슨 완료 수를 더 세밀하게 나눈 7단계 세그먼트를 사용했습니다.  
아래 표는 발표와 A/B 테스트 설계에서 더 직관적으로 활용하기 위해 동일한 기준을 4개 세그먼트로 단순화한 보조 검증입니다.

- **미경험형**: 콘텐츠 경험 없음, 레슨 완료 0개
- **탐색형**: 콘텐츠 경험 있음, 레슨 완료 0개
- **약한 학습형**: 레슨 1~6개 완료
- **강한 학습형**: 레슨 7개 이상 완료


In [ ]:
# 발표용 4개 유저 세그먼트 기준 성과 산출
# 기존 본문의 7단계 세그먼트를 A/B 테스트와 기대효과 추정에 활용하기 쉽도록 4개 단계로 단순화합니다.
from statsmodels.stats.proportion import proportions_ztest

post_segment4 = post.copy()

segment4_numeric_cols = [
    'paid_30d',
    'lesson_complete_prepay_7d',
    'unique_content_prepay_7d',
    'content_view_7d',
    'content_start_7d',
    'question_click_7d',
    'payment_page_enter_7d'
]

for col in segment4_numeric_cols:
    if col not in post_segment4.columns:
        post_segment4[col] = 0
    post_segment4[col] = pd.to_numeric(post_segment4[col], errors='coerce').fillna(0)

def make_segment4(row):
    lesson_count = row['lesson_complete_prepay_7d']
    content_experience = (
        (row['unique_content_prepay_7d'] > 0) or
        (row['content_view_7d'] > 0) or
        (row['content_start_7d'] > 0)
    )

    if lesson_count >= 7:
        return '강한 학습형'
    elif lesson_count >= 1:
        return '약한 학습형'
    elif content_experience:
        return '탐색형'
    else:
        return '미경험형'

segment4_order = ['미경험형', '탐색형', '약한 학습형', '강한 학습형']
post_segment4['4개 유저 세그먼트'] = post_segment4.apply(make_segment4, axis=1)

segment4_definition = pd.DataFrame({
    '유저 세그먼트': segment4_order,
    '정의': [
        '콘텐츠 경험 없음, 레슨 완료 0개',
        '콘텐츠 경험 있음, 레슨 완료 0개',
        '레슨 1~6개 완료',
        '레슨 7개 이상 완료'
    ],
    '비즈니스 해석': [
        '가입 후 핵심 행동이 거의 없는 초기 이탈 가능 집단',
        '콘텐츠에는 관심을 보였지만 아직 첫 학습 완료까지 가지 못한 집단',
        '맛보기 학습은 했지만 핵심 학습 기준에는 도달하지 못한 집단',
        '첫 결제 가능성이 높아지는 핵심 학습 집단'
    ],
    '주요 액션': [
        '첫 콘텐츠 추천, 관심사 기반 진입 유도',
        '첫 레슨 완료 유도, 짧은 레슨 추천',
        '7번째 레슨 완료 유도, 학습 루틴 형성',
        '결제 CTA, 유료 플랜 제안, 심화 로드맵 제시'
    ]
})

display(segment4_definition)

total_segment4_users = post_segment4['user_id'].nunique()
total_segment4_payers = post_segment4['paid_30d'].sum()

segment4_summary = post_segment4.groupby('4개 유저 세그먼트').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    평균_레슨_완료수=('lesson_complete_prepay_7d', 'mean'),
    평균_고유_콘텐츠_수=('unique_content_prepay_7d', 'mean'),
    질문_클릭_유저수=('question_click_7d', lambda x: (x > 0).sum()),
    결제페이지_진입_유저수=('payment_page_enter_7d', lambda x: (x > 0).sum())
).reindex(segment4_order).fillna(0).reset_index()

segment4_summary['유저 비중(%)'] = segment4_summary['유저_수'] / total_segment4_users * 100
segment4_summary['첫 결제율(%)'] = segment4_summary['결제자_수'] / segment4_summary['유저_수'] * 100
segment4_summary['결제 기여도(%)'] = np.where(
    total_segment4_payers > 0,
    segment4_summary['결제자_수'] / total_segment4_payers * 100,
    0
)
segment4_summary['질문 클릭률(%)'] = segment4_summary['질문_클릭_유저수'] / segment4_summary['유저_수'] * 100
segment4_summary['결제 페이지 진입률(%)'] = segment4_summary['결제페이지_진입_유저수'] / segment4_summary['유저_수'] * 100

segment4_display = segment4_summary[[
    '4개 유저 세그먼트', '유저_수', '유저 비중(%)', '결제자_수', '첫 결제율(%)',
    '결제 기여도(%)', '평균_레슨_완료수', '평균_고유_콘텐츠_수', '질문 클릭률(%)', '결제 페이지 진입률(%)'
]].copy()
segment4_display.columns = [
    '유저 세그먼트', '유저 수', '유저 비중(%)', '결제자 수', '첫 결제율(%)',
    '결제 기여도(%)', '평균 레슨 완료 수', '평균 고유 콘텐츠 수', '질문 클릭률(%)', '결제 페이지 진입률(%)'
]

for col in ['유저 비중(%)', '첫 결제율(%)', '결제 기여도(%)', '평균 레슨 완료 수', '평균 고유 콘텐츠 수', '질문 클릭률(%)', '결제 페이지 진입률(%)']:
    segment4_display[col] = segment4_display[col].round(2)

display(segment4_display)

segment4_plot = segment4_summary[[
    '4개 유저 세그먼트', '유저 비중(%)', '첫 결제율(%)', '결제 기여도(%)'
]].melt(
    id_vars='4개 유저 세그먼트',
    var_name='지표',
    value_name='비율(%)'
)
segment4_plot = add_percent_label(segment4_plot, '비율(%)')

fig = px.bar(
    segment4_plot,
    x='4개 유저 세그먼트',
    y='비율(%)',
    color='지표',
    barmode='group',
    text='비율 라벨',
    title='4개 유저 세그먼트별 규모, 첫 결제율, 결제 기여도',
    labels={'4개 유저 세그먼트': '유저 세그먼트', '비율(%)': '비율(%)', '지표': '지표'}
)
fig = apply_common_layout(
    fig,
    height=560,
    y_max=segment4_plot['비율(%)'].max() * 1.25,
    xangle=0
)
fig.show()


### 4-4. 유저 세그먼트 간 첫 결제율 차이 검정

세그먼트 이동 전략이 의미 있으려면, 다음 단계 세그먼트의 첫 결제율이 이전 단계보다 실제로 높은지 확인해야 합니다.  
아래에서는 `미경험형 → 탐색형`, `약한 학습형 → 강한 학습형` 두 구간의 첫 결제율 차이를 모비율 Z검정으로 확인합니다.


In [ ]:
# 유저 세그먼트 간 첫 결제율 차이 검정
# H0: 두 세그먼트의 첫 결제율은 같다
# H1: 두 세그먼트의 첫 결제율은 다르다

def run_segment4_ztest(summary_df, group_a, group_b):
    row_a = summary_df[summary_df['4개 유저 세그먼트'] == group_a].iloc[0]
    row_b = summary_df[summary_df['4개 유저 세그먼트'] == group_b].iloc[0]

    count = np.array([row_a['결제자_수'], row_b['결제자_수']])
    nobs = np.array([row_a['유저_수'], row_b['유저_수']])

    z_stat, p_value = proportions_ztest(
        count=count,
        nobs=nobs,
        alternative='two-sided'
    )

    rate_a = row_a['결제자_수'] / row_a['유저_수'] * 100
    rate_b = row_b['결제자_수'] / row_b['유저_수'] * 100
    diff = rate_b - rate_a

    if p_value < 0.05 and diff > 0:
        interpretation = f'{group_b}의 첫 결제율이 {group_a}보다 통계적으로 유의하게 높음'
    elif p_value < 0.05 and diff < 0:
        interpretation = f'{group_b}의 첫 결제율이 {group_a}보다 통계적으로 유의하게 낮음'
    else:
        interpretation = f'{group_a}와 {group_b}의 첫 결제율 차이가 통계적으로 뚜렷하다고 보기 어려움'

    return {
        '비교 구간': f'{group_a} → {group_b}',
        '기준 그룹 유저 수': row_a['유저_수'],
        '비교 그룹 유저 수': row_b['유저_수'],
        '기준 그룹 결제자 수': row_a['결제자_수'],
        '비교 그룹 결제자 수': row_b['결제자_수'],
        '기준 그룹 첫 결제율(%)': rate_a,
        '비교 그룹 첫 결제율(%)': rate_b,
        '결제율 차이(%p)': diff,
        'Z통계량': z_stat,
        'p-value': p_value,
        '검정 결과': '유의미함' if p_value < 0.05 else '유의미하지 않음',
        '해석': interpretation
    }

ztest_result = pd.DataFrame([
    run_segment4_ztest(segment4_summary, '미경험형', '탐색형'),
    run_segment4_ztest(segment4_summary, '약한 학습형', '강한 학습형')
])

ztest_display = ztest_result.copy()
for col in ['기준 그룹 첫 결제율(%)', '비교 그룹 첫 결제율(%)', '결제율 차이(%p)', 'Z통계량', 'p-value']:
    ztest_display[col] = ztest_display[col].round(4)

display(ztest_display)


### 모비율 Z검정 해석

- `미경험형 → 탐색형` 검정은 콘텐츠 경험 자체가 첫 결제 전환 가능성 차이와 연결되는지 확인하기 위한 보조 근거입니다.
- `약한 학습형 → 강한 학습형` 검정은 레슨 7개 이상 완료를 전환 후보 기준으로 사용할 수 있는지 확인하기 위한 보조 근거입니다.
- 단, 통계적으로 유의하더라도 인과효과가 확정되는 것은 아니므로 실제 액션 효과는 A/B 테스트로 별도 검증해야 합니다.


> **핵심 결론:** 첫 결제 전환 전략은 레슨 7개 이상 완료라는 단일 목표뿐 아니라, 무경험 → 탐색 → 초기 학습 → 핵심 학습으로 이동시키는 세그먼트별 액션 체계로 설계하는 과정이 필요함.


## 5. 무료체험 폐지 후 첫 결제자 품질 변화

해당 분석은 첫 결제율 하락과 별개로, 실제 결제한 유저의 구독 품질이 어떻게 달라졌는지 확인합니다.  
취소 버튼 클릭률은 실제 해지 완료가 아니라 해지 의도 추정 액션으로 해석하며, 60일 정기결제율과 함께 결제자 품질을 판단합니다.


### 5-1. 폐지 전후 핵심 리텐션 지표 비교


In [ ]:
# 무료체험 폐지 전/후 비교군 생성
retention_compare = retention[
    (retention['is_pre_retention_comparable'] == 1) |
    (retention['is_post_retention_comparable'] == 1)
].copy()

retention_compare['비교군'] = np.select(
    [
        retention_compare['is_pre_retention_comparable'] == 1,
        retention_compare['is_post_retention_comparable'] == 1
    ],
    ['폐지 전 비교군', '폐지 후 비교군'],
    default='비교군 외'
)

metric_info = [
    ('7일 내 취소 버튼 클릭률', 'cancel_7d', None),
    ('30일 내 취소 버튼 클릭률', 'cancel_30d', 'can_observe_30d'),
    ('60일 내 취소 버튼 클릭률', 'cancel_60d', 'can_observe_60d'),
    ('30일 내 정기결제율', 'renewed_30d', 'can_observe_30d'),
    ('60일 내 정기결제율', 'renewed_60d', 'can_observe_60d'),
    ('정기결제 전 취소 버튼 클릭률', 'cancel_before_first_renew', None)
]

metric_rows = []

for metric_name, metric_col, observe_col in metric_info:
    for group_name in ['폐지 전 비교군', '폐지 후 비교군']:
        temp = retention_compare[retention_compare['비교군'] == group_name].copy()

        if observe_col is not None:
            temp = temp[temp[observe_col] == 1]

        metric_rows.append({
            '지표': metric_name,
            '비교군': group_name,
            '분모 유저 수': temp['user_id'].nunique(),
            '해당 유저 수': int(temp[metric_col].sum()),
            '비율(%)': temp[metric_col].mean() * 100 if len(temp) > 0 else np.nan,
            '관측 조건': observe_col if observe_col is not None else '전체'
        })

retention_metric_summary = pd.DataFrame(metric_rows)

retention_metric_pivot = retention_metric_summary.pivot_table(
    index=['지표', '관측 조건'],
    columns='비교군',
    values='비율(%)',
    aggfunc='first'
).reset_index()

retention_metric_pivot['차이(%p)'] = (
    retention_metric_pivot['폐지 후 비교군'] - retention_metric_pivot['폐지 전 비교군']
)

display(retention_metric_pivot.round(2))

metric_order = [
    '7일 내 취소 버튼 클릭률',
    '30일 내 취소 버튼 클릭률',
    '60일 내 취소 버튼 클릭률',
    '정기결제 전 취소 버튼 클릭률',
    '30일 내 정기결제율',
    '60일 내 정기결제율'
]

metric_plot = retention_metric_summary.copy()
metric_plot['지표'] = pd.Categorical(metric_plot['지표'], categories=metric_order, ordered=True)
metric_plot = metric_plot.sort_values(['지표', '비교군'])
metric_plot = add_percent_label(metric_plot, '비율(%)')

fig = px.bar(
    metric_plot,
    x='지표',
    y='비율(%)',
    color='비교군',
    barmode='group',
    text='비율 라벨',
    hover_data=['분모 유저 수', '해당 유저 수', '관측 조건'],
    title='무료체험 폐지 후 취소 버튼 클릭률 감소와 60일 정기결제율 개선',
    labels={'지표': '리텐션 지표', '비율(%)': '비율(%)', '비교군': '비교군'}
)

fig = apply_common_layout(
    fig,
    height=620,
    y_max=metric_plot['비율(%)'].max() * 1.25,
    xangle=0
)
fig.show()


**현상**

- 폐지 후 7일 내 취소 버튼 클릭률 9.78%, 폐지 전 14.18% 대비 4.40%p 감소
- 폐지 후 30일 내 취소 버튼 클릭률 25.40%, 폐지 전 43.06% 대비 17.66%p 감소
- 폐지 후 60일 내 취소 버튼 클릭률 40.32%, 폐지 전 54.39% 대비 14.07%p 감소
- 폐지 후 60일 정기결제율 55.04%, 폐지 전 50.64% 대비 4.40%p 증가

**인사이트**

- 무료체험 폐지 후 첫 결제자의 초기 취소 의도 감소 가능성 존재
- 30일 내 정기결제율은 낮아졌으나, 60일 기준 정기결제율은 개선된 흐름 발견
- 첫 결제율 하락 가능성과 별개로 첫 결제자의 초기 구독 품질 개선 가능성 확인


### 5-2. 폐지 전후 결제 후 학습 행동 비교


In [ ]:
# 폐지 전/후 결제 후 학습 행동 비교
learning_cols = {
    'lesson_complete_postpay_7d': '7일 내 레슨 완료 수',
    'lesson_complete_postpay_30d': '30일 내 레슨 완료 수',
    'content_end_postpay_30d': '30일 내 콘텐츠 완료 수',
    'active_learning_days_postpay_30d': '30일 내 학습 활동일 수'
}

learning_rows = []

for group_name in ['폐지 전 비교군', '폐지 후 비교군']:
    temp = retention_compare[retention_compare['비교군'] == group_name].copy()

    for col, label in learning_cols.items():
        learning_rows.append({
            '비교군': group_name,
            '지표': label,
            '평균': temp[col].mean(),
            '중앙값': temp[col].median(),
            '유저 수': temp['user_id'].nunique()
        })

learning_compare = pd.DataFrame(learning_rows)

display(learning_compare.round(2))

learning_plot = learning_compare.copy()
learning_plot['평균 라벨'] = learning_plot['평균'].round(1).astype(str)

fig = px.bar(
    learning_plot,
    x='지표',
    y='평균',
    color='비교군',
    barmode='group',
    text='평균 라벨',
    hover_data=['중앙값', '유저 수'],
    title='무료체험 폐지 후 첫 결제자의 결제 후 학습 행동 증가',
    labels={'지표': '결제 후 학습 행동 지표', '평균': '평균값', '비교군': '비교군'}
)

fig = apply_common_layout(
    fig,
    height=560,
    y_max=learning_plot['평균'].max() * 1.25,
    xangle=-15
)
fig.show()


**현상**

- 폐지 후 결제 후 7일 내 레슨 완료 수 평균 52.15개, 폐지 전 34.18개 대비 증가
- 폐지 후 결제 후 30일 내 레슨 완료 수 평균 100.09개, 폐지 전 70.28개 대비 증가
- 폐지 후 결제 후 30일 내 학습 활동일 수 평균 10.02일, 폐지 전 7.06일 대비 증가
- 중앙값 기준으로도 7일 내 레슨 완료 수 11개 → 35개, 30일 내 학습 활동일 수 5일 → 8일 증가

**인사이트**

- 무료체험 폐지 후 첫 결제자의 결제 후 학습 몰입도 개선 가능성
- 취소 버튼 클릭률 감소와 결제 후 학습 행동 증가가 함께 나타나는 구조
- 결제 이후 학습 지속성이 리텐션 품질 개선과 연결될 가능성


> **핵심 결론:** 무료체험 폐지 후 첫 결제율은 하락했지만, 첫 결제자의 취소 버튼 클릭률은 낮아지고 결제 후 학습 행동은 증가해 결제자 품질이 개선되었을 가능성이 있음.


## 6. 결제 후 학습 행동과 정기구독 유지

해당 분석은 결제 이후 어떤 학습 행동이 정기구독 유지와 연결되는지 확인합니다.  
첫 결제 전에는 레슨 완료 수가 중요했지만, 결제 이후에는 30일 내 학습 활동일 수가 더 직관적인 리텐션 신호인지 검토합니다.


### 6-1. 결제 후 7일 내 학습 여부별 리텐션 비교


In [ ]:
# 결제 후 7일 내 학습 여부별 리텐션 지표
learning_7d_summary = make_retention_summary(post_retention, '결제후_7일내_학습여부')
learning_7d_order = ['7일 내 학습 O', '7일 내 학습 X']
learning_7d_summary['결제후_7일내_학습여부'] = pd.Categorical(
    learning_7d_summary['결제후_7일내_학습여부'],
    categories=learning_7d_order,
    ordered=True
)
learning_7d_summary = learning_7d_summary.sort_values('결제후_7일내_학습여부')

display(learning_7d_summary.round(2))

learning_7d_plot = learning_7d_summary[
    [
        '결제후_7일내_학습여부',
        '60일 정기결제율(%)',
        '30일 내 취소 버튼 클릭률(%)',
        '초기 이탈 위험 그룹 비중(%)'
    ]
].copy()

learning_7d_plot = learning_7d_plot.melt(
    id_vars='결제후_7일내_학습여부',
    var_name='지표',
    value_name='비율(%)'
)
learning_7d_plot = add_percent_label(learning_7d_plot, '비율(%)')

fig = px.bar(
    learning_7d_plot,
    x='지표',
    y='비율(%)',
    color='결제후_7일내_학습여부',
    barmode='group',
    text='비율 라벨',
    title='결제 후 7일 내 학습 유저의 60일 정기결제율 우위',
    labels={'지표': '리텐션 지표', '비율(%)': '비율(%)', '결제후_7일내_학습여부': '결제 후 7일 내 학습 여부'}
)

fig = apply_common_layout(
    fig,
    height=560,
    y_max=learning_7d_plot['비율(%)'].max() * 1.25,
    xangle=0
)
fig.show()


**현상**

- 결제 후 7일 내 학습 유저의 60일 정기결제율 60.28%, 미학습 유저 44.29% 대비 15.99%p 높음
- 결제 후 7일 내 학습 유저의 30일 내 취소 버튼 클릭률 21.78%, 미학습 유저 25.26% 대비 낮은 수준
- 결제 후 7일 내 학습 유저의 초기 이탈 위험 그룹 비중 18.24%, 미학습 유저 24.21% 대비 낮은 수준

**인사이트**

- 결제 직후 7일 내 학습 재개 여부는 60일 정기결제 유지와 연결되는 초기 신호
- 취소 버튼 클릭률 차이보다 정기결제율 차이가 더 크게 나타나는 구조
- 첫 결제 직후 학습 재개 유도가 초기 리텐션 관리의 핵심 후보


### 6-2. 결제 후 30일 내 학습 활동일 수별 리텐션 비교


In [ ]:
# 결제 후 30일 내 학습 활동일 수별 리텐션 지표
active_day_order = ['0일', '1~2일', '3~5일', '6~10일', '11~20일', '21일 이상']

active_day_summary = make_retention_summary(post_retention, '결제후_30일내_학습활동일구간')
active_day_summary['결제후_30일내_학습활동일구간'] = pd.Categorical(
    active_day_summary['결제후_30일내_학습활동일구간'],
    categories=active_day_order,
    ordered=True
)
active_day_summary = active_day_summary.sort_values('결제후_30일내_학습활동일구간')

display(active_day_summary.round(2))

active_day_plot = active_day_summary[
    [
        '결제후_30일내_학습활동일구간',
        '60일 정기결제율(%)',
        '30일 내 취소 버튼 클릭률(%)',
        '초기 이탈 위험 그룹 비중(%)'
    ]
].copy()

active_day_plot = active_day_plot.melt(
    id_vars='결제후_30일내_학습활동일구간',
    var_name='지표',
    value_name='비율(%)'
)
active_day_plot = add_percent_label(active_day_plot, '비율(%)')

fig = px.line(
    active_day_plot,
    x='결제후_30일내_학습활동일구간',
    y='비율(%)',
    color='지표',
    markers=True,
    text='비율 라벨',
    title='결제 후 30일 내 학습 활동일 수 증가에 따른 60일 정기결제율 상승',
    labels={'결제후_30일내_학습활동일구간': '결제 후 30일 내 학습 활동일 수', '비율(%)': '비율(%)', '지표': '리텐션 지표'}
)

fig.update_traces(textposition='top center')
fig.add_annotation(
    x='6~10일',
    y=active_day_plot['비율(%)'].max(),
    text='6일 이상 구간',
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-40
)
fig.update_layout(
    height=580,
    legend_title_text='',
    margin=dict(l=40, r=40, t=90, b=80)
)
fig.update_yaxes(range=[0, active_day_plot['비율(%)'].max() * 1.25])
fig.show()


**현상**

- 학습 활동일 0일 그룹의 60일 정기결제율 26.47%, 30일 내 취소 버튼 클릭률 27.42%
- 학습 활동일 6~10일 그룹의 60일 정기결제율 60.49%, 0일 그룹 대비 34.02%p 높음
- 학습 활동일 11~20일 그룹의 60일 정기결제율 63.76%, 전체 구간 중 가장 높은 수준
- 학습 활동일 증가에 따라 30일 내 취소 버튼 클릭률과 초기 이탈 위험 그룹 비중은 전반적으로 감소 경향

**인사이트**

- 결제 후 30일 내 학습 활동일 수는 정기구독 유지와 가장 직관적으로 연결되는 지표
- 6일 이상 학습 구간부터 60일 정기결제율이 60% 이상으로 상승하는 흐름
- 첫 결제 후 30일 내 6일 이상 학습을 리텐션 아하 모먼트 후보로 해석 가능


> **핵심 결론:** 첫 결제 직후 7일 내 학습 재개와 결제 후 30일 내 6일 이상 학습은 정기구독 유지와 직접적으로 연결되는 리텐션 아하 모먼트 후보임.


## 7. 첫 결제 전후 학습 조합 분석

해당 분석은 첫 결제 전 학습 신호와 결제 후 리텐션 신호를 함께 확인합니다.  
첫 결제 전 레슨 7개 이상 완료가 전환 신호라면, 결제 후 30일 내 6일 이상 학습은 유지 신호인지 구분해 해석합니다.


### 7-1. 결제 전 레슨 완료와 결제 후 학습 지속 조합


In [ ]:
# 첫 결제 전환 마트와 리텐션 마트 병합
# 앞단에서 이미 로드한 user 데이터의 결제 전 학습 행동 컬럼을 활용합니다.
activation = user.copy()

activation_merge_cols = [
    'user_id',
    'lesson_complete_prepay_7d',
    'content_end_prepay_7d',
    'unique_content_prepay_7d'
]
activation_merge_cols = [col for col in activation_merge_cols if col in activation.columns]
activation_for_merge = activation[activation_merge_cols].copy()

for col in activation_for_merge.columns:
    if col != 'user_id':
        activation_for_merge[col] = pd.to_numeric(activation_for_merge[col], errors='coerce')

retention_merged = retention.merge(
    activation_for_merge,
    on='user_id',
    how='left',
    indicator=True
)
retention_merged['merge_status'] = np.where(retention_merged['_merge'] == 'both', '매칭 O', '매칭 X')

merge_summary = retention_merged.groupby('merge_status').agg(
    유저_수=('user_id', 'nunique')
).reset_index()
merge_summary['비율(%)'] = merge_summary['유저_수'] / retention_merged['user_id'].nunique() * 100

display(merge_summary.round(2))

post_retention_merged = retention_merged[
    (retention_merged['is_post_removed'] == 1) &
    (retention_merged['merge_status'] == '매칭 O')
].copy()

post_retention_merged['결제전_학습수준'] = np.where(
    post_retention_merged['lesson_complete_prepay_7d'] >= 7,
    '결제 전 7개 이상',
    '결제 전 7개 미만'
)

post_retention_merged['결제후_학습지속수준'] = np.where(
    post_retention_merged['active_learning_days_postpay_30d'].fillna(0) >= 6,
    '결제 후 6일 이상 학습',
    '결제 후 0~5일 학습'
)

post_retention_merged['결제전후_학습조합'] = (
    post_retention_merged['결제전_학습수준'] +
    ' / ' +
    post_retention_merged['결제후_학습지속수준']
)

learning_combo_order = [
    '결제 전 7개 미만 / 결제 후 0~5일 학습',
    '결제 전 7개 미만 / 결제 후 6일 이상 학습',
    '결제 전 7개 이상 / 결제 후 0~5일 학습',
    '결제 전 7개 이상 / 결제 후 6일 이상 학습'
]

learning_combo_summary = make_retention_summary(post_retention_merged, '결제전후_학습조합')
learning_combo_summary['결제전후_학습조합'] = pd.Categorical(
    learning_combo_summary['결제전후_학습조합'],
    categories=learning_combo_order,
    ordered=True
)
learning_combo_summary = learning_combo_summary.sort_values('결제전후_학습조합')

display(learning_combo_summary.round(2))

learning_combo_plot = learning_combo_summary[
    [
        '결제전후_학습조합',
        '60일 정기결제율(%)',
        '30일 내 취소 버튼 클릭률(%)',
        '초기 이탈 위험 그룹 비중(%)'
    ]
].copy()

learning_combo_plot = learning_combo_plot.melt(
    id_vars='결제전후_학습조합',
    var_name='지표',
    value_name='비율(%)'
)
learning_combo_plot = add_percent_label(learning_combo_plot, '비율(%)')

fig = px.bar(
    learning_combo_plot,
    x='결제전후_학습조합',
    y='비율(%)',
    color='지표',
    barmode='group',
    text='비율 라벨',
    title='첫 결제 전보다 결제 후 30일 내 학습 지속성이 리텐션에 직접 연결',
    labels={'결제전후_학습조합': '첫 결제 전후 학습 조합', '비율(%)': '비율(%)', '지표': '리텐션 지표'}
)

fig = apply_common_layout(
    fig,
    height=660,
    y_max=learning_combo_plot['비율(%)'].max() * 1.25,
    xangle=-20
)
fig.show()


**현상**

- 폐지 후 전체 첫 결제자 중 첫 결제 전환 마트와 매칭된 유저 3,138명 기준 분석
- 결제 전 7개 미만 / 결제 후 0~5일 학습 그룹의 60일 정기결제율 47.60%
- 결제 전 7개 미만 / 결제 후 6일 이상 학습 그룹의 60일 정기결제율 63.20%
- 결제 전 7개 이상 / 결제 후 0~5일 학습 그룹의 60일 정기결제율 57.41%
- 결제 전 7개 이상 / 결제 후 6일 이상 학습 그룹의 60일 정기결제율 61.07%

**인사이트**

- 첫 결제 전 레슨 7개 이상 완료보다 결제 후 6일 이상 학습 여부가 리텐션과 더 직접적으로 연결
- 첫 결제 전 학습량이 낮아도 결제 후 학습 루틴이 형성되면 60일 정기결제율 개선 가능성
- 첫 결제 전 아하 모먼트는 첫 결제 전환 신호, 결제 후 30일 내 6일 이상 학습은 리텐션 신호로 분리 해석 필요


> **핵심 결론:** 결제 전 레슨 7개 이상 완료는 첫 결제 전환 신호이고, 결제 후 30일 내 6일 이상 학습은 정기구독 유지 신호로 분리해 관리해야 함.


### 7-2. 결제 후 학습 활동일 3구간 기준 보조 검증

본문에서는 결제 후 30일 내 6일 이상 학습 여부를 핵심 리텐션 신호로 보았습니다.  
아래 분석은 이를 한 단계 더 세분화해 결제 후 학습 활동일을 `0~4일`, `5~13일`, `14~32일`로 나누고, 결제 전 7개 이상 학습 여부와 함께 비교합니다.


In [ ]:
# 결제 전 학습 수준과 결제 후 학습 지속 수준을 3구간으로 확장해 비교합니다.
# 첫 결제율을 함께 보기 위해 activation 마트의 paid_30d도 함께 가져옵니다.

activation = user.copy()

activation_merge_cols = [
    'user_id',
    'paid_30d',
    'lesson_complete_prepay_7d',
    'content_end_prepay_7d',
    'unique_content_prepay_7d'
]

activation_merge_cols = [col for col in activation_merge_cols if col in activation.columns]
activation_for_merge = activation[activation_merge_cols].copy()

for col in activation_for_merge.columns:
    if col != 'user_id':
        activation_for_merge[col] = pd.to_numeric(activation_for_merge[col], errors='coerce').fillna(0)

retention_merged = retention.merge(
    activation_for_merge,
    on='user_id',
    how='left',
    indicator=True
)

retention_merged['merge_status'] = np.where(
    retention_merged['_merge'] == 'both',
    '매칭 O',
    '매칭 X'
)

merge_summary = retention_merged.groupby('merge_status').agg(
    유저_수=('user_id', 'nunique')
).reset_index()

merge_summary['비율(%)'] = (
    merge_summary['유저_수'] / retention_merged['user_id'].nunique() * 100
)

display(merge_summary.round(2))


# 폐지 후 + 병합 성공 유저만 사용
post_retention_merged = retention_merged[
    (retention_merged['is_post_removed'] == 1) &
    (retention_merged['merge_status'] == '매칭 O')
].copy()


# 1) 결제 전 학습 수준: 7개 미만 / 7개 이상
post_retention_merged['결제전_학습수준'] = np.where(
    post_retention_merged['lesson_complete_prepay_7d'] >= 7,
    '결제 전 7개 이상',
    '결제 전 7개 미만'
)


# 2) 결제 후 학습 지속 수준: 0~4일 / 5~13일 / 14~32일
post_retention_merged['결제후_학습지속수준'] = pd.cut(
    post_retention_merged['active_learning_days_postpay_30d'].fillna(0),
    bins=[-1, 4, 13, 32],
    labels=[
        '결제 후 0~4일 학습',
        '결제 후 5~13일 학습',
        '결제 후 14~32일 학습'
    ]
)


# 3) 결제 전후 학습 조합 생성
post_retention_merged['결제전후_학습조합'] = (
    post_retention_merged['결제전_학습수준'] +
    ' / ' +
    post_retention_merged['결제후_학습지속수준'].astype(str)
)


learning_combo_order = [
    '결제 전 7개 미만 / 결제 후 0~4일 학습',
    '결제 전 7개 미만 / 결제 후 5~13일 학습',
    '결제 전 7개 미만 / 결제 후 14~32일 학습',
    '결제 전 7개 이상 / 결제 후 0~4일 학습',
    '결제 전 7개 이상 / 결제 후 5~13일 학습',
    '결제 전 7개 이상 / 결제 후 14~32일 학습'
]


# 4) 리텐션 지표 요약
learning_combo_summary = make_retention_summary(
    post_retention_merged,
    '결제전후_학습조합'
)


# 5) 첫 결제율 계산용: 폐지 후 전체 가입자 기준 분모 생성
# 주의: 결제 후 학습일 수는 결제자에게만 있으므로,
# 여기서 첫 결제율은 "전체 가입자 중 해당 조합까지 도달한 비율"로 해석합니다.

post_activation_base = activation[
    activation['period_group'] == 'post_2023_05_11'
].copy()

post_activation_base['paid_30d'] = pd.to_numeric(
    post_activation_base['paid_30d'],
    errors='coerce'
).fillna(0)

post_activation_base['lesson_complete_prepay_7d'] = pd.to_numeric(
    post_activation_base['lesson_complete_prepay_7d'],
    errors='coerce'
).fillna(0)

post_activation_base['결제전_학습수준'] = np.where(
    post_activation_base['lesson_complete_prepay_7d'] >= 7,
    '결제 전 7개 이상',
    '결제 전 7개 미만'
)

prepay_denominator = post_activation_base.groupby('결제전_학습수준').agg(
    결제전_학습수준별_가입자수=('user_id', 'nunique'),
    결제전_학습수준별_첫결제자수=('paid_30d', 'sum')
).reset_index()

prepay_denominator['결제전_학습수준별_첫결제율(%)'] = (
    prepay_denominator['결제전_학습수준별_첫결제자수'] /
    prepay_denominator['결제전_학습수준별_가입자수'] * 100
)


# 6) 각 결제 전후 조합별 첫 결제자 수 계산
combo_payment_summary = post_retention_merged.groupby(
    ['결제전_학습수준', '결제후_학습지속수준'],
    observed=False
).agg(
    조합_유저수=('user_id', 'nunique'),
    조합_30일내_첫결제자수=('paid_30d', 'sum')
).reset_index()

combo_payment_summary['결제전후_학습조합'] = (
    combo_payment_summary['결제전_학습수준'] +
    ' / ' +
    combo_payment_summary['결제후_학습지속수준'].astype(str)
)

combo_payment_summary = combo_payment_summary.merge(
    prepay_denominator,
    on='결제전_학습수준',
    how='left'
)

combo_payment_summary['첫 결제율(%)'] = (
    combo_payment_summary['조합_30일내_첫결제자수'] /
    combo_payment_summary['결제전_학습수준별_가입자수'] * 100
)


# 7) 리텐션 요약 테이블에 첫 결제율 붙이기
learning_combo_summary = learning_combo_summary.merge(
    combo_payment_summary[
        [
            '결제전후_학습조합',
            '첫 결제율(%)',
            '조합_유저수',
            '조합_30일내_첫결제자수',
            '결제전_학습수준별_가입자수',
            '결제전_학습수준별_첫결제율(%)'
        ]
    ],
    on='결제전후_학습조합',
    how='left'
)

learning_combo_summary['결제전후_학습조합'] = pd.Categorical(
    learning_combo_summary['결제전후_학습조합'],
    categories=learning_combo_order,
    ordered=True
)

learning_combo_summary = learning_combo_summary.sort_values('결제전후_학습조합')

display(learning_combo_summary.round(2))


# 8) 그래프용 데이터 생성
learning_combo_plot = learning_combo_summary[
    [
        '결제전후_학습조합',
        '첫 결제율(%)',
        '60일 정기결제율(%)',
        '30일 내 취소 버튼 클릭률(%)',
        '초기 이탈 위험 그룹 비중(%)'
    ]
].copy()

learning_combo_plot = learning_combo_plot.melt(
    id_vars='결제전후_학습조합',
    var_name='지표',
    value_name='비율(%)'
)

learning_combo_plot = add_percent_label(learning_combo_plot, '비율(%)')


fig = px.bar(
    learning_combo_plot,
    x='결제전후_학습조합',
    y='비율(%)',
    color='지표',
    barmode='group',
    text='비율 라벨',
    title='결제 전 7개 이상 학습과 결제 후 학습 지속성이 첫 결제·리텐션에 연결',
    labels={
        '결제전후_학습조합': '첫 결제 전후 학습 조합',
        '비율(%)': '비율(%)',
        '지표': '지표'
    }
)

fig = apply_common_layout(
    fig,
    height=720,
    y_max=learning_combo_plot['비율(%)'].max() * 1.25,
    xangle=-25
)

fig.show()


### 결제 전후 학습 조합 보조 검증 해석

- 결제 전 7개 이상 완료 여부는 첫 결제 전환 가능성을 설명하는 신호로 해석합니다.
- 결제 후 학습 활동일 구간은 첫 결제 이후 정기구독 유지 가능성을 설명하는 신호로 해석합니다.
- 두 신호를 함께 보면, 전환 전 학습량보다 결제 후 학습 루틴 재형성이 리텐션 관리에서 더 직접적인 운영 과제가 될 수 있습니다.


## 8. 종합 결론 및 실행 제안

전체 분석을 종합하면 무료체험 폐지 이후의 핵심 과제는 단순 가입자 확대나 첫 결제율 회복이 아니라,  
첫 결제 전에는 충분한 학습 경험을 만들고, 첫 결제 이후에는 학습 루틴을 빠르게 재개·유지시키는 것입니다.


### 핵심 발견

- 무료체험 폐지 이후 가입자 수는 증가했지만, 가입 후 30일 내 첫 결제율은 하락.
- 가입 후 7일 내 레슨 7개 이상 완료는 첫 결제 전환의 1차 아하 모먼트 후보로 해석 가능.
- 콘텐츠 완료 경험은 단순 탐색보다 결제 전환 가능성을 높이는 중요한 보조 신호로 판단.
- 무료체험 폐지 후 첫 결제자의 취소 버튼 클릭률은 감소하고, 결제 후 학습 행동은 증가하는 흐름 확인.
- 결제 후 7일 내 학습 재개 여부는 60일 정기결제율과 연결되는 초기 리텐션 신호.
- 결제 후 30일 내 6일 이상 학습은 정기구독 유지의 핵심 행동 기준으로 활용 가능.
- 결제 전 행동과 결제 후 행동은 같은 지표로 묶기보다, 전환 신호와 리텐션 신호로 분리해 관리 필요.


### 실행 제안 매트릭스


In [ ]:
# 최종 실행 제안 매트릭스 생성
# 첫 결제 전환 분석과 리텐션 분석을 하나의 운영 액션 체계로 연결합니다.
final_story_action = pd.DataFrame({
    '우선순위': [1, 2, 3, 4, 5, 6, 7],
    '대상 유저': [
        '무행동형',
        '탐색형',
        '약한 학습형·임계 직전형',
        '핵심 학습형',
        '첫 결제 직후 유저',
        '결제 후 0~5일 학습 유저',
        '중도 이탈·초기 이탈 위험 그룹'
    ],
    '현재 상태': [
        '가입 후 콘텐츠 경험 부족',
        '콘텐츠 경험은 있으나 학습 완료 부족',
        '레슨 1~6개 완료로 아하 모먼트 직전',
        '레슨 7개 이상 완료로 첫 결제 전환 가능성 상승',
        '첫 결제는 완료했으나 학습 루틴 재형성 전',
        '결제 후 30일 내 학습 활동일 수 부족',
        '정기결제 전후 취소 의도 또는 이탈 위험 발생'
    ],
    '이동 목표': [
        '첫 콘텐츠 경험',
        '첫 레슨 완료 또는 첫 콘텐츠 완료',
        '가입 후 7일 내 레슨 7개 완료',
        '반복 학습 및 첫 결제 전환',
        '결제 후 7일 내 학습 재개',
        '결제 후 30일 내 6일 이상 학습',
        '취소 의도 감소 및 정기구독 유지'
    ],
    '제안 액션': [
        '관심사 기반 첫 콘텐츠 추천, 온보딩 메시지 노출',
        '짧은 레슨 추천, 이어보기, 첫 완료 보상 제공',
        '7번째 레슨 완료 유도 메시지, 다음 레슨 자동 추천',
        '유료 플랜 CTA, 심화 로드맵, 학습 성취 메시지 제공',
        '결제 완료 직후 다음 학습 CTA, 7일 내 학습 리마인드',
        '주간 학습 목표, 루틴 알림, 진행률 피드백, 복귀 메시지 운영',
        '취소 버튼 클릭 전후 설문, 혜택 재안내, 콘텐츠 만족도·가격 부담 추가 분석'
    ],
    '확인 KPI': [
        '콘텐츠 조회율, 콘텐츠 시작률',
        '첫 레슨 완료율, 콘텐츠 완료율',
        '레슨 7개 이상 완료율',
        '첫 결제율, 결제 기여도',
        '결제 후 7일 내 학습률',
        '30일 내 6일 이상 학습률, 60일 정기결제율',
        '30일 내 취소 버튼 클릭률, 60일 정기결제율, 재구독률'
    ]
})

display(final_story_action)


### 실행 제안 기대매출 시뮬레이션

이전 분석에서 확인한 핵심 전환 단계가 실제 매출 관점에서 어느 정도 의미가 있는지 가늠하기 위해, 보수적·기준·공격적 시나리오로 기대효과를 추정합니다.  
해당 결과는 실제 A/B 테스트 결과가 아니라, 현재 데이터의 유저 규모와 세그먼트별 전환율·갱신율 차이를 기반으로 한 가정 기반 시뮬레이션입니다.


In [ ]:

# ============================================================
# Appendix. 세그먼트별 기대매출 시뮬레이션 - 첫 결제 파트
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# ------------------------------------------------------------
# 0. 공통 유틸 함수
# ------------------------------------------------------------
def find_csv_file(base_names):
    """
    CSV 파일을 여러 실행 환경에서 찾기 위한 보조 함수입니다.
    - VS Code: 현재 폴더, data 폴더
    - Colab: /content, Google Drive 일부 경로
    - ChatGPT sandbox: data/private
    """
    base_dir = Path.cwd()
    search_dirs = [
        base_dir,
        base_dir / 'data',
        base_dir.parent,
        base_dir.parent / 'data',
        Path('data/private'),
        Path('/content'),
        Path('/content/data'),
        Path('data/private'),
        Path('data/private'),
        Path('data/private'),
        Path('data/private'),
    ]

    candidates = []
    for name in base_names:
        candidates.append(name)
        stem = name.replace('.csv', '')
        for i in range(1, 21):
            candidates.append(f'{stem}({i}).csv')

    for directory in search_dirs:
        for name in candidates:
            path = directory / name
            if path.exists():
                return path

    checked = [str(d / n) for d in search_dirs[:5] for n in candidates[:5]]
    raise FileNotFoundError(
        'CSV 파일을 찾을 수 없습니다. 노트북과 같은 폴더 또는 data 폴더에 CSV를 두고 다시 실행해주세요.\n'
        + '\n'.join(checked[:20])
    )

def valid_amount_series(series):
    s = pd.to_numeric(series, errors='coerce')
    return s[(s.notna()) & (s > 0)]

def safe_mean(series):
    s = valid_amount_series(series)
    return float(s.mean()) if len(s) > 0 else 0.0

def safe_median(series):
    s = valid_amount_series(series)
    return float(s.median()) if len(s) > 0 else 0.0

def format_money_columns(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce').round(0).astype('Int64')
    return out

def round_numeric(df, digits=2):
    out = df.copy()
    for col in out.select_dtypes(include=[np.number]).columns:
        out[col] = out[col].round(digits)
    return out

def display_df(title, df, digits=2):
    display(Markdown(f'### {title}'))
    display(round_numeric(df, digits))

# ------------------------------------------------------------
# 1. 데이터 로드
# ------------------------------------------------------------
try:
    activation_raw = user.copy()
except NameError:
    activation_path = find_csv_file(['dm_user_activation_payment(8).csv', 'dm_user_activation_payment.csv'])
    activation_raw = pd.read_csv(activation_path, low_memory=False)
    print(f'첫 결제 데이터 로드 경로: {activation_path}')

# ------------------------------------------------------------
# 2. 첫 결제 분석 대상 및 기존 4개 세그먼트 기준 적용
# ------------------------------------------------------------
activation = activation_raw.copy()

if 'period_group' in activation.columns:
    activation_post = activation[activation['period_group'] == 'post_2023_05_11'].copy()
else:
    activation_post = activation.copy()
    print('period_group 컬럼이 없어 전체 activation 데이터를 사용합니다.')

needed_numeric_cols = [
    'paid_30d',
    'lesson_complete_prepay_7d',
    'unique_content_prepay_7d',
    'content_view_7d',
    'content_start_7d',
    'first_paid_amount',
    'first_plan_price'
]

for col in needed_numeric_cols:
    if col not in activation_post.columns:
        activation_post[col] = 0
    activation_post[col] = pd.to_numeric(activation_post[col], errors='coerce').fillna(0)

activation_post['콘텐츠 경험 여부'] = np.where(
    (activation_post['unique_content_prepay_7d'] > 0) |
    (activation_post['content_view_7d'] > 0) |
    (activation_post['content_start_7d'] > 0),
    '콘텐츠 경험 O',
    '콘텐츠 경험 X'
)

def make_first_payment_segment(row):
    lesson_count = row['lesson_complete_prepay_7d']
    content_experience = row['콘텐츠 경험 여부'] == '콘텐츠 경험 O'

    if lesson_count >= 7:
        return '강한 학습형'
    elif lesson_count >= 1:
        return '약한 학습형'
    elif content_experience:
        return '탐색형'
    else:
        return '미경험형'

first_segment_order = ['미경험형', '탐색형', '약한 학습형', '강한 학습형']
activation_post['유저 세그먼트'] = activation_post.apply(make_first_payment_segment, axis=1)

total_post_users = activation_post['user_id'].nunique()
total_post_payers = int(activation_post['paid_30d'].sum())

first_segment_rows = []
for segment in first_segment_order:
    temp = activation_post[activation_post['유저 세그먼트'] == segment].copy()
    paid_temp = temp[(temp['paid_30d'] == 1) & (temp['first_paid_amount'] > 0)].copy()

    users = temp['user_id'].nunique()
    payers = int(temp['paid_30d'].sum())

    first_segment_rows.append({
        '유저 세그먼트': segment,
        '유저 수': users,
        '결제자 수': payers,
        '첫 결제율(%)': payers / users * 100 if users > 0 else 0,
        '평균 첫 결제금액': safe_mean(paid_temp['first_paid_amount']),
        '중앙값 첫 결제금액': safe_median(paid_temp['first_paid_amount']),
        '결제 기여도(%)': payers / total_post_payers * 100 if total_post_payers > 0 else 0
    })

first_payment_segment_table = pd.DataFrame(first_segment_rows)

# 첫 결제 기대매출에는 폐지 후 첫 결제자 전체의 금액 평균/중앙값을 적용합니다.
# 이유: 시나리오 간 비교를 단순화하고, 특정 세그먼트의 가격 믹스 차이가 기대매출 비교를 과도하게 흔들지 않도록 하기 위함입니다.
first_paid_users = activation_post[(activation_post['paid_30d'] == 1) & (activation_post['first_paid_amount'] > 0)].copy()
first_payment_amount_avg = safe_mean(first_paid_users['first_paid_amount'])
first_payment_amount_median = safe_median(first_paid_users['first_paid_amount'])

first_amount_basis = {
    '평균 첫 결제금액 기준': first_payment_amount_avg,
    '중앙값 첫 결제금액 기준': first_payment_amount_median
}

display(Markdown('### 세그먼트별 기대매출 시뮬레이션'))
display(Markdown('''
- 이번 분석은 세그먼트 간 성과 차이를 활용한 What-if 시뮬레이션입니다.
- 1%, 3%, 5%, 10%는 전환율/갱신율 상승폭이 아니라 **출발 세그먼트 유저 중 다음 단계로 이동한다고 가정하는 유저 비중**입니다.
- 기대매출은 액션 비용을 제외한 매출 잠재력입니다.
- 본 분석은 실제 인과효과를 확정하는 것이 아니라, 세그먼트 간 성과 차이를 기반으로 한 기대매출 잠재력 추정이며, 실제 효과 검증은 A/B 테스트가 필요합니다.
'''))

display_df('2. 첫 결제 세그먼트 기준표', format_money_columns(first_payment_segment_table, ['평균 첫 결제금액', '중앙값 첫 결제금액']))

# ------------------------------------------------------------
# 3. 첫 결제 기대매출 시뮬레이션
# ------------------------------------------------------------
movement_rates = [0.01, 0.03, 0.05, 0.10]

first_scenarios = [
    {
        '액션 시나리오': 'A. 미경험형 → 탐색형',
        '출발 세그먼트': '미경험형',
        '도착 세그먼트': '탐색형'
    },
    {
        '액션 시나리오': 'B. 약한 학습형 → 강한 학습형',
        '출발 세그먼트': '약한 학습형',
        '도착 세그먼트': '강한 학습형'
    }
]

first_simulation_rows = []
segment_metric = first_payment_segment_table.set_index('유저 세그먼트')

for scenario in first_scenarios:
    source = scenario['출발 세그먼트']
    target = scenario['도착 세그먼트']

    source_users = float(segment_metric.loc[source, '유저 수']) if source in segment_metric.index else 0
    source_rate = float(segment_metric.loc[source, '첫 결제율(%)']) / 100 if source in segment_metric.index else 0
    target_rate = float(segment_metric.loc[target, '첫 결제율(%)']) / 100 if target in segment_metric.index else 0
    rate_diff = target_rate - source_rate

    for amount_label, amount_value in first_amount_basis.items():
        for move_rate in movement_rates:
            incremental_payers = source_users * move_rate * rate_diff
            expected_revenue = incremental_payers * amount_value

            first_simulation_rows.append({
                '액션 시나리오': scenario['액션 시나리오'],
                '출발 세그먼트': source,
                '도착 세그먼트': target,
                '출발 세그먼트 유저 수': source_users,
                '출발 세그먼트 전환율(%)': source_rate * 100,
                '도착 세그먼트 전환율(%)': target_rate * 100,
                '전환율 차이(%p)': rate_diff * 100,
                '다음 단계 이동 비중(%)': move_rate * 100,
                '증분 결제자 수': incremental_payers,
                '적용 금액 기준': amount_label,
                '적용 결제금액': amount_value,
                '기대매출': expected_revenue,
                '기대매출(만원)': expected_revenue / 10000
            })

first_payment_simulation = pd.DataFrame(first_simulation_rows)

first_payment_simulation_avg = first_payment_simulation[
    first_payment_simulation['적용 금액 기준'] == '평균 첫 결제금액 기준'
].copy()

first_payment_simulation_median = first_payment_simulation[
    first_payment_simulation['적용 금액 기준'] == '중앙값 첫 결제금액 기준'
].copy()

display_df(
    '3. 첫 결제 기대매출 시뮬레이션 표 - 평균 결제금액 기준',
    format_money_columns(first_payment_simulation_avg, ['출발 세그먼트 유저 수', '적용 결제금액', '기대매출'])
)

display_df(
    '4. 첫 결제 기대매출 시뮬레이션 표 - 중앙값 결제금액 기준',
    format_money_columns(first_payment_simulation_median, ['출발 세그먼트 유저 수', '적용 결제금액', '기대매출'])
)

first_payment_pivot = first_payment_simulation.pivot_table(
    index=['적용 금액 기준', '액션 시나리오'],
    columns='다음 단계 이동 비중(%)',
    values='기대매출(만원)',
    aggfunc='sum'
).reset_index()

first_payment_pivot.columns.name = None
display_df('9-1. 첫 결제 기대매출 요약 피벗 테이블 - 기대매출(만원)', first_payment_pivot)

# ------------------------------------------------------------
# 4. 첫 결제 세그먼트 합계 검증
# ------------------------------------------------------------
first_payment_validation = pd.DataFrame({
    '검증 항목': [
        '세그먼트 유저 수 합계',
        '폐지 후 전체 유저 수',
        '세그먼트 결제자 수 합계',
        '폐지 후 전체 결제자 수'
    ],
    '값': [
        int(first_payment_segment_table['유저 수'].sum()),
        int(total_post_users),
        int(first_payment_segment_table['결제자 수'].sum()),
        int(total_post_payers)
    ],
    '일치 여부': [
        int(first_payment_segment_table['유저 수'].sum()) == int(total_post_users),
        True,
        int(first_payment_segment_table['결제자 수'].sum()) == int(total_post_payers),
        True
    ]
})

display_df('첫 결제 세그먼트 합계 검증', first_payment_validation)


In [ ]:

# ============================================================
# Appendix. 세그먼트별 기대매출 시뮬레이션 - 정기결제 갱신 파트
# ============================================================

# ------------------------------------------------------------
# 1. 정기결제 데이터 로드 및 기본 준비
# ------------------------------------------------------------
try:
    retention_raw = retention.copy()
except NameError:
    retention_path = find_csv_file(['dm_user_subscription_retention_core(6).csv', 'dm_user_subscription_retention_core.csv'])
    retention_raw = pd.read_csv(retention_path, low_memory=False)
    print(f'정기결제 데이터 로드 경로: {retention_path}')

retention_base = retention_raw.copy()

# 폐지 후 첫 결제자 기준
if 'is_post_removed' in retention_base.columns:
    retention_base = retention_base[retention_base['is_post_removed'] == 1].copy()
elif 'period_group' in retention_base.columns:
    retention_base = retention_base[retention_base['period_group'] == 'post_2023_05_11'].copy()
else:
    print('is_post_removed/period_group 컬럼이 없어 전체 retention 데이터를 사용합니다.')

retention_numeric_cols = [
    'active_learning_days_postpay_30d',
    'renewed_30d',
    'renewed_60d',
    'renewed_90d',
    'can_observe_30d',
    'can_observe_60d',
    'can_observe_90d',
    'first_paid_amount',
    'first_plan_price'
]

for col in retention_numeric_cols:
    if col not in retention_base.columns:
        retention_base[col] = 0
    retention_base[col] = pd.to_numeric(retention_base[col], errors='coerce').fillna(0)

# 기존 ipynb의 리텐션 해석 기준 유지:
# 결제 후 30일 내 학습 활동일 수를 0~4일 / 5~13일 / 14~32일로 구분
retention_base['정기결제 유저 세그먼트'] = pd.cut(
    retention_base['active_learning_days_postpay_30d'].fillna(0),
    bins=[-1, 4, 13, 32],
    labels=[
        '저활동 Seg1: 결제 후 0~4일 학습',
        '중간활동 Seg2: 결제 후 5~13일 학습',
        '고활동 Seg3: 결제 후 14~32일 학습'
    ]
)

retention_segment_order = [
    '저활동 Seg1: 결제 후 0~4일 학습',
    '중간활동 Seg2: 결제 후 5~13일 학습',
    '고활동 Seg3: 결제 후 14~32일 학습'
]

# 실제 갱신 결제금액 컬럼 탐색
actual_renew_amount_candidates = [
    col for col in retention_base.columns
    if ('renew' in col.lower()) and (('amount' in col.lower()) or ('paid' in col.lower()) or ('price' in col.lower()))
]
actual_renew_amount_candidates = [
    col for col in actual_renew_amount_candidates
    if col not in ['renewed_7d', 'renewed_30d', 'renewed_60d', 'renewed_90d', 'renew_count_after_payment']
]

if len(actual_renew_amount_candidates) > 0:
    print('실제 갱신 결제금액 후보 컬럼이 발견되었습니다:', actual_renew_amount_candidates)
    print('다만 본 노트북 출력 요구사항에 맞춰 first_paid_amount와 first_plan_price 대리값도 함께 산출합니다.')
else:
    print('실제 갱신 결제금액 컬럼을 찾지 못했습니다. first_paid_amount와 first_plan_price를 대리 갱신금액으로 사용합니다.')

# ------------------------------------------------------------
# 2. 관측기간별 정기결제 세그먼트 기준표
# ------------------------------------------------------------
retention_windows = [
    {'관측기간': '30일 갱신', 'renew_col': 'renewed_30d', 'observe_col': 'can_observe_30d'},
    {'관측기간': '60일 갱신', 'renew_col': 'renewed_60d', 'observe_col': 'can_observe_60d'},
    {'관측기간': '90일 갱신', 'renew_col': 'renewed_90d', 'observe_col': 'can_observe_90d'},
]

retention_segment_tables = {}
retention_segment_all_rows = []

for window in retention_windows:
    period_label = window['관측기간']
    renew_col = window['renew_col']
    observe_col = window['observe_col']

    if observe_col in retention_base.columns:
        temp_base = retention_base[retention_base[observe_col] == 1].copy()
        observe_note = f'{observe_col} == 1'
    else:
        temp_base = retention_base.copy()
        observe_note = f'{observe_col} 컬럼 없음 → 전체 유저 기준'

    total_renewers = int(temp_base[renew_col].sum()) if renew_col in temp_base.columns else 0

    rows = []
    for segment in retention_segment_order:
        temp = temp_base[temp_base['정기결제 유저 세그먼트'].astype(str) == segment].copy()
        users = temp['user_id'].nunique()
        renewers = int(temp[renew_col].sum()) if renew_col in temp.columns else 0

        row = {
            '관측기간': period_label,
            '관측 기준': observe_note,
            '유저 세그먼트': segment,
            '유저 수': users,
            '갱신자 수': renewers,
            '갱신율(%)': renewers / users * 100 if users > 0 else 0,
            'first_paid_amount 평균': safe_mean(temp['first_paid_amount']),
            'first_paid_amount 중앙값': safe_median(temp['first_paid_amount']),
            'first_plan_price 평균': safe_mean(temp['first_plan_price']),
            'first_plan_price 중앙값': safe_median(temp['first_plan_price']),
            '갱신 기여도(%)': renewers / total_renewers * 100 if total_renewers > 0 else 0
        }
        rows.append(row)
        retention_segment_all_rows.append(row)

    table = pd.DataFrame(rows)
    retention_segment_tables[period_label] = table

    display_df(
        f'정기결제 세그먼트 기준표 - {period_label}',
        format_money_columns(
            table,
            ['first_paid_amount 평균', 'first_paid_amount 중앙값', 'first_plan_price 평균', 'first_plan_price 중앙값']
        )
    )

retention_segment_table_all = pd.DataFrame(retention_segment_all_rows)

# ------------------------------------------------------------
# 3. 정기결제 기대매출 시뮬레이션
# ------------------------------------------------------------
retention_scenarios = [
    {
        '액션 시나리오': 'C. 저활동 Seg1 → 중간활동 Seg2',
        '출발 세그먼트': '저활동 Seg1: 결제 후 0~4일 학습',
        '도착 세그먼트': '중간활동 Seg2: 결제 후 5~13일 학습'
    },
    {
        '액션 시나리오': 'D. 중간활동 Seg2 → 고활동 Seg3',
        '출발 세그먼트': '중간활동 Seg2: 결제 후 5~13일 학습',
        '도착 세그먼트': '고활동 Seg3: 결제 후 14~32일 학습'
    }
]

amount_basis_cols = {
    'first_paid_amount 평균': 'first_paid_amount 평균',
    'first_paid_amount 중앙값': 'first_paid_amount 중앙값',
    'first_plan_price 평균': 'first_plan_price 평균',
    'first_plan_price 중앙값': 'first_plan_price 중앙값'
}

retention_sim_rows = []

for period_label, table in retention_segment_tables.items():
    metric = table.set_index('유저 세그먼트')

    for scenario in retention_scenarios:
        source = scenario['출발 세그먼트']
        target = scenario['도착 세그먼트']

        source_users = float(metric.loc[source, '유저 수']) if source in metric.index else 0
        source_rate = float(metric.loc[source, '갱신율(%)']) / 100 if source in metric.index else 0
        target_rate = float(metric.loc[target, '갱신율(%)']) / 100 if target in metric.index else 0
        rate_diff = target_rate - source_rate

        # 적용 금액은 출발 세그먼트의 대리 갱신금액을 사용합니다.
        # 이유: 실제로 이동시키려는 대상은 출발 세그먼트 유저이며, 기존 플랜/결제금액 구조를 유지한다고 보는 것이 보수적입니다.
        for amount_label, amount_col in amount_basis_cols.items():
            amount_value = float(metric.loc[source, amount_col]) if source in metric.index else 0

            for move_rate in movement_rates:
                incremental_renewers = source_users * move_rate * rate_diff
                expected_revenue = incremental_renewers * amount_value

                retention_sim_rows.append({
                    '관측기간': period_label,
                    '액션 시나리오': scenario['액션 시나리오'],
                    '출발 세그먼트': source,
                    '도착 세그먼트': target,
                    '출발 세그먼트 유저 수': source_users,
                    '출발 세그먼트 갱신율(%)': source_rate * 100,
                    '도착 세그먼트 갱신율(%)': target_rate * 100,
                    '갱신율 차이(%p)': rate_diff * 100,
                    '다음 단계 이동 비중(%)': move_rate * 100,
                    '증분 갱신자 수': incremental_renewers,
                    '적용 금액 기준': amount_label,
                    '적용 금액': amount_value,
                    '기대매출': expected_revenue,
                    '기대매출(만원)': expected_revenue / 10000
                })

retention_simulation = pd.DataFrame(retention_sim_rows)

display_df(
    '8. 정기결제 기대매출 시뮬레이션 표',
    format_money_columns(retention_simulation, ['출발 세그먼트 유저 수', '적용 금액', '기대매출'])
)

retention_pivot = retention_simulation.pivot_table(
    index=['관측기간', '액션 시나리오', '적용 금액 기준'],
    columns='다음 단계 이동 비중(%)',
    values='기대매출(만원)',
    aggfunc='sum'
).reset_index()
retention_pivot.columns.name = None

display_df('9-2. 정기결제 기대매출 요약 피벗 테이블 - 기대매출(만원)', retention_pivot)

# 관측기간별 유저 수·갱신자 수 검증
retention_validation_rows = []
for window in retention_windows:
    period_label = window['관측기간']
    renew_col = window['renew_col']
    observe_col = window['observe_col']

    if observe_col in retention_base.columns:
        temp_base = retention_base[retention_base[observe_col] == 1].copy()
    else:
        temp_base = retention_base.copy()

    table = retention_segment_tables[period_label]
    retention_validation_rows.append({
        '관측기간': period_label,
        '세그먼트 유저 수 합계': int(table['유저 수'].sum()),
        '분석 대상 전체 유저 수': int(temp_base['user_id'].nunique()),
        '유저 수 일치 여부': int(table['유저 수'].sum()) == int(temp_base['user_id'].nunique()),
        '세그먼트 갱신자 수 합계': int(table['갱신자 수'].sum()),
        '분석 대상 전체 갱신자 수': int(temp_base[renew_col].sum()),
        '갱신자 수 일치 여부': int(table['갱신자 수'].sum()) == int(temp_base[renew_col].sum())
    })

retention_validation = pd.DataFrame(retention_validation_rows)
display_df('정기결제 관측기간별 합계 검증', retention_validation)


In [ ]:

# ============================================================
# Appendix. 발표용 해석 문장
# ============================================================

# 첫 결제 최고 기대매출 시나리오: 평균 결제금액 기준, 10% 이동 가정
first_focus = first_payment_simulation[
    (first_payment_simulation['적용 금액 기준'] == '평균 첫 결제금액 기준') &
    (first_payment_simulation['다음 단계 이동 비중(%)'] == 10)
].copy()

if len(first_focus) > 0:
    first_best = first_focus.sort_values('기대매출', ascending=False).iloc[0]
else:
    first_best = None

# 정기결제 최고 기대매출 시나리오: first_paid_amount 평균 기준, 10% 이동 가정
retention_focus = retention_simulation[
    (retention_simulation['적용 금액 기준'] == 'first_paid_amount 평균') &
    (retention_simulation['다음 단계 이동 비중(%)'] == 10)
].copy()

if len(retention_focus) > 0:
    retention_best = retention_focus.sort_values('기대매출', ascending=False).iloc[0]
else:
    retention_best = None

# 관측기간별 차이가 큰 구간 확인
window_diff_summary = (
    retention_simulation[
        (retention_simulation['적용 금액 기준'] == 'first_paid_amount 평균') &
        (retention_simulation['다음 단계 이동 비중(%)'] == 10)
    ]
    .groupby('관측기간', as_index=False)
    .agg(기대매출_만원=('기대매출(만원)', 'sum'))
    .sort_values('기대매출_만원', ascending=False)
)

if len(window_diff_summary) > 0:
    best_window = window_diff_summary.iloc[0]['관측기간']
else:
    best_window = '확인 필요'

first_best_text = ''
if first_best is not None:
    first_best_text = (
        f"- 첫 결제 파트에서는 평균 첫 결제금액 기준·10% 이동 가정 시, "
        f"**{first_best['액션 시나리오']}** 시나리오의 기대매출 잠재력이 가장 큽니다. "
        f"해당 시나리오의 기대매출은 약 **{first_best['기대매출(만원)']:.1f}만 원**입니다.\n"
    )
else:
    first_best_text = '- 첫 결제 파트의 최고 기대매출 시나리오는 계산 결과를 확인해야 합니다.\n'

retention_best_text = ''
if retention_best is not None:
    retention_best_text = (
        f"- 정기결제 파트에서는 first_paid_amount 평균 기준·10% 이동 가정 시, "
        f"**{retention_best['관측기간']} / {retention_best['액션 시나리오']}** 시나리오의 기대매출 잠재력이 가장 큽니다. "
        f"해당 시나리오의 기대매출은 약 **{retention_best['기대매출(만원)']:.1f}만 원**입니다.\n"
    )
else:
    retention_best_text = '- 정기결제 파트의 최고 기대매출 시나리오는 계산 결과를 확인해야 합니다.\n'

interpretation_markdown = f'''
### 기대매출 시뮬레이션 해석

{first_best_text}
{retention_best_text}
- 정기결제 관측기간은 30일·60일·90일을 모두 분리해 보았습니다. 이 중 기대매출 잠재력이 가장 크게 나타난 관측기간은 **{best_window}** 기준입니다. 다만 관측기간이 길어질수록 관측 가능한 유저 수가 줄어들 수 있으므로, 단순 기대매출뿐 아니라 표본 수와 갱신율 안정성을 함께 확인해야 합니다.
- `first_paid_amount` 기준은 실제 첫 결제 실결제금액을 반영하므로 할인·쿠폰 적용 이후의 보수적 매출 추정에 가깝습니다.
- `first_plan_price` 기준은 플랜 정가에 가까운 대리값이므로, 할인 전 가격 기준의 매출 잠재력을 보는 참고 지표입니다. 실제 기대매출로는 `first_paid_amount` 기준을 우선 보는 것이 더 안전합니다.
- 평균 기준 기대매출은 일부 고액 결제 플랜의 영향을 반영하므로 전체 매출 규모 추정에 적합합니다.
- 중앙값 기준 기대매출은 일반적인 유저 1명의 결제금액에 가까워 보수적인 추정치로 해석하는 것이 적절합니다.
- A/B 테스트 우선순위는 기대매출이 큰 시나리오만 볼 것이 아니라, 실제 액션 가능성도 함께 고려해야 합니다. 첫 결제 파트에서는 `미경험형 → 탐색형`과 `약한 학습형 → 강한 학습형` 중 기대매출과 실행 난이도를 함께 비교하고, 정기결제 파트에서는 결제 후 30일 내 학습활동을 늘리는 액션을 우선 실험 후보로 볼 수 있습니다.
- 본 분석은 비용 제외 기대매출이며, 실제 순효과는 푸시·리워드·개발·운영 비용과 A/B 테스트 결과를 함께 봐야 합니다.
- 본 분석은 실제 인과효과를 확정하는 것이 아니라, 세그먼트 간 성과 차이를 기반으로 한 기대매출 잠재력 추정이며, 실제 효과 검증은 A/B 테스트가 필요합니다.
'''

display(Markdown(interpretation_markdown))


### A/B 테스트 계산기 입력값 산출

아래 표는 세그먼트 이동 가정을 A/B 테스트 계산기에 입력할 수 있는 형태로 변환한 값입니다.  
전환율 개선 목표는 실제 운영 액션의 효과를 단정한 값이 아니라, 실험 설계에 필요한 최소 감지 효과와 필요 표본 수를 가늠하기 위한 기준값입니다.


In [ ]:
# ============================================================
# A/B 테스트 계산기 입력값 산출
# - 첫 결제율 기준
# - 콘텐츠 경험률(탐색 전환율) 기준
# ============================================================

import pandas as pd
import numpy as np
import glob
import math

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# ------------------------------------------------------------
# 1. 데이터 로드
# ------------------------------------------------------------
# 이미 노트북에 post 또는 dm_user_activation_payment가 있으면 그 변수를 사용하고,
# 없으면 현재 폴더 또는 data/private에서 dm_user_activation_payment csv를 자동 탐색합니다.

if 'post' in globals():
    activation = post.copy()
elif 'dm_user_activation_payment' in globals():
    activation = dm_user_activation_payment.copy()
else:
    candidate_paths = (
        glob.glob('**/dm_user_activation_payment*.csv', recursive=True)
        + glob.glob('data/private/dm_user_activation_payment*.csv')
    )
    if len(candidate_paths) == 0:
        raise FileNotFoundError("dm_user_activation_payment CSV 파일을 찾을 수 없습니다.")
    
    activation_path = sorted(candidate_paths)[-1]
    print("사용 파일:", activation_path)
    activation = pd.read_csv(activation_path, low_memory=False)

# ------------------------------------------------------------
# 2. 분석 대상 필터링
# ------------------------------------------------------------
# 기존 분석 기준과 동일하게 무료체험 폐지 후 가입자 기준 사용

if 'period_group' in activation.columns:
    post_df = activation[activation['period_group'] == 'post_2023_05_11'].copy()
else:
    post_df = activation.copy()

# 날짜 컬럼 처리
if 'signup_time' in post_df.columns:
    post_df['signup_time'] = pd.to_datetime(post_df['signup_time'], errors='coerce')
    post_df['signup_date'] = post_df['signup_time'].dt.date

# 결측치 방어
num_cols = [
    'paid_30d',
    'lesson_complete_prepay_7d',
    'unique_content_prepay_7d',
    'content_view_7d',
    'content_start_7d',
    'unique_content_view_7d',
    'unique_content_start_7d'
]

for col in num_cols:
    if col in post_df.columns:
        post_df[col] = post_df[col].fillna(0)

# ------------------------------------------------------------
# 3. 기존 세그먼트 기준 재현
# ------------------------------------------------------------
# 콘텐츠 경험 기준: 기존 4개 세그먼트 기준과 동일하게 사용
# unique_content_view_7d 단일 기준만 쓰지 않음

post_df['content_experience_flag'] = (
    (post_df.get('unique_content_prepay_7d', 0) > 0)
    | (post_df.get('content_view_7d', 0) > 0)
    | (post_df.get('content_start_7d', 0) > 0)
)

post_df['view_or_start_flag'] = (
    (post_df.get('content_view_7d', 0) > 0)
    | (post_df.get('content_start_7d', 0) > 0)
)

lesson_7d = post_df['lesson_complete_prepay_7d']

conditions = [
    (lesson_7d == 0) & (~post_df['content_experience_flag']),
    (lesson_7d == 0) & (post_df['content_experience_flag']),
    (lesson_7d.between(1, 6)),
    (lesson_7d >= 7)
]

choices = ['미경험형', '탐색형', '약한 학습형', '강한 학습형']

post_df['user_segment'] = np.select(conditions, choices, default='기타')

segment_order = ['미경험형', '탐색형', '약한 학습형', '강한 학습형']

segment_summary = (
    post_df[post_df['user_segment'].isin(segment_order)]
    .groupby('user_segment')
    .agg(
        유저수=('user_id', 'nunique'),
        결제자수=('paid_30d', 'sum'),
        첫결제율=('paid_30d', 'mean')
    )
    .reindex(segment_order)
)

segment_summary['첫결제율(%)'] = segment_summary['첫결제율'] * 100

display(segment_summary[['유저수', '결제자수', '첫결제율(%)']])

# ------------------------------------------------------------
# 4. 첫 결제율 기준 AB Tasty 입력값 계산
# ------------------------------------------------------------

mis_rate = segment_summary.loc['미경험형', '첫결제율']
explore_rate = segment_summary.loc['탐색형', '첫결제율']
rate_gap = explore_rate - mis_rate

move_rates = [0.03, 0.05, 0.10]

first_payment_ab_inputs = []

for move_rate in move_rates:
    expected_rate = mis_rate + (move_rate * rate_gap)
    absolute_uplift_pp = (expected_rate - mis_rate) * 100
    relative_uplift_pct = ((expected_rate / mis_rate) - 1) * 100
    
    first_payment_ab_inputs.append({
        '지표': '가입 후 30일 내 첫 결제율',
        '목표 해석': f'미경험형 {int(move_rate*100)}% → 탐색형 이동 가정',
        'AB Tasty Conversion Rate 입력값(%)': mis_rate * 100,
        '기대 전환율(%)': expected_rate * 100,
        '절대 상승폭(%p)': absolute_uplift_pp,
        'AB Tasty Expected uplift 입력값(%)': relative_uplift_pct
    })

first_payment_ab_inputs = pd.DataFrame(first_payment_ab_inputs)

display(first_payment_ab_inputs)

# ------------------------------------------------------------
# 5. 콘텐츠 경험률 / 조회·시작률 기준 AB Tasty 입력값 계산
# ------------------------------------------------------------
# 현재 데이터마트만으로는 "가입 후 24시간 내 콘텐츠 경험 없는 유저가 이후 콘텐츠를 조회/시작했는지"를
# 완벽히 재현하기 어렵습니다.
# 따라서 현재 분석에서는 proxy로 '레슨 완료 0개 유저 중 콘텐츠 경험이 있는 비율'을 사용합니다.

no_lesson_pool = post_df[post_df['lesson_complete_prepay_7d'] == 0].copy()

content_experience_rate = no_lesson_pool['content_experience_flag'].mean()
view_or_start_rate = no_lesson_pool['view_or_start_flag'].mean()

content_mde_pp_list = [0.03, 0.05, 0.10]

content_ab_inputs = []

for mde_pp in content_mde_pp_list:
    expected_rate = content_experience_rate + mde_pp
    relative_uplift_pct = (mde_pp / content_experience_rate) * 100
    
    content_ab_inputs.append({
        '지표': '콘텐츠 경험률 / 탐색 전환율',
        '목표 해석': f'탐색 전환율 +{int(mde_pp*100)}%p',
        'AB Tasty Conversion Rate 입력값(%)': content_experience_rate * 100,
        '기대 전환율(%)': expected_rate * 100,
        '절대 상승폭(%p)': mde_pp * 100,
        'AB Tasty Expected uplift 입력값(%)': relative_uplift_pct
    })

content_ab_inputs = pd.DataFrame(content_ab_inputs)

display(content_ab_inputs)

# ------------------------------------------------------------
# 6. 필요 샘플 수 계산
# ------------------------------------------------------------
# AB Tasty 계산기와 완전히 동일한 내부 로직은 아닐 수 있지만,
# 일반적인 95% 유의수준, 80% 검정력 기준의 두 집단 모비율 비교 샘플 수입니다.

power_analysis = NormalIndPower()

def calc_sample_size_per_group(p1, p2, alpha=0.05, power=0.8):
    if p1 <= 0 or p2 <= 0 or p1 == p2:
        return np.nan
    
    effect_size = abs(proportion_effectsize(p1, p2))
    n = power_analysis.solve_power(
        effect_size=effect_size,
        power=power,
        alpha=alpha,
        ratio=1,
        alternative='two-sided'
    )
    return math.ceil(n)

sample_rows = []

# 첫 결제율 기준
for _, row in first_payment_ab_inputs.iterrows():
    p1 = row['AB Tasty Conversion Rate 입력값(%)'] / 100
    p2 = row['기대 전환율(%)'] / 100
    n_per_group = calc_sample_size_per_group(p1, p2)
    
    sample_rows.append({
        '지표': row['지표'],
        '목표 해석': row['목표 해석'],
        '기준 전환율(%)': row['AB Tasty Conversion Rate 입력값(%)'],
        '기대 전환율(%)': row['기대 전환율(%)'],
        'Expected uplift 입력값(%)': row['AB Tasty Expected uplift 입력값(%)'],
        '필요 샘플 수 / 그룹': n_per_group,
        '필요 전체 샘플 수': n_per_group * 2
    })

# 콘텐츠 경험률 기준
for _, row in content_ab_inputs.iterrows():
    p1 = row['AB Tasty Conversion Rate 입력값(%)'] / 100
    p2 = row['기대 전환율(%)'] / 100
    n_per_group = calc_sample_size_per_group(p1, p2)
    
    sample_rows.append({
        '지표': row['지표'],
        '목표 해석': row['목표 해석'],
        '기준 전환율(%)': row['AB Tasty Conversion Rate 입력값(%)'],
        '기대 전환율(%)': row['기대 전환율(%)'],
        'Expected uplift 입력값(%)': row['AB Tasty Expected uplift 입력값(%)'],
        '필요 샘플 수 / 그룹': n_per_group,
        '필요 전체 샘플 수': n_per_group * 2
    })

sample_size_table = pd.DataFrame(sample_rows)

display(sample_size_table)

# ------------------------------------------------------------
# 7. 하루 실험 대상 유입 수와 테스트 기간 계산
# ------------------------------------------------------------

# 실험 대상 proxy 1: 레슨 완료 0개 유저
# 실험 대상 proxy 2: 최종 미경험형 유저
# 실제 실험에서는 "가입 후 24시간 내 콘텐츠 경험 없는 신규 유저" 기준으로 별도 산출하는 것이 가장 좋음

if 'signup_date' in post_df.columns:
    daily_no_lesson_users = (
        no_lesson_pool
        .groupby('signup_date')['user_id']
        .nunique()
    )
    
    daily_mis_users = (
        post_df[post_df['user_segment'] == '미경험형']
        .groupby('signup_date')['user_id']
        .nunique()
    )
    
    avg_daily_no_lesson = daily_no_lesson_users.mean()
    avg_daily_mis = daily_mis_users.mean()
else:
    avg_daily_no_lesson = np.nan
    avg_daily_mis = np.nan

duration_rows = []

for _, row in sample_size_table.iterrows():
    total_required = row['필요 전체 샘플 수']
    
    duration_rows.append({
        '지표': row['지표'],
        '목표 해석': row['목표 해석'],
        '필요 전체 샘플 수': total_required,
        '일평균 대상 유저 수 기준': '레슨 완료 0개 유저 proxy',
        '일평균 대상 유저 수': avg_daily_no_lesson,
        '예상 테스트 기간(일)': math.ceil(total_required / avg_daily_no_lesson) if avg_daily_no_lesson > 0 else np.nan
    })
    
    duration_rows.append({
        '지표': row['지표'],
        '목표 해석': row['목표 해석'],
        '필요 전체 샘플 수': total_required,
        '일평균 대상 유저 수 기준': '최종 미경험형 유저 proxy',
        '일평균 대상 유저 수': avg_daily_mis,
        '예상 테스트 기간(일)': math.ceil(total_required / avg_daily_mis) if avg_daily_mis > 0 else np.nan
    })

duration_table = pd.DataFrame(duration_rows)

display(duration_table)

# ------------------------------------------------------------
# 8. AB Tasty에 바로 입력할 추천값 요약
# ------------------------------------------------------------

abtasty_recommendation = pd.DataFrame([
    {
        '계산기': 'Sample Size Calculator',
        '목적': '콘텐츠 경험률 +3%p 감지',
        'Conversion Rate 입력값(%)': content_experience_rate * 100,
        'Expected uplift 입력값(%)': (0.03 / content_experience_rate) * 100,
        'Average Daily Visitors 입력값': avg_daily_no_lesson,
        'Number of Variations 입력값': 2
    },
    {
        '계산기': 'Sample Size Calculator',
        '목적': '콘텐츠 경험률 +5%p 감지',
        'Conversion Rate 입력값(%)': content_experience_rate * 100,
        'Expected uplift 입력값(%)': (0.05 / content_experience_rate) * 100,
        'Average Daily Visitors 입력값': avg_daily_no_lesson,
        'Number of Variations 입력값': 2
    },
    {
        '계산기': 'Sample Size Calculator',
        '목적': '첫 결제율: 미경험형 3% 이동 효과 감지',
        'Conversion Rate 입력값(%)': mis_rate * 100,
        'Expected uplift 입력값(%)': first_payment_ab_inputs.loc[0, 'AB Tasty Expected uplift 입력값(%)'],
        'Average Daily Visitors 입력값': avg_daily_mis,
        'Number of Variations 입력값': 2
    },
    {
        '계산기': 'Sample Size Calculator',
        '목적': '첫 결제율: 미경험형 5% 이동 효과 감지',
        'Conversion Rate 입력값(%)': mis_rate * 100,
        'Expected uplift 입력값(%)': first_payment_ab_inputs.loc[1, 'AB Tasty Expected uplift 입력값(%)'],
        'Average Daily Visitors 입력값': avg_daily_mis,
        'Number of Variations 입력값': 2
    }
])

display(abtasty_recommendation.round(2))


### 후속 분석 과제

- 첫 결제 전후 학습 행동이 실제 인과적으로 리텐션을 개선하는지 A/B 테스트로 검증
- 결제 후 학습 루틴 형성을 방해하는 요인에 대한 콘텐츠 만족도, 가격 부담, 알림 반응 데이터 추가 수집
- 취소 버튼 클릭 후 실제 해지, 재구독, 고객센터 문의 등 후속 행동 연결
- 콘텐츠 ID별 이름, 난이도, 길이, 커리큘럼 위치를 결합한 온보딩 콘텐츠 추천 후보 재검증
- 할인율, 플랜 유형, 프로모션 시점이 리텐션에 미치는 영향은 별도 통제 분석 필요

> **최종 결론:** 무료체험 폐지 이후 최우선 과제는 가입 후 7일 내 레슨 7개 완료를 통해 첫 결제 전환 가능성을 높이고, 첫 결제 직후 30일 내 6일 이상 학습 루틴을 형성시켜 정기구독 유지로 연결하는 것.


> # **Appendix. 보조 분석 및 상세 검증**

Main Analysis의 결론을 보조하거나 세부 기준을 검증하는 분석입니다.  
핵심 내용 위주의 분석 흐름을 유지하기 위해 필요한 내용만 Main에 사용하고, 상세표·보조 그래프·기준선 검증은 Appendix에서 참고합니다.


## Appendix 1. 레슨 0개 유저 및 레슨 완료 구간 상세 분석


In [ ]:
# 레슨 0개 유저 행동 유형 세분화 및 결제 기여도 산출
zero = post[post['lesson_complete_prepay_7d'] == 0].copy()

print('레슨 0개 세분화 기준')
print('- 무행동형: 콘텐츠 조회와 콘텐츠 시작이 모두 없는 유저')
print('- 조회 후 미시작형: 콘텐츠 조회는 있으나 콘텐츠 시작이 없는 유저')
print('- 시작 후 미완료형: 콘텐츠 시작은 있으나 레슨 완료가 없는 유저')
print('- 검토형: 후기 또는 질문 클릭은 있으나 레슨 완료가 없는 유저')
print('- 기타 탐색형: 위 조건 외 콘텐츠 경험이 있는 유저를 판별하기 위한 그룹')
print('- 레슨 미완료 결제자: 행동 유형이 아니라 레슨 완료 0개이면서 30일 내 첫 결제가 발생한 결과 flag')
print('')
print('분석 원칙')
print('- 행동 유형은 무행동형, 조회 후 미시작형, 시작 후 미완료형, 검토형으로 분류')
print('- 결제 여부는 행동 유형과 섞지 않고, 레슨 미완료 결제 flag 및 결제율·결제 기여도로 별도 확인')

def classify_zero_behavior(row):
    if row['content_start_7d'] > 0:
        return '시작 후 미완료형'
    elif row['content_view_7d'] > 0:
        return '조회 후 미시작형'
    elif row['review_click_7d'] > 0 or row['question_click_7d'] > 0:
        return '검토형'
    else:
        return '무행동형'

zero['레슨 0개 행동 유형'] = zero.apply(classify_zero_behavior, axis=1)
zero['레슨 미완료 결제 flag'] = np.where(zero['paid_30d'] == 1, '레슨 미완료 결제자', '미결제')

zero_order_all = ['무행동형', '조회 후 미시작형', '시작 후 미완료형', '검토형', '기타 탐색형']
zero_order_display = ['무행동형', '조회 후 미시작형', '시작 후 미완료형', '검토형']

zero_summary_all = zero.groupby('레슨 0개 행동 유형').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reindex(zero_order_all).fillna(0).reset_index()

zero_summary_all['레슨 0개 내 유저 비중(%)'] = zero_summary_all['유저_수'] / len(zero) * 100
zero_summary_all['전체 결제 기여도(%)'] = zero_summary_all['결제자_수'] / post['paid_30d'].sum() * 100
zero_summary_all['레슨 0개 결제 기여도(%)'] = zero_summary_all['결제자_수'] / zero['paid_30d'].sum() * 100 if zero['paid_30d'].sum() > 0 else 0
zero_summary_all['첫 결제율(%)'] = zero_summary_all['첫_결제율'] * 100

etc_count = int(zero_summary_all.loc[zero_summary_all['레슨 0개 행동 유형'] == '기타 탐색형', '유저_수'].iloc[0])
print('')
print(f'기타 탐색형 산출 결과: {etc_count:,}건')
print('- 기타 탐색형은 산출 결과 0건으로 본문 표와 그래프에서는 제외')

zero_summary = zero_summary_all[zero_summary_all['레슨 0개 행동 유형'].isin(zero_order_display)].copy()

zero_action = pd.DataFrame({
    '구분': ['무행동형', '조회 후 미시작형', '시작 후 미완료형', '검토형', '레슨 미완료 결제자'],
    '비즈니스 해석': [
        '가입 후 주요 콘텐츠 행동이 없는 초기 이탈 가능 집단',
        '관심은 있으나 콘텐츠 시작까지 연결되지 않은 집단',
        '콘텐츠 시작은 했지만 첫 레슨 완료까지 도달하지 못한 집단',
        '후기 또는 질문을 확인했으나 학습 완료 전 단계에 머문 집단',
        '레슨 완료 없이 결제가 발생한 유저로, 결제 후 학습 정착이 필요한 집단'
    ],
    '우선 액션': [
        '첫 콘텐츠 추천, 관심사 기반 온보딩 메시지',
        '상세페이지 설득 요소, 난이도·소요시간 안내 개선',
        '첫 레슨 완료 유도, 짧은 레슨 추천, 이어보기 강조',
        '신뢰 요소 강화, 질문·후기 영역에서 학습 시작 CTA 연결',
        '결제 후 첫 학습 안내, 학습 로드맵 제안, 초기 사용 정착 유도'
    ]
})

zero_display = zero_summary[['레슨 0개 행동 유형', '유저_수', '레슨 0개 내 유저 비중(%)', '결제자_수', '첫 결제율(%)', '전체 결제 기여도(%)', '레슨 0개 결제 기여도(%)']].copy()
zero_display.columns = ['레슨 0개 행동 유형', '유저 수', '레슨 0개 내 유저 비중(%)', '결제자 수', '첫 결제율(%)', '전체 결제 기여도(%)', '레슨 0개 결제 기여도(%)']
for col in ['레슨 0개 내 유저 비중(%)', '첫 결제율(%)', '전체 결제 기여도(%)', '레슨 0개 결제 기여도(%)']:
    zero_display[col] = zero_display[col].round(2)

display(zero_display)
display(zero_action)

fig = px.bar(
    zero_summary.sort_values('유저_수'),
    y='레슨 0개 행동 유형', x='유저_수', orientation='h',
    text='유저_수',
    hover_data={'결제자_수': ':,', '첫 결제율(%)': ':.2f', '전체 결제 기여도(%)': ':.2f', '레슨 0개 결제 기여도(%)': ':.2f'},
    labels={'레슨 0개 행동 유형': '레슨 0개 행동 유형', '유저_수': '유저 수'},
    title='레슨 0개 행동 유형별 유저 수'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=520)
fig.show()


### 레슨 0개 유저 세분화 해석

- 현상
    - 레슨 0개 유저는 33,556명으로 폐지 후 전체 가입 유저의 72.86%를 차지
    - 시작 후 미완료형은 유저 수 6,929명, 첫 결제율 18.13%, 전체 결제 기여도 45.05%로 레슨 0개 집단 내에서 가장 중요한 결제 기여 집단
    - 무행동형은 유저 수 20,739명으로 가장 크지만 첫 결제율은 1.09%로 낮음
    - 기타 탐색형은 산출 결과 0건으로 본문 해석에서는 제외

- 인사이트
    - 레슨 0개 유저를 단순 비활성 집단으로만 보면 안 되며, 콘텐츠 시작 여부와 결제 여부를 분리해서 해석 필요
    - 특히 시작 후 미완료형은 레슨 완료는 없지만 결제자가 많이 발생하는 집단이므로, 결제 전환과 결제 후 학습 정착을 동시에 관리해야 함
    - 레슨 미완료 결제자는 행동 유형이 아니라 결과 flag이므로, 무행동형·조회 후 미시작형·시작 후 미완료형·검토형 안에서 결제 여부로 별도 확인하는 구조가 적절

- 액션
    - 무행동형에는 첫 콘텐츠 추천과 관심사 기반 온보딩 메시지를 제공
    - 조회 후 미시작형에는 상세페이지 설득 요소, 난이도, 소요시간, 첫 레슨 안내를 개선
    - 시작 후 미완료형에는 짧은 레슨 추천, 이어보기, 첫 레슨 완료 보상을 통해 첫 완료 경험을 유도
    - 레슨 미완료 결제자에게는 결제 직후 첫 학습 안내와 학습 로드맵을 제공해 결제 후 이탈을 방지


In [ ]:
print('버블차트 해석 기준')
print('- X축: 폐지 후 전체 가입 유저 중 해당 레슨 완료 구간이 차지하는 비중')
print('- Y축: 해당 레슨 완료 구간의 30일 내 첫 결제율')
print('- 버블 크기: 전체 결제자 중 해당 구간이 차지하는 결제 기여도')
print(f'- 세로 기준선: 레슨 완료 구간별 유저 비중 중앙값 {lesson_band_user_share_median:.2f}%')
print(f'- 가로 기준선: 폐지 후 전체 평균 첫 결제율 {post_overall_paid_rate:.2f}%')
print('- 유저 비중 중앙값은 각 구간의 유저 비중을 정렬했을 때 가운데 위치한 값이며, 구간 규모를 상대적으로 구분하기 위한 보조선으로 참고')

quadrant_rule = pd.DataFrame({
    '사분면': ['규모 큼 + 결제율 높음', '규모 큼 + 결제율 낮음', '규모 작음 + 결제율 높음', '규모 작음 + 결제율 낮음'],
    '해석': ['이미 성과가 큰 핵심 구간', '개선 임팩트가 큰 구간', '효율은 높지만 규모가 작은 구간', '후순위 또는 모니터링 구간'],
    '전략': ['유지·확장', '완료 유도와 전환 설계 개선', '상위 구간으로 이동 가능한 유저 확대', '낮은 우선순위 관리']
})
display(quadrant_rule)

fig = px.scatter(
    lesson_band,
    x='유저 비중(%)', y='첫 결제율(%)', size='결제 기여도(%)', color='레슨 완료 구간',
    hover_name='레슨 완료 구간',
    hover_data={'유저_수': ':,', '결제자_수': ':,', '유저 비중(%)': ':.2f', '첫 결제율(%)': ':.2f', '결제 기여도(%)': ':.2f'},
    labels={'유저 비중(%)': '전체 유저 비중(%)', '첫 결제율(%)': '첫 결제율(%)'},
    title='레슨 완료 구간별 규모, 첫 결제율, 결제 기여도'
)
fig.add_vline(x=lesson_band_user_share_median, line_dash='dash', line_color='gray', annotation_text=f'유저 비중 중앙값 {lesson_band_user_share_median:.2f}%')
fig.add_hline(y=post_overall_paid_rate, line_dash='dash', line_color='gray', annotation_text=f'전체 평균 결제율 {post_overall_paid_rate:.2f}%')
fig.update_layout(height=620, legend_title_text='레슨 완료 구간')
fig.show()


### Appendix 1 해석

- 레슨 0개 유저는 단일 집단이 아니라 무경험, 조회 후 미시작, 시작 후 미완료 등 서로 다른 병목을 가진 집단으로 분리 필요.
- 레슨 완료 구간별 사분면은 규모, 첫 결제율, 결제 기여도를 함께 보는 보조 도구로 활용 가능.


## Appendix 2. 콘텐츠 후보 및 콘텐츠 성과 보조 분석


In [ ]:
# 콘텐츠별 기간 전체 성과 집계
content_period = content.groupby(['period_group', 'content_id']).agg(
    활성_월수=('cohort_month', 'nunique'),
    조회_유저_수=('content_view_users_7d', 'sum'),
    시작_유저_수=('content_start_users_7d', 'sum'),
    레슨_완료_유저_수=('lesson_complete_users_7d', 'sum'),
    콘텐츠_완료_유저_수=('content_end_users_7d', 'sum'),
    결제_유저_수=('paid_30d_users', 'sum')
).reset_index()

content_period['시작률(%)'] = content_period['시작_유저_수'] / content_period['조회_유저_수'].replace(0, np.nan) * 100
content_period['완료율(%)'] = content_period['콘텐츠_완료_유저_수'] / content_period['시작_유저_수'].replace(0, np.nan) * 100
content_period['경험 유저 proxy'] = content_period[['조회_유저_수', '시작_유저_수', '레슨_완료_유저_수', '콘텐츠_완료_유저_수']].max(axis=1)
content_period['결제 연결성 proxy(%)'] = content_period['결제_유저_수'] / content_period['경험 유저 proxy'].replace(0, np.nan) * 100

post_content = content_period[content_period['period_group'] == 'post_2023_05_11'].copy()

view_median = post_content['조회_유저_수'].median()
start_rate_median = post_content['시작률(%)'].median()
completion_rate_median = post_content['완료율(%)'].median()
paid_proxy_median = post_content['결제 연결성 proxy(%)'].median()

print('proxy 지표 정의 및 사용 이유')
print('- 현재 콘텐츠 성과 마트는 콘텐츠별 조회 유저 수, 시작 유저 수, 완료 유저 수, 결제 유저 수가 이미 집계된 형태')
print('- 따라서 user_id-content_id 단위의 정확한 경험 유저 합집합을 해당 데이터 마트만으로는 직접 계산하기 어려움')
print('- 경험 유저 proxy = 조회 유저 수, 시작 유저 수, 레슨 완료 유저 수, 콘텐츠 완료 유저 수 중 최댓값')
print('- 결제 연결성 proxy = 결제 유저 수 / 경험 유저 proxy')
print('- 결제 연결성 proxy는 정확한 콘텐츠별 결제율이 아니라, 콘텐츠 간 결제 연결성을 비교하기 위한 보조 지표')
print('- 최종 추천 후보 확정 전에는 월별 통합 로그에서 user_id-content_id 단위 합집합과 결제 여부를 재검증 필요')
print('')
print('콘텐츠 후보 분류 기준')
print(f'- 조회 유저 수 중앙값: {view_median:.0f}명')
print(f'- 시작률 중앙값: {start_rate_median:.2f}%')
print(f'- 완료율 중앙값: {completion_rate_median:.2f}%')
print(f'- 결제 연결성 proxy 중앙값: {paid_proxy_median:.2f}%')

candidate_rule = pd.DataFrame({
    '후보 유형': ['온보딩 후보', '개선 후보', '숨은 후보', '일반 콘텐츠'],
    '분류 기준': [
        '시작률, 완료율, 결제 연결성 proxy가 모두 중앙값 이상',
        '조회 유저 수는 중앙값 이상, 시작률은 중앙값 미만',
        '조회 유저 수는 중앙값 미만, 시작률·완료율·결제 연결성 proxy는 중앙값 이상',
        '온보딩 후보, 개선 후보, 숨은 후보 조건 중 어느 것도 충족하지 않는 콘텐츠'
    ],
    '해석': [
        '신규 유저 첫 추천 후보',
        '관심은 받지만 시작 전환이 약한 상세페이지·CTA 개선 대상',
        '노출은 적지만 성과 효율이 좋은 노출 확대 실험 후보',
        '현재 기준에서는 우선순위가 낮거나 추가 메타데이터 확인이 필요한 콘텐츠'
    ],
    '우선 액션': [
        '첫 추천 영역 노출',
        '상세페이지, 난이도 안내, 시작 CTA 개선',
        '세그먼트 대상 추천 실험 및 노출 확대',
        '콘텐츠명, 난이도, 길이, 커리큘럼 위치 확인 후 후순위 검토'
    ]
})

display(candidate_rule)
print('='*100)
print('일반 콘텐츠 보충 설명')
print('- 일반 콘텐츠는 조회 규모가 충분하면서 시작률이 낮은 개선 후보도 아니고, 저노출 고효율의 숨은 후보도 아니며, 시작률·완료율·결제 연결성이 모두 우수한 온보딩 후보도 아닌 콘텐츠를 의미')


In [ ]:
# 중앙값 기준 콘텐츠 후보 분류
post_content['후보 유형'] = '일반 콘텐츠'

post_content.loc[
    (post_content['시작률(%)'] >= start_rate_median) &
    (post_content['완료율(%)'] >= completion_rate_median) &
    (post_content['결제 연결성 proxy(%)'] >= paid_proxy_median),
    '후보 유형'
] = '온보딩 후보'

post_content.loc[
    (post_content['조회_유저_수'] >= view_median) &
    (post_content['시작률(%)'] < start_rate_median),
    '후보 유형'
] = '개선 후보'

post_content.loc[
    (post_content['조회_유저_수'] < view_median) &
    (post_content['시작률(%)'] >= start_rate_median) &
    (post_content['완료율(%)'] >= completion_rate_median) &
    (post_content['결제 연결성 proxy(%)'] >= paid_proxy_median),
    '후보 유형'
] = '숨은 후보'

content_type_summary = post_content.groupby('후보 유형').agg(
    콘텐츠_수=('content_id', 'nunique'),
    조회_유저_수=('조회_유저_수', 'sum'),
    결제_유저_수=('결제_유저_수', 'sum')
).reset_index()

content_type_summary['조회 유저 비중(%)'] = content_type_summary['조회_유저_수'] / content_type_summary['조회_유저_수'].sum() * 100
content_type_summary['결제 유저 비중(%)'] = content_type_summary['결제_유저_수'] / content_type_summary['결제_유저_수'].sum() * 100

content_type_display = content_type_summary.copy()
content_type_display.columns = ['후보 유형', '콘텐츠 수', '조회 유저 수', '결제 유저 수', '조회 유저 비중(%)', '결제 유저 비중(%)']
for col in ['조회 유저 비중(%)', '결제 유저 비중(%)']:
    content_type_display[col] = content_type_display[col].round(2)

display(content_type_display)

fig = px.bar(
    content_type_summary,
    x='후보 유형', y='콘텐츠_수',
    text='콘텐츠_수',
    hover_data={'조회_유저_수': ':,', '결제_유저_수': ':,', '결제 유저 비중(%)': ':.2f'},
    labels={'후보 유형': '후보 유형', '콘텐츠_수': '콘텐츠 수'},
    title='폐지 후 콘텐츠 후보 유형별 콘텐츠 수'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=480)
fig.show()


### 콘텐츠 후보 유형 해석

- 현상
    - 콘텐츠 후보는 단순 조회 수가 아니라 시작률, 완료율, 결제 연결성 proxy를 함께 고려해 분류
    - 온보딩 후보는 시작률, 완료율, 결제 연결성 proxy가 모두 중앙값 이상인 균형형 콘텐츠
    - 개선 후보는 조회 규모는 있으나 시작률이 낮아 상세페이지, 난이도 안내, 시작 CTA 점검이 필요한 콘텐츠
    - 숨은 후보는 조회 규모는 작지만 시작률, 완료율, 결제 연결성 proxy가 모두 높은 노출 확대 실험 후보

- 인사이트
    - 조회 수가 높다고 항상 좋은 콘텐츠는 아니며, 조회 이후 시작·완료·결제 연결성을 함께 확인 필요
    - 결제 연결성 proxy는 정확한 결제율이 아니라 현재 집계 마트에서 콘텐츠 간 상대 비교를 위한 보조 지표
    - 일반 콘텐츠는 현재 기준에서 우선순위가 낮지만, 콘텐츠명, 난이도, 길이, 커리큘럼 위치 같은 메타데이터 연결 후 재평가 가능

- 액션
    - 온보딩 후보는 신규 유저 첫 추천 영역에 우선 노출
    - 개선 후보는 상세페이지, 시작 버튼, 난이도·소요시간 안내, 커리큘럼 설명 개선 대상으로 관리
    - 숨은 후보는 특정 세그먼트 대상 추천 실험 또는 노출 확대 실험 대상으로 활용


In [ ]:
# 콘텐츠 후보 점수 계산 및 유형별 TOP 콘텐츠 확인
candidate_top = post_content[post_content['후보 유형'] != '일반 콘텐츠'].copy()

print('='*100)
print('후보 점수 계산 로직')
print('- 후보 점수(점) = 시작률, 완료율, 결제 연결성 proxy의 단순 평균')
print('- 구성 지표는 % 단위이지만, 후보 점수는 실제 결제율이 아니라 후보 정렬용 점수로 %가 아니라 점으로 표기')
print('- 콘텐츠명, 난이도, 길이, 커리큘럼 위치 연결 후 최종 추천 후보 확정 필요')
print('='*100)

candidate_top['후보 점수(점)'] = (
    candidate_top['시작률(%)'].fillna(0) +
    candidate_top['완료율(%)'].fillna(0) +
    candidate_top['결제 연결성 proxy(%)'].fillna(0)
) / 3

candidate_action_map = {
    '온보딩 후보': '첫 추천 영역 노출',
    '개선 후보': '상세페이지·CTA 개선',
    '숨은 후보': '세그먼트 대상 노출 확대 실험'
}
candidate_top['추천 액션'] = candidate_top['후보 유형'].map(candidate_action_map)

candidate_top = candidate_top.sort_values(['후보 유형', '후보 점수(점)'], ascending=[True, False])

candidate_display = candidate_top[[
    '후보 유형', 'content_id', '조회_유저_수', '시작률(%)', '완료율(%)', '결제_유저_수', '결제 연결성 proxy(%)', '후보 점수(점)', '추천 액션'
]].copy()

candidate_display.columns = [
    '후보 유형', '콘텐츠 ID', '조회 유저 수', '시작률(%)', '완료율(%)', '결제 유저 수', '결제 연결성 proxy(%)', '후보 점수(점)', '추천 액션'
]

for col in ['시작률(%)', '완료율(%)', '결제 연결성 proxy(%)', '후보 점수(점)']:
    candidate_display[col] = candidate_display[col].round(2)

# 유형별 상위 5개 콘텐츠 표시
for candidate_type in ['온보딩 후보', '개선 후보', '숨은 후보']:
    print(f'{candidate_type} TOP 5')
    display(candidate_display[candidate_display['후보 유형'] == candidate_type].head(5).reset_index(drop=True))


### 콘텐츠 후보 TOP 해석

- 현상
    - 후보 콘텐츠는 `content_id` 기준으로 도출되며, 후보 점수(점)는 시작률·완료율·결제 연결성 proxy의 단순 평균값
    - 온보딩 후보, 개선 후보, 숨은 후보는 서로 다른 문제와 액션을 가지므로 하나의 표에 섞기보다 유형별 TOP 5로 분리해 보는 것이 적절
    - 후보 점수는 실제 결제율이 아니라 후보 정렬용 참고 점수이므로, 최종 추천 확정 기준으로 단독 사용하면 안 됨

- 인사이트
    - 온보딩 후보는 시작·완료·결제 연결성이 모두 양호해 신규 유저 첫 추천 후보로 활용 가능
    - 개선 후보는 조회 규모는 있으나 시작률이 낮아 콘텐츠 상세페이지나 시작 동선의 병목 가능성이 있음
    - 숨은 후보는 노출 규모는 작지만 효율 지표가 좋아 세그먼트 대상 추천 실험 가치가 있음

- 액션
    - 온보딩 후보 TOP 5는 신규 유저 첫 추천 영역 또는 온보딩 커리큘럼 후보로 검토
    - 개선 후보 TOP 5는 제목, 난이도, 소요시간, CTA, 커리큘럼 설명을 우선 점검
    - 숨은 후보 TOP 5는 특정 관심사·난이도 세그먼트 대상 추천 실험으로 검증


## Appendix 3. 콘텐츠 탐색·질문 행동 세그먼트 상세 분석


In [ ]:
# 콘텐츠 탐색·학습 세그먼트별 첫 결제율 복원
# 고유 콘텐츠 수는 탐색 폭, 레슨 완료 수와 콘텐츠 완료 수는 학습 깊이를 의미

def make_content_learning_segment(row):
    lesson_count = row['lesson_complete_prepay_7d']
    content_end = row['content_end_prepay_7d']
    unique_content = row['unique_content_prepay_7d']
    content_experience = (unique_content > 0) or (row['content_view_7d'] > 0) or (row['content_start_7d'] > 0)

    if (not content_experience) and lesson_count == 0 and content_end == 0:
        return '무경험형'
    elif content_experience and lesson_count == 0 and content_end == 0:
        return '탐색만 한 유저'
    elif lesson_count > 0 and content_end == 0 and unique_content <= 1:
        return '단일 콘텐츠 부분학습형'
    elif lesson_count > 0 and content_end == 0 and unique_content >= 2:
        return '다중 콘텐츠 부분학습형'
    elif content_end >= 1 and unique_content <= 1:
        return '단일 콘텐츠 완주형'
    elif content_end >= 1 and unique_content >= 2:
        return '다중 콘텐츠 완주형'
    else:
        return '기타'

content_learning_order = [
    '무경험형', '탐색만 한 유저', '단일 콘텐츠 부분학습형', '다중 콘텐츠 부분학습형', '단일 콘텐츠 완주형', '다중 콘텐츠 완주형'
]

post['콘텐츠 탐색·학습 세그먼트'] = post.apply(make_content_learning_segment, axis=1)

content_learning_segment = post.groupby('콘텐츠 탐색·학습 세그먼트').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean'),
    평균_고유_콘텐츠_수=('unique_content_prepay_7d', 'mean'),
    평균_레슨_완료_수=('lesson_complete_prepay_7d', 'mean'),
    평균_콘텐츠_완료_수=('content_end_prepay_7d', 'mean')
).reindex(content_learning_order).fillna(0).reset_index()

content_learning_segment['유저 비중(%)'] = content_learning_segment['유저_수'] / len(post) * 100
content_learning_segment['결제 기여도(%)'] = content_learning_segment['결제자_수'] / post['paid_30d'].sum() * 100
content_learning_segment['첫 결제율(%)'] = content_learning_segment['첫_결제율'] * 100

print('콘텐츠 탐색·학습 세그먼트 기준')
print('- 무경험형: 콘텐츠 경험과 학습 완료가 모두 없는 유저')
print('- 탐색만 한 유저: 콘텐츠 조회·시작 등 경험은 있으나 레슨 완료와 콘텐츠 완료가 없는 유저')
print('- 단일/다중 콘텐츠 부분학습형: 레슨 완료는 있으나 콘텐츠 완주는 없는 유저')
print('- 단일/다중 콘텐츠 완주형: 콘텐츠 완료 경험이 있는 유저')
print('- 고유 콘텐츠 수는 탐색 폭을 의미하며, 독립적인 판단 지표가 아니라 레슨 완료와 콘텐츠 완료를 함께 해석 필요')

content_learning_display = content_learning_segment[[
    '콘텐츠 탐색·학습 세그먼트', '유저_수', '유저 비중(%)', '결제자_수', '첫 결제율(%)', '결제 기여도(%)',
    '평균_고유_콘텐츠_수', '평균_레슨_완료_수', '평균_콘텐츠_완료_수'
]].copy()
content_learning_display.columns = [
    '콘텐츠 탐색·학습 세그먼트', '유저 수', '유저 비중(%)', '결제자 수', '첫 결제율(%)', '결제 기여도(%)',
    '평균 고유 콘텐츠 수', '평균 레슨 완료 수', '평균 콘텐츠 완료 수'
]
for col in ['유저 비중(%)', '첫 결제율(%)', '결제 기여도(%)', '평균 고유 콘텐츠 수', '평균 레슨 완료 수', '평균 콘텐츠 완료 수']:
    content_learning_display[col] = content_learning_display[col].round(2)

display(content_learning_display)

fig = px.bar(
    content_learning_segment,
    x='콘텐츠 탐색·학습 세그먼트', y='첫 결제율(%)',
    text=content_learning_segment['첫 결제율(%)'].round(1),
    hover_data={'유저_수': ':,', '결제자_수': ':,', '유저 비중(%)': ':.2f', '결제 기여도(%)': ':.2f'},
    labels={'콘텐츠 탐색·학습 세그먼트': '콘텐츠 탐색·학습 세그먼트', '첫 결제율(%)': '첫 결제율(%)'},
    title='콘텐츠 탐색·학습 세그먼트별 첫 결제율'
)
fig.add_hline(y=post_overall_paid_rate, line_dash='dash', line_color='gray', annotation_text=f'전체 평균 {post_overall_paid_rate:.2f}%')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=540, xaxis_tickangle=-20)
fig.show()


### 콘텐츠 탐색·학습 세그먼트 해석

- 현상
    - 콘텐츠 탐색·학습 세그먼트는 고유 콘텐츠 수, 레슨 완료 수, 콘텐츠 완료 수를 함께 사용해 탐색 폭과 학습 깊이를 동시에 확인하는 보조 분석
    - 탐색만 한 유저는 콘텐츠 경험은 있으나 레슨 완료와 콘텐츠 완료가 없는 집단으로 결제 CTA보다 첫 레슨 완료 유도가 우선
    - 콘텐츠 완주형 세그먼트는 탐색 폭과 관계없이 첫 결제율이 높게 나타나며 콘텐츠 완료 경험 분석의 보조 근거로 활용 가능

- 인사이트
    - 고유 콘텐츠 수가 많다고 항상 좋은 것은 아니며, 레슨 완료나 콘텐츠 완료 없이 탐색만 많다면 선택 실패 또는 산만한 탐색일 수 있음
    - 단일 콘텐츠에 집중해도 완주 경험이 있으면 결제 가능성이 높아질 수 있음
    - 다중 콘텐츠 완주형은 탐색 폭과 학습 깊이가 함께 나타나는 고몰입 유저 후보

- 액션
    - 탐색만 한 유저에게는 결제 CTA보다 첫 레슨 완료를 유도하는 콘텐츠 추천과 시작 동선 개선을 우선 적용
    - 단일 콘텐츠 부분학습형에게는 같은 콘텐츠 내 이어보기와 첫 완주 유도를 제공
    - 완주형 유저에게는 다음 콘텐츠 추천, 커리큘럼 로드맵, 유료 플랜 CTA를 연결


In [ ]:
# 질문 클릭과 레슨 완료 강도 세그먼트 정의
question_definition = pd.DataFrame({
    '세그먼트': ['비학습형', '질문 탐색형', '약한 학습형', '약한 학습+질문형', '핵심 학습형', '고몰입 질문형'],
    '정의': [
        '레슨 완료 0개, 질문 클릭 X',
        '레슨 완료 0개, 질문 클릭 O',
        '레슨 1~6개 완료, 질문 클릭 X',
        '레슨 1~6개 완료, 질문 클릭 O',
        '레슨 7개 이상 완료, 질문 클릭 X',
        '레슨 7개 이상 완료, 질문 클릭 O'
    ],
    '해석': [
        '학습 참여 부재',
        '질문 확인은 있으나 학습 완료 부재',
        '임계점 전 학습 상태',
        '학습 중 도움 또는 심화 니즈 가능성',
        '자기주도 고몰입 학습',
        '고관여 및 질문 니즈 동시 발생'
    ]
})

display(question_definition)

# 질문 클릭과 레슨 완료 강도별 성과 산출
def make_question_segment(row):
    lesson_count = row['lesson_complete_prepay_7d']
    question_click = row['question_click_7d'] > 0

    if lesson_count == 0 and not question_click:
        return '비학습형'
    elif lesson_count == 0 and question_click:
        return '질문 탐색형'
    elif 1 <= lesson_count <= 6 and not question_click:
        return '약한 학습형'
    elif 1 <= lesson_count <= 6 and question_click:
        return '약한 학습+질문형'
    elif lesson_count >= 7 and not question_click:
        return '핵심 학습형'
    elif lesson_count >= 7 and question_click:
        return '고몰입 질문형'
    else:
        return '기타'

post['질문·레슨 세그먼트'] = post.apply(make_question_segment, axis=1)
question_order = ['비학습형', '질문 탐색형', '약한 학습형', '약한 학습+질문형', '핵심 학습형', '고몰입 질문형']

question_segment = post.groupby('질문·레슨 세그먼트').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reindex(question_order).reset_index()

question_segment['첫 결제율(%)'] = question_segment['첫_결제율'] * 100

display(question_segment[['질문·레슨 세그먼트', '유저_수', '결제자_수', '첫 결제율(%)']].rename(columns={
    '유저_수': '유저 수', '결제자_수': '결제자 수'
}).round(2))

fig = px.bar(
    question_segment,
    x='질문·레슨 세그먼트', y='첫 결제율(%)',
    text=question_segment['첫 결제율(%)'].round(1),
    hover_data={'유저_수': ':,', '결제자_수': ':,'},
    labels={'질문·레슨 세그먼트': '질문·레슨 세그먼트', '첫 결제율(%)': '첫 결제율(%)'},
    title='질문 클릭과 레슨 완료 강도별 첫 결제율'
)
fig.add_hline(y=post_overall_paid_rate, line_dash='dash', line_color='gray', annotation_text=f'전체 평균 {post_overall_paid_rate:.2f}%')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=520, xaxis_tickangle=-20)
fig.show()


### 질문 행동 해석

- 현상
    - 질문 클릭은 레슨 완료 수와 결합할 때 세그먼트별 의미가 달라지는 행동 신호
    - 레슨 완료가 없는 상태의 질문 클릭은 학습 몰입이라기보다 결제 전 정보 탐색 또는 도움이 필요한 신호일 수 있음
    - 레슨 7개 이상 완료와 질문 클릭이 함께 나타나면 고관여 학습과 심화 니즈가 동시에 발생한 집단으로 해석 가능

- 인사이트
    - 질문 클릭은 단독으로 고몰입 신호라고 단정하기 어렵고, 반드시 레슨 완료 수와 함께 해석 필요
    - 질문 탐색형의 결제율이 높다면 단순 학습 몰입이 아니라 결제 전 의사결정에 필요한 정보 확인 행동일 수 있음
    - 고몰입 질문형은 유료 기능, Q&A, 심화 콘텐츠 제안에 반응할 가능성이 있는 세그먼트

- 액션
    - 질문 탐색형에게는 질문 영역에서 학습 시작 CTA와 신뢰 요소를 함께 제공
    - 약한 학습+질문형에게는 첫 완료를 방해하는 난이도, 이해도, 도움말 요소를 점검
    - 고몰입 질문형에게는 심화 콘텐츠, Q&A, 유료 기능, 멘토링형 혜택을 제안


## Appendix 4. 최종 유저 세그먼트 기준선 검증


In [ ]:
# 최종 유저 세그먼트 버블차트 기준선 상세 검증
print('중앙값 계산 방식')
print(f'- 현재 최종 세그먼트 수는 {segment_count}개')
print(f'- 각 세그먼트의 유저 비중을 작은 값부터 큰 값으로 정렬한 뒤, 가운데 위치한 {median_position}번째 값을 중앙값으로 사용')
print(f'- 현재 중앙값은 {user_share_median:.2f}%이며, 정렬 기준 가운데 위치한 세그먼트는 {median_segment_name}')
print('- 해당 기준선은 세그먼트 규모를 상대적으로 구분하기 위한 보조선이며, 세그먼트 정의가 바뀌면 중앙값도 달라질 수 있음')

median_reference_display = median_reference.copy()
median_reference_display['정렬 순서'] = range(1, len(median_reference_display) + 1)
median_reference_display = median_reference_display[['정렬 순서', '최종 유저 세그먼트', '유저 비중(%)']]
median_reference_display['유저 비중(%)'] = median_reference_display['유저 비중(%)'].round(2)
display(median_reference_display)

quadrant_rule = pd.DataFrame({
    '사분면': ['규모 큼 + 결제율 높음', '규모 큼 + 결제율 낮음', '규모 작음 + 결제율 높음', '규모 작음 + 결제율 낮음'],
    '의미': ['이미 성과가 큰 핵심 집단', '개선 임팩트가 큰 집단', '효율은 좋지만 규모가 작은 집단', '후순위 또는 모니터링 집단'],
    '전략': ['유지·확장', '전환 병목 개선', '노출·유입 확대 실험', '낮은 우선순위 관리']
})
display(quadrant_rule)


### Appendix 4 해석

- 세그먼트별 중앙값 기준선은 유저 규모를 상대적으로 구분하기 위한 보조선.
- 세그먼트 정의가 바뀌면 중앙값과 사분면 해석도 달라질 수 있으므로, 기준선 산출 방식만 간단히 언급.


## Appendix 5. 구독 생애주기 세그먼트별 상세 행동 차이


In [ ]:
# 구독 생애주기 세그먼트별 행동 요약
post_segment = post_retention[
    post_retention['subscription_lifecycle_segment'].isin([
        'retention_core',
        'renew_then_cancel',
        'early_cancel_before_renew',
        'cancel_then_resubscribe',
        'silent_no_followup'
    ])
].copy()

post_segment['세그먼트'] = post_segment['subscription_lifecycle_segment'].map(segment_label)

segment_summary_rows = []

for segment, temp in post_segment.groupby('subscription_lifecycle_segment', observed=False):
    segment_summary_rows.append({
        'subscription_lifecycle_segment': segment,
        '세그먼트': segment_label.get(segment, segment),
        '유저 수': temp['user_id'].nunique(),
        '유저 비중(%)': temp['user_id'].nunique() / post_segment['user_id'].nunique() * 100,
        '7일 내 학습률(%)': ((temp['lesson_complete_postpay_7d'] > 0) | (temp['content_end_postpay_7d'] > 0)).mean() * 100,
        '30일 내 레슨 완료 수 중앙값': temp['lesson_complete_postpay_30d'].median(),
        '30일 내 학습 활동일 수 중앙값': temp['active_learning_days_postpay_30d'].median(),
        '결제 후 학습 없음 비율(%)': temp['days_to_first_learning_after_payment'].isna().mean() * 100,
        '첫 결제금액 중앙값': temp['first_paid_amount'].median()
    })

segment_summary = pd.DataFrame(segment_summary_rows)
segment_summary['subscription_lifecycle_segment'] = pd.Categorical(
    segment_summary['subscription_lifecycle_segment'],
    categories=[
        'retention_core',
        'renew_then_cancel',
        'early_cancel_before_renew',
        'cancel_then_resubscribe',
        'silent_no_followup'
    ],
    ordered=True
)
segment_summary = segment_summary.sort_values('subscription_lifecycle_segment')

display(segment_summary.round(2))

segment_activity_plot = segment_summary.copy()
segment_activity_plot['중앙값 라벨'] = segment_activity_plot['30일 내 학습 활동일 수 중앙값'].round(1).astype(str) + '일'
segment_activity_plot['학습 없음 라벨'] = segment_activity_plot['결제 후 학습 없음 비율(%)'].round(1).astype(str) + '%'

fig = px.bar(
    segment_activity_plot,
    x='세그먼트',
    y='30일 내 학습 활동일 수 중앙값',
    text='중앙값 라벨',
    hover_data=['유저 수', '7일 내 학습률(%)', '결제 후 학습 없음 비율(%)', '첫 결제금액 중앙값'],
    title='초기 이탈 위험 그룹의 결제 후 학습 지속성 약화',
    labels={'세그먼트': '구독 생애주기 세그먼트', '30일 내 학습 활동일 수 중앙값': '30일 내 학습 활동일 수 중앙값'}
)

fig = apply_common_layout(
    fig,
    height=560,
    y_max=segment_activity_plot['30일 내 학습 활동일 수 중앙값'].max() * 1.35,
    xangle=-20
)
fig.show()


**현상**

- 리텐션 우수 그룹의 30일 내 학습 활동일 수 중앙값 10일, 초기 이탈 위험 그룹 7일 대비 높은 수준
- 리텐션 우수 그룹의 결제 후 학습 없음 비율 1.62%, 초기 이탈 위험 그룹 4.36% 대비 낮은 수준
- 중도 이탈 그룹의 30일 내 학습 활동일 수 중앙값 11일로 초기 학습량 자체는 낮지 않은 구조
- 후속 행동 없음 그룹의 첫 결제금액 중앙값 118,440원으로 다른 그룹 대비 높은 수준

**인사이트**

- 초기 취소 의도 방어에는 결제 후 학습 지속성 관리가 중요
- 중도 이탈은 결제 직후 학습량 부족만으로 설명하기 어려운 영역
- 후속 행동 없음 그룹은 장기 플랜 또는 관측 기간 이슈 가능성으로 해석 주의 필요


## Appendix 6. 결제금액·할인율 보조 분석


In [ ]:
# 결제금액 구간 및 할인율 구간 생성
payment_segment_order = ['월간 정가 결제', '월간 할인 결제', '연간 정가 결제', '연간 할인 결제', '기타 가격대']

def make_payment_segment(row):
    plan_price = row['first_plan_price']
    paid_amount = row['first_paid_amount']

    if plan_price == 15920 and paid_amount == 15920:
        return '월간 정가 결제'
    elif plan_price == 15920 and paid_amount < 15920:
        return '월간 할인 결제'
    elif plan_price == 131600 and paid_amount == 131600:
        return '연간 정가 결제'
    elif plan_price == 131600 and paid_amount < 131600:
        return '연간 할인 결제'
    else:
        return '기타 가격대'

post_retention['첫결제금액구간'] = post_retention.apply(make_payment_segment, axis=1)

payment_segment_summary = make_retention_summary(post_retention, '첫결제금액구간')
payment_segment_summary['첫결제금액구간'] = pd.Categorical(
    payment_segment_summary['첫결제금액구간'],
    categories=payment_segment_order,
    ordered=True
)
payment_segment_summary = payment_segment_summary.sort_values('첫결제금액구간')

display(payment_segment_summary.round(2))

# 할인율 구간 생성
def make_discount_segment(x):
    if pd.isna(x):
        return '결측'
    elif x == 0:
        return '할인 없음'
    elif x == 10:
        return '10% 할인'
    elif x == 15:
        return '15% 할인'
    else:
        return '기타 할인'

discount_order = ['할인 없음', '10% 할인', '15% 할인', '기타 할인']
post_retention['첫할인율구간'] = post_retention['first_discount_rate'].apply(make_discount_segment)

discount_summary = make_retention_summary(post_retention, '첫할인율구간')
discount_summary['첫할인율구간'] = pd.Categorical(
    discount_summary['첫할인율구간'],
    categories=discount_order,
    ordered=True
)
discount_summary = discount_summary.sort_values('첫할인율구간')

display(discount_summary.round(2))

discount_plot = discount_summary[
    [
        '첫할인율구간',
        '60일 정기결제율(%)',
        '30일 내 취소 버튼 클릭률(%)',
        '정기결제 전 취소 버튼 클릭률(%)'
    ]
].copy()

discount_plot = discount_plot.melt(
    id_vars='첫할인율구간',
    var_name='지표',
    value_name='비율(%)'
)
discount_plot = add_percent_label(discount_plot, '비율(%)')

fig = px.bar(
    discount_plot,
    x='첫할인율구간',
    y='비율(%)',
    color='지표',
    barmode='group',
    text='비율 라벨',
    title='15% 할인 그룹의 낮은 60일 정기결제율과 높은 취소 버튼 클릭률',
    labels={'첫할인율구간': '첫 할인율 구간', '비율(%)': '비율(%)', '지표': '리텐션 지표'}
)

fig = apply_common_layout(
    fig,
    height=560,
    y_max=discount_plot['비율(%)'].max() * 1.25,
    xangle=0
)
fig.show()


**현상**

- 월간 정가 결제자의 60일 정기결제율 69.89%, 월간 할인 결제자 70.96%로 유사한 수준
- 월간 할인 결제자의 30일 내 취소 버튼 클릭률 30.90%, 월간 정가 결제자 22.63% 대비 높은 수준
- 15% 할인 그룹의 60일 정기결제율 49.19%, 할인 없음 그룹 59.57% 대비 낮은 수준
- 15% 할인 그룹의 30일 내 취소 버튼 클릭률 29.64%, 할인 없음 그룹 21.45% 대비 높은 수준

**인사이트**

- 결제금액과 할인율은 리텐션을 설명하는 보조 변수로 활용 가능
- 연간 플랜은 60일 내 정기결제가 발생하지 않는 구조이므로 월간 플랜과 동일 기준 비교 시 해석 왜곡 가능성
- 할인율은 프로모션 시기, 플랜 종류, 유저 특성과 함께 움직일 수 있어 인과적 해석 주의 필요
- 리텐션 핵심 신호는 결제 속성보다 결제 후 학습 루틴 유지에 가까운 구조


### Appendix 6 해석

- 결제금액과 할인율은 리텐션을 설명하는 보조 변수로 활용 가능하지만, Main 결론의 중심 지표는 아닌 것으로 판단.
- 연간 플랜은 60일 내 정기결제가 발생하지 않을 수 있어 월간 플랜과 동일한 기준으로 비교하면 해석이 왜곡될 수 있음.
- 따라서 결제 속성은 학습 루틴 분석 이후의 보조 검증 또는 후속 분석 과제로 다루는 것이 적절함.


## Appendix 7. 기존 4개 세그먼트 기준 검증표

아래 표는 발표용 4개 유저 세그먼트와 동일한 콘텐츠 경험 기준을 사용해, 레슨 완료 여부와 콘텐츠 경험 여부가 첫 결제 성과와 어떻게 연결되는지 단순 검증합니다.


In [ ]:
# ============================================================
# 발표용 4개 세그먼트 기준으로 만든 검증용 요약표 2개
# - 표 1: 레슨 완료 여부별 유저 수, 결제자 수, 결제율, 결제 기여도
# - 표 2: 레슨 완료 여부 × 콘텐츠 경험 여부별 유저 수, 결제자 수, 결제율, 결제 기여도
#
# 중요:
# - 콘텐츠 경험 여부는 unique_content_view_7d 단일 컬럼이 아니라,
#   기존 4개 세그먼트에서 사용한 기준과 동일하게 계산합니다.
# - 콘텐츠 경험 O = unique_content_prepay_7d > 0 or content_view_7d > 0 or content_start_7d > 0
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. 분석 데이터 준비
#    - 앞선 셀을 실행했다면 user/post 변수를 그대로 사용
#    - 이 셀만 단독 실행하는 경우 CSV를 자동 탐색해 로드
# ------------------------------------------------------------
try:
    _user_base = user.copy()
except NameError:
    base_dir = Path.cwd()
    possible_dirs = [
        base_dir / 'data',
        base_dir,
        base_dir.parent / 'data',
        Path('data/private'),
        Path('data/private'),
        Path('data/private'),
        Path('data/private')
    ]

    file_candidates = ['dm_user_activation_payment.csv']
    file_candidates += [f'dm_user_activation_payment({i}).csv' for i in range(1, 11)]

    activation_path = None
    for directory in possible_dirs:
        for file_name in file_candidates:
            candidate_path = directory / file_name
            if candidate_path.exists():
                activation_path = candidate_path
                break
        if activation_path is not None:
            break

    if activation_path is None:
        checked_paths = [str(directory / file_name) for directory in possible_dirs for file_name in file_candidates]
        raise FileNotFoundError(
            'dm_user_activation_payment.csv 파일을 찾을 수 없습니다.\n'
            '노트북 파일과 같은 폴더 또는 data 폴더에 CSV를 넣은 뒤 다시 실행해주세요.\n\n'
            '확인한 경로 예시:\n' + '\n'.join(checked_paths[:20])
        )

    _user_base = pd.read_csv(activation_path, low_memory=False)

# 폐지 후 가입 코호트 기준으로 분석
post_segment_check = _user_base[_user_base['period_group'] == 'post_2023_05_11'].copy()

# ------------------------------------------------------------
# 2. 기존 4개 세그먼트와 동일한 기준 적용
# ------------------------------------------------------------
numeric_cols = [
    'paid_30d',
    'lesson_complete_prepay_7d',
    'unique_content_prepay_7d',
    'content_view_7d',
    'content_start_7d'
]

for col in numeric_cols:
    if col not in post_segment_check.columns:
        post_segment_check[col] = 0
    post_segment_check[col] = pd.to_numeric(post_segment_check[col], errors='coerce').fillna(0)

# 기존 4개 세그먼트의 콘텐츠 경험 기준과 동일
post_segment_check['콘텐츠 경험 여부'] = np.where(
    (post_segment_check['unique_content_prepay_7d'] > 0) |
    (post_segment_check['content_view_7d'] > 0) |
    (post_segment_check['content_start_7d'] > 0),
    '1개 이상',
    '0'
)

post_segment_check['결제 전 7일 이내 완료한 수강 수'] = np.where(
    post_segment_check['lesson_complete_prepay_7d'] >= 1,
    '1개 이상',
    '0'
)

lesson_order = ['0', '1개 이상']
content_order = ['0', '1개 이상']
total_paid_users = post_segment_check['paid_30d'].sum()

# ------------------------------------------------------------
# 3. 표 1: 레슨 완료 여부별 요약
# ------------------------------------------------------------
lesson_completion_summary = (
    post_segment_check
    .groupby('결제 전 7일 이내 완료한 수강 수')
    .agg(
        총_유저_수=('user_id', 'nunique'),
        결제_유저_수=('paid_30d', 'sum')
    )
    .reindex(lesson_order)
    .fillna(0)
    .reset_index()
)

lesson_completion_summary['총_유저_수'] = lesson_completion_summary['총_유저_수'].astype(int)
lesson_completion_summary['결제_유저_수'] = lesson_completion_summary['결제_유저_수'].astype(int)
lesson_completion_summary['결제율'] = np.where(
    lesson_completion_summary['총_유저_수'] > 0,
    lesson_completion_summary['결제_유저_수'] / lesson_completion_summary['총_유저_수'] * 100,
    0
)
lesson_completion_summary['결제_기여도'] = np.where(
    total_paid_users > 0,
    lesson_completion_summary['결제_유저_수'] / total_paid_users * 100,
    0
)

lesson_completion_summary[['결제율', '결제_기여도']] = lesson_completion_summary[['결제율', '결제_기여도']].round(2)

# ------------------------------------------------------------
# 4. 표 2: 레슨 완료 여부 × 콘텐츠 경험 여부별 요약
# ------------------------------------------------------------
summary_index = pd.MultiIndex.from_product(
    [lesson_order, content_order],
    names=['결제 전 7일 이내 완료한 수강 수', '콘텐츠 경험 여부']
)

lesson_content_summary = (
    post_segment_check
    .groupby(['결제 전 7일 이내 완료한 수강 수', '콘텐츠 경험 여부'])
    .agg(
        총_유저_수=('user_id', 'nunique'),
        결제_유저_수=('paid_30d', 'sum')
    )
    .reindex(summary_index)
    .fillna(0)
    .reset_index()
)

lesson_content_summary['총_유저_수'] = lesson_content_summary['총_유저_수'].astype(int)
lesson_content_summary['결제_유저_수'] = lesson_content_summary['결제_유저_수'].astype(int)
lesson_content_summary['결제율'] = np.where(
    lesson_content_summary['총_유저_수'] > 0,
    lesson_content_summary['결제_유저_수'] / lesson_content_summary['총_유저_수'] * 100,
    0
)
lesson_content_summary['결제_기여도'] = np.where(
    total_paid_users > 0,
    lesson_content_summary['결제_유저_수'] / total_paid_users * 100,
    0
)

lesson_content_summary[['결제율', '결제_기여도']] = lesson_content_summary[['결제율', '결제_기여도']].round(2)

# ------------------------------------------------------------
# 5. 출력
# ------------------------------------------------------------
print('표 1. 결제 전 7일 이내 레슨 완료 여부별 첫 결제 성과')
display(lesson_completion_summary)

print('표 2. 결제 전 7일 이내 레슨 완료 여부 × 콘텐츠 경험 여부별 첫 결제 성과')
display(lesson_content_summary)


## 실행 시 필요한 데이터 파일 목록

최종 노트북 실행을 위해 아래 CSV 파일이 필요합니다.

1. `dm_user_activation_payment.csv`
2. `dm_content_7d_performance.csv`
3. `dm_user_subscription_retention_core.csv`

권장 위치:

`data/private`

파일명이 `(2)` 등으로 저장된 경우 일부 후보명은 자동 탐색하도록 처리되어 있습니다.  
단, 파일 구조와 컬럼명이 기존 분석 마트와 다르면 일부 셀은 수정이 필요할 수 있습니다.


# Appendix A. 이서희 팀원 개별 분석 원본

> 본 섹션은 팀원별 개별 분석 산출물을 원본 흐름에 가깝게 보존한 제출용 부록입니다.
> 전체 노트북은 순차 실행을 전제로 하지 않으며, 분석 과정과 산출물 취합을 목적으로 합니다.


# 무료체험 폐지 전후 및 신규 유저 콘텐츠 탐색 통합 분석


## 전체 분석 흐름

1. **분석 환경 설정 및 데이터 불러오기**
2. **무료체험 폐지 전후 전체 유저 지표 비교**
3. **폐지 이전 신규 유저의 콘텐츠 탐색 로그 분석**
4. **콘텐츠 탐색 → 수강 시작 → 레슨 진입/완료 흐름 분석**
5. **최종 인사이트 및 팀 공유용 정리**

## 분석 기준

- 무료체험 폐지 전후 비교는 가입일 기준으로 기간을 나눕니다.
- 콘텐츠 탐색 로그 분석은 무료체험 폐지 이전 가입 유저를 대상으로, 가입 후 7일 내 행동을 봅니다.
- 같은 지표가 중복 계산되지 않도록, 전후 비교는 앞부분에만 두고 로그 기반 세부 분석은 뒷부분에 배치했습니다.


## 0. 라이브러리 및 시각화 환경 설정

데이터 처리, 집계, 시각화를 위해 필요한 라이브러리를 불러옵니다.  
한글 그래프 저장을 위해 나눔고딕 폰트 설정도 함께 진행합니다.


In [ ]:
# 코랩 환경에서 한글 폰트가 깨질 경우를 대비한 설정입니다.
# 로컬/Jupyter 환경에서 오류가 나면 이 셀의 apt-get 줄은 주석 처리해도 됩니다.
pass  # 환경 설치 명령 제거. requirements-analysis.txt를 참고하세요.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import matplotlib.font_manager as fm

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

SAVE_DIR = "./"


## 1. 데이터 불러오기

분석에 필요한 유저 단위 데이터, 콘텐츠 단위 성과 데이터, 무료체험 폐지 이전 월별 로그 데이터를 불러옵니다.

- 유저 단위 데이터: 가입, 학습, 결제 여부 확인용
- 콘텐츠 단위 데이터: 콘텐츠별 7일 성과 확인용
- 월별 로그 데이터: 가입 후 콘텐츠 페이지 진입, 후기 클릭, 수강 시작 버튼 클릭 등 행동 분석용


In [ ]:
# 데이터 불러오기
content = pd.read_csv("dm_content_7d_performance.csv")
user = pd.read_csv("dm_user_activation_payment.csv")

log_22_04 = pd.read_csv("event_log_thin_2022_04.csv")
log_22_05 = pd.read_csv("event_log_thin_2022_05.csv")
log_22_06 = pd.read_csv("event_log_thin_2022_06.csv")
log_22_07 = pd.read_csv("event_log_thin_2022_07.csv")
log_22_08 = pd.read_csv("event_log_thin_2022_08.csv")
log_22_09 = pd.read_csv("event_log_thin_2022_09.csv")
log_22_10 = pd.read_csv("event_log_thin_2022_10.csv")
log_22_11 = pd.read_csv("event_log_thin_2022_11.csv")
log_22_12 = pd.read_csv("event_log_thin_2022_12.csv")

print("user 데이터 크기:", user.shape)
print("content 데이터 크기:", content.shape)
user.head()


# 파트 1. 무료체험 폐지 전후 전체 유저 지표 비교

무료체험 폐지 전후로 유저 수, 콘텐츠 조회, 결제 전환, 수강 행동 등이 어떻게 달라졌는지 요약합니다.


## 2. 분석 대상 기간 필터링

가입일 기준으로 무료체험 폐지 전후 7개월 유저만 남기고, 기간 구분 컬럼을 만듭니다.  
이 셀이 먼저 실행되어야 아래에서 분석 대상 유저 데이터를 사용할 수 있습니다.


In [ ]:
user["signup_time"] = pd.to_datetime(user["signup_time"], errors="coerce")

user_filtered = user[
    (user["signup_time"] >= "2022-10-01") &
    (user["signup_time"] < "2023-12-01")
].copy()

# 무료체험 폐지 전후 기간 구분
user_filtered["period"] = np.select(
    [
        (user_filtered["signup_time"] >= "2022-10-01") & (user_filtered["signup_time"] < "2023-05-01"),
        (user_filtered["signup_time"] >= "2023-05-01") & (user_filtered["signup_time"] < "2023-12-01")
    ],
    ["폐지 전", "폐지 후"],
    default="기타"
)

user_filtered = user_filtered[user_filtered["period"] != "기타"].copy()

period_user_count = (
    user_filtered.groupby("period")["user_id"]
    .nunique()
    .reset_index()
    .rename(columns={"period": "기간 구분", "user_id": "유저 수"})
)

print("분석 대상 유저 수:", user_filtered["user_id"].nunique())
display(period_user_count)


## 3. 전처리: 날짜, 결측치, 행동 여부 컬럼 만들기

횟수형 컬럼은 결측치를 0으로 채우고, 주요 행동은 행동 여부를 나타내는 0/1 컬럼으로 변환합니다.  
이렇게 해두면 이후 집계에서 유저 수와 비율을 한 번에 계산할 수 있습니다.


In [ ]:
# 날짜형 컬럼 변환
for col in ["signup_time", "first_payment_time"]:
    if col in user_filtered.columns:
        user_filtered[col] = pd.to_datetime(user_filtered[col], errors="coerce")

# 횟수형 컬럼 결측치 처리
count_cols = [
    "content_view_7d",
    "unique_content_view_7d",
    "content_start_7d",
    "unique_content_start_7d",
    "lesson_enter_7d",
    "unique_lesson_enter_7d",
    "lesson_complete_7d",
    "unique_lesson_complete_7d",
    "lesson_complete_prepay_7d",
    "payment_page_enter_7d"
]

count_cols = [col for col in count_cols if col in user_filtered.columns]
user_filtered[count_cols] = user_filtered[count_cols].fillna(0)

# 결제/활성화 여부 컬럼 결측치 처리
binary_cols = [
    "paid_30d",
    "is_activated_lesson_7d",
    "is_activated_strong_7d"
]

binary_cols = [col for col in binary_cols if col in user_filtered.columns]
user_filtered[binary_cols] = user_filtered[binary_cols].fillna(0).astype(int)

# 주요 행동 여부 컬럼 생성
if "content_view_7d" in user_filtered.columns:
    user_filtered["has_content_view_7d"] = (user_filtered["content_view_7d"] > 0).astype(int)

if "content_start_7d" in user_filtered.columns:
    user_filtered["has_content_start_7d"] = (user_filtered["content_start_7d"] > 0).astype(int)

if "lesson_complete_7d" in user_filtered.columns:
    user_filtered["has_lesson_complete_7d"] = (user_filtered["lesson_complete_7d"] > 0).astype(int)

if "lesson_complete_prepay_7d" in user_filtered.columns:
    user_filtered["has_prepay_lesson_7d"] = (user_filtered["lesson_complete_prepay_7d"] > 0).astype(int)

if "payment_page_enter_7d" in user_filtered.columns:
    user_filtered["has_payment_page_enter_7d"] = (user_filtered["payment_page_enter_7d"] > 0).astype(int)

user_filtered.head()


## 4. 무료체험 폐지 전후 핵심 지표 비교

기간별 유저 수와 주요 행동 유저 수를 집계합니다.  
비율 컬럼은 팀 공유용으로 보기 쉽게 `%` 단위로 계산합니다.


In [ ]:
agg_dict = {
    "user_count": ("user_id", "nunique")
}

# 컬럼이 있을 때만 집계에 포함합니다.
optional_metrics = {
    "paid_30d_users": "paid_30d",
    "content_view_users_7d": "has_content_view_7d",
    "content_start_users_7d": "has_content_start_7d",
    "lesson_complete_users_7d": "has_lesson_complete_7d",
    "prepay_lesson_users_7d": "has_prepay_lesson_7d",
    "payment_page_enter_users_7d": "has_payment_page_enter_7d"
}

for metric_name, col_name in optional_metrics.items():
    if col_name in user_filtered.columns:
        agg_dict[metric_name] = (col_name, "sum")

period_summary = user_filtered.groupby("period").agg(**agg_dict).reset_index()

# 유저 수 대비 비율 계산
for col in period_summary.columns:
    if col.endswith("_users") or col.endswith("_users_7d"):
        rate_col = col.replace("users", "rate")
        period_summary[rate_col] = (period_summary[col] / period_summary["user_count"] * 100).round(2)

period_summary_kor = period_summary.rename(columns={
    "period": "기간 구분",
    "user_count": "유저 수",
    "paid_30d_users": "30일 내 결제 유저 수",
    "content_view_users_7d": "7일 내 콘텐츠 조회 유저 수",
    "content_start_users_7d": "7일 내 수강 시작 유저 수",
    "lesson_complete_users_7d": "7일 내 레슨 완료 유저 수",
    "prepay_lesson_users_7d": "결제 전 7일 내 레슨 완료 유저 수",
    "payment_page_enter_users_7d": "7일 내 결제 페이지 진입 유저 수",
    "paid_30d_rate": "30일 내 결제율",
    "content_view_rate_7d": "7일 내 콘텐츠 조회율",
    "content_start_rate_7d": "7일 내 수강 시작률",
    "lesson_complete_rate_7d": "7일 내 레슨 완료율",
    "prepay_lesson_rate_7d": "결제 전 7일 내 레슨 완료율",
    "payment_page_enter_rate_7d": "7일 내 결제 페이지 진입률"
})

period_summary_kor


## 5. 가입 후 7일 내 학습 행동 구간별 30일 내 결제율

결제 전 레슨 완료 수를 구간화해서, 초기 학습량에 따라 결제율이 달라지는지 확인합니다.  
이 표는 “초기 학습 경험이 결제로 이어지는가?”를 설명할 때 사용할 수 있습니다.


In [ ]:
if {"lesson_complete_prepay_7d", "paid_30d"}.issubset(user_filtered.columns):
    user_filtered["lesson_complete_prepay_7d_group"] = pd.cut(
        user_filtered["lesson_complete_prepay_7d"],
        bins=[-1, 0, 3, 10, float("inf")],
        labels=["0개", "1~3개", "4~10개", "11개 이상"]
    )

    lesson_payment_summary = (
        user_filtered
        .groupby(["period", "lesson_complete_prepay_7d_group"], observed=False)
        .agg(
            user_count=("user_id", "nunique"),
            paid_30d_users=("paid_30d", "sum")
        )
        .reset_index()
    )

    lesson_payment_summary["paid_30d_rate"] = (
        lesson_payment_summary["paid_30d_users"] /
        lesson_payment_summary["user_count"] * 100
    ).round(2)

    lesson_payment_summary_kor = lesson_payment_summary.rename(columns={
        "period": "기간 구분",
        "lesson_complete_prepay_7d_group": "결제 전 7일 내 레슨 완료 수",
        "user_count": "유저 수",
        "paid_30d_users": "30일 내 결제 유저 수",
        "paid_30d_rate": "30일 내 결제율"
    })

    display(lesson_payment_summary_kor)
else:
    print("필요 컬럼이 없어 해당 분석을 건너뜁니다.")


## 파트 1 정리

- 여기까지는 무료체험 폐지 전후의 **전체 유저 단위 요약**입니다.
- 이후부터는 폐지 이전 신규 유저가 가입 직후 콘텐츠를 어떻게 탐색했는지 로그 단위로 더 깊게 확인합니다.
- 따라서 같은 전후 비교 지표를 반복하지 않고, 원인 후보를 찾는 세부 행동 분석으로 이어집니다.


# 파트 2. 폐지 이전 신규 유저 콘텐츠 탐색 로그 분석

무료체험 폐지 이전에 가입한 유저를 대상으로, 가입 후 7일 안에 어떤 콘텐츠 탐색 행동을 보였는지 확인합니다.

핵심 질문은 다음과 같습니다.

1. 신규 유저가 가입 후 콘텐츠 페이지에 얼마나 진입하는가?
2. 콘텐츠 페이지 진입이 수강 시작 버튼 클릭과 실제 수강 시작으로 이어지는가?
3. 후기 클릭, 콘텐츠 탐색 수, 콘텐츠별 차이가 수강 시작률과 관련 있는가?


## 6. 무료체험 폐지 이전 가입 유저 정의

무료체험 폐지 이전 가입 유저만 분리해 가입 후 7일 행동을 볼 기준 유저 집단을 정의합니다.


In [ ]:
# 1. 무료체험 폐지 이전 가입 유저 만들기
user["signup_time"] = pd.to_datetime(user["signup_time"], errors="coerce")

pre_user = user[
    (user["signup_time"] >= "2022-05-01") &
    (user["signup_time"] < "2022-12-01")
].copy()

pre_user["period"] = "폐지 전"

pre_total_users = pre_user["user_id"].nunique()

print("무료체험 폐지 이전 가입 유저 수:", pre_total_users)


## 7. 폐지 이전 월별 로그 통합

가입 후 7일 행동을 보기 위해 2022년 5월~12월 로그를 하나로 합칩니다. 11월 말 가입자의 7일 행동까지 보려면 12월 로그도 필요합니다.


In [ ]:
# 2. 로그 합치기
pre_logs = pd.concat([
    log_22_05,
    log_22_06,
    log_22_07,
    log_22_08,
    log_22_09,
    log_22_10,
    log_22_11,
    log_22_12
], ignore_index=True)

pre_logs["event_time"] = pd.to_datetime(pre_logs["event_time"], errors="coerce")
pre_user["signup_time"] = pd.to_datetime(pre_user["signup_time"], errors="coerce")

pre_signup = pre_user[["user_id", "signup_time"]].copy()


## 8. 콘텐츠 페이지 진입 로그 생성

콘텐츠 페이지 진입 이벤트만 추출하고 가입 시점과 연결해 가입 후 7일 내 콘텐츠 페이지 진입 로그를 만듭니다.


In [ ]:
# 3. 콘텐츠 페이지 진입 로그 만들기
content_page_log = pre_logs[
    pre_logs["event_name"] == "enter_content_page"
].copy()

content_page_joined = content_page_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

content_page_joined["days_after_signup"] = (
    content_page_joined["event_time"] - content_page_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

content_page_7d = content_page_joined[
    (content_page_joined["days_after_signup"] >= 0) &
    (content_page_joined["days_after_signup"] < 7)
].copy()


## 9. 가입 후 1일/3일/7일 내 콘텐츠 페이지 진입률

가입 직후 콘텐츠 탐색이 어느 시점에 집중되는지 확인합니다.


In [ ]:
# 1번 표: 가입 후 1일/3일/7일 내 콘텐츠 페이지 진입률
users_1d = content_page_joined[
    (content_page_joined["days_after_signup"] >= 0) &
    (content_page_joined["days_after_signup"] < 1)
]["user_id"].nunique()

users_3d = content_page_joined[
    (content_page_joined["days_after_signup"] >= 0) &
    (content_page_joined["days_after_signup"] < 3)
]["user_id"].nunique()

users_7d = content_page_joined[
    (content_page_joined["days_after_signup"] >= 0) &
    (content_page_joined["days_after_signup"] < 7)
]["user_id"].nunique()

table_1 = pd.DataFrame({
    "구분": ["무료체험 폐지 이전"],
    "가입 유저 수": [pre_total_users],
    "1일 내 진입 유저 수": [users_1d],
    "1일 내 진입률": [round(users_1d / pre_total_users * 100, 2)],
    "3일 내 진입 유저 수": [users_3d],
    "3일 내 진입률": [round(users_3d / pre_total_users * 100, 2)],
    "7일 내 진입 유저 수": [users_7d],
    "7일 내 진입률": [round(users_7d / pre_total_users * 100, 2)]
})

table_1


## 10. 가입 후 7일 내 고유 콘텐츠 탐색 수

신규 유저가 가입 후 7일 동안 몇 개의 서로 다른 콘텐츠를 탐색했는지 집계합니다.


In [ ]:
# 2번 표: 가입 후 7일 내 고유 콘텐츠 탐색 수
user_unique_content = (
    content_page_7d.groupby("user_id")["content_id"]
    .nunique()
    .reset_index(name="unique_content_count")
)

pre_unique_content = pre_signup[["user_id"]].merge(
    user_unique_content,
    on="user_id",
    how="left"
)

pre_unique_content["unique_content_count"] = (
    pre_unique_content["unique_content_count"].fillna(0).astype(int)
)

pre_unique_content["고유 콘텐츠 탐색 수"] = pd.cut(
    pre_unique_content["unique_content_count"],
    bins=[-1, 0, 1, 2, 5, np.inf],
    labels=["0개", "1개", "2개", "3~5개", "6개 이상"]
)

table_2 = (
    pre_unique_content.groupby("고유 콘텐츠 탐색 수", observed=False)
    .agg(유저수=("user_id", "nunique"))
    .reset_index()
)

table_2["구분"] = "무료체험 폐지 이전"
table_2["비율"] = (table_2["유저수"] / pre_total_users * 100).round(2)

table_2 = table_2[["구분", "고유 콘텐츠 탐색 수", "유저수", "비율"]]

table_2


## 11. 가입 후 7일 내 많이 본 콘텐츠 상위 10개

신규 유저가 가장 많이 조회한 콘텐츠를 확인해 첫 추천/노출 후보를 검토합니다.


In [ ]:
# 3번 표: 가입 후 7일 내 많이 본 콘텐츠 상위 10개
table_3 = (
    content_page_7d.groupby("content_id")
    .agg(
        조회_유저_수=("user_id", "nunique"),
        조회_이벤트_수=("id", "count")
    )
    .reset_index()
    .sort_values(["조회_유저_수", "조회_이벤트_수"], ascending=False)
    .head(10)
    .reset_index(drop=True)
)

table_3["순위"] = table_3.index + 1

table_3 = table_3[["순위", "content_id", "조회_유저_수", "조회_이벤트_수"]]

table_3.rename(columns={
    "content_id": "콘텐츠 아이디",
    "조회_유저_수": "조회 유저 수",
    "조회_이벤트_수": "조회 이벤트 수"
})


## 12. 콘텐츠 페이지 진입 유저-콘텐츠 조합 생성

이후 전환율 계산을 위해 유저-콘텐츠 단위의 중복 없는 조합을 만듭니다.


In [ ]:
# 4~8번 공통 기준: 콘텐츠 페이지 진입 user-content 조합
content_entry_pairs = (
    content_page_7d[["user_id", "content_id"]]
    .dropna(subset=["content_id"])
    .drop_duplicates()
    .copy()
)

page_entry_count = len(content_entry_pairs)

print("콘텐츠 페이지 진입 건수(유저-콘텐츠 기준):", page_entry_count)


## 13. 콘텐츠 페이지 진입 후 수강 시작 버튼 클릭률

콘텐츠 페이지 진입이 수강 의도 버튼 클릭까지 이어지는지 확인합니다.


In [ ]:
# 4번 표: 콘텐츠 페이지 진입 후 수강 시작 버튼 클릭률
start_button_log = pre_logs[
    pre_logs["event_name"] == "click_content_page_start_content_button"
].copy()

start_button_joined = start_button_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

start_button_joined["days_after_signup"] = (
    start_button_joined["event_time"] - start_button_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

start_button_7d_pairs = (
    start_button_joined[
        (start_button_joined["days_after_signup"] >= 0) &
        (start_button_joined["days_after_signup"] < 7)
    ][["user_id", "content_id"]]
    .dropna(subset=["content_id"])
    .drop_duplicates()
)

entry_with_start_button = content_entry_pairs.merge(
    start_button_7d_pairs.assign(start_button_clicked=1),
    on=["user_id", "content_id"],
    how="left"
)

entry_with_start_button["start_button_clicked"] = (
    entry_with_start_button["start_button_clicked"].fillna(0).astype(int)
)

start_button_click_count = entry_with_start_button["start_button_clicked"].sum()
start_button_click_rate = round(start_button_click_count / page_entry_count * 100, 2)

table_4 = pd.DataFrame({
    "콘텐츠 페이지 진입 건수": [page_entry_count],
    "수강 시작 버튼 클릭 수": [start_button_click_count],
    "수강 시작 버튼 클릭률": [start_button_click_rate]
})

table_4


## 14. 콘텐츠 페이지 진입 후 실제 수강 시작률

콘텐츠 페이지 진입이 실제 수강 시작 행동으로 이어지는지 확인합니다.


In [ ]:
# 5번 표: 콘텐츠 페이지 진입 후 실제 수강 시작률
start_content_log = pre_logs[
    pre_logs["event_name"] == "start_content"
].copy()

start_content_joined = start_content_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

start_content_joined["days_after_signup"] = (
    start_content_joined["event_time"] - start_content_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

start_content_7d_pairs = (
    start_content_joined[
        (start_content_joined["days_after_signup"] >= 0) &
        (start_content_joined["days_after_signup"] < 7)
    ][["user_id", "content_id"]]
    .dropna(subset=["content_id"])
    .drop_duplicates()
)

entry_with_start_content = content_entry_pairs.merge(
    start_content_7d_pairs.assign(started_content=1),
    on=["user_id", "content_id"],
    how="left"
)

entry_with_start_content["started_content"] = (
    entry_with_start_content["started_content"].fillna(0).astype(int)
)

actual_start_count = entry_with_start_content["started_content"].sum()
actual_start_rate = round(actual_start_count / page_entry_count * 100, 2)

table_5 = pd.DataFrame({
    "콘텐츠 페이지 진입 건수": [page_entry_count],
    "실제 수강 시작 수": [actual_start_count],
    "실제 수강 시작률": [actual_start_rate]
})

table_5


## 15. 후기 더보기 클릭률

콘텐츠 페이지 진입 후 후기 확인 행동이 얼마나 발생하는지 확인합니다.


In [ ]:
# 6번 표: 후기 더보기 클릭률
review_click_log = pre_logs[
    pre_logs["event_name"] == "click_content_page_more_review_button"
].copy()

review_click_joined = review_click_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

review_click_joined["days_after_signup"] = (
    review_click_joined["event_time"] - review_click_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

review_click_7d_pairs = (
    review_click_joined[
        (review_click_joined["days_after_signup"] >= 0) &
        (review_click_joined["days_after_signup"] < 7)
    ][["user_id", "content_id"]]
    .dropna(subset=["content_id"])
    .drop_duplicates()
)

entry_with_review_click = content_entry_pairs.merge(
    review_click_7d_pairs.assign(review_clicked=1),
    on=["user_id", "content_id"],
    how="left"
)

entry_with_review_click["review_clicked"] = (
    entry_with_review_click["review_clicked"].fillna(0).astype(int)
)

review_click_count = entry_with_review_click["review_clicked"].sum()
review_click_rate = round(review_click_count / page_entry_count * 100, 2)

table_6 = pd.DataFrame({
    "콘텐츠 페이지 진입 건수": [page_entry_count],
    "후기 더보기 클릭 수": [review_click_count],
    "후기 더보기 클릭률": [review_click_rate]
})

table_6


## 16. 후기 클릭 유저 대비 미클릭 유저 수강 시작률

후기 클릭 여부에 따라 실제 수강 시작률이 달라지는지 비교합니다.


In [ ]:
# 7번 표: 후기 클릭 유저 대비 미클릭 유저의 수강 시작률 차이
entry_review_start = entry_with_review_click.merge(
    start_content_7d_pairs.assign(started_content=1),
    on=["user_id", "content_id"],
    how="left"
)

entry_review_start["started_content"] = (
    entry_review_start["started_content"].fillna(0).astype(int)
)

table_7 = (
    entry_review_start.groupby("review_clicked")
    .agg(
        콘텐츠_페이지_진입_건수=("user_id", "count"),
        실제_수강_시작_수=("started_content", "sum")
    )
    .reset_index()
)

table_7["구분"] = np.where(
    table_7["review_clicked"] == 1,
    "후기 더보기 클릭",
    "후기 더보기 미클릭"
)

table_7["수강 시작률"] = (
    table_7["실제_수강_시작_수"] / table_7["콘텐츠_페이지_진입_건수"] * 100
).round(2)

table_7 = table_7[
    ["구분", "콘텐츠_페이지_진입_건수", "실제_수강_시작_수", "수강 시작률"]
]

table_7


## 17. 조회 상위 10개 콘텐츠별 실제 수강 시작률

조회가 많은 콘텐츠와 실제 수강 시작으로 잘 이어지는 콘텐츠가 같은지 비교합니다.


In [ ]:
# 8번 표: 신규 유저 조회 콘텐츠 대비 실제 수강 시작률
view_by_content = (
    content_entry_pairs.groupby("content_id")
    .agg(조회_유저_수=("user_id", "nunique"))
    .reset_index()
)

start_by_content = (
    start_content_7d_pairs.groupby("content_id")
    .agg(수강_시작_유저_수=("user_id", "nunique"))
    .reset_index()
)

table_8 = view_by_content.merge(
    start_by_content,
    on="content_id",
    how="left"
)

table_8["수강_시작_유저_수"] = (
    table_8["수강_시작_유저_수"].fillna(0).astype(int)
)

table_8["수강_시작률"] = (
    table_8["수강_시작_유저_수"] / table_8["조회_유저_수"] * 100
).round(2)

table_8 = (
    table_8.sort_values("조회_유저_수", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

table_8["순위"] = table_8.index + 1

table_8 = table_8[
    ["순위", "content_id", "조회_유저_수", "수강_시작_유저_수", "수강_시작률"]
]

table_8.rename(columns={
    "content_id": "콘텐츠 아이디",
    "조회_유저_수": "조회 유저 수",
    "수강_시작_유저_수": "수강 시작 유저 수",
    "수강_시작률": "수강 시작률"
})


# 파트 3. 추가 분석: 퍼널, 탐색 수, 시각화

앞에서 만든 표를 바탕으로 팀 발표/보고서에 넣기 좋은 요약표와 그래프를 만듭니다.  
이 파트는 분석 결과를 설명하기 위한 시각화 중심입니다.


## 추가분석


In [ ]:
# 추가분석 1.퍼널분석
funnel_summary = pd.DataFrame({
    "단계": [
        "콘텐츠 페이지 진입",
        "수강 시작 버튼 클릭",
        "실제 수강 시작"
    ],
    "건수": [
        page_entry_count,
        start_button_click_count,
        actual_start_count
    ]
})

funnel_summary["이전 단계 대비 전환율"] = [
    100,
    round(start_button_click_count / page_entry_count * 100, 2),
    round(actual_start_count / start_button_click_count * 100, 2)
]

funnel_summary["콘텐츠 페이지 진입 대비 전환율"] = [
    100,
    round(start_button_click_count / page_entry_count * 100, 2),
    round(actual_start_count / page_entry_count * 100, 2)
]

funnel_summary


In [ ]:
# 추가분석 2. 고유 콘텐츠 탐색 수별 실제 수강 시작률
# 유저별 고유 콘텐츠 탐색 수
user_unique_content = (
    content_page_7d
    .groupby("user_id")["content_id"]
    .nunique()
    .reset_index(name="unique_content_count")
)

# 콘텐츠 페이지 진입 조합 안에서 실제 수강 시작한 유저
matched_start_pairs = content_entry_pairs.merge(
    start_content_7d_pairs.assign(started_content=1),
    on=["user_id", "content_id"],
    how="inner"
)

user_started = (
    matched_start_pairs
    .groupby("user_id")
    .agg(started_content_count=("content_id", "nunique"))
    .reset_index()
)

user_started["has_started_content"] = (
    user_started["started_content_count"] > 0
).astype(int)

# 전체 pre 가입자 기준으로 병합
explore_start_user = pre_signup[["user_id"]].merge(
    user_unique_content,
    on="user_id",
    how="left"
).merge(
    user_started[["user_id", "has_started_content"]],
    on="user_id",
    how="left"
)

explore_start_user["unique_content_count"] = (
    explore_start_user["unique_content_count"].fillna(0).astype(int)
)

explore_start_user["has_started_content"] = (
    explore_start_user["has_started_content"].fillna(0).astype(int)
)

# 구간화
explore_start_user["고유 콘텐츠 탐색 수"] = pd.cut(
    explore_start_user["unique_content_count"],
    bins=[-1, 0, 1, 2, 5, np.inf],
    labels=["0개", "1개", "2개", "3~5개", "6개 이상"]
)

# 구간별 수강 시작률
explore_start_summary = (
    explore_start_user
    .groupby("고유 콘텐츠 탐색 수", observed=False)
    .agg(
        유저수=("user_id", "nunique"),
        실제_수강_시작_유저수=("has_started_content", "sum")
    )
    .reset_index()
)

explore_start_summary["수강 시작률"] = (
    explore_start_summary["실제_수강_시작_유저수"] /
    explore_start_summary["유저수"] * 100
).round(2)

explore_start_summary


In [ ]:
# 1. 가입 후 1일/3일/7일 내 콘텐츠 페이지 진입률
entry_rate_df = pd.DataFrame({
    "구분": ["1일 내", "3일 내", "7일 내"],
    "진입률": [26.84, 28.34, 30.09]
})

plt.figure(figsize=(7, 4))
plt.bar(entry_rate_df["구분"], entry_rate_df["진입률"])

plt.title("가입 후 콘텐츠 페이지 진입률")
plt.xlabel("가입 후 기간")
plt.ylabel("진입률(%)")
plt.ylim(0, 40)

for i, v in enumerate(entry_rate_df["진입률"]):
    plt.text(i, v + 0.8, f"{v:.2f}%", ha="center")

plt.tight_layout()
plt.savefig(SAVE_DIR + "01_content_entry_rate.png", dpi=300, bbox_inches="tight")
plt.show()


- 가입 후 1일 내 콘텐츠 페이지 진입률은 26.84%, 7일 내 진입률은 30.09%로 나타났다. 1일 내 진입률과 7일 내 진입률의 차이가 크지 않기 때문에, 콘텐츠 탐색은 대부분 가입 직후 1일 안에 발생한다고 볼 수 있다.

- 즉, 가입 직후 콘텐츠 탐색으로 이어지지 않은 유저는 이후에도 콘텐츠 페이지로 진입할 가능성이 상대적으로 낮다. 따라서 신규 유저의 첫 화면, 온보딩 이후 추천 콘텐츠, 가입 직후 행동 유도 버튼 설계가 중요하다.


In [ ]:
#2. 가입 후 7일 내 고유 콘텐츠 탐색 수 분포
unique_content_plot = table_2.copy()

# 컬럼명 혹시 다르면 맞춰주기
unique_content_plot = unique_content_plot.rename(columns={
    "유저수": "유저 수"
})

plt.figure(figsize=(8, 4))
plt.bar(unique_content_plot["고유 콘텐츠 탐색 수"], unique_content_plot["비율"])

plt.title("가입 후 7일 내 고유 콘텐츠 탐색 수 분포")
plt.xlabel("고유 콘텐츠 탐색 수")
plt.ylabel("비율(%)")
plt.ylim(0, 80)

for i, v in enumerate(unique_content_plot["비율"]):
    plt.text(i, v + 1, f"{v:.2f}%", ha="center")

plt.tight_layout()
plt.savefig(SAVE_DIR + "02_unique_content_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


- 가입 후 7일 내 콘텐츠를 하나도 탐색하지 않은 유저가 69.91%로 가장 많았다. 콘텐츠를 탐색한 유저 중에서도 1개만 본 유저가 16.49%로 나타났고, 3개 이상 여러 콘텐츠를 비교·탐색한 유저는 소수에 그쳤다.

- 이는 신규 유저가 가입 이후 콘텐츠 목록을 충분히 둘러보지 못하고 있음을 보여준다. 따라서 가입 직후 관심사 기반 콘텐츠 추천이나 목표별 콘텐츠 묶음 제공을 통해 다양한 콘텐츠 탐색을 유도할 필요가 있다.


In [ ]:
# 3. 콘텐츠 페이지 진입 → 버튼 클릭 → 실제 수강 시작 퍼널
funnel_plot = funnel_summary.copy()

plt.figure(figsize=(8, 4))
plt.bar(funnel_plot["단계"], funnel_plot["건수"])

plt.title("콘텐츠 페이지 진입 후 수강 시작 퍼널")
plt.xlabel("단계")
plt.ylabel("건수")

for i, v in enumerate(funnel_plot["건수"]):
    rate = funnel_plot.loc[i, "콘텐츠 페이지 진입 대비 전환율"]
    plt.text(i, v + 300, f"{v:,}\n({rate:.2f}%)", ha="center")

plt.tight_layout()
plt.savefig(SAVE_DIR + "03_content_start_funnel.png", dpi=300, bbox_inches="tight")
plt.show()


- 콘텐츠 페이지 진입 22,319건 중 수강 시작 버튼 클릭으로 이어진 건수는 5,297건으로, 전환율은 23.73%였다. 이후 실제 수강 시작까지 이어진 건수는 2,837건으로, 콘텐츠 페이지 진입 대비 최종 수강 시작률은 12.71%였다.

- 가장 큰 이탈은 콘텐츠 페이지 진입 후 수강 시작 버튼을 클릭하기 전 구간에서 발생했다. 즉, 상세 페이지까지 들어온 유저가 수강을 시작할 만큼 충분한 확신을 얻지 못했을 가능성이 있다. 콘텐츠의 학습 목표, 난이도, 커리큘럼, 기대 효과 등을 더 명확히 보여주는 개선이 필요하다.


In [ ]:
# 4. 고유 콘텐츠 탐색 수별 실제 수강 시작률
explore_plot = explore_start_summary.copy()

plt.figure(figsize=(8, 4))
plt.bar(explore_plot["고유 콘텐츠 탐색 수"], explore_plot["수강 시작률"])

plt.title("고유 콘텐츠 탐색 수별 실제 수강 시작률")
plt.xlabel("고유 콘텐츠 탐색 수")
plt.ylabel("수강 시작률(%)")
plt.ylim(0, 45)

for i, v in enumerate(explore_plot["수강 시작률"]):
    plt.text(i, v + 1, f"{v:.2f}%", ha="center")

plt.tight_layout()
plt.savefig(SAVE_DIR + "04_start_rate_by_unique_content.png", dpi=300, bbox_inches="tight")
plt.show()


- 고유 콘텐츠 탐색 수가 많을수록 실제 수강 시작률이 높아지는 경향이 나타났다. 콘텐츠를 1개 탐색한 유저의 수강 시작률은 5.37%였지만, 2개 탐색 유저는 15.48%, 3~5개 탐색 유저는 23.08%, 6개 이상 탐색 유저는 37.83%로 증가했다.

- 이는 여러 콘텐츠를 비교하고 탐색한 유저일수록 실제 수강 시작까지 이어질 가능성이 높다는 점을 보여준다. 다만 이는 인과관계로 단정할 수는 없으며, 원래 학습 의향이 높은 유저가 더 많은 콘텐츠를 탐색했을 가능성도 있다.


In [ ]:
# 5. 조회 상위 10개 콘텐츠별 수강 시작률
top_content_plot = table_8.copy()

# 콘텐츠 아이디가 너무 길어서 앞 6자리만 사용
top_content_plot["content_short"] = top_content_plot["content_id"].astype(str).str[:6] + "..."

plt.figure(figsize=(9, 5))
plt.barh(top_content_plot["content_short"], top_content_plot["수강_시작률"])

plt.title("조회 상위 10개 콘텐츠별 수강 시작률")
plt.xlabel("수강 시작률(%)")
plt.ylabel("콘텐츠 아이디")
plt.gca().invert_yaxis()

for i, v in enumerate(top_content_plot["수강_시작률"]):
    plt.text(v + 0.5, i, f"{v:.2f}%", va="center")

plt.tight_layout()
plt.savefig(SAVE_DIR + "05_top_content_start_rate.png", dpi=300, bbox_inches="tight")
plt.show()


- 조회 유저 수 1위 콘텐츠인 f4be1a...의 수강 시작률은 13.10%였지만, 조회 유저 수 2위 콘텐츠인 61b646...의 수강 시작률은 25.12%로 더 높게 나타났다. 또한 조회 상위 콘텐츠 간에도 수강 시작률 차이가 존재했다.

- 따라서 많이 조회되는 콘텐츠가 반드시 실제 수강 시작으로 잘 이어지는 콘텐츠라고 보기는 어렵다. 신규 유저에게 추천할 콘텐츠를 선정할 때는 조회 유저 수뿐 아니라 조회 이후 실제 수강 시작률을 함께 고려해야 한다.


## 콘텐츠 별 레슨 별 분석


# 파트 4. 콘텐츠 탐색 수와 레슨 진입/완료 관계

콘텐츠를 많이 탐색한 유저가 실제 레슨 진입과 완료까지 더 잘 이어지는지 확인합니다.  
이 파트는 “콘텐츠 탐색이 초기 학습 활성화로 이어지는가?”를 설명할 때 사용할 수 있습니다.


In [ ]:
# 1. 유저별 고유 콘텐츠 탐색 수 만들기
# 유저별 고유 콘텐츠 탐색 수
user_unique_content = (
    content_page_7d
    .groupby("user_id")["content_id"]
    .nunique()
    .reset_index(name="unique_content_count")
)

# 전체 pre 가입자 기준으로 붙이기
user_content_lesson = pre_signup[["user_id"]].merge(
    user_unique_content,
    on="user_id",
    how="left"
)

user_content_lesson["unique_content_count"] = (
    user_content_lesson["unique_content_count"].fillna(0).astype(int)
)

# 고유 콘텐츠 탐색 수 구간화
user_content_lesson["고유 콘텐츠 탐색 수"] = pd.cut(
    user_content_lesson["unique_content_count"],
    bins=[-1, 0, 1, 2, 5, np.inf],
    labels=["0개", "1개", "2개", "3~5개", "6개 이상"]
)


In [ ]:
# 2. 가입 후 7일 내 레슨 진입 만들기
lesson_enter_log = pre_logs[
    pre_logs["event_name"] == "enter_lesson_page"
].copy()

lesson_enter_joined = lesson_enter_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

lesson_enter_joined["days_after_signup"] = (
    lesson_enter_joined["event_time"] - lesson_enter_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

lesson_enter_7d = lesson_enter_joined[
    (lesson_enter_joined["days_after_signup"] >= 0) &
    (lesson_enter_joined["days_after_signup"] < 7)
].copy()


In [ ]:
# 3. 가입 후 7일 내 레슨 완료 만들기
lesson_complete_log = pre_logs[
    pre_logs["event_name"] == "complete_lesson"
].copy()

lesson_complete_joined = lesson_complete_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

lesson_complete_joined["days_after_signup"] = (
    lesson_complete_joined["event_time"] - lesson_complete_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

lesson_complete_7d = lesson_complete_joined[
    (lesson_complete_joined["days_after_signup"] >= 0) &
    (lesson_complete_joined["days_after_signup"] < 7)
].copy()


In [ ]:
# 4. 유저별 레슨 진입/완료 수 만들기
# 유저별 고유 레슨 진입 수
user_lesson_enter = (
    lesson_enter_7d
    .dropna(subset=["lesson_id"])
    .groupby("user_id")["lesson_id"]
    .nunique()
    .reset_index(name="unique_lesson_enter_count")
)

# 유저별 고유 레슨 완료 수
user_lesson_complete = (
    lesson_complete_7d
    .dropna(subset=["lesson_id"])
    .groupby("user_id")["lesson_id"]
    .nunique()
    .reset_index(name="unique_lesson_complete_count")
)

# 붙이기
user_content_lesson = user_content_lesson.merge(
    user_lesson_enter,
    on="user_id",
    how="left"
).merge(
    user_lesson_complete,
    on="user_id",
    how="left"
)

user_content_lesson["unique_lesson_enter_count"] = (
    user_content_lesson["unique_lesson_enter_count"].fillna(0).astype(int)
)

user_content_lesson["unique_lesson_complete_count"] = (
    user_content_lesson["unique_lesson_complete_count"].fillna(0).astype(int)
)

# 여부 컬럼
user_content_lesson["has_lesson_enter"] = (
    user_content_lesson["unique_lesson_enter_count"] > 0
).astype(int)

user_content_lesson["has_lesson_complete"] = (
    user_content_lesson["unique_lesson_complete_count"] > 0
).astype(int)


In [ ]:
# 최종 표 1. 고유 콘텐츠 탐색 수별 레슨 진입/완료율
content_lesson_summary = (
    user_content_lesson
    .groupby("고유 콘텐츠 탐색 수", observed=False)
    .agg(
        유저수=("user_id", "nunique"),
        레슨_진입_유저수=("has_lesson_enter", "sum"),
        레슨_완료_유저수=("has_lesson_complete", "sum"),
        평균_고유_레슨_진입수=("unique_lesson_enter_count", "mean"),
        평균_고유_레슨_완료수=("unique_lesson_complete_count", "mean")
    )
    .reset_index()
)

content_lesson_summary["레슨 진입률"] = (
    content_lesson_summary["레슨_진입_유저수"] /
    content_lesson_summary["유저수"] * 100
).round(2)

content_lesson_summary["레슨 완료율"] = (
    content_lesson_summary["레슨_완료_유저수"] /
    content_lesson_summary["유저수"] * 100
).round(2)

content_lesson_summary["평균_고유_레슨_진입수"] = (
    content_lesson_summary["평균_고유_레슨_진입수"].round(2)
)

content_lesson_summary["평균_고유_레슨_완료수"] = (
    content_lesson_summary["평균_고유_레슨_완료수"].round(2)
)

content_lesson_summary = content_lesson_summary[
    [
        "고유 콘텐츠 탐색 수",
        "유저수",
        "레슨_진입_유저수",
        "레슨 진입률",
        "레슨_완료_유저수",
        "레슨 완료율",
        "평균_고유_레슨_진입수",
        "평균_고유_레슨_완료수"
    ]
]

content_lesson_summary


### 고유 콘텐츠 탐색 수 0개 찾아보기


In [ ]:
# 1. 콘텐츠 탐색 0개인데 레슨 진입한 유저 뽑기
zero_content_lesson_users = user_content_lesson[
    (user_content_lesson["unique_content_count"] == 0) &
    (user_content_lesson["has_lesson_enter"] == 1)
]["user_id"].unique()

len(zero_content_lesson_users)


In [ ]:
#  2. 이 유저들의 레슨 진입 로그 확인
zero_lesson_enter_logs = lesson_enter_7d[
    lesson_enter_7d["user_id"].isin(zero_content_lesson_users)
].copy()

zero_lesson_enter_logs[
    ["user_id", "event_time", "event_name", "content_id", "lesson_id", "source_table"]
].head(20).rename(columns={
    "user_id": "유저 아이디",
    "event_time": "이벤트 발생 시간",
    "event_name": "이벤트명",
    "content_id": "콘텐츠 아이디",
    "lesson_id": "레슨 아이디",
    "source_table": "원본 테이블"
})


In [ ]:
# 3. 레슨 진입 로그에 콘텐츠 아이디가 있는지 확인
zero_lesson_enter_logs.assign(
    has_content_id=zero_lesson_enter_logs["content_id"].notna(),
    has_lesson_id=zero_lesson_enter_logs["lesson_id"].notna()
).groupby(["has_content_id", "has_lesson_id"]).size().reset_index(name="row_count").rename(columns={
    "has_content_id": "콘텐츠 아이디 있음",
    "has_lesson_id": "레슨 아이디 있음",
    "row_count": "로그 수"
})


In [ ]:
# 4. 같은 유저가 콘텐츠 페이지 진입 이벤트는 진짜 없는지 확인
zero_users_content_page = content_page_7d[
    content_page_7d["user_id"].isin(zero_content_lesson_users)
]

zero_users_content_page.shape


In [ ]:
# 5. 콘텐츠 아이디가 비어 있어서 0개로 잡힌 건지 확인
zero_users_content_page.assign(
    has_content_id=zero_users_content_page["content_id"].notna()
).groupby("has_content_id").size().reset_index(name="row_count").rename(columns={
    "has_content_id": "콘텐츠 아이디 있음",
    "row_count": "로그 수"
})


# 맞춰가야할 분석


# 파트 5. 최종 검토용 질문 분석

**부록/검증용**


In [ ]:
# 1. 콘텐츠 페이지 진입을 초기 활성화 기준으로 볼 수 있을까?
# 콘텐츠 페이지 진입 여부
content_enter_users = (
    content_page_7d[["user_id"]]
    .drop_duplicates()
    .assign(has_content_enter=1)
)

activation_base = pre_signup[["user_id"]].merge(
    content_enter_users,
    on="user_id",
    how="left"
)

activation_base["has_content_enter"] = (
    activation_base["has_content_enter"].fillna(0).astype(int)
)

# 실제 수강 시작 여부
matched_start_pairs = content_entry_pairs.merge(
    start_content_7d_pairs.assign(started_content=1),
    on=["user_id", "content_id"],
    how="inner"
)

start_users = (
    matched_start_pairs[["user_id"]]
    .drop_duplicates()
    .assign(has_started_content=1)
)

activation_base = activation_base.merge(
    start_users,
    on="user_id",
    how="left"
)

activation_base["has_started_content"] = (
    activation_base["has_started_content"].fillna(0).astype(int)
)

# 30일 내 결제 여부가 user 데이터에 있으면 붙이기
payment_cols = ["user_id", "paid_30d"]
if "paid_30d" in pre_user.columns:
    activation_base = activation_base.merge(
        pre_user[payment_cols],
        on="user_id",
        how="left"
    )
    activation_base["paid_30d"] = activation_base["paid_30d"].fillna(0).astype(int)

# 콘텐츠 진입 여부별 수강 시작률/결제율
content_enter_activation_summary = (
    activation_base
    .groupby("has_content_enter")
    .agg(
        유저수=("user_id", "nunique"),
        실제_수강_시작_유저수=("has_started_content", "sum"),
        결제_유저수=("paid_30d", "sum") if "paid_30d" in activation_base.columns else ("user_id", "size")
    )
    .reset_index()
)

content_enter_activation_summary["구분"] = content_enter_activation_summary["has_content_enter"].map({
    0: "콘텐츠 페이지 미진입",
    1: "콘텐츠 페이지 진입"
})

content_enter_activation_summary["수강 시작률"] = (
    content_enter_activation_summary["실제_수강_시작_유저수"] /
    content_enter_activation_summary["유저수"] * 100
).round(2)

if "paid_30d" in activation_base.columns:
    content_enter_activation_summary["30일 내 결제율"] = (
        content_enter_activation_summary["결제_유저수"] /
        content_enter_activation_summary["유저수"] * 100
    ).round(2)

content_enter_activation_summary = content_enter_activation_summary[
    ["구분", "유저수", "실제_수강_시작_유저수", "수강 시작률"] +
    (["결제_유저수", "30일 내 결제율"] if "paid_30d" in activation_base.columns else [])
]

content_enter_activation_summary


In [ ]:
# 2. 고유 콘텐츠 2개 이상 탐색을 관심 형성 기준으로 볼 수 있을까?
# 유저별 고유 콘텐츠 탐색 수
user_unique_content = (
    content_page_7d
    .groupby("user_id")["content_id"]
    .nunique()
    .reset_index(name="unique_content_count")
)

interest_base = pre_signup[["user_id"]].merge(
    user_unique_content,
    on="user_id",
    how="left"
)

interest_base["unique_content_count"] = (
    interest_base["unique_content_count"].fillna(0).astype(int)
)

# 관심 형성 그룹
interest_base["콘텐츠 탐색 그룹"] = np.select(
    [
        interest_base["unique_content_count"] == 0,
        interest_base["unique_content_count"] == 1,
        interest_base["unique_content_count"] >= 2
    ],
    [
        "0개",
        "1개",
        "2개 이상"
    ],
    default="기타"
)

# 실제 수강 시작 여부
start_users = (
    matched_start_pairs[["user_id"]]
    .drop_duplicates()
    .assign(has_started_content=1)
)

interest_base = interest_base.merge(
    start_users,
    on="user_id",
    how="left"
)

interest_base["has_started_content"] = (
    interest_base["has_started_content"].fillna(0).astype(int)
)

# 레슨 완료 여부까지 붙이기
lesson_complete_log = pre_logs[
    pre_logs["event_name"] == "complete_lesson"
].copy()

lesson_complete_joined = lesson_complete_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

lesson_complete_joined["days_after_signup"] = (
    lesson_complete_joined["event_time"] - lesson_complete_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

lesson_complete_7d = lesson_complete_joined[
    (lesson_complete_joined["days_after_signup"] >= 0) &
    (lesson_complete_joined["days_after_signup"] < 7)
].copy()

lesson_complete_users = (
    lesson_complete_7d[["user_id"]]
    .drop_duplicates()
    .assign(has_lesson_complete=1)
)

interest_base = interest_base.merge(
    lesson_complete_users,
    on="user_id",
    how="left"
)

interest_base["has_lesson_complete"] = (
    interest_base["has_lesson_complete"].fillna(0).astype(int)
)

interest_summary = (
    interest_base
    .groupby("콘텐츠 탐색 그룹")
    .agg(
        유저수=("user_id", "nunique"),
        실제_수강_시작_유저수=("has_started_content", "sum"),
        레슨_완료_유저수=("has_lesson_complete", "sum")
    )
    .reset_index()
)

interest_summary["수강 시작률"] = (
    interest_summary["실제_수강_시작_유저수"] /
    interest_summary["유저수"] * 100
).round(2)

interest_summary["레슨 완료율"] = (
    interest_summary["레슨_완료_유저수"] /
    interest_summary["유저수"] * 100
).round(2)

interest_summary


In [ ]:
# 3. 콘텐츠 탐색 수는 실제 레슨 진입/완료와 연결되는가?
# 레슨 진입 로그
lesson_enter_log = pre_logs[
    pre_logs["event_name"] == "enter_lesson_page"
].copy()

lesson_enter_joined = lesson_enter_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

lesson_enter_joined["days_after_signup"] = (
    lesson_enter_joined["event_time"] - lesson_enter_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

lesson_enter_7d = lesson_enter_joined[
    (lesson_enter_joined["days_after_signup"] >= 0) &
    (lesson_enter_joined["days_after_signup"] < 7)
].copy()

# 유저별 고유 레슨 진입 수
user_lesson_enter = (
    lesson_enter_7d
    .dropna(subset=["lesson_id"])
    .groupby("user_id")["lesson_id"]
    .nunique()
    .reset_index(name="unique_lesson_enter_count")
)

# 유저별 고유 레슨 완료 수
user_lesson_complete = (
    lesson_complete_7d
    .dropna(subset=["lesson_id"])
    .groupby("user_id")["lesson_id"]
    .nunique()
    .reset_index(name="unique_lesson_complete_count")
)

content_lesson_base = pre_signup[["user_id"]].merge(
    user_unique_content,
    on="user_id",
    how="left"
).merge(
    user_lesson_enter,
    on="user_id",
    how="left"
).merge(
    user_lesson_complete,
    on="user_id",
    how="left"
)

content_lesson_base[["unique_content_count", "unique_lesson_enter_count", "unique_lesson_complete_count"]] = (
    content_lesson_base[["unique_content_count", "unique_lesson_enter_count", "unique_lesson_complete_count"]]
    .fillna(0)
    .astype(int)
)

content_lesson_base["고유 콘텐츠 탐색 수"] = pd.cut(
    content_lesson_base["unique_content_count"],
    bins=[-1, 0, 1, 2, 5, np.inf],
    labels=["0개", "1개", "2개", "3~5개", "6개 이상"]
)

content_lesson_base["has_lesson_enter"] = (
    content_lesson_base["unique_lesson_enter_count"] > 0
).astype(int)

content_lesson_base["has_lesson_complete"] = (
    content_lesson_base["unique_lesson_complete_count"] > 0
).astype(int)

content_lesson_summary = (
    content_lesson_base
    .groupby("고유 콘텐츠 탐색 수", observed=False)
    .agg(
        유저수=("user_id", "nunique"),
        레슨_진입_유저수=("has_lesson_enter", "sum"),
        레슨_완료_유저수=("has_lesson_complete", "sum"),
        평균_고유_레슨_진입수=("unique_lesson_enter_count", "mean"),
        평균_고유_레슨_완료수=("unique_lesson_complete_count", "mean")
    )
    .reset_index()
)

content_lesson_summary["레슨 진입률"] = (
    content_lesson_summary["레슨_진입_유저수"] /
    content_lesson_summary["유저수"] * 100
).round(2)

content_lesson_summary["레슨 완료율"] = (
    content_lesson_summary["레슨_완료_유저수"] /
    content_lesson_summary["유저수"] * 100
).round(2)

content_lesson_summary["평균_고유_레슨_진입수"] = (
    content_lesson_summary["평균_고유_레슨_진입수"].round(2)
)

content_lesson_summary["평균_고유_레슨_완료수"] = (
    content_lesson_summary["평균_고유_레슨_완료수"].round(2)
)

content_lesson_summary


In [ ]:
# 4. 콘텐츠 페이지 진입 후 수강 시작 버튼 클릭 전 이탈 원인은 무엇일까?
# 질문 클릭 이벤트
question_click_log = pre_logs[
    pre_logs["event_name"] == "click_lesson_page_related_question_box"
].copy()

question_click_joined = question_click_log.merge(
    pre_signup,
    on="user_id",
    how="inner"
)

question_click_joined["days_after_signup"] = (
    question_click_joined["event_time"] - question_click_joined["signup_time"]
).dt.total_seconds() / (60 * 60 * 24)

question_click_7d_users = (
    question_click_joined[
        (question_click_joined["days_after_signup"] >= 0) &
        (question_click_joined["days_after_signup"] < 7)
    ][["user_id"]]
    .drop_duplicates()
    .assign(has_question_click=1)
)

# 후기 클릭 유저
review_click_users = (
    review_click_7d_pairs[["user_id"]]
    .drop_duplicates()
    .assign(has_review_click=1)
)

behavior_base = pre_signup[["user_id"]].merge(
    start_users,
    on="user_id",
    how="left"
).merge(
    question_click_7d_users,
    on="user_id",
    how="left"
).merge(
    review_click_users,
    on="user_id",
    how="left"
)

for col in ["has_started_content", "has_question_click", "has_review_click"]:
    behavior_base[col] = behavior_base[col].fillna(0).astype(int)

# 질문 클릭 여부별 수강 시작률
question_behavior_summary = (
    behavior_base
    .groupby("has_question_click")
    .agg(
        유저수=("user_id", "nunique"),
        수강_시작_유저수=("has_started_content", "sum")
    )
    .reset_index()
)

question_behavior_summary["구분"] = question_behavior_summary["has_question_click"].map({
    0: "질문 클릭 없음",
    1: "질문 클릭 있음"
})

question_behavior_summary["수강 시작률"] = (
    question_behavior_summary["수강_시작_유저수"] /
    question_behavior_summary["유저수"] * 100
).round(2)

question_behavior_summary = question_behavior_summary[
    ["구분", "유저수", "수강_시작_유저수", "수강 시작률"]
]

question_behavior_summary


In [ ]:
# 5. 조회 수가 높은 콘텐츠를 첫 추천 콘텐츠로 두는 것이 적절할까?
# 조회 수 상위 10개
view_top10 = table_8.copy()
view_top10["분류"] = "조회 수 상위 10개"

# 수강 시작률 상위 10개: 조회 유저 수 100명 이상 조건
content_conversion = view_by_content.merge(
    start_by_content,
    on="content_id",
    how="left"
)

content_conversion["수강_시작_유저_수"] = (
    content_conversion["수강_시작_유저_수"].fillna(0).astype(int)
)

content_conversion["수강_시작률"] = (
    content_conversion["수강_시작_유저_수"] /
    content_conversion["조회_유저_수"] * 100
).round(2)

start_rate_top10 = (
    content_conversion[content_conversion["조회_유저_수"] >= 100]
    .sort_values(["수강_시작률", "조회_유저_수"], ascending=False)
    .head(10)
    .reset_index(drop=True)
)

start_rate_top10["분류"] = "수강 시작률 상위 10개"

# 조회 수 상위 10개에 수강 시작률 순위 붙이기
content_conversion["조회수_순위"] = (
    content_conversion["조회_유저_수"]
    .rank(method="min", ascending=False)
)

content_conversion["수강시작률_순위"] = (
    content_conversion["수강_시작률"]
    .rank(method="min", ascending=False)
)

recommend_check = content_conversion[
    content_conversion["content_id"].isin(table_8["content_id"])
].sort_values("조회수_순위")

recommend_check[
    ["content_id", "조회_유저_수", "수강_시작_유저_수", "수강_시작률", "조회수_순위", "수강시작률_순위"]
].rename(columns={
    "content_id": "콘텐츠 아이디",
    "조회_유저_수": "조회 유저 수",
    "수강_시작_유저_수": "수강 시작 유저 수",
    "수강_시작률": "수강 시작률",
    "조회수_순위": "조회 수 순위",
    "수강시작률_순위": "수강 시작률 순위"
})


In [ ]:
# 6. 콘텐츠 탐색 0개인데 레슨 진입한 유저는 어떤 경로인가?
zero_content_lesson_users = content_lesson_base[
    (content_lesson_base["unique_content_count"] == 0) &
    (content_lesson_base["has_lesson_enter"] == 1)
]["user_id"].unique()

print("콘텐츠 탐색 0개 + 레슨 진입 유저 수:", len(zero_content_lesson_users))

zero_lesson_enter_logs = lesson_enter_7d[
    lesson_enter_7d["user_id"].isin(zero_content_lesson_users)
].copy()

# 콘텐츠 아이디, 레슨 아이디 존재 여부 확인
zero_lesson_check = (
    zero_lesson_enter_logs
    .assign(
        has_content_id=zero_lesson_enter_logs["content_id"].notna(),
        has_lesson_id=zero_lesson_enter_logs["lesson_id"].notna()
    )
    .groupby(["has_content_id", "has_lesson_id"])
    .size()
    .reset_index(name="row_count")
)

zero_lesson_check


## 시각화


# 파트 6. 발표/보고서용 최종 시각화



In [ ]:
# 1. 고유 콘텐츠 탐색 수별 레슨 진입률 / 레슨 완료율
content_lesson_df = pd.DataFrame({
    "고유 콘텐츠 탐색 수": ["0개", "1개", "2개", "3~5개", "6개 이상"],
    "유저수": [19255, 4542, 1331, 1482, 933],
    "레슨 진입률": [33.21, 48.70, 62.13, 73.89, 87.14],
    "레슨 완료율": [0.74, 10.81, 32.38, 51.69, 74.49],
    "평균 고유 레슨 진입수": [1.08, 3.33, 11.55, 27.36, 78.77],
    "평균 고유 레슨 완료수": [0.28, 1.78, 8.99, 23.38, 66.09]
})

plt.figure(figsize=(10, 6))
plt.plot(content_lesson_df["고유 콘텐츠 탐색 수"], content_lesson_df["레슨 진입률"], marker="o", label="레슨 진입률")
plt.plot(content_lesson_df["고유 콘텐츠 탐색 수"], content_lesson_df["레슨 완료율"], marker="o", label="레슨 완료율")
plt.xlabel("고유 콘텐츠 탐색 수")
plt.ylabel("비율(%)")
plt.title("고유 콘텐츠 탐색 수별 레슨 진입률 / 완료율")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


- 고유 콘텐츠 탐색 수가 많아질수록 레슨 진입률과 레슨 완료율이 모두 증가하는 경향이 나타났다.  
- 특히 1개 탐색 유저의 레슨 완료율은 10.81%였지만, 2개 탐색 유저는 32.38%, 3~5개 탐색 유저는 51.69%, 6개 이상 탐색 유저는 74.49%까지 상승했다.  
- 이를 통해 여러 콘텐츠를 탐색한 유저일수록 실제 학습 행동으로 이어질 가능성이 높다는 점을 확인할 수 있었다.


In [ ]:
# 2. 고유 콘텐츠 탐색 수별 평균 고유 레슨 진입수 / 완료수
plt.figure(figsize=(10, 6))
plt.plot(content_lesson_df["고유 콘텐츠 탐색 수"], content_lesson_df["평균 고유 레슨 진입수"], marker="o", label="평균 고유 레슨 진입수")
plt.plot(content_lesson_df["고유 콘텐츠 탐색 수"], content_lesson_df["평균 고유 레슨 완료수"], marker="o", label="평균 고유 레슨 완료수")
plt.xlabel("고유 콘텐츠 탐색 수")
plt.ylabel("평균 개수")
plt.title("고유 콘텐츠 탐색 수별 평균 고유 레슨 진입수 / 완료수")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


- 고유 콘텐츠 탐색 수가 증가할수록 평균 고유 레슨 진입수와 평균 고유 레슨 완료수도 함께 증가했다.  
- 특히 6개 이상 콘텐츠를 탐색한 유저는 평균 66.09개의 레슨을 완료해, 1개 콘텐츠 탐색 유저의 1.78개보다 매우 높게 나타났다.  
- 이는 콘텐츠를 많이 탐색한 유저가 단순 조회에 그치지 않고 실제 학습을 깊게 수행하는 경향이 있음을 보여준다.


In [ ]:
# 3. 콘텐츠 페이지 진입 여부별 수강 시작률 / 30일 내 결제율
entry_activation_df = pd.DataFrame({
    "구분": ["콘텐츠 페이지 미진입", "콘텐츠 페이지 진입"],
    "유저수": [19255, 8288],
    "수강 시작률": [0.00, 13.82],
    "30일 내 결제율": [2.57, 27.50]
})

x = range(len(entry_activation_df))

plt.figure(figsize=(8, 6))
plt.bar([i - 0.15 for i in x], entry_activation_df["수강 시작률"], width=0.3, label="수강 시작률")
plt.bar([i + 0.15 for i in x], entry_activation_df["30일 내 결제율"], width=0.3, label="30일 내 결제율")
plt.xticks(x, entry_activation_df["구분"])
plt.ylabel("비율(%)")
plt.title("콘텐츠 페이지 진입 여부별 수강 시작률 / 30일 내 결제율")
plt.legend()
plt.show()


- 콘텐츠 페이지에 진입한 유저는 미진입 유저보다 수강 시작률과 30일 내 결제율이 모두 높게 나타났다.  
- 특히 콘텐츠 페이지 진입 유저의 30일 내 결제율은 27.50%로, 미진입 유저의 2.57%보다 큰 차이를 보였다.  
- 따라서 콘텐츠 페이지 진입은 단순 조회 행동이 아니라, 이후 학습 및 결제와 연결되는 초기 활성화 신호로 해석할 수 있다.


In [ ]:
# 4. 0개 / 1개 / 2개 이상 콘텐츠 탐색 그룹별 수강 시작률 / 레슨 완료율
interest_group_df = pd.DataFrame({
    "콘텐츠 탐색 그룹": ["0개", "1개", "2개 이상"],
    "유저수": [19255, 4542, 3746],
    "수강 시작률": [0.00, 5.37, 24.05],
    "레슨 완료율": [0.74, 10.81, 50.51]
})

x = range(len(interest_group_df))

plt.figure(figsize=(8, 6))
plt.bar([i - 0.15 for i in x], interest_group_df["수강 시작률"], width=0.3, label="수강 시작률")
plt.bar([i + 0.15 for i in x], interest_group_df["레슨 완료율"], width=0.3, label="레슨 완료율")
plt.xticks(x, interest_group_df["콘텐츠 탐색 그룹"])
plt.ylabel("비율(%)")
plt.title("콘텐츠 탐색 그룹별 수강 시작률 / 레슨 완료율")
plt.legend()
plt.show()


- 고유 콘텐츠를 2개 이상 탐색한 유저는 1개만 탐색한 유저보다 수강 시작률과 레슨 완료율이 모두 높았다.  
- 1개 탐색 유저의 수강 시작률은 5.37%, 레슨 완료율은 10.81%였지만, 2개 이상 탐색 유저는 수강 시작률 24.05%, 레슨 완료율 50.51%로 나타났다.  
- 따라서 2개 이상 콘텐츠 탐색은 신규 유저의 관심 형성 및 학습 전환 가능성을 보여주는 기준으로 활용할 수 있다.


In [ ]:
# 5. 질문 클릭 여부별 수강 시작률
question_click_df = pd.DataFrame({
    "구분": ["질문 클릭 없음", "질문 클릭 있음"],
    "유저수": [23713, 3830],
    "수강 시작률": [1.55, 20.29]
})

plt.figure(figsize=(8, 6))
plt.bar(question_click_df["구분"], question_click_df["수강 시작률"])
plt.ylabel("수강 시작률(%)")
plt.title("질문 클릭 여부별 수강 시작률")
plt.show()


- 질문 클릭이 있는 유저의 수강 시작률은 20.29%로, 질문 클릭이 없는 유저의 1.55%보다 높게 나타났다.  
- 이는 질문 클릭 행동이 단순 부가 행동이 아니라, 콘텐츠나 레슨에 대한 관심이 높아진 유저에게서 나타나는 신호일 가능성을 보여준다.  
- 다만 질문 클릭이 수강 시작을 직접 유도했다고 단정하기보다는, 관심도 높은 유저의 행동 신호로 해석하는 것이 적절하다.


In [ ]:
#6. 조회 유저 수 상위 콘텐츠 10개
top_content_df = pd.DataFrame({
    "콘텐츠 아이디": [
        "f4be1a...", "61b646...", "f49151...", "68a365...", "7cebec...",
        "c269eb...", "afe201...", "b314d5...", "595cb4...", "ff7998..."
    ],
    "조회 유저 수": [2970, 1855, 1071, 1040, 956, 892, 854, 699, 667, 629],
    "수강 시작률": [13.10, 25.12, 22.13, 16.35, 16.63, 11.10, 14.64, 8.44, 15.59, 11.45]
})

plt.figure(figsize=(12, 6))
plt.bar(top_content_df["콘텐츠 아이디"], top_content_df["조회 유저 수"])
plt.xticks(rotation=45)
plt.ylabel("조회 유저 수")
plt.title("조회 유저 수 상위 콘텐츠 10개")
plt.show()


- 무료체험 폐지 이전 신규 유저가 가입 후 7일 내 가장 많이 조회한 콘텐츠는 콘텐츠 아이디 기준 `f4be1a...`, `61b646...`, `f49151...` 순으로 나타났다.  
- 특정 콘텐츠에 신규 유저의 관심이 집중되는 경향이 있었으며, 상위 몇 개 콘텐츠에 조회가 몰리는 패턴을 확인할 수 있었다.  
- 다만 많이 조회된 콘텐츠가 실제 수강 시작으로도 잘 이어지는지는 별도로 확인할 필요가 있다.


In [ ]:
# 7. 조회 유저 수와 수강 시작률 비교
fig, ax1 = plt.subplots(figsize=(12, 6))

# 막대: 조회 유저 수
bars = ax1.bar(
    top_content_df["콘텐츠 아이디"],
    top_content_df["조회 유저 수"],
    label="조회 유저 수",
    zorder=1
)

ax1.set_ylabel("조회 유저 수")
ax1.set_xlabel("콘텐츠 아이디")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(axis="y", alpha=0.3, zorder=0)

# 선: 수강 시작률
ax2 = ax1.twinx()

line = ax2.plot(
    top_content_df["콘텐츠 아이디"],
    top_content_df["수강 시작률"],
    marker="o",
    linewidth=2.5,
    markersize=7,
    label="수강 시작률",
    zorder=5
)

ax2.set_ylabel("수강 시작률(%)")

# 선 그래프가 막대 뒤로 숨지 않도록 설정
ax2.set_zorder(ax1.get_zorder() + 1)
ax2.patch.set_visible(False)

# 수강 시작률 라벨 표시
for i, v in enumerate(top_content_df["수강 시작률"]):
    ax2.text(
        i,
        v + 0.8,
        f"{v:.2f}%",
        ha="center",
        fontsize=9,
        zorder=6
    )

# 막대 위 조회 유저 수 라벨 표시
for i, v in enumerate(top_content_df["조회 유저 수"]):
    ax1.text(
        i,
        v + 80,
        f"{v:,}",
        ha="center",
        fontsize=9,
        zorder=2
    )

# 범례 합치기
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper right"
)

plt.title("상위 콘텐츠별 조회 유저 수와 수강 시작률 비교")
plt.tight_layout()
plt.savefig("07_상위콘텐츠_조회수_수강시작률.png", dpi=300, bbox_inches="tight")
plt.show()


- 조회 유저 수가 높은 콘텐츠와 수강 시작률이 높은 콘텐츠는 완전히 일치하지 않았다.  
- 예를 들어 조회 유저 수 1위인 `f4be1a...`는 수강 시작률이 13.10%였지만, 조회 유저 수 2위인 `61b646...`은 수강 시작률이 25.12%로 더 높게 나타났다.  
- 이는 신규 유저에게 추천할 콘텐츠를 단순 조회 수 기준으로 선정하기보다, 조회 이후 실제 수강 시작률까지 함께 고려해야 함을 보여준다.


# 최종 인사이트 요약

## 1. 무료체험 폐지 전후 비교

- 무료체험 폐지 전후로 전체 유저의 행동 지표를 비교해, 폐지 이후 전환/학습 행동이 어떻게 달라졌는지 확인할 수 있습니다.
- 이 결과는 프로젝트에서 “무료체험 폐지가 유저 전환 구조에 어떤 영향을 줬는가?”를 설명하는 근거로 사용할 수 있습니다.

## 2. 신규 유저 콘텐츠 탐색 분석

- 폐지 이전 신규 유저의 가입 후 7일 행동을 보면, 콘텐츠 페이지 진입 자체가 초기 활성화의 중요한 기준이 될 수 있습니다.
- 단순 조회 수가 높은 콘텐츠와 실제 수강 시작률이 높은 콘텐츠가 반드시 일치하지 않기 때문에, 첫 추천 콘텐츠는 조회 수만이 아니라 수강 시작률도 함께 봐야 합니다.


# 콘텐츠별 개선 우선순위 점수 수식


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# =================================================
# 0. 파일 불러오기
# =================================================

# 가입/결제 데이터
act = pd.read_csv("dm_user_activation_payment.csv")

# 이벤트 로그 데이터: 2023년 5월~11월
log_files = [
    "event_log_thin_2023_05.csv",
    "event_log_thin_2023_06.csv",
    "event_log_thin_2023_07.csv",
    "event_log_thin_2023_08.csv",
    "event_log_thin_2023_09.csv",
    "event_log_thin_2023_10.csv",
    "event_log_thin_2023_11.csv"
]

log_list = []

for file in log_files:
    print("불러오는 중:", file)

    temp = pd.read_csv(
        file,
        engine="python",
        on_bad_lines="skip",
        encoding="utf-8"
    )

    print(file, temp.shape)
    log_list.append(temp)

log_raw = pd.concat(log_list, ignore_index=True)

# 컬럼명 앞뒤 공백 제거
act.columns = act.columns.str.strip()
log_raw.columns = log_raw.columns.str.strip()

print("\nact 컬럼명:")
print(act.columns.tolist())

print("\nlog 컬럼명:")
print(log_raw.columns.tolist())


# =================================================
# 1. 필수 컬럼명 정리
# =================================================

# 로그 시간 컬럼 자동 찾기
time_col_candidates = [
    "client_event_time",
    "event_time",
    "event_datetime",
    "client_event_datetime",
    "client_event_datetime_kst",
    "timestamp",
    "created_at"
]

time_col = None

for col in time_col_candidates:
    if col in log_raw.columns:
        time_col = col
        break

if time_col is None:
    raise KeyError(f"로그 시간 컬럼을 못 찾음. 현재 로그 컬럼명: {log_raw.columns.tolist()}")

print("\n사용할 로그 시간 컬럼:", time_col)

# 이후 코드에서 client_event_time으로 통일
if time_col != "client_event_time":
    log_raw = log_raw.rename(columns={time_col: "client_event_time"})


# 이벤트명 컬럼 자동 찾기
event_col_candidates = [
    "event_name",
    "event_type",
    "event",
    "name"
]

event_col = None

for col in event_col_candidates:
    if col in log_raw.columns:
        event_col = col
        break

if event_col is None:
    raise KeyError(f"이벤트명 컬럼을 못 찾음. 현재 로그 컬럼명: {log_raw.columns.tolist()}")

print("사용할 이벤트명 컬럼:", event_col)

if event_col != "event_name":
    log_raw = log_raw.rename(columns={event_col: "event_name"})


# 콘텐츠 ID 컬럼 확인
content_id_candidates = [
    "content_id",
    "contentId",
    "content_id ",
    "content"
]

content_id_col = None

for col in content_id_candidates:
    if col in log_raw.columns:
        content_id_col = col
        break

if content_id_col is None:
    raise KeyError(f"content_id 컬럼을 못 찾음. 현재 로그 컬럼명: {log_raw.columns.tolist()}")

print("사용할 콘텐츠 ID 컬럼:", content_id_col)

if content_id_col != "content_id":
    log_raw = log_raw.rename(columns={content_id_col: "content_id"})


# 콘텐츠 난이도 컬럼 확인
difficulty_candidates = [
    "content_difficulty",
    "difficulty",
    "content_level",
    "level"
]

difficulty_col = None

for col in difficulty_candidates:
    if col in log_raw.columns:
        difficulty_col = col
        break

if difficulty_col is None:
    raise KeyError(f"content_difficulty 컬럼을 못 찾음. 현재 로그 컬럼명: {log_raw.columns.tolist()}")

print("사용할 콘텐츠 난이도 컬럼:", difficulty_col)

if difficulty_col != "content_difficulty":
    log_raw = log_raw.rename(columns={difficulty_col: "content_difficulty"})


# =================================================
# 2. 날짜 변환
# =================================================

act["signup_time"] = pd.to_datetime(act["signup_time"], errors="coerce")
act["first_payment_time"] = pd.to_datetime(act["first_payment_time"], errors="coerce")
log_raw["client_event_time"] = pd.to_datetime(log_raw["client_event_time"], errors="coerce")


# =================================================
# 3. 무료체험 폐지 후 가입자 기준
# 2023-05-01 ~ 2023-11-30
# =================================================

post_users = act[
    (act["signup_time"] >= "2023-05-01") &
    (act["signup_time"] < "2023-12-01")
][
    [
        "user_id",
        "signup_time",
        "first_payment_time",
        "paid_30d"
    ]
].copy()

post_users["paid_30d"] = post_users["paid_30d"].fillna(0).astype(int)

print("\n폐지 후 가입자 수:", post_users["user_id"].nunique())


# =================================================
# 4. 가입 후 7일 이내 + 첫 결제 전 로그 필터
# =================================================

log_post = log_raw.merge(
    post_users,
    on="user_id",
    how="inner"
)

log_post_7d = log_post[
    (log_post["client_event_time"] >= log_post["signup_time"]) &
    (log_post["client_event_time"] < log_post["signup_time"] + pd.Timedelta(days=7))
].copy()

df_post_log_7d_prepay = log_post_7d[
    (log_post_7d["first_payment_time"].isna()) |
    (log_post_7d["client_event_time"] < log_post_7d["first_payment_time"])
].copy()

log = df_post_log_7d_prepay.copy()

print("가입 후 7일 & 첫 결제 전 로그 수:", log.shape)
print("가입 후 7일 & 첫 결제 전 고유 유저 수:", log["user_id"].nunique())


# =================================================
# 5. 이벤트명 확인
# =================================================

print("\n주요 이벤트명 확인:")
print(log["event_name"].value_counts().head(30))


# =================================================
# 6. 콘텐츠 시작 유저
# 시작유저수 = start_content 기준
# =================================================

content_start_user = (
    log[
        (log["event_name"] == "start_content") &
        (log["content_id"].notna()) &
        (log["content_difficulty"].notna())
    ][["user_id", "content_id", "content_difficulty"]]
    .drop_duplicates()
)

print("\n콘텐츠 시작 row 수:", len(content_start_user))
print("콘텐츠 시작 고유 유저 수:", content_start_user["user_id"].nunique())


# =================================================
# 7. 레슨 완료 유저
# 레슨완료유저수 = complete_lesson 기준
# =================================================

lesson_complete_user = (
    log[
        (log["event_name"] == "complete_lesson") &
        (log["content_id"].notna())
    ][["user_id", "content_id"]]
    .drop_duplicates()
)

print("레슨 완료 row 수:", len(lesson_complete_user))
print("레슨 완료 고유 유저 수:", lesson_complete_user["user_id"].nunique())


# =================================================
# 8. 시작 유저 기준으로 레슨 완료 여부 붙이기
# =================================================

content_user_base = content_start_user.merge(
    lesson_complete_user.assign(lesson_completed=1),
    on=["user_id", "content_id"],
    how="left"
)

content_user_base["lesson_completed"] = (
    content_user_base["lesson_completed"]
    .fillna(0)
    .astype(int)
)


# =================================================
# 9. 결제 여부 붙이기
# 결제율 분모 = 시작유저수
# =================================================

payment_user = (
    post_users[["user_id", "paid_30d"]]
    .drop_duplicates("user_id")
    .copy()
)

content_user_base = content_user_base.merge(
    payment_user,
    on="user_id",
    how="left"
)

content_user_base["paid_30d"] = (
    content_user_base["paid_30d"]
    .fillna(0)
    .astype(int)
)


# =================================================
# 10. 비기너 콘텐츠만 필터링
# =================================================

print("\ncontent_difficulty 값 확인:")
print(content_user_base["content_difficulty"].value_counts(dropna=False).head(20))

beginner_user_base = content_user_base[
    content_user_base["content_difficulty"]
    .astype(str)
    .str.contains("beginner|초급|입문", case=False, na=False)
].copy()

print("\n비기너 콘텐츠-유저 row 수:", len(beginner_user_base))
print("비기너 콘텐츠 고유 유저 수:", beginner_user_base["user_id"].nunique())


# =================================================
# 11. 콘텐츠별 집계
# =================================================

content_result_lesson = (
    beginner_user_base
    .groupby(["content_id", "content_difficulty"])
    .agg(
        시작유저수=("user_id", "nunique"),
        레슨완료유저수=("lesson_completed", "sum"),
        결제유저수=("paid_30d", "sum")
    )
    .reset_index()
)

content_result_lesson["레슨미완료유저수"] = (
    content_result_lesson["시작유저수"] -
    content_result_lesson["레슨완료유저수"]
)

content_result_lesson["미결제유저수"] = (
    content_result_lesson["시작유저수"] -
    content_result_lesson["결제유저수"]
)


# =================================================
# 12. 비율 계산
# 모든 비율의 분모 = 시작유저수
# =================================================

content_result_lesson["레슨완료율(%)"] = (
    content_result_lesson["레슨완료유저수"] /
    content_result_lesson["시작유저수"] * 100
)

content_result_lesson["레슨미완료율(%)"] = (
    content_result_lesson["레슨미완료유저수"] /
    content_result_lesson["시작유저수"] * 100
)

content_result_lesson["결제율(%)"] = (
    content_result_lesson["결제유저수"] /
    content_result_lesson["시작유저수"] * 100
)

content_result_lesson["미결제율(%)"] = (
    content_result_lesson["미결제유저수"] /
    content_result_lesson["시작유저수"] * 100
)


# =================================================
# 13. 시작유저수 중앙값 이상 필터
# 비율 극단값 방지
# =================================================

median_start_users = content_result_lesson["시작유저수"].median()

score_df = content_result_lesson[
    content_result_lesson["시작유저수"] >= median_start_users
].copy()

print("\n전체 비기너 콘텐츠 수:", len(content_result_lesson))
print("시작유저수 중앙값:", median_start_users)
print("필터 후 콘텐츠 수:", len(score_df))
print("필터 후 시작유저수 최소값:", score_df["시작유저수"].min())


# =================================================
# 14. 최대-최소 정규화
# =================================================

def min_max_scale(series):
    if series.max() == series.min():
        return 0
    return (series - series.min()) / (series.max() - series.min())

score_df["시작유저수_정규화점수"] = min_max_scale(score_df["시작유저수"])

score_df["레슨미완료율_정규화점수"] = min_max_scale(score_df["레슨미완료율(%)"])

score_df["미결제율_정규화점수"] = min_max_scale(score_df["미결제율(%)"])


# =================================================
# 15. 점수 계산
# =================================================

# 1안-1. 온보딩 자체 중요도
# 시작유저수 0.3 + 레슨미완료율 0.5 + 미결제율 0.2
score_df["온보딩중요도점수"] = (
    score_df["시작유저수_정규화점수"] * 0.3 +
    score_df["레슨미완료율_정규화점수"] * 0.5 +
    score_df["미결제율_정규화점수"] * 0.2
) * 100

# 1안-2. 결제 전환 중요도
# 시작유저수 0.3 + 레슨미완료율 0.2 + 미결제율 0.5
score_df["결제전환중요도점수"] = (
    score_df["시작유저수_정규화점수"] * 0.3 +
    score_df["레슨미완료율_정규화점수"] * 0.2 +
    score_df["미결제율_정규화점수"] * 0.5
) * 100


# =================================================
# 16. 소수점 둘째 자리 반올림
# =================================================

round_cols = [
    "레슨완료율(%)",
    "레슨미완료율(%)",
    "결제율(%)",
    "미결제율(%)",
    "시작유저수_정규화점수",
    "레슨미완료율_정규화점수",
    "미결제율_정규화점수",
    "온보딩중요도점수",
    "결제전환중요도점수"
]

score_df[round_cols] = score_df[round_cols].round(2)


# =================================================
# 17. TOP 5 표 생성
# =================================================

onboarding_top5_lesson_median = (
    score_df
    .sort_values("온보딩중요도점수", ascending=False)
    .head(5)
    [
        [
            "content_id",
            "시작유저수",
            "레슨완료유저수",
            "결제유저수",
            "레슨완료율(%)",
            "레슨미완료율(%)",
            "결제율(%)",
            "미결제율(%)",
            "온보딩중요도점수"
        ]
    ]
    .copy()
)

payment_top5_lesson_median = (
    score_df
    .sort_values("결제전환중요도점수", ascending=False)
    .head(5)
    [
        [
            "content_id",
            "시작유저수",
            "레슨완료유저수",
            "결제유저수",
            "레슨완료율(%)",
            "레슨미완료율(%)",
            "결제율(%)",
            "미결제율(%)",
            "결제전환중요도점수"
        ]
    ]
    .copy()
)


# =================================================
# 18. 콘텐츠 ID 줄이기
# =================================================

onboarding_top5_lesson_median["content_id"] = (
    onboarding_top5_lesson_median["content_id"].astype(str).str[:4]
    + "..."
    + onboarding_top5_lesson_median["content_id"].astype(str).str[-2:]
)

payment_top5_lesson_median["content_id"] = (
    payment_top5_lesson_median["content_id"].astype(str).str[:4]
    + "..."
    + payment_top5_lesson_median["content_id"].astype(str).str[-2:]
)

onboarding_top5_lesson_median = onboarding_top5_lesson_median.rename(
    columns={"content_id": "콘텐츠 ID"}
)

payment_top5_lesson_median = payment_top5_lesson_median.rename(
    columns={"content_id": "콘텐츠 ID"}
)


# =================================================
# 19. 일반 표 출력
# =================================================

display(Markdown("## 1안-1. 온보딩 자체 중요도 TOP 5"))
display(onboarding_top5_lesson_median)

display(Markdown("## 1안-2. 결제 전환 중요도 TOP 5"))
display(payment_top5_lesson_median)


# Appendix B. 이지현 팀원 개별 분석 원본

> 본 섹션은 팀원별 개별 분석 산출물을 원본 흐름에 가깝게 보존한 제출용 부록입니다.
> 전체 노트북은 순차 실행을 전제로 하지 않으며, 분석 과정과 산출물 취합을 목적으로 합니다.


# 1\. 라이브러리 및 데이터 불러오기


In [ ]:
import os
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
pass  # 환경 설치 명령 제거. requirements-analysis.txt를 참고하세요.
import koreanize_matplotlib


In [ ]:
# 저장소 정리: Google Drive 자동 연결을 제거했습니다. 로컬 입력 파일을 사용하세요.


In [ ]:
# 저장소 정리: 개인 Drive 경로로 작업 폴더를 바꾸는 동작을 제거했습니다.


In [ ]:
df_user = pd.read_csv("data/private/dm_user_activation_payment.csv")
df_content = pd.read_csv("data/private/dm_content_7d_performance.csv")


In [ ]:
df_202204 = pd.read_csv("event_log_thin_2022_04.csv")
df_202205 = pd.read_csv("event_log_thin_2022_05.csv")
df_202206 = pd.read_csv("event_log_thin_2022_06.csv")
df_202207 = pd.read_csv("event_log_thin_2022_07.csv")
df_202208 = pd.read_csv("event_log_thin_2022_08.csv")
df_202209 = pd.read_csv("event_log_thin_2022_09.csv")
df_202210 = pd.read_csv("event_log_thin_2022_10.csv")
df_202211 = pd.read_csv("event_log_thin_2022_11.csv")
df_202212 = pd.read_csv("event_log_thin_2022_12.csv")

df_202304 = pd.read_csv("event_log_thin_2023_04.csv")
df_202305 = pd.read_csv("event_log_thin_2023_05.csv")
df_202306 = pd.read_csv("event_log_thin_2023_06.csv")
df_202307 = pd.read_csv("event_log_thin_2023_07.csv")
df_202308 = pd.read_csv("event_log_thin_2023_08.csv")
df_202309 = pd.read_csv("event_log_thin_2023_09.csv")
df_202310 = pd.read_csv("event_log_thin_2023_10.csv")
df_202311 = pd.read_csv("event_log_thin_2023_11.csv", low_memory=False, on_bad_lines="skip")
df_202312 = pd.read_csv("event_log_thin_2023_12.csv")


In [ ]:
# 분석 부분에 결과표가 보기 불편한 것 같아서 한국어로 컬럼명 변경하는 코드입니다!
COLUMN_KR_MAP = {
    'user_id': '유저 ID',
    'content_id': '콘텐츠 ID',
    'lesson_id': '레슨 ID',
    'signup_users': '가입 유저 수',
    'lesson_enter_users_7d': '7일 내 레슨 진입 유저 수',
    'lesson_enter_rate_7d': '7일 내 레슨 진입률(%)',
    'lesson_complete_users_7d': '7일 내 레슨 완료 유저 수',
    'lesson_complete_rate_7d': '7일 내 레슨 완료율(%)',
    'total_lesson_enter_cnt_7d': '7일 내 레슨 진입 수',
    'total_lesson_complete_cnt_7d': '7일 내 레슨 완료 수',
    'enter_to_complete_rate': '레슨 진입 수 대비 완료율(%)',
    'lesson_complete_users_after_enter': '레슨 진입 후 완료 유저 수',
    'enter_user_to_complete_rate': '레슨 진입 유저 기준 완료율(%)',
    'lesson_complete_bucket_7d': '7일 내 레슨 완료 수 구간',
    'lesson_group_7d': '7일 내 레슨 완료 그룹',
    'user_cnt': '유저 수',
    'user_rate': '유저 비율(%)',
    'paid_users': '결제 유저 수',
    'unpaid_users': '미결제 유저 수',
    'paid_rate': '결제율(%)',
    'paid_rate_diff': '직전 기준 대비 결제율 차이(%p)',
    'paid_user_share': '결제 유저 기여도(%)',
    'lesson_complete_users': '레슨 완료 유저 수',
    'lesson_complete_cnt': '레슨 완료 수',
    'avg_lesson_complete_cnt': '평균 레슨 완료 수',
    'lesson_complete_rate': '레슨 완료율(%)',
    'avg_unique_lesson_complete': '평균 고유 레슨 완료 수',
    'avg_lesson_enter_cnt': '평균 레슨 진입 수',
    'avg_content_view_cnt': '평균 콘텐츠 조회 수',
    'avg_content_start_cnt': '평균 콘텐츠 시작 수',
    'avg_content_end_cnt': '평균 콘텐츠 완료 수',
    'avg_content_start_cnt_7d': '7일 내 평균 콘텐츠 시작 수',
    'avg_content_end_cnt_7d': '7일 내 평균 콘텐츠 완료 수',
    'question_click_users': '질문 클릭 유저 수',
    'question_click_rate': '질문 클릭률(%)',
    'review_click_users': '리뷰 클릭 유저 수',
    'review_click_rate': '리뷰 클릭률(%)',
    'payment_page_enter_users': '결제 페이지 진입 유저 수',
    'payment_page_enter_rate': '결제 페이지 진입률(%)',
    'question_click_group': '질문 클릭 여부',
    'activation_group': '활성화 그룹',
    'high_active_group': '고활성 그룹',
    'enter_complete_group': '레슨 진입 후 완료 여부',
    'not_complete_segment': '미완료 유형',
    'user_rate_in_not_complete': '미완료 유저 내 비율(%)',
    'first_lesson_enter_users': '첫 레슨 진입 유저 수',
    'same_first_lesson_completed_users': '첫 레슨 완료 유저 수',
    'same_first_lesson_complete_rate': '첫 레슨 완료율(%)',
    'first_lesson_group': '첫 레슨 완료 여부',
    'high_active_users': '7개 이상 고활성 유저 수',
    'high_active_rate_7plus': '7개 이상 고활성 비율(%)',
    'unique_lesson_detail_group': '고유 레슨 완료 수 구간',
    'threshold': 'N개 이상 기준',
    'threshold_num': '기준값',
    'excluded_group': '제외 기준',
    'remaining_7plus_users': '남은 7개 이상 완료 유저 수',
    'difficulty_final': '콘텐츠 난이도',
    'content_difficulty': '콘텐츠 난이도',
    'start_users': '시작 유저 수',
    'completed_users': '완료 유저 수',
    'completed_users_after_start': '시작 후 완료 유저 수',
    'complete_rate': '완료율(%)',
    'complete_after_start_rate': '시작 후 완료율(%)',
    'incomplete_users': '미완료 유저 수',
    'improvement_priority_score': '개선 우선순위 점수',
    'row_cnt': '행 수',
    'content_cnt': '콘텐츠 수',
    'avg_view_users': '평균 조회 유저 수',
    'avg_start_users': '평균 시작 유저 수',
    'avg_view_to_start_rate': '평균 조회 대비 시작률(%)',
    'months_cnt': '집계 월 수',
    'total_view_users': '총 조회 유저 수',
    'total_start_users': '총 시작 유저 수',
    'start_minus_view': '시작 유저 수 - 조회 유저 수',
    'difficulty_path_group': '난이도 이동 유형',
    'avg_unique_content_start': '평균 고유 콘텐츠 시작 수'
}

VALUE_KR_MAP = {
    'clicked_question': '질문 클릭',
    'not_clicked_question': '질문 미클릭',
    '0': '0개',
    '1': '1개',
    '2-3': '2~3개',
    '4+': '4개 이상',
    '4-10': '4~10개',
    '11-30': '11~30개',
    '31+': '31개 이상',
    '11~20': '11~20개',
    '21~50': '21~50개',
    '51~100': '51~100개',
    '101~200': '101~200개',
    '201+': '201개 이상',
}

def make_korean_table(df):
    df_kr = df.copy()
    df_kr = df_kr.rename(columns=COLUMN_KR_MAP)
    return df_kr.replace(VALUE_KR_MAP)


# 2\. 데이터 확인 및 전처리


## 2-1. 데이터 확인


In [ ]:
df_user.shape


In [ ]:
df_user.head()


In [ ]:
df_user.columns


In [ ]:
df_user.info()


In [ ]:
df_content.shape


In [ ]:
df_content.head()


In [ ]:
df_content.columns


In [ ]:
df_2022 = pd.concat([
    df_202205, df_202206, df_202207, df_202208,
    df_202209, df_202210, df_202211
], ignore_index=True)

df_2023 = pd.concat([
    df_202305, df_202306, df_202307, df_202308,
    df_202309, df_202310, df_202311
], ignore_index=True)


In [ ]:
df_2022.shape


In [ ]:
df_2023.shape


## 2-2. 유저 마트 전처리


In [ ]:
print("df_user shape:", df_user.shape)

print("\n컬럼 목록:")
print(df_user.columns.tolist())


In [ ]:
df_user.dtypes


In [ ]:
missing_user = pd.DataFrame({
    'missing_count': df_user.isnull().sum(),
    'missing_ratio': (df_user.isnull().mean() * 100).round(2)
}).sort_values('missing_count', ascending=False)

missing_user


In [ ]:
df_user['paid_30d'].value_counts(dropna=False)


In [ ]:
df_user.groupby('paid_30d')['first_pg_type'].apply(lambda x: x.isnull().sum())


In [ ]:
for col in ['signup_platform', 'signup_device_type']:
    print(f"\n===== {col} =====")
    print(df_user[col].value_counts(dropna=False))


In [ ]:
print("전체 행 수:", len(df_user))
print("user_id 고유 수:", df_user['user_id'].nunique())
print("user_id 중복 행 수:", df_user['user_id'].duplicated().sum())


In [ ]:
print("missing:", df_user['first_payment_time'].isnull().sum())

print("\n값이 있는 first_payment_time 샘플:")
print(df_user.loc[df_user['first_payment_time'].notnull(), 'first_payment_time'].head(10))

print("\n값의 타입 분포:")
print(df_user['first_payment_time'].map(type).value_counts())


In [ ]:
date_cols = ['signup_time', 'cohort_month', 'first_payment_time']

for col in date_cols:
    df_user[col] = pd.to_datetime(df_user[col], errors='coerce')


In [ ]:
df_user[['signup_time', 'cohort_month', 'first_payment_time']].dtypes


In [ ]:
for col in ['signup_time', 'cohort_month', 'first_payment_time']:
    print(f"\n===== {col} =====")
    print("min:", df_user[col].min())
    print("max:", df_user[col].max())
    print("missing:", df_user[col].isnull().sum())


In [ ]:
df_user.loc[df_user['paid_30d'] == 1, 'days_to_payment'].describe()


In [ ]:
payment_check = df_user[df_user['paid_30d'] == 1].copy()

payment_check['calc_days_to_payment'] = (
    payment_check['first_payment_time'] - payment_check['signup_time']
).dt.days

payment_check[['days_to_payment', 'calc_days_to_payment']].head(10)


In [ ]:
(payment_check['days_to_payment'] != payment_check['calc_days_to_payment']).sum()


In [ ]:
for col in ['period_group', 'signup_type']:
    print(f"\n===== {col} =====")
    print(df_user[col].value_counts(dropna=False))


In [ ]:
behavior_cols = [
    'content_view_7d',
    'unique_content_view_7d',
    'content_start_7d',
    'unique_content_start_7d',
    'lesson_enter_7d',
    'unique_lesson_enter_7d',
    'lesson_complete_7d',
    'unique_lesson_complete_7d',
    'content_end_7d',
    'unique_content_end_7d',
    'review_click_7d',
    'question_click_7d',
    'payment_page_enter_7d',
    'lesson_complete_prepay_7d',
    'content_end_prepay_7d',
    'unique_content_prepay_7d',
    'lesson_complete_prepay_30d',
    'content_end_prepay_30d',
    'unique_content_prepay_30d'
]

df_user[behavior_cols].agg(['min', 'max']).T


In [ ]:
df_user.sort_values('lesson_enter_7d', ascending=False)[[
    'user_id',
    'signup_time',
    'paid_30d',
    'content_view_7d',
    'content_start_7d',
    'lesson_enter_7d',
    'unique_lesson_enter_7d',
    'lesson_complete_7d',
    'unique_lesson_complete_7d',
    'content_end_7d',
    'question_click_7d'
]].head(10)


In [ ]:
df_user[behavior_cols].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).T


In [ ]:
for col in ['is_activated_lesson_7d', 'is_activated_strong_7d']:
    print(f"\n===== {col} =====")
    print(df_user[col].value_counts(dropna=False))


In [ ]:
(df_user['is_activated_lesson_7d'] != df_user['is_activated_strong_7d']).sum()


In [ ]:
df_user[['first_plan_price', 'first_paid_amount', 'first_discount_amount']].describe().T


In [ ]:
pay_check = df_user[df_user['paid_30d'] == 1].copy()

pay_check['calc_paid_amount'] = (
    pay_check['first_plan_price'] - pay_check['first_discount_amount']
)

(pay_check['calc_paid_amount'] != pay_check['first_paid_amount']).sum()


In [ ]:
df_user['signup_device_type'].value_counts(dropna=False).head(30)


In [ ]:
def device_group(x):
    if pd.isna(x):
        return 'unknown'

    x = str(x).lower()

    if 'windows' in x:
        return 'Windows'
    elif 'mac' in x:
        return 'Mac'
    elif 'iphone' in x:
        return 'iPhone'
    elif 'ipad' in x:
        return 'iPad'
    elif 'android' in x:
        return 'Android'
    elif 'samsung' in x or 'galaxy' in x or 'lge' in x or 'huawei' in x or 'xiaomi' in x:
        return 'Android'
    elif 'linux' in x:
        return 'Linux'
    else:
        return 'Other'

df_user['signup_device_group'] = df_user['signup_device_type'].apply(device_group)


In [ ]:
df_user['signup_device_group'].value_counts(dropna=False)


In [ ]:
df_user['signup_platform_group'] = df_user['signup_platform'].fillna('unknown')


In [ ]:
df_user['signup_platform_group'].value_counts(dropna=False)


In [ ]:
df_user['first_pg_type'].value_counts(dropna=False)


In [ ]:
df_user[['signup_device_type', 'signup_device_group', 'signup_platform', 'signup_platform_group']].head()


In [ ]:
print("df_user shape:", df_user.shape)

print("\n데이터 타입:")
print(df_user.dtypes)

print("\n추가된 그룹 컬럼 분포:")
print(df_user['signup_device_group'].value_counts(dropna=False))
print()
print(df_user['signup_platform_group'].value_counts(dropna=False))


## 2-3. 콘텐츠 마트 전처리


In [ ]:
print("df_content shape:", df_content.shape)

print("\n컬럼 목록:")
print(df_content.columns.tolist())


In [ ]:
df_content.dtypes


In [ ]:
print("period_group 분포")
print(df_content['period_group'].value_counts(dropna=False))

print("\ncohort_month 범위")
print("min:", df_content['cohort_month'].min())
print("max:", df_content['cohort_month'].max())
print("missing:", df_content['cohort_month'].isnull().sum())

print("\ncohort_month 샘플")
print(df_content['cohort_month'].head())


In [ ]:
df_content['cohort_month'] = pd.to_datetime(df_content['cohort_month'], errors='coerce')


In [ ]:
df_content[['cohort_month']].dtypes


In [ ]:
df_content.isnull().sum().sort_values(ascending=False)


In [ ]:
print("view_to_start_rate 결측 행의 content_view_users_7d 분포")
print(df_content.loc[df_content['view_to_start_rate'].isnull(), 'content_view_users_7d'].value_counts(dropna=False))

print("\nstart_to_end_rate 결측 행의 content_start_users_7d 분포")
print(df_content.loc[df_content['start_to_end_rate'].isnull(), 'content_start_users_7d'].value_counts(dropna=False))


In [ ]:
print("전체 행 수:", len(df_content))
print("content_id 고유 수:", df_content['content_id'].nunique())
print("content_id 중복 행 수:", df_content['content_id'].duplicated().sum())

print("\nperiod_group + cohort_month + content_id 기준 중복 행 수:")
print(df_content.duplicated(subset=['period_group', 'cohort_month', 'content_id']).sum())


In [ ]:
df_content[['view_to_start_rate', 'start_to_end_rate', 'paid_30d_rate']].describe().T


In [ ]:
print("view_to_start_rate > 100 행 수:")
print((df_content['view_to_start_rate'] > 100).sum())

print("\nstart_to_end_rate > 100 행 수:")
print((df_content['start_to_end_rate'] > 100).sum())


In [ ]:
df_content[
    (df_content['view_to_start_rate'] > 100) |
    (df_content['start_to_end_rate'] > 100)
][[
    'period_group',
    'cohort_month',
    'content_id',
    'content_view_users_7d',
    'content_start_users_7d',
    'content_end_users_7d',
    'view_to_start_rate',
    'start_to_end_rate',
    'paid_30d_users',
    'paid_30d_rate'
]].sort_values(['cohort_month', 'content_id'])


In [ ]:
content_rate_check = df_content.copy()

content_rate_check['calc_view_to_start_rate'] = (
    content_rate_check['content_start_users_7d']
    / content_rate_check['content_view_users_7d']
    * 100
)

content_rate_check['calc_start_to_end_rate'] = (
    content_rate_check['content_end_users_7d']
    / content_rate_check['content_start_users_7d']
    * 100
)

content_rate_check['view_to_start_diff'] = (
    content_rate_check['view_to_start_rate']
    - content_rate_check['calc_view_to_start_rate']
).round(2)

content_rate_check['start_to_end_diff'] = (
    content_rate_check['start_to_end_rate']
    - content_rate_check['calc_start_to_end_rate']
).round(2)

print("view_to_start_rate 불일치 행 수:")
print((content_rate_check['view_to_start_diff'].abs() > 0.01).sum())

print("\nstart_to_end_rate 불일치 행 수:")
print((content_rate_check['start_to_end_diff'].abs() > 0.01).sum())


일부 비율이 100%를 초과하지만 계산식 자체는 일치함.
이는 조회 없이 시작, 시작 없이 종료 이벤트가 잡힌 구조 때문일 수 있어 퍼널 전환율로 강하게 해석하지 않음


## 2-4. 이벤트 로그 전처리 및 7일 기준 검증


In [ ]:
print("df_2022 shape:", df_2022.shape)
print("df_2023 shape:", df_2023.shape)

print("\ndf_2022 columns:")
print(df_2022.columns.tolist())

print("\ndf_2023 columns:")
print(df_2023.columns.tolist())


In [ ]:
print("df_2022 dtypes")
print(df_2022.dtypes)

print("\n====================\n")

print("df_2023 dtypes")
print(df_2023.dtypes)


In [ ]:
for name, df in [('df_2022', df_2022), ('df_2023', df_2023)]:
    print(f"\n===== {name} =====")

    for col in ['cohort_month', 'event_month', 'event_time']:
        print(f"\n--- {col} ---")
        print("missing:", df[col].isnull().sum())
        print("sample:")
        print(df[col].head())


In [ ]:
date_cols = ['cohort_month', 'event_month', 'event_time']

for col in date_cols:
    df_2022[col] = pd.to_datetime(df_2022[col], errors='coerce')
    df_2023[col] = pd.to_datetime(df_2023[col], errors='coerce')


In [ ]:
print("df_2022 날짜 타입")
print(df_2022[['cohort_month', 'event_month', 'event_time']].dtypes)

print("\ndf_2023 날짜 타입")
print(df_2023[['cohort_month', 'event_month', 'event_time']].dtypes)


In [ ]:
for name, df in [('df_2022', df_2022), ('df_2023', df_2023)]:
    print(f"\n===== {name} =====")

    for col in ['cohort_month', 'event_month', 'event_time']:
        print(f"\n--- {col} ---")
        print("min:", df[col].min())
        print("max:", df[col].max())
        print("missing:", df[col].isnull().sum())


In [ ]:
for col in ['pg_type', 'is_trial', 'is_free_trial']:
    print(f"\n================ {col} ================")

    print("\n[df_2022]")
    print(df_2022[col].value_counts(dropna=False).head(20))

    print("\n[df_2023]")
    print(df_2023[col].value_counts(dropna=False).head(20))


In [ ]:
for col in ['event_name', 'source_table']:
    print(f"\n================ {col} ================")

    print("\n[df_2022]")
    print(df_2022[col].value_counts(dropna=False).head(30))

    print("\n[df_2023]")
    print(df_2023[col].value_counts(dropna=False).head(30))


In [ ]:
print("df_2022 결측치")
print(df_2022.isnull().sum().sort_values(ascending=False))

print("\n====================\n")

print("df_2023 결측치")
print(df_2023.isnull().sum().sort_values(ascending=False))


In [ ]:
df_log = pd.concat([df_2022, df_2023], ignore_index=True)

print("df_log shape:", df_log.shape)
print(df_log[['event_time', 'event_month', 'cohort_month']].dtypes)


In [ ]:
df_log['event_name'].value_counts()


In [ ]:
print("전체 로그 행 수:", len(df_log))
print("user_id 결측 수:", df_log['user_id'].isnull().sum())
print("user_id 고유 수:", df_log['user_id'].nunique())


In [ ]:
user_ids_from_user = set(df_user['user_id'])
user_ids_from_log = set(df_log['user_id'])

print("df_user에는 있는데 df_log에는 없는 user_id 수:", len(user_ids_from_user - user_ids_from_log))
print("df_log에는 있는데 df_user에는 없는 user_id 수:", len(user_ids_from_log - user_ids_from_user))


In [ ]:
print("전체 로그 행 수:", len(df_log))
print("id 고유 수:", df_log['id'].nunique())
print("id 중복 행 수:", df_log['id'].duplicated().sum())


In [ ]:
df_log = df_log.merge(
    df_user[['user_id', 'signup_time']],
    on='user_id',
    how='left'
)

print("df_log shape:", df_log.shape)
print("signup_time 결측 수:", df_log['signup_time'].isnull().sum())
print(df_log[['user_id', 'event_time', 'signup_time']].head())


In [ ]:
df_log['is_within_7d'] = (
    (df_log['event_time'] >= df_log['signup_time']) &
    (df_log['event_time'] < df_log['signup_time'] + pd.Timedelta(days=7))
)

df_log['days_after_signup'] = (
    df_log['event_time'] - df_log['signup_time']
).dt.days


In [ ]:
print(df_log['is_within_7d'].value_counts(dropna=False))

print("\ndays_after_signup 범위:")
print("min:", df_log['days_after_signup'].min())
print("max:", df_log['days_after_signup'].max())


In [ ]:
df_log[df_log['days_after_signup'] < 0]['event_name'].value_counts()


In [ ]:
pre_signup_log = df_log[df_log['event_time'] < df_log['signup_time']]

print("가입 전 이벤트 행 수:", len(pre_signup_log))
print("가입 전 이벤트 발생 유저 수:", pre_signup_log['user_id'].nunique())


In [ ]:
df_log_7d = df_log[df_log['is_within_7d']].copy()

print("df_log 전체 행 수:", len(df_log))
print("df_log_7d 행 수:", len(df_log_7d))
print("df_log_7d 유저 수:", df_log_7d['user_id'].nunique())

print("\n7일 내 이벤트 분포:")
print(df_log_7d['event_name'].value_counts())


In [ ]:
event_user_summary_7d = (
    df_log_7d
    .groupby('event_name')
    .agg(
        event_count=('id', 'count'),
        user_count=('user_id', 'nunique')
    )
    .sort_values('event_count', ascending=False)
)

event_user_summary_7d


In [ ]:
print("df_user lesson 활성화 유저 수:")
print(df_user['is_activated_lesson_7d'].sum())

print("\ndf_log_7d complete_lesson 유저 수:")
print(df_log_7d.loc[df_log_7d['event_name'] == 'complete_lesson', 'user_id'].nunique())


In [ ]:
print("is_activated_lesson_7d = 1 유저 수")
print(df_user['is_activated_lesson_7d'].sum())

print("\nlesson_complete_7d > 0 유저 수")
print((df_user['lesson_complete_7d'] > 0).sum())

print("\nunique_lesson_complete_7d > 0 유저 수")
print((df_user['unique_lesson_complete_7d'] > 0).sum())

print("\nlesson_enter_7d > 0 유저 수")
print((df_user['lesson_enter_7d'] > 0).sum())


In [ ]:
complete_lesson_users_by_daydiff = df_log[
    (df_log['event_name'] == 'complete_lesson') &
    (df_log['days_after_signup'] >= 0) &
    (df_log['days_after_signup'] <= 7)
]['user_id'].nunique()

print("days_after_signup 0~7 기준 complete_lesson 유저 수:")
print(complete_lesson_users_by_daydiff)

print("\ndf_user lesson_complete_7d > 0 유저 수:")
print((df_user['lesson_complete_7d'] > 0).sum())


In [ ]:
# df_user 기준 lesson_complete_7d > 0 유저
user_complete_from_user = set(
    df_user.loc[df_user['lesson_complete_7d'] > 0, 'user_id']
)

# df_log 기준: 가입 후 168시간 이내 complete_lesson 유저
user_complete_from_log_168h = set(
    df_log_7d.loc[df_log_7d['event_name'] == 'complete_lesson', 'user_id']
)

print("df_user에는 있는데 df_log_168h에는 없는 유저 수:")
print(len(user_complete_from_user - user_complete_from_log_168h))

print("\ndf_log_168h에는 있는데 df_user에는 없는 유저 수:")
print(len(user_complete_from_log_168h - user_complete_from_user))


In [ ]:
diff_users = user_complete_from_user - user_complete_from_log_168h

diff_complete_logs = df_log[
    (df_log['user_id'].isin(diff_users)) &
    (df_log['event_name'] == 'complete_lesson')
].copy()

diff_complete_logs['hours_after_signup'] = (
    diff_complete_logs['event_time'] - diff_complete_logs['signup_time']
).dt.total_seconds() / 3600

diff_complete_logs[[
    'user_id',
    'signup_time',
    'event_time',
    'days_after_signup',
    'hours_after_signup',
    'cohort_month',
    'event_month'
]].sort_values(['user_id', 'event_time']).head(30)


In [ ]:
# df_user에는 lesson_complete_7d > 0인데 df_log_7d complete_lesson에는 없는 110명

diff_users = user_complete_from_user - user_complete_from_log_168h

print("차이 유저 수:", len(diff_users))

df_log[df_log['user_id'].isin(diff_users)]['event_name'].value_counts()


df_user의 lesson_complete_7d와 df_log의 complete_lesson 이벤트 기준은 완전히 일치하지 않았다.  
다만 차이는 약 0.68% 수준으로 크지 않기 때문에, 전체 경향 분석에는 제한적인 영향으로 판단한다.

따라서 이후 분석에서는 다음 기준을 사용한다.

- 이벤트 흐름 분석: df_log 기준
- 유저 단위 최종 지표 비교: df_user 및 df_log 집계 기준 병행


# 3\. 분석 대상 및 핵심 분석 테이블 생성


## 3-1. 무료체험 폐지 후 신규 유저 정의


In [ ]:
# 무료체험 폐지 후 신규 가입자 정의
post_users = df_user[
    (df_user['signup_time'] >= '2023-05-01') &
    (df_user['signup_time'] < '2023-12-01')
][[
    'user_id',
    'signup_time',
    'first_payment_time',
    'paid_30d'
]].copy()

print("무료체험 폐지 후 신규 가입 유저 수:", post_users['user_id'].nunique())


# 해당 유저의 로그만 추출
df_post_log = df_log[
    df_log['user_id'].isin(post_users['user_id'])
].copy()


# first_payment_time 붙이기
df_post_log = df_post_log.merge(
    post_users[['user_id', 'first_payment_time', 'paid_30d']],
    on='user_id',
    how='left'
)


# 가입 후 7일 내 이벤트 여부 생성
df_post_log['is_within_7d'] = (
    (df_post_log['event_time'] >= df_post_log['signup_time']) &
    (df_post_log['event_time'] < df_post_log['signup_time'] + pd.Timedelta(days=7))
)


# 첫 결제 전 행동 여부 생성, 미결제 유저는 모든 행동 인정, 결제 유저는 first_payment_time 이전 행동만 인정
df_post_log['is_before_first_payment'] = (
    df_post_log['first_payment_time'].isna() |
    (df_post_log['event_time'] < df_post_log['first_payment_time'])
)


# 가입 후 7일 내 + 첫 결제 전 행동 로그
df_post_log_7d_prepay = df_post_log[
    (df_post_log['is_within_7d']) &
    (df_post_log['is_before_first_payment'])
].copy()


# 가입 후 7일 내 전체 로그, complete_subscription 같은 결제 이벤트 확인용
df_post_log_7d = df_post_log[
    df_post_log['is_within_7d']
].copy()


print("df_post_log_7d 행 수:", len(df_post_log_7d))
print("df_post_log_7d_prepay 행 수:", len(df_post_log_7d_prepay))
print("df_post_log_7d 유저 수:", df_post_log_7d['user_id'].nunique())


## 3-2. 유저별 초기 행동 요약 테이블 생성


In [ ]:
# 유저별 7일 내 첫 결제 전 행동 요약 테이블
analysis_user_behavior = post_users[['user_id', 'signup_time', 'first_payment_time', 'paid_30d']].copy()

# 이벤트 수 집계
event_count_table = (
    df_post_log_7d_prepay
    .groupby(['user_id', 'event_name'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

analysis_user_behavior = analysis_user_behavior.merge(
    event_count_table,
    on='user_id',
    how='left'
)

# 없는 이벤트 컬럼은 0으로 생성
event_cols = [
    'enter_content_page',
    'start_content',
    'end_content',
    'enter_lesson_page',
    'complete_lesson',
    'click_lesson_page_related_question_box',
    'click_content_page_more_review_button',
    'enter_payment_page',
    'complete_subscription'
]

for col in event_cols:
    if col not in analysis_user_behavior.columns:
        analysis_user_behavior[col] = 0

# NaN -> 0
for col in event_cols:
    analysis_user_behavior[col] = analysis_user_behavior[col].fillna(0).astype(int)

# 고유 레슨 완료 수
unique_lesson_complete = (
    df_post_log_7d_prepay[
        (df_post_log_7d_prepay['event_name'] == 'complete_lesson') &
        (df_post_log_7d_prepay['lesson_id'].notnull())
    ]
    .groupby('user_id')['lesson_id']
    .nunique()
    .reset_index(name='unique_lesson_complete_cnt_7d_prepay')
)

analysis_user_behavior = analysis_user_behavior.merge(
    unique_lesson_complete,
    on='user_id',
    how='left'
)

analysis_user_behavior['unique_lesson_complete_cnt_7d_prepay'] = (
    analysis_user_behavior['unique_lesson_complete_cnt_7d_prepay']
    .fillna(0)
    .astype(int)
)

# 고유 콘텐츠 시작 수
unique_content_start = (
    df_post_log_7d_prepay[
        (df_post_log_7d_prepay['event_name'] == 'start_content') &
        (df_post_log_7d_prepay['content_id'].notnull())
    ]
    .groupby('user_id')['content_id']
    .nunique()
    .reset_index(name='unique_content_start_cnt_7d_prepay')
)

analysis_user_behavior = analysis_user_behavior.merge(
    unique_content_start,
    on='user_id',
    how='left'
)

analysis_user_behavior['unique_content_start_cnt_7d_prepay'] = (
    analysis_user_behavior['unique_content_start_cnt_7d_prepay']
    .fillna(0)
    .astype(int)
)

# 여부 컬럼 생성
analysis_user_behavior['has_lesson_enter'] = (analysis_user_behavior['enter_lesson_page'] > 0).astype(int)
analysis_user_behavior['has_lesson_complete'] = (analysis_user_behavior['unique_lesson_complete_cnt_7d_prepay'] > 0).astype(int)
analysis_user_behavior['has_content_view'] = (analysis_user_behavior['enter_content_page'] > 0).astype(int)
analysis_user_behavior['has_content_start'] = (analysis_user_behavior['start_content'] > 0).astype(int)
analysis_user_behavior['has_content_end'] = (analysis_user_behavior['end_content'] > 0).astype(int)
analysis_user_behavior['has_question_click'] = (analysis_user_behavior['click_lesson_page_related_question_box'] > 0).astype(int)
analysis_user_behavior['has_review_click'] = (analysis_user_behavior['click_content_page_more_review_button'] > 0).astype(int)
analysis_user_behavior['has_payment_page_enter'] = (analysis_user_behavior['enter_payment_page'] > 0).astype(int)

print("analysis_user_behavior shape:", analysis_user_behavior.shape)
analysis_user_behavior.head()


## 3-3. 유저별 콘텐츠 난이도 시작 순서 생성


In [ ]:
difficulty_sequence_logs = df_post_log_7d_prepay[
    (df_post_log_7d_prepay['event_name'] == 'start_content') &
    (df_post_log_7d_prepay['content_difficulty'].notnull()) &
    (df_post_log_7d_prepay['content_id'].notnull())
][[
    'user_id',
    'event_time',
    'content_id',
    'content_difficulty'
]].drop_duplicates().copy()

difficulty_sequence_logs = difficulty_sequence_logs.sort_values(
    ['user_id', 'event_time']
)

user_difficulty_sequence = (
    difficulty_sequence_logs
    .groupby('user_id')['content_difficulty']
    .apply(list)
    .reset_index(name='difficulty_sequence')
)

user_difficulty_sequence['first_difficulty'] = user_difficulty_sequence['difficulty_sequence'].apply(
    lambda x: x[0] if len(x) > 0 else np.nan
)

user_difficulty_sequence['used_beginner'] = user_difficulty_sequence['difficulty_sequence'].apply(
    lambda x: 'beginner' in x
)

user_difficulty_sequence['used_advanced'] = user_difficulty_sequence['difficulty_sequence'].apply(
    lambda x: 'advanced' in x
)

user_difficulty_sequence['used_intermediate'] = user_difficulty_sequence['difficulty_sequence'].apply(
    lambda x: 'intermediate' in x
)

user_difficulty_sequence['used_hard'] = user_difficulty_sequence['difficulty_sequence'].apply(
    lambda x: 'hard' in x
)

user_difficulty_sequence.head()


# 4\. 분석


## 4-1. 가입 후 7일 내 레슨 학습 퍼널 진단



### 4-1-1. 무료체험 폐지 후 신규 유저 가입 후 7일 내 레슨 페이지 진입률


In [ ]:
signup_users = post_users['user_id'].nunique()

lesson_enter_users = df_post_log_7d_prepay.loc[
    df_post_log_7d_prepay['event_name'] == 'enter_lesson_page',
    'user_id'
].nunique()

result_1 = pd.DataFrame({
    'signup_users': [signup_users],
    'lesson_enter_users_7d': [lesson_enter_users],
    'lesson_enter_rate_7d': [round(lesson_enter_users / signup_users * 100, 2)]
})

result_1_kr = make_korean_table(result_1)
display(result_1_kr)


- 무료체험 폐지 후 신규 가입자 46,055명 중 32,642명이 가입 후 7일 이내 레슨 페이지에 진입했다.
- 레슨 페이지 진입률은 70.88%로, 가입 직후 많은 유저가 일단 학습 화면까지는 도달하고 있음을 보여준다.
- 즉, 무료체험이 사라졌다고 해서 유저가 학습 화면 자체를 보지 않는 상황은 아니다.
- 다만 레슨 페이지에 들어온 것만으로 실제 학습이 이루어졌다고 보기는 어렵기 때문에, 뒤에서 레슨 완료까지 이어졌는지 확인해야 한다.


### 4-1-2. 무료체험 폐지 후 신규 유저 가입 후 7일 내 레슨 완료율


In [ ]:
lesson_complete_users = df_post_log_7d_prepay.loc[
    df_post_log_7d_prepay['event_name'] == 'complete_lesson',
    'user_id'
].nunique()

result_2 = pd.DataFrame({
    'signup_users': [signup_users],
    'lesson_complete_users_7d': [lesson_complete_users],
    'lesson_complete_rate_7d': [round(lesson_complete_users / signup_users * 100, 2)]
})

result_2_kr = make_korean_table(result_2)
display(result_2_kr)


- 가입 후 7일 이내 실제로 레슨을 완료한 유저는 12,473명이다.
- 전체 신규 가입자 기준 레슨 완료율은 27.08%로, 위의 레슨 페이지 진입률 70.88%와 비교하면 큰 차이가 있다.
- 이는 많은 유저가 레슨 페이지까지는 들어오지만, 실제 완료 경험까지 이어지는 유저는 제한적이라는 의미이다.
- 따라서 무료체험 폐지 이후에는 단순히 레슨 페이지로 유입시키는 것보다 첫 레슨 완료 경험을 만들도록 유도하는 것이 더 중요해 보인다.


### 4-1-3. 레슨 진입 유저 기준 완료율 분석


In [ ]:
lesson_enter_user_set = set(
    df_post_log_7d_prepay.loc[
        df_post_log_7d_prepay['event_name'] == 'enter_lesson_page',
        'user_id'
    ]
)

lesson_complete_user_set = set(
    df_post_log_7d_prepay.loc[
        df_post_log_7d_prepay['event_name'] == 'complete_lesson',
        'user_id'
    ]
)

enter_and_complete_users = lesson_enter_user_set & lesson_complete_user_set

result_3 = pd.DataFrame({
    'lesson_enter_users_7d': [len(lesson_enter_user_set)],
    'lesson_complete_users_after_enter': [len(enter_and_complete_users)],
    'enter_user_to_complete_rate': [
        round(len(enter_and_complete_users) / len(lesson_enter_user_set) * 100, 2)
    ]
})

result_3_kr = make_korean_table(result_3)
display(result_3_kr)


- 레슨 페이지에 진입한 유저 32,642명 중 실제 레슨 완료까지 이어진 유저는 12,177명이다.
- 레슨 진입 유저 기준 완료율은 37.3%이다.
- 즉, 레슨 페이지에 들어온 유저 중 약 3분의 1 정도만 완료 경험을 만든다.
- 이 결과는 레슨 학습 분석에서 중요한 개선 지점이 레슨 진입 이후 완료까지의 구간임을 보여준다.


## 4-2. 레슨 완료 기반 활성화 기준 탐색


### 4-2-1. 최소 활성화 기준과 고활성 기준 산정


In [ ]:
activation_df = analysis_user_behavior.copy()

def activation_group(x):
    if x == 0:
        return '0개 완료'
    elif x < 7:
        return '1~6개 완료'
    else:
        return '7개 이상 완료'

activation_df['activation_group'] = activation_df['unique_lesson_complete_cnt_7d_prepay'].apply(activation_group)

activation_order = ['0개 완료', '1~6개 완료', '7개 이상 완료']

result_activation_group = (
    activation_df
    .groupby('activation_group')
    .agg(
        user_cnt=('user_id', 'count'),
        paid_users=('paid_30d', 'sum'),
        avg_unique_lesson_complete=('unique_lesson_complete_cnt_7d_prepay', 'mean'),
        avg_lesson_enter_cnt=('enter_lesson_page', 'mean'),
        avg_content_start_cnt=('start_content', 'mean'),
        question_click_users=('has_question_click', 'sum'),
        payment_page_enter_users=('has_payment_page_enter', 'sum')
    )
    .reindex(activation_order)
    .reset_index()
)

result_activation_group['paid_rate'] = round(
    result_activation_group['paid_users'] / result_activation_group['user_cnt'] * 100, 2
)

result_activation_group['question_click_rate'] = round(
    result_activation_group['question_click_users'] / result_activation_group['user_cnt'] * 100, 2
)

result_activation_group['payment_page_enter_rate'] = round(
    result_activation_group['payment_page_enter_users'] / result_activation_group['user_cnt'] * 100, 2
)

round_cols = ['avg_unique_lesson_complete', 'avg_lesson_enter_cnt', 'avg_content_start_cnt']

for col in round_cols:
    result_activation_group[col] = result_activation_group[col].round(2)

result_activation_group_kr = make_korean_table(result_activation_group)
display(result_activation_group_kr)

activation_rate_plot = result_activation_group.melt(
    id_vars='activation_group',
    value_vars=['paid_rate', 'question_click_rate', 'payment_page_enter_rate'],
    var_name='지표',
    value_name='비율'
)

activation_rate_plot['지표'] = activation_rate_plot['지표'].replace({
    'paid_rate': '결제율',
    'question_click_rate': '질문 클릭률',
    'payment_page_enter_rate': '결제 페이지 진입률'
})

plt.figure(figsize=(10, 5))
sns.barplot(
    data=activation_rate_plot,
    x='activation_group',
    y='비율',
    hue='지표'
)

plt.title('활성화 수준별 주요 행동 비율')
plt.xlabel('활성화 그룹')
plt.ylabel('비율(%)')
plt.legend(title='')
plt.show()


- 레슨 0개 완료 유저는 33,582명, 결제율은 4.74%이다.
- 1~6개 완료 유저는 6,989명, 결제율은 3.39%이다.
- 7개 이상 완료 유저는 5,484명, 결제율은 17.47%이다.
- 여기서 중요한 점은 1~6개 완료 유저의 결제율이 높지 않다는 것이다.
- 즉, 레슨을 1개라도 완료했다고 해서 바로 결제 가능성이 높아진다고 보기는 어렵다.
- 반면 7개 이상 완료 유저는 질문 클릭률 57.31%, 결제 페이지 진입률 65.57%로 다른 그룹보다 확실히 높다.
- 따라서 이 분석에서는 1개 이상 완료 = 최소 학습 경험, 7개 이상 완료 = 고활성 학습 유저로 분리해서 보는 것이 적절하다고 판단된다.


### 4-2-2. 미완료 원인 추정


In [ ]:
lesson_enter_df = analysis_user_behavior[
    analysis_user_behavior['has_lesson_enter'] == 1
].copy()

lesson_enter_df['enter_complete_group'] = np.where(
    lesson_enter_df['has_lesson_complete'] == 1,
    '진입 후 완료',
    '진입 후 미완료'
)

result_enter_complete_detail = (
    lesson_enter_df
    .groupby('enter_complete_group')
    .agg(
        user_cnt=('user_id', 'count'),
        avg_lesson_enter_cnt=('enter_lesson_page', 'mean'),
        avg_content_view_cnt=('enter_content_page', 'mean'),
        avg_content_start_cnt=('start_content', 'mean'),
        avg_content_end_cnt=('end_content', 'mean'),
        question_click_users=('has_question_click', 'sum'),
        review_click_users=('has_review_click', 'sum'),
        payment_page_enter_users=('has_payment_page_enter', 'sum'),
        paid_users=('paid_30d', 'sum')
    )
    .reset_index()
)

result_enter_complete_detail['question_click_rate'] = round(
    result_enter_complete_detail['question_click_users'] / result_enter_complete_detail['user_cnt'] * 100, 2
)

result_enter_complete_detail['review_click_rate'] = round(
    result_enter_complete_detail['review_click_users'] / result_enter_complete_detail['user_cnt'] * 100, 2
)

result_enter_complete_detail['payment_page_enter_rate'] = round(
    result_enter_complete_detail['payment_page_enter_users'] / result_enter_complete_detail['user_cnt'] * 100, 2
)

result_enter_complete_detail['paid_rate'] = round(
    result_enter_complete_detail['paid_users'] / result_enter_complete_detail['user_cnt'] * 100, 2
)

for col in [
    'avg_lesson_enter_cnt',
    'avg_content_view_cnt',
    'avg_content_start_cnt',
    'avg_content_end_cnt'
]:
    result_enter_complete_detail[col] = result_enter_complete_detail[col].round(2)

result_enter_complete_detail_kr = make_korean_table(result_enter_complete_detail)
display(result_enter_complete_detail_kr)


- 레슨 페이지에 진입했지만 완료하지 못한 유저는 20,465명이다.
- 레슨 진입 후 완료한 유저는 12,177명이다.
- 진입 후 미완료 유저의 결제율은 3.20%, 진입 후 완료 유저의 결제율은 9.62%다.
- 질문 클릭률도 미완료 유저는 4.86%, 완료 유저는 37.93%로 큰 차이가 난다.
- 즉, 레슨 페이지 진입 자체보다 완료 경험을 만들었는지가 학습 몰입과 결제 가능성 차이를 더 잘 설명한다.


### 4-2-3. 미완료 유저 세분화


In [ ]:
not_complete_users = analysis_user_behavior[
    (analysis_user_behavior['has_lesson_enter'] == 1) &
    (analysis_user_behavior['has_lesson_complete'] == 0)
].copy()

def not_complete_segment(row):
    if row['has_content_start'] == 0 and row['has_question_click'] == 0:
        return '진입만 하고 추가 행동 없음'
    elif row['has_content_start'] == 1 and row['has_question_click'] == 0:
        return '콘텐츠 시작은 했지만 질문 클릭 없음'
    elif row['has_content_start'] == 0 and row['has_question_click'] == 1:
        return '질문 클릭은 했지만 콘텐츠 시작 없음'
    else:
        return '콘텐츠 시작 + 질문 클릭'

not_complete_users['not_complete_segment'] = not_complete_users.apply(
    not_complete_segment,
    axis=1
)

result_not_complete_segment = (
    not_complete_users
    .groupby('not_complete_segment')
    .agg(
        user_cnt=('user_id', 'count'),
        avg_lesson_enter_cnt=('enter_lesson_page', 'mean'),
        avg_content_start_cnt=('start_content', 'mean'),
        payment_page_enter_users=('has_payment_page_enter', 'sum'),
        paid_users=('paid_30d', 'sum')
    )
    .reset_index()
)

result_not_complete_segment['user_rate_in_not_complete'] = round(
    result_not_complete_segment['user_cnt'] / len(not_complete_users) * 100, 2
)

result_not_complete_segment['payment_page_enter_rate'] = round(
    result_not_complete_segment['payment_page_enter_users'] / result_not_complete_segment['user_cnt'] * 100, 2
)

result_not_complete_segment['paid_rate'] = round(
    result_not_complete_segment['paid_users'] / result_not_complete_segment['user_cnt'] * 100, 2
)

for col in ['avg_lesson_enter_cnt', 'avg_content_start_cnt']:
    result_not_complete_segment[col] = result_not_complete_segment[col].round(2)

result_not_complete_segment_sorted = result_not_complete_segment.sort_values('user_cnt', ascending=False)

result_not_complete_segment_kr = make_korean_table(result_not_complete_segment.sort_values('user_cnt', ascending=False))
display(result_not_complete_segment_kr)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=result_not_complete_segment_sorted,
    x='not_complete_segment',
    y='user_cnt'
)

plt.title('레슨 진입 후 미완료 유저 세분화')
plt.xlabel('미완료 유형')
plt.ylabel('유저 수')
plt.xticks(rotation=20)

for i, row in result_not_complete_segment_sorted.reset_index(drop=True).iterrows():
    plt.text(i, row['user_cnt'], f"{row['user_cnt']:,}", ha='center', va='bottom')

plt.show()


- 미완료 유저 중 가장 큰 그룹은 진입만 하고 추가 행동이 없는 유저로, 14,217명이며 미완료 유저의 69.47%를 차지한다.
- 이 그룹의 결제율은 2.60%로 낮다.
- 반면 콘텐츠 시작은 했지만 질문 클릭이 없는 유저는 결제율 4.64%, 콘텐츠 시작과 질문 클릭이 모두 있는 유저는 5.30%로 상대적으로 높다.
- 따라서 미완료 유저 안에서도 완전히 이탈한 유저와, 콘텐츠 시작 또는 질문 클릭까지는 한 유저를 구분해야 한다고 생각한다.
- 개선 액션은 전체 미완료 유저에게 동일하게 적용하기보다, 진입만 하고 멈춘 유저에게는 첫 행동 유도, 콘텐츠 시작 후 멈춘 유저에게는 이어하기/힌트 제공처럼 나눠 설계하는 것이 좋다.
- 즉, 레슨을 완료시키는 것도 좋은데 작은 행동 하나하나를 늘리는 것도 학습 몰입과 결제율 개선에 영향이 있을 것이라고 본다.


## 4-3. 첫 레슨 완료 경험의 영향 분석


### 4-3-1. 첫 레슨 완료 경험이 이후 활성화와 결제 전환의 초기 신호가 될 수 있을까?


In [ ]:
lesson_enter_logs = df_post_log_7d_prepay[
    (df_post_log_7d_prepay['event_name'] == 'enter_lesson_page') &
    (df_post_log_7d_prepay['lesson_id'].notnull())
].copy()

lesson_complete_logs = df_post_log_7d_prepay[
    (df_post_log_7d_prepay['event_name'] == 'complete_lesson') &
    (df_post_log_7d_prepay['lesson_id'].notnull())
].copy()

# 유저별 첫 레슨 진입
first_lesson_enter = (
    lesson_enter_logs
    .sort_values('event_time')
    .groupby('user_id')
    .first()
    .reset_index()[['user_id', 'event_time', 'lesson_id', 'content_id']]
    .rename(columns={
        'event_time': 'first_lesson_enter_time',
        'lesson_id': 'first_lesson_id',
        'content_id': 'first_content_id'
    })
)

# 같은 user_id + 같은 lesson_id 완료 매칭
first_lesson_complete_check = first_lesson_enter.merge(
    lesson_complete_logs[['user_id', 'event_time', 'lesson_id', 'content_id']],
    left_on=['user_id', 'first_lesson_id'],
    right_on=['user_id', 'lesson_id'],
    how='left'
)

first_lesson_complete_check['same_first_lesson_completed'] = (
    first_lesson_complete_check['event_time'] >=
    first_lesson_complete_check['first_lesson_enter_time']
)

first_lesson_user_summary = (
    first_lesson_complete_check
    .groupby('user_id')
    .agg(
        first_lesson_enter_time=('first_lesson_enter_time', 'first'),
        first_lesson_id=('first_lesson_id', 'first'),
        first_content_id=('first_content_id', 'first'),
        same_first_lesson_completed=('same_first_lesson_completed', 'max')
    )
    .reset_index()
)

first_lesson_user_summary['same_first_lesson_completed'] = (
    first_lesson_user_summary['same_first_lesson_completed']
    .fillna(False)
    .astype(int)
)

result_first_lesson_base = pd.DataFrame({
    'first_lesson_enter_users': [first_lesson_user_summary['user_id'].nunique()],
    'same_first_lesson_completed_users': [first_lesson_user_summary['same_first_lesson_completed'].sum()]
})

result_first_lesson_base['same_first_lesson_complete_rate'] = round(
    result_first_lesson_base['same_first_lesson_completed_users'] /
    result_first_lesson_base['first_lesson_enter_users'] * 100,
    2
)

result_first_lesson_base_kr = make_korean_table(result_first_lesson_base)
display(result_first_lesson_base_kr)


- 첫 레슨에 진입한 유저는 32,642명이다.
- 이 중 처음 진입한 레슨을 실제로 완료한 유저는 9,961명이다.
- 첫 레슨 완료율은 30.52%이다.
- 즉, 레슨 페이지까지 온 유저 중 약 70%는 첫 레슨을 완료하지 못한다.
- 이 구간은 무료체험 폐지 후 초기 학습 경험을 개선할 때 가장 우선적으로 봐야 하는 구간이라고 본다.


### 4-3-2. 첫 레슨을 완료한 유저와 완료하지 못한 유저 비교


In [ ]:
first_lesson_impact = first_lesson_user_summary.merge(
    analysis_user_behavior,
    on='user_id',
    how='left'
)

first_lesson_impact['first_lesson_group'] = np.where(
    first_lesson_impact['same_first_lesson_completed'] == 1,
    '첫 레슨 완료',
    '첫 레슨 미완료'
)

result_first_lesson_impact = (
    first_lesson_impact
    .groupby('first_lesson_group')
    .agg(
        user_cnt=('user_id', 'count'),
        avg_unique_lesson_complete=('unique_lesson_complete_cnt_7d_prepay', 'mean'),
        avg_lesson_enter_cnt=('enter_lesson_page', 'mean'),
        avg_content_start_cnt=('start_content', 'mean'),
        question_click_users=('has_question_click', 'sum'),
        payment_page_enter_users=('has_payment_page_enter', 'sum'),
        paid_users=('paid_30d', 'sum')
    )
    .reset_index()
)

result_first_lesson_impact['question_click_rate'] = round(
    result_first_lesson_impact['question_click_users'] / result_first_lesson_impact['user_cnt'] * 100, 2
)

result_first_lesson_impact['payment_page_enter_rate'] = round(
    result_first_lesson_impact['payment_page_enter_users'] / result_first_lesson_impact['user_cnt'] * 100, 2
)

result_first_lesson_impact['paid_rate'] = round(
    result_first_lesson_impact['paid_users'] / result_first_lesson_impact['user_cnt'] * 100, 2
)

first_lesson_impact['is_high_active_7plus'] = (
    first_lesson_impact['unique_lesson_complete_cnt_7d_prepay'] >= 7
).astype(int)

high_active_by_first_lesson = (
    first_lesson_impact
    .groupby('first_lesson_group')
    .agg(
        high_active_users=('is_high_active_7plus', 'sum'),
        user_cnt=('user_id', 'count')
    )
    .reset_index()
)

high_active_by_first_lesson['high_active_rate_7plus'] = round(
    high_active_by_first_lesson['high_active_users'] /
    high_active_by_first_lesson['user_cnt'] * 100,
    2
)

result_first_lesson_impact = result_first_lesson_impact.merge(
    high_active_by_first_lesson[['first_lesson_group', 'high_active_rate_7plus']],
    on='first_lesson_group',
    how='left'
)

for col in ['avg_unique_lesson_complete', 'avg_lesson_enter_cnt', 'avg_content_start_cnt']:
    result_first_lesson_impact[col] = result_first_lesson_impact[col].round(2)

result_first_lesson_impact_kr = make_korean_table(result_first_lesson_impact)
display(result_first_lesson_impact_kr)

first_lesson_rate_plot = result_first_lesson_impact.melt(
    id_vars='first_lesson_group',
    value_vars=['question_click_rate', 'payment_page_enter_rate', 'paid_rate', 'high_active_rate_7plus'],
    var_name='지표',
    value_name='비율'
)

first_lesson_rate_plot['지표'] = first_lesson_rate_plot['지표'].replace({
    'question_click_rate': '질문 클릭률',
    'payment_page_enter_rate': '결제 페이지 진입률',
    'paid_rate': '결제율',
    'high_active_rate_7plus': '7개 이상 고활성 비율'
})

plt.figure(figsize=(10, 5))
sns.barplot(
    data=first_lesson_rate_plot,
    x='first_lesson_group',
    y='비율',
    hue='지표'
)

plt.title('첫 레슨 완료 여부별 주요 행동 비율')
plt.xlabel('')
plt.ylabel('비율(%)')
plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


- 첫 레슨 미완료 유저의 결제율은 3.75%이고, 첫 레슨 완료 유저의 결제율은 9.80%이다.
- 첫 레슨 완료 유저는 평균 고유 레슨 완료 수가 10.94개로, 미완료 유저의 1.04개보다 훨씬 높다.
- 질문 클릭률도 첫 레슨 완료 유저가 38.00%, 미완료 유저가 8.06%로 차이가 크다.
- 특히 7개 이상 레슨을 완료한 고활성 비율은 첫 레슨 완료 유저가 44.84%, 미완료 유저가 3.98%다.
- 따라서 첫 레슨 완료는 단순히 하나의 행동이 아니라, 이후 반복 학습(고활성 유저로 전환?)과 결제 전환으로 이어질 가능성을 보여주는 강한 초기 신호로 해석할 수 있다.


## 4-4. 레슨 완료 수 임계점 탐색


### 4-4-1. 고유 레슨 완료 수 세분화 구간별 결제율


In [ ]:
threshold_detail_df = analysis_user_behavior.copy()

def unique_lesson_detail_bucket(x):
    if x == 0:
        return '0'
    elif x <= 10:
        return str(int(x))
    elif x <= 20:
        return '11~20'
    elif x <= 50:
        return '21~50'
    elif x <= 100:
        return '51~100'
    elif x <= 200:
        return '101~200'
    else:
        return '201+'

threshold_detail_df['unique_lesson_detail_group'] = (
    threshold_detail_df['unique_lesson_complete_cnt_7d_prepay']
    .apply(unique_lesson_detail_bucket)
)

detail_order = [str(i) for i in range(0, 11)] + ['11~20', '21~50', '51~100', '101~200', '201+']

result_threshold_detail = (
    threshold_detail_df
    .groupby('unique_lesson_detail_group')
    .agg(
        user_cnt=('user_id', 'count'),
        paid_users=('paid_30d', 'sum'),
        avg_unique_lesson_complete=('unique_lesson_complete_cnt_7d_prepay', 'mean'),
        question_click_users=('has_question_click', 'sum'),
        payment_page_enter_users=('has_payment_page_enter', 'sum')
    )
    .reindex(detail_order)
    .reset_index()
)

result_threshold_detail['unpaid_users'] = (
    result_threshold_detail['user_cnt'] - result_threshold_detail['paid_users']
)

result_threshold_detail['paid_rate'] = round(
    result_threshold_detail['paid_users'] / result_threshold_detail['user_cnt'] * 100,
    2
)

result_threshold_detail['question_click_rate'] = round(
    result_threshold_detail['question_click_users'] / result_threshold_detail['user_cnt'] * 100,
    2
)

result_threshold_detail['payment_page_enter_rate'] = round(
    result_threshold_detail['payment_page_enter_users'] / result_threshold_detail['user_cnt'] * 100,
    2
)

result_threshold_detail['avg_unique_lesson_complete'] = (
    result_threshold_detail['avg_unique_lesson_complete'].round(2)
)

result_threshold_detail_kr = make_korean_table(result_threshold_detail)
display(result_threshold_detail_kr)

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=result_threshold_detail,
    x='unique_lesson_detail_group',
    y='paid_rate',
    marker='o'
)

plt.title('고유 레슨 완료 수 구간별 30일 내 결제율')
plt.xlabel('가입 후 7일 내 첫 결제 전 고유 레슨 완료 수')
plt.ylabel('결제율(%)')
plt.xticks(rotation=30)

for i, row in result_threshold_detail.iterrows():
    if pd.notnull(row['paid_rate']):
        plt.text(i, row['paid_rate'], f"{row['paid_rate']:.1f}%", ha='center', va='bottom')

plt.show()


- 레슨 완료 수를 세부 구간으로 나누면, 7개 이상부터 결제율이 눈에 띄게 높아지는 흐름이 보인다.
- 7개 완료 유저의 결제율은 12.84%, 11\~20개 완료 유저는 19.54%, 21\~50개 완료 유저는 27.67%다.
- 반면 1~6개 완료 구간은 결제율이 상대적으로 낮고 완만하다.
- 따라서 “1개 완료”보다는 7개 이상 완료가 결제 전환과 연결되는 고활성 기준으로 더 적절해 보인다.


### 4-4-2. N개 이상 누적 기준 결제율


In [ ]:
threshold_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 21, 51, 101, 201]

nplus_results = []

for n in threshold_list:
    temp = analysis_user_behavior[
        analysis_user_behavior['unique_lesson_complete_cnt_7d_prepay'] >= n
    ]

    user_cnt = temp['user_id'].nunique()
    paid_users = temp['paid_30d'].sum()

    nplus_results.append({
        'threshold': f'{n}+',
        'threshold_num': n,
        'user_cnt': user_cnt,
        'paid_users': paid_users,
        'unpaid_users': user_cnt - paid_users,
        'paid_rate': round(paid_users / user_cnt * 100, 2) if user_cnt > 0 else np.nan
    })

result_threshold_nplus_detail = pd.DataFrame(nplus_results)

result_threshold_nplus_detail['paid_rate_diff'] = (
    result_threshold_nplus_detail['paid_rate'].diff().round(2)
)

result_threshold_nplus_detail_kr = make_korean_table(result_threshold_nplus_detail)
display(result_threshold_nplus_detail_kr)

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=result_threshold_nplus_detail,
    x='threshold',
    y='paid_rate',
    marker='o'
)

plt.title('N개 이상 고유 레슨 완료 기준별 30일 내 결제율')
plt.xlabel('고유 레슨 완료 기준')
plt.ylabel('결제율(%)')
plt.xticks(rotation=30)

for i, row in result_threshold_nplus_detail.iterrows():
    plt.text(i, row['paid_rate'], f"{row['paid_rate']:.1f}%", ha='center', va='bottom')

plt.show()


- 1개 이상 완료 유저의 결제율은 9.58%다.
- 7개 이상 완료 유저의 결제율은 17.47%로 높아진다.
- 11개 이상 완료 유저의 결제율은 20.11%로 더 높아진다.
- 즉, 완료 기준을 높일수록 결제율은 상승하지만, 동시에 해당 유저 수는 줄어든다.
- 운영 지표로는 너무 높은 기준보다, 유저 규모와 결제율이 모두 어느 정도 확보되는 7개 이상 완료가 현실적인 고활성 기준으로 보인다.


### 4-4-3. 초고활동 유저 제외 후 7개 이상 결제율


In [ ]:
cutoff_results = []

for cutoff in [21, 51, 101, 201]:
    temp = analysis_user_behavior[
        analysis_user_behavior['unique_lesson_complete_cnt_7d_prepay'] < cutoff
    ].copy()

    high_7 = temp[
        temp['unique_lesson_complete_cnt_7d_prepay'] >= 7
    ]

    user_cnt = high_7['user_id'].nunique()
    paid_users = high_7['paid_30d'].sum()

    cutoff_results.append({
        'excluded_group': f'{cutoff}개 이상 제외',
        'remaining_7plus_users': user_cnt,
        'paid_users': paid_users,
        'paid_rate': round(paid_users / user_cnt * 100, 2) if user_cnt > 0 else np.nan
    })

result_cutoff_7plus = pd.DataFrame(cutoff_results)

result_cutoff_7plus_kr = make_korean_table(result_cutoff_7plus)
display(result_cutoff_7plus_kr)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=result_cutoff_7plus,
    x='excluded_group',
    y='paid_rate'
)

plt.title('초고활동 유저 제외 후 7개 이상 완료 유저 결제율')
plt.xlabel('제외 기준')
plt.ylabel('결제율(%)')
plt.xticks(rotation=20)

for i, row in result_cutoff_7plus.iterrows():
    plt.text(i, row['paid_rate'], f"{row['paid_rate']:.2f}%", ha='center', va='bottom')

plt.show()


- 21개 이상 완료 유저를 제외해도 7개 이상 완료 그룹의 결제율은 16.27%다.
- 51개 이상, 101개 이상, 201개 이상 초고활동 유저를 제외해도 결제율은 대체로 17~18% 수준으로 유지된다.
- 즉, 7개 이상 완료 그룹의 높은 결제율은 일부 초고활동 유저만의 영향으로 보기 어렵다.
- 이는 7개 이상 완료가 비교적 안정적인 고활성 기준일 수 있음을 뒷받침한다.


### 4-4-4. 완료 수 구간별 결제 유저 기여도


In [ ]:
result_threshold_detail_contribution = result_threshold_detail.copy()

total_paid_users = result_threshold_detail_contribution['paid_users'].sum()

result_threshold_detail_contribution['paid_user_share'] = round(
    result_threshold_detail_contribution['paid_users'] / total_paid_users * 100,
    2
)

result_threshold_detail_contribution_kr = make_korean_table(
    result_threshold_detail_contribution[
        [
            'unique_lesson_detail_group',
            'user_cnt',
            'paid_users',
            'paid_rate',
            'paid_user_share'
        ]
    ]
)

display(result_threshold_detail_contribution_kr)


- 결제 유저 수 기준으로 보면 0개 완료 유저에서도 결제자는 많이 존재한다.
- 이는 결제 전 레슨을 하지 않고 바로 결제 페이지로 이동하거나, 레슨 외 다른 요인으로 결제한 유저가 있다는 의미다.
- 하지만 결제율 관점에서는 7개 이상, 특히 11~50개 완료 구간에서 더 높은 전환 가능성이 나타난다.
- 따라서 결제자 수와 결제율을 함께 봐야 하며, 레슨 완료 수는 전체 결제를 모두 설명하는 요인이 아니라 학습 기반 결제 가능성을 설명하는 핵심 행동 지표로 해석하는 것이 적절하다.


### 4-4-5. 7개 이상 완료 유저가 왜 결제율이 높아질까?


In [ ]:
high_active_df = analysis_user_behavior.copy()

def high_active_group(x):
    if x == 0:
        return '0개 완료'
    elif x < 7:
        return '1~6개 완료'
    else:
        return '7개 이상 완료'

high_active_df['high_active_group'] = high_active_df['unique_lesson_complete_cnt_7d_prepay'].apply(
    high_active_group
)

high_active_order = ['0개 완료', '1~6개 완료', '7개 이상 완료']

result_high_active_pattern = (
    high_active_df
    .groupby('high_active_group')
    .agg(
        user_cnt=('user_id', 'count'),
        paid_users=('paid_30d', 'sum'),
        avg_unique_lesson_complete=('unique_lesson_complete_cnt_7d_prepay', 'mean'),
        avg_lesson_enter_cnt=('enter_lesson_page', 'mean'),
        avg_content_view_cnt=('enter_content_page', 'mean'),
        avg_content_start_cnt=('start_content', 'mean'),
        avg_content_end_cnt=('end_content', 'mean'),
        question_click_users=('has_question_click', 'sum'),
        review_click_users=('has_review_click', 'sum'),
        payment_page_enter_users=('has_payment_page_enter', 'sum')
    )
    .reindex(high_active_order)
    .reset_index()
)

result_high_active_pattern['paid_rate'] = round(
    result_high_active_pattern['paid_users'] / result_high_active_pattern['user_cnt'] * 100, 2
)

result_high_active_pattern['question_click_rate'] = round(
    result_high_active_pattern['question_click_users'] / result_high_active_pattern['user_cnt'] * 100, 2
)

result_high_active_pattern['review_click_rate'] = round(
    result_high_active_pattern['review_click_users'] / result_high_active_pattern['user_cnt'] * 100, 2
)

result_high_active_pattern['payment_page_enter_rate'] = round(
    result_high_active_pattern['payment_page_enter_users'] / result_high_active_pattern['user_cnt'] * 100, 2
)

for col in [
    'avg_unique_lesson_complete',
    'avg_lesson_enter_cnt',
    'avg_content_view_cnt',
    'avg_content_start_cnt',
    'avg_content_end_cnt'
]:
    result_high_active_pattern[col] = result_high_active_pattern[col].round(2)

result_high_active_pattern_kr = make_korean_table(result_high_active_pattern)
display(result_high_active_pattern_kr)


- 7개 이상 완료 유저는 평균 레슨 진입 수가 80.70회로 매우 높고, 질문 클릭률도 57.31%다.
- 결제 페이지 진입률은 65.57%, 결제율은 17.47%로 다른 그룹보다 높다.
- 즉, 이들은 단순히 레슨 완료 수만 많은 것이 아니라 콘텐츠 조회, 질문 클릭, 결제 페이지 진입까지 전반적인 서비스 행동이 활발하다.
- 따라서 7개 이상 완료는 단순 학습량 지표가 아니라, 서비스에 충분히 몰입한 고활성 유저를 식별하는 기준으로 볼 수 있다.


## 4-5. beginner 콘텐츠 중심의 초기 학습 개선 후보 탐색


### 4-5-1. 난이도별 콘텐츠 시작 후 레슨 완료 분석


In [ ]:
content_start = df_post_log_7d_prepay[
    (df_post_log_7d_prepay['event_name'] == 'start_content') &
    (df_post_log_7d_prepay['content_id'].notnull()) &
    (df_post_log_7d_prepay['content_difficulty'].notnull())
][[
    'user_id',
    'content_id',
    'content_difficulty'
]].drop_duplicates()

lesson_complete = df_post_log_7d_prepay[
    (df_post_log_7d_prepay['event_name'] == 'complete_lesson') &
    (df_post_log_7d_prepay['content_id'].notnull())
][[
    'user_id',
    'content_id'
]].drop_duplicates()

content_start_complete = content_start.merge(
    lesson_complete.assign(completed_after_start=1),
    on=['user_id', 'content_id'],
    how='left'
)

content_start_complete['completed_after_start'] = (
    content_start_complete['completed_after_start']
    .fillna(0)
    .astype(int)
)

result_10 = (
    content_start_complete
    .groupby('content_difficulty')
    .agg(
        start_users=('user_id', 'nunique'),
        completed_users_after_start=('completed_after_start', 'sum')
    )
    .reset_index()
)

result_10['complete_after_start_rate'] = round(
    result_10['completed_users_after_start'] / result_10['start_users'] * 100,
    2
)

result_10 = result_10.sort_values('start_users', ascending=False)

result_10_kr = make_korean_table(result_10)
display(result_10_kr)


- beginner 콘텐츠 시작 후 완료율은 77.29%다.
- advanced, intermediate보다 낮게 나타나지만, beginner는 시작 유저 수 자체가 압도적으로 많다.
- 따라서 beginner 완료율을 조금만 개선해도 전체 신규 유저의 초기 학습 경험에 미치는 영향이 클 수 있다.
- 이후에는 beginner 콘텐츠 중 어떤 콘텐츠가 특히 완료율이 낮은지 확인해야 한다. (뒤에서 진행)


### 4-5-2. beginner 콘텐츠가 많이 시작되는 원인 추정


In [ ]:
df_content['cohort_month'] = pd.to_datetime(df_content['cohort_month'], errors='coerce')

post_content = df_content[
    (df_content['cohort_month'] >= '2023-05-01') &
    (df_content['cohort_month'] < '2023-12-01')
].copy()

# 시작 수가 조회 수보다 많은 콘텐츠-월
content_start_over_view = post_content[
    post_content['content_start_users_7d'] > post_content['content_view_users_7d']
].copy()

print("조회 유저보다 시작 유저가 많은 행 수:", len(content_start_over_view))
print("해당 content_id 고유 수:", content_start_over_view['content_id'].nunique())

content_start_over_view_kr = (
    content_start_over_view[
        [
            'period_group',
            'cohort_month',
            'content_id',
            'content_view_users_7d',
            'content_start_users_7d',
            'view_to_start_rate'
        ]
    ]
    .sort_values('view_to_start_rate', ascending=False)
    .head(30)
    .rename(columns={
        'period_group': '기간 구분',
        'cohort_month': '가입 월',
        'content_id': '콘텐츠 ID',
        'content_view_users_7d': '콘텐츠 조회 유저 수',
        'content_start_users_7d': '콘텐츠 시작 유저 수',
        'view_to_start_rate': '조회 대비 시작률(%)'
    })
)

content_start_over_view_kr


- 콘텐츠 페이지 조회 유저 수보다 시작 유저 수가 많은 행이 21개 확인되었다.
- 이는 일반적인 `조회 → 시작` 흐름만으로 설명하기 어렵다.
- 유저가 콘텐츠 상세 페이지를 거치지 않고 추천, 온보딩, 버튼, 자동 진입 등의 경로로 바로 시작했을 가능성이 있다.
- 따라서 시작 수가 많은 콘텐츠는 유저의 완전한 자발 선택이라기보다, 서비스가 초기에 노출한 콘텐츠일 가능성도 함께 고려해야 한다.


In [ ]:
difficulty_map = (
    df_post_log_7d_prepay[
        df_post_log_7d_prepay['content_difficulty'].notnull() &
        df_post_log_7d_prepay['content_id'].notnull()
    ]
    .groupby('content_id')['content_difficulty']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0])
    .reset_index()
)

content_start_over_view = content_start_over_view.merge(
    difficulty_map,
    on='content_id',
    how='left'
)

result_start_over_view_by_difficulty = (
    content_start_over_view
    .groupby('content_difficulty')
    .agg(
        row_cnt=('content_id', 'count'),
        content_cnt=('content_id', 'nunique'),
        avg_view_users=('content_view_users_7d', 'mean'),
        avg_start_users=('content_start_users_7d', 'mean'),
        avg_view_to_start_rate=('view_to_start_rate', 'mean')
    )
    .reset_index()
)

for col in ['avg_view_users', 'avg_start_users', 'avg_view_to_start_rate']:
    result_start_over_view_by_difficulty[col] = result_start_over_view_by_difficulty[col].round(2)

result_start_over_view_by_difficulty_kr = make_korean_table(result_start_over_view_by_difficulty)
display(result_start_over_view_by_difficulty_kr)


- 조회보다 시작이 많은 콘텐츠는 beginner에서 더 많이 나타난다.
- beginner 콘텐츠는 평균 조회 대비 시작 비율이 190.82%로 높게 나타났다.
- 이는 beginner 콘텐츠가 신규 유저에게 초기에 강하게 노출되거나, 바로 시작되는 구조와 연결되어 있을 가능성을 보여준다.
- 따라서 beginner 콘텐츠의 품질과 완료 경험은 신규 유저의 첫인상에 큰 영향을 줄 수 있다.


### 4-5-3. 콘텐츠 페이지 진입 수보다 시작 수가 많은 beginner 콘텐츠


In [ ]:
beginner_start_over_view = content_start_over_view[
    content_start_over_view['content_difficulty'] == 'beginner'
].copy()

beginner_start_over_view_result = (
    beginner_start_over_view
    .groupby('content_id')
    .agg(
        months_cnt=('cohort_month', 'nunique'),
        avg_view_users=('content_view_users_7d', 'mean'),
        avg_start_users=('content_start_users_7d', 'mean'),
        avg_view_to_start_rate=('view_to_start_rate', 'mean'),
        total_view_users=('content_view_users_7d', 'sum'),
        total_start_users=('content_start_users_7d', 'sum')
    )
    .reset_index()
)

for col in [
    'avg_view_users',
    'avg_start_users',
    'avg_view_to_start_rate'
]:
    beginner_start_over_view_result[col] = beginner_start_over_view_result[col].round(2)

beginner_start_over_view_result['start_minus_view'] = (
    beginner_start_over_view_result['total_start_users'] -
    beginner_start_over_view_result['total_view_users']
)

beginner_start_over_view_result_kr = make_korean_table(
    beginner_start_over_view_result.sort_values(
        'start_minus_view',
        ascending=False
    ).head(30)
)

display(beginner_start_over_view_result_kr)


- 특히 `f4be...` 콘텐츠는 6개월 동안 평균 시작 유저 수가 2,072.67명으로 높고, 조회 대비 시작 차이도 가장 크다.
- `61b6...` 콘텐츠 역시 여러 달에 걸쳐 시작 유저 수가 많다.
- 이 콘텐츠들은 단순히 일부 달에 튄 값이 아니라, 반복적으로 신규 유저의 초기 학습 흐름에 등장한 콘텐츠로 보인다.
- 따라서 이 콘텐츠들의 완료율과 이후 행동을 확인하는 것이 중요하다.


### 4-5-4. 콘텐츠 개선 후보


In [ ]:
# 유저-콘텐츠 단위 시작
content_start_user = df_post_log_7d_prepay[
    (df_post_log_7d_prepay['event_name'] == 'start_content') &
    (df_post_log_7d_prepay['content_id'].notnull()) &
    (df_post_log_7d_prepay['content_difficulty'].notnull())
][[
    'user_id',
    'content_id',
    'content_difficulty'
]].drop_duplicates()

# 유저-콘텐츠 단위 완료
content_complete_user = df_post_log_7d_prepay[
    (df_post_log_7d_prepay['event_name'].isin(['complete_lesson', 'end_content'])) &
    (df_post_log_7d_prepay['content_id'].notnull())
][[
    'user_id',
    'content_id'
]].drop_duplicates()

content_start_complete_user = content_start_user.merge(
    content_complete_user.assign(completed=1),
    on=['user_id', 'content_id'],
    how='left'
)

content_start_complete_user['completed'] = (
    content_start_complete_user['completed']
    .fillna(0)
    .astype(int)
)

# 결제 여부 붙이기
content_start_complete_user = content_start_complete_user.merge(
    post_users[['user_id', 'paid_30d']],
    on='user_id',
    how='left'
)

beginner_content_result = (
    content_start_complete_user[
        content_start_complete_user['content_difficulty'] == 'beginner'
    ]
    .groupby('content_id')
    .agg(
        start_users=('user_id', 'nunique'),
        completed_users=('completed', 'sum'),
        paid_users=('paid_30d', 'sum')
    )
    .reset_index()
)

beginner_content_result['complete_rate'] = round(
    beginner_content_result['completed_users'] /
    beginner_content_result['start_users'] * 100,
    2
)

beginner_content_result['paid_rate'] = round(
    beginner_content_result['paid_users'] /
    beginner_content_result['start_users'] * 100,
    2
)

# 개선 우선순위 점수: 시작 유저가 많고 완료율이 낮을수록 높음
beginner_content_result['incomplete_users'] = (
    beginner_content_result['start_users'] -
    beginner_content_result['completed_users']
)

beginner_content_result['improvement_priority_score'] = (
    beginner_content_result['incomplete_users']
)

# 표본 너무 작은 콘텐츠 제외
beginner_content_result_filtered = beginner_content_result[
    beginner_content_result['start_users'] >= 30
].copy()

beginner_content_result_filtered_kr = make_korean_table(
    beginner_content_result_filtered.sort_values(
        ['improvement_priority_score', 'start_users'],
        ascending=False
    ).head(30)
)

display(beginner_content_result_filtered_kr)


- 개선 우선순위 1순위는 `f4be...` 콘텐츠다.
- 이 콘텐츠는 시작 유저가 가장 많고, 미완료 유저도 5,535명으로 가장 많다.
- 즉, 완료율을 개선했을 때 영향을 받을 수 있는 유저 규모가 가장 크다.
- 개선 후보를 정할 때 단순히 완료율만 낮은 콘텐츠가 아니라, 시작 유저 수가 많고 미완료 유저가 많은 콘텐츠를 우선적으로 보는 것이 적절하다.


## 4-6. 난이도 이동 경로별 행동 차이 분석


### 4-6-1. 난이도 이동 유형 정의


In [ ]:
def is_beginner_to_advanced(seq):
    if 'beginner' not in seq or 'advanced' not in seq:
        return False

    first_beginner_idx = seq.index('beginner')
    first_advanced_idx = seq.index('advanced')

    return first_beginner_idx < first_advanced_idx

user_difficulty_sequence['beginner_to_advanced'] = (
    user_difficulty_sequence['difficulty_sequence'].apply(is_beginner_to_advanced)
)

def difficulty_path_group(row):
    seq = row['difficulty_sequence']

    if row['beginner_to_advanced']:
        return 'beginner 이후 advanced'
    elif row['first_difficulty'] == 'advanced':
        return '처음부터 advanced'
    elif row['used_beginner'] and not row['used_advanced']:
        return 'beginner만 시작'
    elif row['used_advanced'] and not row['used_beginner']:
        return 'advanced만 시작'
    else:
        return '기타'

user_difficulty_sequence['difficulty_path_group'] = user_difficulty_sequence.apply(
    difficulty_path_group,
    axis=1
)

result_difficulty_path_count = (
    user_difficulty_sequence['difficulty_path_group']
    .value_counts()
    .reset_index()
)

result_difficulty_path_count.columns = ['difficulty_path_group', 'user_cnt']
result_difficulty_path_count_kr = make_korean_table(result_difficulty_path_count)

display(result_difficulty_path_count_kr)


- 대부분의 유저는 beginner만 시작한 그룹에 속한다.
- `beginner 이후 advanced`로 이동한 유저는 157명으로 규모는 작지만, 학습 확장 행동을 보인 유저군이다.
- `처음부터 advanced` 유저는 631명으로 존재하지만, 신규 유저 전체에서 큰 비중은 아니다.
- 따라서 초기 학습 설계는 여전히 beginner 중심으로 두되, 일부 고의도 유저에게는 advanced로 확장할 수 있는 경로를 제공하는 방식이 적절해 보인다.


### 4-6-2. 난이도 이동 유형별 행동/결제 비교


In [ ]:
difficulty_path_behavior = user_difficulty_sequence.merge(
    analysis_user_behavior[[
        'user_id',
        'enter_lesson_page',
        'unique_lesson_complete_cnt_7d_prepay',
        'start_content',
        'unique_content_start_cnt_7d_prepay',
        'has_question_click',
        'has_payment_page_enter',
        'paid_30d'
    ]],
    on='user_id',
    how='left'
)

result_difficulty_path = (
    difficulty_path_behavior
    .groupby('difficulty_path_group')
    .agg(
        user_cnt=('user_id', 'count'),
        avg_lesson_enter_cnt=('enter_lesson_page', 'mean'),
        avg_unique_lesson_complete=('unique_lesson_complete_cnt_7d_prepay', 'mean'),
        avg_content_start_cnt=('start_content', 'mean'),
        avg_unique_content_start=('unique_content_start_cnt_7d_prepay', 'mean'),
        question_click_users=('has_question_click', 'sum'),
        payment_page_enter_users=('has_payment_page_enter', 'sum'),
        paid_users=('paid_30d', 'sum')
    )
    .reset_index()
)

result_difficulty_path['question_click_rate'] = round(
    result_difficulty_path['question_click_users'] /
    result_difficulty_path['user_cnt'] * 100,
    2
)

result_difficulty_path['payment_page_enter_rate'] = round(
    result_difficulty_path['payment_page_enter_users'] /
    result_difficulty_path['user_cnt'] * 100,
    2
)

result_difficulty_path['paid_rate'] = round(
    result_difficulty_path['paid_users'] /
    result_difficulty_path['user_cnt'] * 100,
    2
)

# 고활성 유저 비율
difficulty_path_behavior['is_high_active_7plus'] = (
    difficulty_path_behavior['unique_lesson_complete_cnt_7d_prepay'] >= 7
).astype(int)

high_active_by_path = (
    difficulty_path_behavior
    .groupby('difficulty_path_group')
    .agg(
        high_active_users=('is_high_active_7plus', 'sum'),
        user_cnt=('user_id', 'count')
    )
    .reset_index()
)

high_active_by_path['high_active_rate_7plus'] = round(
    high_active_by_path['high_active_users'] /
    high_active_by_path['user_cnt'] * 100,
    2
)

result_difficulty_path = result_difficulty_path.merge(
    high_active_by_path[['difficulty_path_group', 'high_active_rate_7plus']],
    on='difficulty_path_group',
    how='left'
)

for col in [
    'avg_lesson_enter_cnt',
    'avg_unique_lesson_complete',
    'avg_content_start_cnt',
    'avg_unique_content_start'
]:
    result_difficulty_path[col] = result_difficulty_path[col].round(2)

result_difficulty_path_kr = make_korean_table(result_difficulty_path.sort_values('user_cnt', ascending=False))
display(result_difficulty_path_kr)


- `beginner 이후 advanced` 그룹은 평균 레슨 진입 수와 고유 레슨 완료 수가 가장 높다.
- 이 그룹은 단순 beginner 소비를 넘어, 서비스 안에서 학습을 확장한 유저로 볼 수 있다.
- 따라서 beginner 콘텐츠는 단순히 초보자용 콘텐츠가 아니라, 이후 더 깊은 학습으로 넘어가는 진입점 역할을 할 수 있다.
- 다만 이 그룹의 유저 수가 157명으로 적기 때문에, 결론을 과하게 일반화하기보다는 고활성 유저의 학습 경로 후보로 해석하는 것이 안전하다.


## 4-7. 할인율-결제율


In [ ]:
# 날짜 컬럼 변환
df_user['cohort_month'] = pd.to_datetime(df_user['cohort_month'], errors='coerce')

# 첫 결제 여부 컬럼 만들기
df_user['is_first_paid'] = df_user['first_payment_time'].notna().astype(int)

# 할인률 계산
df_user['discount_rate'] = df_user['first_discount_amount'] / (
    df_user['first_paid_amount'] + df_user['first_discount_amount']
)

# 0으로 나누는 경우 제거
df_user['discount_rate'] = df_user['discount_rate'].replace([float('inf'), -float('inf')], pd.NA)

# 월별 집계
monthly_discount_payment = (
    df_user
    .groupby('cohort_month')
    .agg(
        signup_users=('user_id', 'nunique'),
        first_paid_users=('is_first_paid', 'sum'),
        avg_discount_rate=('discount_rate', 'mean')
    )
    .reset_index()
)

# 결제율 계산
monthly_discount_payment['payment_rate'] = (
    monthly_discount_payment['first_paid_users'] /
    monthly_discount_payment['signup_users'] * 100
)

# 할인률도 %로 변환
monthly_discount_payment['avg_discount_rate'] = (
    monthly_discount_payment['avg_discount_rate'] * 100
)

display(monthly_discount_payment)

# 산점도
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=monthly_discount_payment,
    x='avg_discount_rate',
    y='payment_rate',
    s=100
)

# 추세선 추가
sns.regplot(
    data=monthly_discount_payment,
    x='avg_discount_rate',
    y='payment_rate',
    scatter=False,
    ci=None
)

plt.title('월별 평균 할인률과 첫 결제율의 관계')
plt.xlabel('평균 할인률(%)')
plt.ylabel('첫 결제율(%)')
plt.grid(alpha=0.3)

plt.show()

# 상관계수 확인
corr = monthly_discount_payment[['avg_discount_rate', 'payment_rate']].corr()
display(corr)


In [ ]:
# 날짜 변환
df_user['cohort_month'] = pd.to_datetime(df_user['cohort_month'], errors='coerce')
df_user['first_payment_time'] = pd.to_datetime(df_user['first_payment_time'], errors='coerce')

# 첫 결제 여부
df_user['is_first_paid'] = df_user['first_payment_time'].notna().astype(int)

# 할인률 계산
# 할인률 = 할인금액 / (실결제금액 + 할인금액)
df_user['discount_rate'] = df_user['first_discount_amount'] / (
    df_user['first_paid_amount'] + df_user['first_discount_amount']
)

# inf 처리
df_user['discount_rate'] = df_user['discount_rate'].replace([np.inf, -np.inf], np.nan)

# 결제 안 한 유저는 할인률 계산이 안 되므로 0으로 둘지/제외할지 선택 필요
# 일단 산점도용으로는 0으로 채움
df_user['discount_rate_fill0'] = df_user['discount_rate'].fillna(0)

df_user[['user_id', 'cohort_month', 'is_first_paid', 'first_paid_amount', 'first_discount_amount', 'discount_rate_fill0']].head()


In [ ]:
# 원본 보호
plot_df = df_user.copy()

# 날짜 변환
plot_df['cohort_month'] = pd.to_datetime(plot_df['cohort_month'], errors='coerce')
plot_df['first_payment_time'] = pd.to_datetime(plot_df['first_payment_time'], errors='coerce')

# 첫 결제 여부
plot_df['is_first_paid'] = plot_df['first_payment_time'].notna().astype(int)

# 결측 0 처리
plot_df['first_paid_amount'] = plot_df['first_paid_amount'].fillna(0)
plot_df['first_discount_amount'] = plot_df['first_discount_amount'].fillna(0)

# 월별 집계
monthly_discount = (
    plot_df
    .groupby('cohort_month')
    .agg(
        signup_users=('user_id', 'nunique'),
        first_paid_users=('is_first_paid', 'sum'),
        total_paid_amount=('first_paid_amount', 'sum'),
        total_discount_amount=('first_discount_amount', 'sum')
    )
    .reset_index()
)

# 첫 결제율
monthly_discount['payment_rate'] = (
    monthly_discount['first_paid_users'] /
    monthly_discount['signup_users'] * 100
)

# 할인율 = 총 할인금액 / (총 실결제금액 + 총 할인금액)
monthly_discount['discount_rate'] = (
    monthly_discount['total_discount_amount'] /
    (monthly_discount['total_paid_amount'] + monthly_discount['total_discount_amount']) * 100
)

# inf, 결측 제거
monthly_discount = monthly_discount.replace([np.inf, -np.inf], np.nan)
monthly_discount = monthly_discount.dropna(subset=['discount_rate', 'payment_rate'])

# 월 라벨
monthly_discount['month_label'] = monthly_discount['cohort_month'].dt.strftime('%Y-%m')

display(monthly_discount)


In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    daily_discount['discount_rate'],
    daily_discount['payment_rate'],
    s=50,
    alpha=0.6
)

x = daily_discount['discount_rate']
y = daily_discount['payment_rate']

if len(daily_discount) >= 2:
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)

    x_line = np.linspace(x.min(), x.max(), 100)
    plt.plot(x_line, p(x_line), linewidth=2)

plt.title('일별 할인율과 첫 결제율의 관계')
plt.xlabel('할인율(%)')
plt.ylabel('첫 결제율(%)')
plt.grid(alpha=0.3)

plt.show()

display(daily_discount[['discount_rate', 'payment_rate']].corr())


## 4-8. 유저 세그먼트 분석


In [ ]:
import plotly.express as px

activation_df = analysis_user_behavior.copy()

fill_cols = [
    'unique_lesson_complete_cnt_7d_prepay',
    'enter_content_page',
    'start_content',
    'end_content',
    'paid_30d',
    'has_question_click',
    'has_payment_page_enter'
]

for col in fill_cols:
    activation_df[col] = activation_df[col].fillna(0)

activation_df['has_content_experience'] = (
    (activation_df['enter_content_page'] > 0) |
    (activation_df['start_content'] > 0) |
    (activation_df['end_content'] > 0)
).astype(int)

def make_activation_group(row):
    lesson_count = row['unique_lesson_complete_cnt_7d_prepay']
    has_content_exp = row['has_content_experience']

    if lesson_count == 0 and has_content_exp == 0:
        return '무행동형'
    elif lesson_count == 0 and has_content_exp == 1:
        return '탐색형'
    elif 1 <= lesson_count <= 6:
        return '약한 학습형'
    else:
        return '핵심 학습형'


activation_order = [
    '무행동형',
    '탐색형',
    '약한 학습형',
    '핵심 학습형'
]

activation_df['activation_group'] = activation_df.apply(make_activation_group, axis=1)

result_activation_group = (
    activation_df
    .groupby('activation_group')
    .agg(
        user_cnt=('user_id', 'count'),
        paid_users=('paid_30d', 'sum'),
        avg_unique_lesson_complete=('unique_lesson_complete_cnt_7d_prepay', 'mean'),
        avg_content_view_cnt=('enter_content_page', 'mean'),
        avg_content_start_cnt=('start_content', 'mean'),
        avg_content_end_cnt=('end_content', 'mean'),
        question_click_users=('has_question_click', 'sum'),
        payment_page_enter_users=('has_payment_page_enter', 'sum')
    )
    .reindex(activation_order)
    .reset_index()
)

total_users = result_activation_group['user_cnt'].sum()
total_paid_users = result_activation_group['paid_users'].sum()

result_activation_group['user_rate'] = (
    result_activation_group['user_cnt'] / total_users * 100
)

result_activation_group['paid_rate'] = (
    result_activation_group['paid_users'] / result_activation_group['user_cnt'] * 100
)

result_activation_group['paid_user_share'] = (
    result_activation_group['paid_users'] / total_paid_users * 100
)

result_activation_group['question_click_rate'] = (
    result_activation_group['question_click_users'] / result_activation_group['user_cnt'] * 100
)

result_activation_group['payment_page_enter_rate'] = (
    result_activation_group['payment_page_enter_users'] / result_activation_group['user_cnt'] * 100
)

round_cols = [
    'avg_unique_lesson_complete',
    'avg_content_view_cnt',
    'avg_content_start_cnt',
    'avg_content_end_cnt',
    'user_rate',
    'paid_rate',
    'paid_user_share',
    'question_click_rate',
    'payment_page_enter_rate'
]

for col in round_cols:
    result_activation_group[col] = result_activation_group[col].round(2)

result_activation_group_kr = make_korean_table(result_activation_group)
display(result_activation_group_kr)


In [ ]:
# 날짜 타입 보정
df_user['signup_time'] = pd.to_datetime(df_user['signup_time'], errors='coerce')

# 무료체험 폐지 후 유저 기준
# 팀원 코드 결과 총 유저 수 46,055명 맞추는 기준
post = df_user[
    (df_user['signup_time'] >= '2023-05-01') &
    (df_user['signup_time'] < '2023-12-01')
].copy()

# 필요한 원래 컬럼 결측치 처리
fill_cols = [
    'paid_30d',
    'lesson_complete_prepay_7d',
    'unique_content_prepay_7d',
    'content_view_7d',
    'content_start_7d',
    'question_click_7d',
    'payment_page_enter_7d'
]

for col in fill_cols:
    post[col] = post[col].fillna(0)

# 확인
print('post 유저 수:', post['user_id'].nunique())
print('post shape:', post.shape)


In [ ]:
# 콘텐츠 경험 여부
post['content_experience'] = (
    (post['unique_content_prepay_7d'] > 0) |
    (post['content_view_7d'] > 0) |
    (post['content_start_7d'] > 0)
)

# 4그룹 분류
def make_activation_group(row):
    lesson_count = row['lesson_complete_prepay_7d']
    content_experience = row['content_experience']

    if lesson_count == 0 and not content_experience:
        return '무행동형'
    elif lesson_count == 0 and content_experience:
        return '탐색형'
    elif 1 <= lesson_count <= 6:
        return '약한 학습형'
    else:
        return '핵심 학습형'

activation_order = [
    '무행동형',
    '탐색형',
    '약한 학습형',
    '핵심 학습형'
]

post['활성화 그룹'] = post.apply(make_activation_group, axis=1)

# 그룹별 성과 산출
activation_group_result = (
    post
    .groupby('활성화 그룹')
    .agg(
        유저_수=('user_id', 'nunique'),
        결제자_수=('paid_30d', 'sum'),
        첫_결제율=('paid_30d', 'mean'),
        평균_레슨_완료_수=('lesson_complete_prepay_7d', 'mean'),
        평균_고유_콘텐츠_수=('unique_content_prepay_7d', 'mean'),
        평균_콘텐츠_조회_수=('content_view_7d', 'mean'),
        평균_콘텐츠_시작_수=('content_start_7d', 'mean'),
        질문_클릭_유저_수=('question_click_7d', lambda x: (x > 0).sum()),
        결제_페이지_진입_유저_수=('payment_page_enter_7d', lambda x: (x > 0).sum())
    )
    .reindex(activation_order)
    .fillna(0)
    .reset_index()
)

# 비율 계산
activation_group_result['유저 비중(%)'] = (
    activation_group_result['유저_수'] / post['user_id'].nunique() * 100
)

activation_group_result['첫 결제율(%)'] = (
    activation_group_result['첫_결제율'] * 100
)

activation_group_result['결제 기여도(%)'] = (
    activation_group_result['결제자_수'] / post['paid_30d'].sum() * 100
)

activation_group_result['질문 클릭률(%)'] = (
    activation_group_result['질문_클릭_유저_수'] / activation_group_result['유저_수'] * 100
)

activation_group_result['결제 페이지 진입률(%)'] = (
    activation_group_result['결제_페이지_진입_유저_수'] / activation_group_result['유저_수'] * 100
)

# 출력용 표
activation_display = activation_group_result[
    [
        '활성화 그룹',
        '유저_수',
        '결제자_수',
        '평균_레슨_완료_수',
        '평균_고유_콘텐츠_수',
        '평균_콘텐츠_조회_수',
        '평균_콘텐츠_시작_수',
        '질문_클릭_유저_수',
        '결제_페이지_진입_유저_수',
        '유저 비중(%)',
        '첫 결제율(%)',
        '결제 기여도(%)',
        '질문 클릭률(%)',
        '결제 페이지 진입률(%)'
    ]
].copy()

activation_display.columns = [
    '활성화 그룹',
    '유저 수',
    '결제자 수',
    '평균 레슨 완료 수',
    '평균 고유 콘텐츠 수',
    '평균 콘텐츠 조회 수',
    '평균 콘텐츠 시작 수',
    '질문 클릭 유저 수',
    '결제 페이지 진입 유저 수',
    '유저 비중(%)',
    '첫 결제율(%)',
    '결제 기여도(%)',
    '질문 클릭률(%)',
    '결제 페이지 진입률(%)'
]

for col in [
    '평균 레슨 완료 수',
    '평균 고유 콘텐츠 수',
    '평균 콘텐츠 조회 수',
    '평균 콘텐츠 시작 수',
    '유저 비중(%)',
    '첫 결제율(%)',
    '결제 기여도(%)',
    '질문 클릭률(%)',
    '결제 페이지 진입률(%)'
]:
    activation_display[col] = activation_display[col].round(2)

display(activation_display)


In [ ]:
plot_bubble = activation_group_result.copy()

# 기준선
avg_user_rate = plot_bubble['유저 비중(%)'].mean()
avg_paid_rate = plot_bubble['첫 결제율(%)'].mean()

fig = px.scatter(
    plot_bubble,
    x='유저 비중(%)',
    y='첫 결제율(%)',
    size='결제 기여도(%)',
    color='활성화 그룹',
    text='활성화 그룹',
    category_orders={'활성화 그룹': activation_order},
    hover_data={
        '활성화 그룹': True,
        '유저_수': ':,',
        '결제자_수': ':,',
        '유저 비중(%)': ':.2f',
        '첫 결제율(%)': ':.2f',
        '결제 기여도(%)': ':.2f',
        '질문 클릭률(%)': ':.2f',
        '결제 페이지 진입률(%)': ':.2f'
    },
    size_max=50,
    title='활성화 그룹별 규모, 첫 결제율, 결제 기여도'
)

# 세로 기준선: 유저 비중 평균
fig.add_vline(
    x=avg_user_rate,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'유저 비중 평균 {avg_user_rate:.2f}%',
    annotation_position='top left'
)

# 가로 기준선: 첫 결제율 평균
fig.add_hline(
    y=avg_paid_rate,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'첫 결제율 평균 {avg_paid_rate:.2f}%',
    annotation_position='right'
)

fig.update_traces(
    textposition='top center',
    marker=dict(
        opacity=0.75,
        line=dict(width=1, color='white')
    )
)

fig.update_layout(
    width=1100,
    height=600,
    xaxis_title='전체 유저 비중(%)',
    yaxis_title='첫 결제율(%)',
    legend_title_text='활성화 그룹',
    plot_bgcolor='#EAF0F8'
)

fig.show()


In [ ]:
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

plot_bar = activation_group_result.copy()

plot_bar['활성화 그룹'] = pd.Categorical(
    plot_bar['활성화 그룹'],
    categories=activation_order,
    ordered=True
)

plot_bar = plot_bar.sort_values('활성화 그룹').reset_index(drop=True)

groups = plot_bar['활성화 그룹']
x = np.arange(len(groups))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(
    x - width,
    plot_bar['첫 결제율(%)'],
    width,
    label='첫 결제율'
)

bars2 = ax.bar(
    x,
    plot_bar['질문 클릭률(%)'],
    width,
    label='질문 클릭률'
)

bars3 = ax.bar(
    x + width,
    plot_bar['결제 페이지 진입률(%)'],
    width,
    label='결제 페이지 진입률'
)

ax.set_title(
    '활성화 수준별 주요 행동 비율',
    fontsize=15,
    fontweight='bold'
)

ax.set_xlabel('활성화 그룹')
ax.set_ylabel('비율(%)')
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 막대 위 수치 표시
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f'{height:.1f}%',
            ha='center',
            va='bottom',
            fontsize=9
        )

max_y = max(
    plot_bar['첫 결제율(%)'].max(),
    plot_bar['질문 클릭률(%)'].max(),
    plot_bar['결제 페이지 진입률(%)'].max()
)

ax.set_ylim(0, max_y * 1.2)

plt.tight_layout()
plt.show()


# Appendix C. 장준영 팀원 개별 분석 원본

> 본 섹션은 팀원별 개별 분석 산출물을 원본 흐름에 가깝게 보존한 제출용 부록입니다.
> 전체 노트북은 순차 실행을 전제로 하지 않으며, 분석 과정과 산출물 취합을 목적으로 합니다.


# 데이터 마트 해지 분석


In [ ]:
import pandas as pd
from IPython.display import display

file_path = "/content/dm_user_subscription_retention_core.csv"

df = pd.read_csv(file_path)

print("데이터 크기:", df.shape)

print("\n전체 컬럼")
for col in df.columns:
    print(col)

print("\n데이터 정보")
df.info()

print("\n상위 5개")
display(df.head())


In [ ]:
keywords = [
    "cancel", "cancle", "churn",
    "subscription", "subscribe",
    "renew", "resubscribe",
    "retention", "expire",
    "platform", "device", "os",
    "date", "time", "month", "year"
]

candidate_cols = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in keywords)
]

print("분석 관련 컬럼")
for col in candidate_cols:
    print(col)


In [ ]:
# 해지 분석 핵심 컬럼 값 확인
check_cols = [
    "first_payment_month",
    "first_cancel_time",
    "days_to_first_cancel",
    "cancel_7d",
    "cancel_30d",
    "cancel_60d",
    "cancel_90d",
    "first_renew_time",
    "cancel_before_first_renew",
    "first_resubscribe_time",
    "subscription_lifecycle_segment"
]

for col in check_cols:
    if col in df.columns:
        print(f"\n===== {col} =====")
        print("결측치 수:", df[col].isna().sum())
        print(df[col].value_counts(dropna=False).head(10))


In [ ]:
date_cols = [
    "signup_time",
    "first_payment_time",
    "first_cancel_time",
    "first_renew_time",
    "first_resubscribe_time"
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("첫 결제일 범위")
print(df["first_payment_time"].min(), "~", df["first_payment_time"].max())

print("\n첫 취소일 범위")
print(df["first_cancel_time"].min(), "~", df["first_cancel_time"].max())


In [ ]:
df["first_payment_month"] = (
    pd.to_datetime(df["first_payment_time"], errors="coerce")
    .dt.to_period("M")
    .astype(str)
)

period_check = df[
    df["first_payment_month"].isin([
        "2022-09", "2022-10", "2022-11",
        "2023-09", "2023-10", "2023-11"
    ])
]

print("비교 기간 유저 수")
print(period_check["first_payment_month"].value_counts().sort_index())


## 데이터 검증


In [ ]:
# 유저 단위 데이터인지 확인
print("전체 행 수:", len(df))
print("고유 유저 수:", df["user_id"].nunique())
print("중복 user_id 수:", df["user_id"].duplicated().sum())

# cancel_30d 값 확인
print("\ncancel_30d 분포")
print(df["cancel_30d"].value_counts(dropna=False))


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

df["first_payment_time"] = pd.to_datetime(
    df["first_payment_time"],
    errors="coerce"
)

df["first_payment_month"] = (
    df["first_payment_time"]
    .dt.to_period("M")
    .astype(str)
)

target_months = [
    "2022-09", "2022-10", "2022-11",
    "2023-09", "2023-10", "2023-11"
]

analysis_df = df[
    df["first_payment_month"].isin(target_months)
].copy()

analysis_df["period_group"] = np.where(
    analysis_df["first_payment_month"].str.startswith("2022"),
    "2022년 9~11월",
    "2023년 9~11월"
)

print("분석 대상 유저 수:", len(analysis_df))

display(
    analysis_df["first_payment_month"]
    .value_counts()
    .sort_index()
)


# 분석


## 해지 버튼 클릭 비교 분석

### 분석 목적
2022년과 2023년의 동일 기간에 첫 결제한 유저를 대상으로, 첫 결제 후 30일 내 해지 버튼 클릭률에 차이가 있는지 확인한다.

### 분석 기간
- 2022년 9월~11월
- 2023년 9월~11월
- 각 기간은 3개월이다.

### 분석 기준
- 분모: 해당 기간 첫 결제 고유 유저 수
- 분자: 첫 결제 후 30일 내 해지 버튼을 클릭한 유저 수
- 계산식: `cancel_30d = 1인 유저 수 / 첫 결제 유저 수 × 100`

### 해석 주의
`cancel_30d`는 실제 해지 완료 여부가 아니라 해지 버튼 클릭 행동을 기준으로 한다. 따라서 결과는 실제 해지율보다 ‘30일 내 해지 버튼 클릭률’ 또는 ‘해지 의향률’로 해석한다.


In [ ]:
# =========================================================
# 1. 분석 환경 설정 및 비교 기간 데이터 생성
# =========================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display

# ---------------------------------------------------------
# 한글 폰트 설정
# ---------------------------------------------------------
pass  # 환경 설치 명령 제거. requirements-analysis.txt를 참고하세요.

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

# ---------------------------------------------------------
# 분석에 필요한 컬럼 확인
# ---------------------------------------------------------
required_cols = [
    "user_id",
    "first_payment_time",
    "cancel_7d",
    "cancel_30d",
    "days_to_first_cancel",
    "cancel_before_first_renew",
    "subscription_lifecycle_segment"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"필요한 컬럼이 없습니다: {missing_cols}")

# ---------------------------------------------------------
# 날짜 변환
# ---------------------------------------------------------
df["first_payment_time"] = pd.to_datetime(
    df["first_payment_time"],
    errors="coerce"
)

df["first_payment_month"] = (
    df["first_payment_time"]
    .dt.to_period("M")
    .astype(str)
)

# 0/1 정수형 변환
binary_cols = [
    "cancel_7d",
    "cancel_30d",
    "cancel_before_first_renew"
]

for col in binary_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# ---------------------------------------------------------
# 분석 대상 기간 필터링
# ---------------------------------------------------------
target_months = [
    "2022-09", "2022-10", "2022-11",
    "2023-09", "2023-10", "2023-11"
]

analysis_df = df[
    df["first_payment_month"].isin(target_months)
].copy()

analysis_df["period_group"] = np.where(
    analysis_df["first_payment_time"].dt.year == 2022,
    "2022년 9~11월",
    "2023년 9~11월"
)

analysis_df["month_num"] = analysis_df["first_payment_time"].dt.month
analysis_df["month_label"] = analysis_df["month_num"].astype(str) + "월"

period_order = ["2022년 9~11월", "2023년 9~11월"]

analysis_df["period_group"] = pd.Categorical(
    analysis_df["period_group"],
    categories=period_order,
    ordered=True
)

print("분석 대상 전체 유저 수:", analysis_df["user_id"].nunique())

print("\n기간별 첫 결제 유저 수")
display(
    analysis_df.groupby("period_group", observed=False)["user_id"]
    .nunique()
    .reset_index(name="첫 결제 유저 수")
)

print("\n월별 첫 결제 유저 수")
display(
    analysis_df["first_payment_month"]
    .value_counts()
    .sort_index()
)


In [ ]:
# =========================================================
# 2. 연도별 30일 내 해지 버튼 클릭률
# =========================================================

year_compare = (
    analysis_df
    .groupby("period_group", observed=False)
    .agg(
        first_payment_users=("user_id", "nunique"),
        cancel_30d_users=("cancel_30d", "sum")
    )
    .reset_index()
)

year_compare["cancel_30d_rate"] = round(
    year_compare["cancel_30d_users"] /
    year_compare["first_payment_users"] * 100,
    2
)

year_compare["non_cancel_30d_users"] = (
    year_compare["first_payment_users"] -
    year_compare["cancel_30d_users"]
)

print("[결과 1] 2022년 vs 2023년 30일 내 해지 버튼 클릭률")
display(year_compare)

# 연도 간 차이 계산
rate_2022 = year_compare.loc[
    year_compare["period_group"] == "2022년 9~11월",
    "cancel_30d_rate"
].iloc[0]

rate_2023 = year_compare.loc[
    year_compare["period_group"] == "2023년 9~11월",
    "cancel_30d_rate"
].iloc[0]

rate_diff = round(rate_2023 - rate_2022, 2)

print(f"2022년 30일 내 해지 버튼 클릭률: {rate_2022:.2f}%")
print(f"2023년 30일 내 해지 버튼 클릭률: {rate_2023:.2f}%")
print(f"2023년 - 2022년 차이: {rate_diff:+.2f}%p")


In [ ]:
# =========================================================
# 시각화 1. 연도별 30일 내 해지 버튼 클릭률
# =========================================================

plt.figure(figsize=(8, 5))

bars = plt.bar(
    year_compare["period_group"].astype(str),
    year_compare["cancel_30d_rate"]
)

plt.title("2022년 vs 2023년 30일 내 해지 버튼 클릭률")
plt.xlabel("첫 결제 기간")
plt.ylabel("30일 내 해지 버튼 클릭률(%)")
plt.ylim(0, max(year_compare["cancel_30d_rate"]) + 15)

for bar, rate, users in zip(
    bars,
    year_compare["cancel_30d_rate"],
    year_compare["cancel_30d_users"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{rate:.2f}%\n({int(users):,}명)",
        ha="center"
    )

plt.tight_layout()
plt.show()


2022년 9~11월 첫 결제 유저 1,333명 중 574명이 30일 내 해지 버튼을 클릭해, 클릭률은 **43.06%**로 나타났다.
2023년 같은 기간에는 첫 결제 유저 1,509명 중 377명이 해지 버튼을 클릭해, 클릭률은 **24.98%**였다.
2023년의 30일 내 해지 버튼 클릭률은 2022년보다 18.08%p 감소했다.
첫 결제 유저 수는 2023년에 더 많았지만, 해지 버튼 클릭 유저 수와 클릭률은 모두 감소했다.
따라서 2023년 신규 결제 유저는 2022년보다 첫 결제 후 30일 이내에 해지 의사를 보이는 비중이 낮아진 것으로 해석할 수 있다.
다만 해지 버튼 클릭 이후 실제 구독 해지가 완료되었는지는 확인되지 않으므로, 구독 유지율이 동일한 정도로 개선됐다고 단정할 수는 없다.


In [ ]:
# =========================================================
# 3. 월별 30일 내 해지 버튼 클릭률
# =========================================================

monthly_compare = (
    analysis_df
    .groupby(
        ["period_group", "month_num", "month_label"],
        observed=False
    )
    .agg(
        first_payment_users=("user_id", "nunique"),
        cancel_30d_users=("cancel_30d", "sum")
    )
    .reset_index()
)

# 그룹 조합 때문에 생길 수 있는 0명 행 제거
monthly_compare = monthly_compare[
    monthly_compare["first_payment_users"] > 0
].copy()

monthly_compare["cancel_30d_rate"] = round(
    monthly_compare["cancel_30d_users"] /
    monthly_compare["first_payment_users"] * 100,
    2
)

monthly_compare = monthly_compare.sort_values(
    ["period_group", "month_num"]
)

print("[결과 2] 월별 30일 내 해지 버튼 클릭률")
display(monthly_compare)


In [ ]:
# =========================================================
# 시각화 2. 9월·10월·11월 월별 클릭률 비교
# =========================================================

plt.figure(figsize=(9, 5))

for period in period_order:
    temp = monthly_compare[
        monthly_compare["period_group"] == period
    ].sort_values("month_num")

    plt.plot(
        temp["month_num"],
        temp["cancel_30d_rate"],
        marker="o",
        label=period
    )

    for _, row in temp.iterrows():
        plt.text(
            row["month_num"],
            row["cancel_30d_rate"] + 0.5,
            f'{row["cancel_30d_rate"]:.1f}%',
            ha="center"
        )

plt.title("2022년 vs 2023년 월별 30일 내 해지 버튼 클릭률")
plt.xlabel("첫 결제 월")
plt.ylabel("30일 내 해지 버튼 클릭률(%)")
plt.xticks([9, 10, 11], ["9월", "10월", "11월"])
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


2022년의 월별 클릭률은 9월 33.8%, 10월 32.8%, 11월 47.3%로 나타났다.
2023년은 9월 22.8%, 10월 22.5%, 11월 28.9%로, 모든 월에서 2022년보다 낮았다.
전년 동월과 비교하면 9월은 약 11.0%p, 10월은 약 10.3%p, 11월은 약 18.4%p 감소했다.
두 연도 모두 9월과 10월보다 11월의 해지 버튼 클릭률이 높게 나타났다.
특히 2022년 11월의 클릭률이 크게 상승했기 때문에, 기간 전체의 높은 클릭률에 11월이 상당한 영향을 준 것으로 보인다.
11월에 공통적으로 클릭률이 높아진 원인이 가격, 프로모션 종료, 결제 주기, 콘텐츠 이용 경험 등의 계절적 요인과 관련 있는지 추가 확인할 필요가 있다.


In [ ]:
# =========================================================
# 할인율 x 30일 내 해지 버튼 클릭률 산점도
# 기간: 2022년 9~11월 vs 2023년 9~11월
# 기준: 첫 결제 유저 기준
# x축: 일별 평균 할인율
# y축: 일별 30일 내 해지 버튼 클릭률
# 색상: 무료체험 폐지 전 / 후
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# -----------------------------
# 1. 한글 폰트 설정
# -----------------------------
try:
    import matplotlib.font_manager as fm

    font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

    if not os.path.exists(font_path):
        pass  # 환경 설치 명령 제거. requirements-analysis.txt를 참고하세요.

    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
    plt.rc("axes", unicode_minus=False)

except:
    print("한글 폰트 설정 중 오류가 발생했지만 분석은 계속 진행됩니다.")


# -----------------------------
# 2. 데이터 불러오기
# -----------------------------
file_path = "/content/dm_user_subscription_retention_core.csv"

if not os.path.exists(file_path):
    candidates = glob.glob("/content/*subscription*retention*.csv")
    if len(candidates) > 0:
        file_path = candidates[0]
    else:
        raise FileNotFoundError("dm_user_subscription_retention_core.csv 파일을 찾을 수 없습니다.")

df = pd.read_csv(file_path)

print("데이터 크기:", df.shape)
display(df.head())


# -----------------------------
# 3. 날짜 변환 및 분석 기간 필터링
# -----------------------------
df["first_payment_time"] = pd.to_datetime(df["first_payment_time"], errors="coerce")
df["first_payment_month"] = df["first_payment_time"].dt.to_period("M").astype(str)
df["first_payment_date"] = df["first_payment_time"].dt.date

# 분석 기간: 2022년 9~11월, 2023년 9~11월
target_months = [
    "2022-09", "2022-10", "2022-11",
    "2023-09", "2023-10", "2023-11"
]

analysis_df = df[df["first_payment_month"].isin(target_months)].copy()

analysis_df["period_group"] = np.where(
    analysis_df["first_payment_month"].str.startswith("2022"),
    "무료체험 폐지 전",
    "무료체험 폐지 후"
)

print("분석 대상 데이터 크기:", analysis_df.shape)
display(analysis_df["first_payment_month"].value_counts().sort_index())


# -----------------------------
# 4. 할인율 계산
# -----------------------------
# 가능한 할인 관련 컬럼 확인
discount_cols = [
    "discount_rate",
    "first_discount_rate",
    "first_discount_amount",
    "discount_amount",
    "first_plan_price",
    "plan_price",
    "first_paid_amount",
    "paid_amount"
]

print("현재 데이터에 있는 할인 관련 컬럼:")
print([col for col in discount_cols if col in analysis_df.columns])

# 숫자형 변환
for col in discount_cols:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

# 할인율 계산 우선순위
if "discount_rate" in analysis_df.columns:
    analysis_df["calc_discount_rate"] = analysis_df["discount_rate"]

elif "first_discount_rate" in analysis_df.columns:
    analysis_df["calc_discount_rate"] = analysis_df["first_discount_rate"]

elif "first_discount_amount" in analysis_df.columns and "first_plan_price" in analysis_df.columns:
    analysis_df["calc_discount_rate"] = (
        analysis_df["first_discount_amount"] / analysis_df["first_plan_price"] * 100
    )

elif "first_plan_price" in analysis_df.columns and "first_paid_amount" in analysis_df.columns:
    analysis_df["calc_discount_rate"] = (
        (analysis_df["first_plan_price"] - analysis_df["first_paid_amount"]) /
        analysis_df["first_plan_price"] * 100
    )

elif "plan_price" in analysis_df.columns and "paid_amount" in analysis_df.columns:
    analysis_df["calc_discount_rate"] = (
        (analysis_df["plan_price"] - analysis_df["paid_amount"]) /
        analysis_df["plan_price"] * 100
    )

else:
    raise ValueError(
        "할인율을 계산할 수 있는 컬럼이 없습니다. "
        "discount_rate, first_discount_rate, first_discount_amount, "
        "first_plan_price, first_paid_amount 중 하나가 필요합니다."
    )

# 이상치 처리
analysis_df["calc_discount_rate"] = (
    analysis_df["calc_discount_rate"]
    .replace([np.inf, -np.inf], np.nan)
    .clip(lower=0, upper=100)
)

print("할인율 요약")
display(
    analysis_df.groupby("period_group")["calc_discount_rate"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)


# -----------------------------
# 5. 해지 버튼 클릭 여부 확인
# -----------------------------
# 기존 분석 기준: cancel_30d
analysis_df["cancel_30d"] = (
    pd.to_numeric(analysis_df["cancel_30d"], errors="coerce")
    .fillna(0)
    .astype(int)
)

print("30일 내 해지 버튼 클릭 여부")
display(analysis_df["cancel_30d"].value_counts())


# -----------------------------
# 6. 일별 평균 할인율과 해지 버튼 클릭률 집계
# -----------------------------
daily_discount_cancel = (
    analysis_df
    .groupby(["period_group", "first_payment_date"], as_index=False)
    .agg(
        first_payment_users=("user_id", "nunique"),
        avg_discount_rate=("calc_discount_rate", "mean"),
        cancel_30d_users=("cancel_30d", "sum")
    )
)

daily_discount_cancel["cancel_30d_rate"] = (
    daily_discount_cancel["cancel_30d_users"] /
    daily_discount_cancel["first_payment_users"] * 100
)

# 표본 수가 너무 작은 날짜 제외
daily_discount_cancel = daily_discount_cancel[
    daily_discount_cancel["first_payment_users"] >= 5
].copy()

print("[결과] 일별 평균 할인율과 30일 내 해지 버튼 클릭률")
display(daily_discount_cancel.head())


# -----------------------------
# 7. 산점도 + 추세선
# -----------------------------
plt.figure(figsize=(9, 6))

colors = {
    "무료체험 폐지 전": "tab:blue",
    "무료체험 폐지 후": "tab:orange"
}

for period in ["무료체험 폐지 전", "무료체험 폐지 후"]:
    temp = daily_discount_cancel[
        daily_discount_cancel["period_group"] == period
    ].dropna(subset=["avg_discount_rate", "cancel_30d_rate"])

    # 산점도
    plt.scatter(
        temp["avg_discount_rate"],
        temp["cancel_30d_rate"],
        s=temp["first_payment_users"] * 3,
        alpha=0.6,
        label=period,
        color=colors[period]
    )

    # 추세선
    if len(temp) >= 2:
        z = np.polyfit(temp["avg_discount_rate"], temp["cancel_30d_rate"], 1)
        p = np.poly1d(z)

        x_line = np.linspace(
            temp["avg_discount_rate"].min(),
            temp["avg_discount_rate"].max(),
            100
        )

        plt.plot(
            x_line,
            p(x_line),
            linestyle="--",
            color=colors[period]
        )

plt.title("무료체험 폐지 전/후 할인율과 30일 내 해지 버튼 클릭률")
plt.xlabel("일별 평균 할인율(%)")
plt.ylabel("30일 내 해지 버튼 클릭률(%)")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# -----------------------------
# 8. 상관계수 확인
# -----------------------------
corr_result = []

for period in ["무료체험 폐지 전", "무료체험 폐지 후"]:
    temp = daily_discount_cancel[
        daily_discount_cancel["period_group"] == period
    ].dropna(subset=["avg_discount_rate", "cancel_30d_rate"])

    corr_result.append({
        "period_group": period,
        "상관계수": round(temp["avg_discount_rate"].corr(temp["cancel_30d_rate"]), 3),
        "분석일수": len(temp),
        "평균 할인율": round(temp["avg_discount_rate"].mean(), 2),
        "평균 해지 버튼 클릭률": round(temp["cancel_30d_rate"].mean(), 2)
    })

corr_result = pd.DataFrame(corr_result)

print("[결과] 할인율과 30일 내 해지 버튼 클릭률 상관계수")
display(corr_result)


In [ ]:
# =========================================================
# 시각화 3. 첫 결제 유저와 30일 내 해지 버튼 클릭 유저 수
# =========================================================

x = np.arange(len(year_compare))
width = 0.35

plt.figure(figsize=(9, 5))

bars1 = plt.bar(
    x - width / 2,
    year_compare["first_payment_users"],
    width=width,
    label="첫 결제 유저 수"
)

bars2 = plt.bar(
    x + width / 2,
    year_compare["cancel_30d_users"],
    width=width,
    label="30일 내 해지 버튼 클릭 유저 수"
)

plt.title("기간별 첫 결제 유저 수와 해지 버튼 클릭 유저 수")
plt.xlabel("첫 결제 기간")
plt.ylabel("유저 수")
plt.xticks(x, year_compare["period_group"].astype(str))
plt.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 10,
            f"{int(bar.get_height()):,}",
            ha="center"
        )

plt.tight_layout()
plt.show()


첫 결제 유저 수는 2022년 1,333명에서 2023년 1,509명으로 176명 증가했다.
반면 30일 내 해지 버튼 클릭 유저 수는 2022년 574명에서 2023년 377명으로 197명 감소했다.
즉, 2023년에는 신규 결제 유저의 규모가 커졌음에도 해지 의향 행동을 보인 유저의 절대적인 수는 줄어들었다.
따라서 2023년 클릭률 감소는 단순히 첫 결제 유저 수가 늘어나 분모가 커진 효과만으로 설명하기 어렵다.
첫 결제 유저 증가와 해지 버튼 클릭 유저 감소가 동시에 나타났다는 점에서, 2023년 신규 결제 유저의 초기 구독 경험이 상대적으로 개선되었을 가능성이 있다.


In [ ]:
# =========================================================
# 4. 7일 내 취소율과 30일 내 취소율 비교
# 초기 취소가 얼마나 빠르게 발생하는지 확인
# =========================================================

cancel_window_compare = (
    analysis_df
    .groupby("period_group", observed=False)
    .agg(
        first_payment_users=("user_id", "nunique"),
        cancel_7d_users=("cancel_7d", "sum"),
        cancel_30d_users=("cancel_30d", "sum")
    )
    .reset_index()
)

cancel_window_compare["cancel_7d_rate"] = round(
    cancel_window_compare["cancel_7d_users"] /
    cancel_window_compare["first_payment_users"] * 100,
    2
)

cancel_window_compare["cancel_30d_rate"] = round(
    cancel_window_compare["cancel_30d_users"] /
    cancel_window_compare["first_payment_users"] * 100,
    2
)

print("[결과 3] 7일 내·30일 내 해지 버튼 클릭률")
display(cancel_window_compare)


In [ ]:
# =========================================================
# 시각화 4. 7일 내 vs 30일 내 해지 버튼 클릭률
# =========================================================

x = np.arange(len(cancel_window_compare))
width = 0.35

plt.figure(figsize=(9, 5))

bars1 = plt.bar(
    x - width / 2,
    cancel_window_compare["cancel_7d_rate"],
    width=width,
    label="7일 내 클릭률"
)

bars2 = plt.bar(
    x + width / 2,
    cancel_window_compare["cancel_30d_rate"],
    width=width,
    label="30일 내 클릭률"
)

plt.title("7일 내 vs 30일 내 해지 버튼 클릭률")
plt.xlabel("첫 결제 기간")
plt.ylabel("해지 버튼 클릭률(%)")
plt.xticks(x, cancel_window_compare["period_group"].astype(str))
plt.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{bar.get_height():.2f}%",
            ha="center"
        )

plt.tight_layout()
plt.show()


2022년에는 첫 결제 후 7일 내 클릭률이 14.18%, 30일 내 클릭률이 43.06%였다.
2023년에는 7일 내 클릭률이 10.01%, 30일 내 클릭률이 24.98%로 나타났다.
두 기간 모두 7일 이내보다 30일 이내 클릭률이 크게 높아, 해지 의향이 결제 직후에만 발생하는 것이 아니라 이후 이용 과정에서도 계속 발생한다는 점을 확인할 수 있다.
7일에서 30일 사이 클릭률 증가 폭은 2022년 28.88%p, 2023년 14.97%p였다.
특히 2022년에는 결제 후 8~30일 사이에 해지 의향이 크게 증가한 것으로 보인다.
따라서 해지 방지를 위해서는 결제 직후 온보딩뿐 아니라, 첫 결제 후 2 4주 동안 콘텐츠 추천, 학습 리마인드, 이용 가치 안내 등의 유지 관리가 필요하다.


In [ ]:
# =========================================================
# 5. 첫 정기 갱신 전에 취소 버튼을 클릭한 유저 비율
# =========================================================

before_renew_compare = (
    analysis_df
    .groupby("period_group", observed=False)
    .agg(
        first_payment_users=("user_id", "nunique"),
        cancel_before_renew_users=("cancel_before_first_renew", "sum")
    )
    .reset_index()
)

before_renew_compare["cancel_before_renew_rate"] = round(
    before_renew_compare["cancel_before_renew_users"] /
    before_renew_compare["first_payment_users"] * 100,
    2
)

print("[결과 4] 첫 갱신 전 해지 버튼 클릭률")
display(before_renew_compare)


In [ ]:
# =========================================================
# 시각화 5. 첫 갱신 전 해지 버튼 클릭률
# =========================================================

plt.figure(figsize=(8, 5))

bars = plt.bar(
    before_renew_compare["period_group"].astype(str),
    before_renew_compare["cancel_before_renew_rate"]
)

plt.title("첫 정기 갱신 전 해지 버튼 클릭률")
plt.xlabel("첫 결제 기간")
plt.ylabel("첫 갱신 전 클릭률(%)")
plt.ylim(
    0,
    max(before_renew_compare["cancel_before_renew_rate"]) + 15
)

for bar, rate, users in zip(
    bars,
    before_renew_compare["cancel_before_renew_rate"],
    before_renew_compare["cancel_before_renew_users"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{rate:.2f}%\n({int(users):,}명)",
        ha="center"
    )

plt.tight_layout()
plt.show()


첫 정기 갱신 전에 해지 버튼을 클릭한 유저 비율은 2022년 41.86%, 2023년 26.04%로 나타났다.
2023년의 첫 갱신 전 클릭률은 2022년보다 15.82%p 낮았다.
이는 2023년 유저가 첫 번째 정기결제 시점에 도달하기 전에 해지 의사를 보이는 비중이 감소했다는 의미다.
첫 갱신 전 클릭은 정기구독 유지 여부에 직접적으로 연결될 가능성이 높은 행동이므로, 이 지표의 감소는 2023년 구독 유지 측면에서 긍정적인 신호로 볼 수 있다.
다만 클릭 이후 실제 취소 여부와 첫 갱신 성공 여부를 함께 연결해야 실제 유지 효과를 확정할 수 있다.


In [ ]:
# =========================================================
# 6. 30일 내 취소 유저의 취소 시점 분포
# =========================================================

cancel_timing_df = analysis_df[
    (analysis_df["cancel_30d"] == 1) &
    (analysis_df["days_to_first_cancel"].between(0, 30))
].copy()

cancel_timing_df["cancel_timing_group"] = pd.cut(
    cancel_timing_df["days_to_first_cancel"],
    bins=[-0.01, 7, 14, 30],
    labels=["7일 이내", "8~14일", "15~30일"]
)

cancel_timing_summary = (
    cancel_timing_df
    .groupby(
        ["period_group", "cancel_timing_group"],
        observed=False
    )
    .agg(cancel_users=("user_id", "nunique"))
    .reset_index()
)

cancel_timing_summary["group_total"] = (
    cancel_timing_summary
    .groupby("period_group", observed=False)["cancel_users"]
    .transform("sum")
)

cancel_timing_summary["ratio"] = round(
    cancel_timing_summary["cancel_users"] /
    cancel_timing_summary["group_total"] * 100,
    2
)

print("[결과 5] 30일 내 취소 시점 분포")
display(cancel_timing_summary)


In [ ]:
# =========================================================
# 7. 구독 생애주기 유형 구성 비교
# =========================================================

lifecycle_labels = {
    "renew_then_cancel": "갱신 후 취소",
    "early_cancel_before_renew": "첫 갱신 전 조기 취소",
    "retention_core": "핵심 유지 유저",
    "silent_no_followup": "후속 행동 없음",
    "cancel_then_resubscribe": "취소 후 재구독",
    "other": "기타"
}

analysis_df["lifecycle_label"] = (
    analysis_df["subscription_lifecycle_segment"]
    .map(lifecycle_labels)
    .fillna(analysis_df["subscription_lifecycle_segment"])
)

lifecycle_summary = (
    analysis_df
    .groupby(
        ["period_group", "lifecycle_label"],
        observed=False
    )
    .agg(users=("user_id", "nunique"))
    .reset_index()
)

lifecycle_summary["period_total"] = (
    lifecycle_summary
    .groupby("period_group", observed=False)["users"]
    .transform("sum")
)

lifecycle_summary["ratio"] = round(
    lifecycle_summary["users"] /
    lifecycle_summary["period_total"] * 100,
    2
)

print("[결과 6] 구독 생애주기 유형 구성")
display(lifecycle_summary.sort_values(
    ["period_group", "ratio"],
    ascending=[True, False]
))


In [ ]:
# =========================================================
# 시각화 7. 구독 생애주기 유형 구성
# =========================================================

lifecycle_pivot = lifecycle_summary.pivot(
    index="period_group",
    columns="lifecycle_label",
    values="ratio"
).fillna(0)

lifecycle_pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 6)
)

plt.title("2022년 vs 2023년 구독 생애주기 유형 구성")
plt.xlabel("첫 결제 기간")
plt.ylabel("유저 비중(%)")
plt.xticks(rotation=0)
plt.legend(
    title="구독 생애주기",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


2022년에는 갱신 후 취소와 첫 갱신 전 조기 취소 유형의 비중이 상대적으로 크게 나타났다.
반면 2023년에는 이 두 취소 유형의 비중이 감소하고, 핵심 유지 유저와 후속 행동 없음 유형의 비중이 증가했다.
핵심 유지 유저 비중이 증가한 것은 2023년 구독 유지 상태가 개선되었을 가능성을 보여준다.
하지만 후속 행동 없음 유형도 함께 증가했기 때문에, 모든 유저가 적극적인 유지 행동으로 이동했다고 해석하기는 어렵다.
후속 행동 없음은 취소·갱신·재구독 이벤트가 관측되지 않은 상태이므로, 실제 유지 중인지 단순 비활성 상태인지 추가 확인이 필요하다.
취소 후 재구독 유형은 두 연도 모두 비중이 작아, 일단 취소 의향을 보인 유저를 다시 구독으로 회복시키는 비율은 높지 않은 것으로 보인다.
따라서 재구독 유도보다 첫 갱신 전에 해지 의향이 생기지 않도록 관리하는 전략이 더 중요할 가능성이 있다.


In [ ]:
# =========================================================
# 8. 두 연도의 30일 내 클릭률 차이 통계 검정
# =========================================================

from statsmodels.stats.proportion import proportions_ztest

row_2022 = year_compare[
    year_compare["period_group"] == "2022년 9~11월"
].iloc[0]

row_2023 = year_compare[
    year_compare["period_group"] == "2023년 9~11월"
].iloc[0]

count = np.array([
    row_2022["cancel_30d_users"],
    row_2023["cancel_30d_users"]
])

nobs = np.array([
    row_2022["first_payment_users"],
    row_2023["first_payment_users"]
])

z_stat, p_value = proportions_ztest(count, nobs)

print("두 비율 Z 검정")
print(f"Z 통계량: {z_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("두 연도의 30일 내 해지 버튼 클릭률 차이는 통계적으로 유의합니다.")
else:
    print("두 연도의 30일 내 해지 버튼 클릭률 차이는 통계적으로 유의하지 않습니다.")


In [ ]:
# =========================================================
# 9. 최종 결과 자동 요약
# =========================================================

users_2022 = int(row_2022["first_payment_users"])
cancel_2022 = int(row_2022["cancel_30d_users"])
rate_2022 = float(row_2022["cancel_30d_rate"])

users_2023 = int(row_2023["first_payment_users"])
cancel_2023 = int(row_2023["cancel_30d_users"])
rate_2023 = float(row_2023["cancel_30d_rate"])

diff = round(rate_2023 - rate_2022, 2)

print("========== 최종 요약 ==========")
print(f"2022년 9~11월 첫 결제 유저: {users_2022:,}명")
print(f"2022년 30일 내 해지 버튼 클릭 유저: {cancel_2022:,}명")
print(f"2022년 30일 내 해지 버튼 클릭률: {rate_2022:.2f}%")
print()
print(f"2023년 9~11월 첫 결제 유저: {users_2023:,}명")
print(f"2023년 30일 내 해지 버튼 클릭 유저: {cancel_2023:,}명")
print(f"2023년 30일 내 해지 버튼 클릭률: {rate_2023:.2f}%")
print()
print(f"2023년과 2022년의 차이: {diff:+.2f}%p")
print("================================")

print("""
해석 주의:
- 이 지표는 실제 해지 완료율이 아니라 해지 버튼 클릭 행동 기준이다.
- 비교 기간은 2022년과 2023년의 9월~11월로 각각 3개월이다.
- 동일한 30일 관측 기간을 적용해 연도별 관측 기간 차이를 줄였다.
""")


## 종합 분석


2023년 9~11월의 30일 내 해지 버튼 클릭률은 24.98%로, 2022년 같은 기간의 43.06%보다 18.08%p 낮았다.
2023년에는 첫 결제 유저가 증가했지만 해지 버튼 클릭 유저의 수는 감소해, 초기 해지 의향 행동이 전반적으로 줄어든 것으로 나타났다.
월별로도 2023년의 클릭률이 9월, 10월, 11월 모두 전년보다 낮았으며, 특히 11월의 차이가 가장 컸다.
7일 내 클릭률뿐 아니라 30일 내 클릭률과 첫 정기 갱신 전 클릭률도 함께 감소해, 결제 직후와 첫 갱신 이전 구간 모두에서 2023년의 해지 의향이 낮아진 것으로 볼 수 있다.
통계 검정에서도 두 연도의 차이가 유의하게 나타나, 2023년의 감소는 단순한 우연으로 보기 어렵다.
다만 본 분석은 실제 해지 완료가 아닌 해지 버튼 클릭 행동을 기준으로 하므로, 최종적으로는 실제 갱신 성공 여부, 결제 유지 기간, 결제금액과 함께 확인해야 한다.
현재 결과만 보면 2023년에는 신규 결제 유저의 초기 구독 유지 흐름이 2022년보다 개선된 것으로 보이지만, 후속 행동 없음 유저의 증가 원인과 실제 구독 상태를 추가로 확인할 필요가 있다.


# Appendix D. 김은기 팀원 개별 분석 원본

> 본 섹션은 팀원별 개별 분석 산출물을 원본 흐름에 가깝게 보존한 제출용 부록입니다.
> 전체 노트북은 순차 실행을 전제로 하지 않으며, 분석 과정과 산출물 취합을 목적으로 합니다.


In [ ]:
# 저장소 정리: Google Drive 자동 연결을 제거했습니다. 로컬 입력 파일을 사용하세요.


In [ ]:
# 라이브러리 로드
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# 그래프 한글 사용을 위한 라이브러리

pass  # 환경 설치 명령 제거. requirements-analysis.txt를 참고하세요.


In [ ]:
# 추가적인 세팅
import koreanize_matplotlib

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


# 데이터 불러오기


In [ ]:
# 데이터 불러오기
content_7d_performance_dm = pd.read_csv("data/private/dm_content_7d_performance.csv")
pre_content_7d_performance_dm = content_7d_performance_dm[content_7d_performance_dm['period_group'] == 'pre_2022_05_11']
user_activation_payment_dm = pd.read_csv("data/private/dm_user_activation_payment.csv")
pre_user_activation_payment_dm = user_activation_payment_dm[user_activation_payment_dm['period_group'] == 'pre_2022_05_11']
event_log_2022_05 = pd.read_csv("data/private/event_log_thin_2022_05.csv")
event_log_2022_06 = pd.read_csv("data/private/event_log_thin_2022_06.csv")
event_log_2022_07 = pd.read_csv("data/private/event_log_thin_2022_07.csv")
event_log_2022_08 = pd.read_csv("data/private/event_log_thin_2022_08.csv")
event_log_2022_09 = pd.read_csv("data/private/event_log_thin_2022_09.csv")
event_log_2022_10 = pd.read_csv("data/private/event_log_thin_2022_10.csv")
event_log_2022_11 = pd.read_csv("data/private/event_log_thin_2022_11.csv")
event_log_2022_12 = pd.read_csv("data/private/event_log_thin_2022_12.csv")
event_log_2022 = pd.concat([event_log_2022_05,event_log_2022_06,event_log_2022_07,event_log_2022_08,event_log_2022_09,event_log_2022_10,event_log_2022_11,event_log_2022_12])


# 파트 1
- 왜 무료 체험 서비스를 폐지했을까?


## 무료 체험 서비스 폐지 이전 가입 후 7일 이내 유저의 레슨 수강 수 분포
- 가입 직후 사이트에 매력을 느껴 활용했는가에 대해 확인한 분석입니다.
- 무료 체험 서비스가 일주일이기 때문에 가입 직후 서비스를 신청했을 거란 가정을 두고 확인하고자 하였습니다.


In [ ]:
# 가입 후 7일 내 레슨 수강 수 별 유저 수 테이블 생성
lesson_complete_attribution = pre_user_activation_payment_dm.groupby('unique_lesson_complete_7d')['user_id'].count().reset_index()
# 레슨 수강 수에 따른 그룹 생성
lesson_complete_attribution['complete_lesson_group'] = pd.cut(
    lesson_complete_attribution['unique_lesson_complete_7d'],
    bins=[-1, 0, 20, 50, 100, 200, float('inf')],
    labels = ['0', '1~20', '21~50', '51~100', '101~200', '200+']
)
# 그룹 별 유저 수 확인
lesson_complete_attribution_df = lesson_complete_attribution.groupby('complete_lesson_group').agg({
    'user_id' : 'sum',
    'unique_lesson_complete_7d' : 'mean'
}).reset_index()
lesson_complete_attribution_df


In [ ]:
# 간단한 시각화
plt.pie(
    lesson_complete_attribution_df['user_id'],
    labels=lesson_complete_attribution_df['complete_lesson_group'],
    autopct='%1.1f%%'
)
plt.show()


- 신규 가입 후 7일동안 레슨 수강을 하지 않은 유저는 약 91%, 수강한 유저는 약 9%로 확인
- 1~20회 수강한 그룹이 약 3.8%, 21~50회 수강한 그룹이 약 2.3%, 51~100회 수강한 그룹이 약 1.8%로 수강을 시작하면 고활동 유저로 전환될 가능성이 있는 것으로 보임
- 레슨 완료 수가 200 이상인부분을 하드컷 하면 예쁘게 보일 수 있을 것 같음


## 무료 체험 서비스 폐지 이전 가입 후 유저의 결제 전 컨텐츠 수강 수 분포 (30일 내)
- 결제 유저의 30일 내 컨텐츠 완료 수 분포를 확인하여 컨텐츠 경험 / 완료가 결제에 유의미한 영향을 미쳤을 지 확인하기 위해 분석 진행하였습니다
- 유의미한 영향을 미치지 않는 것으로 보여 세부적인 분석으로 나아가지 않았습니다
- 필요하다면 추후 분석할 예정입니다


In [ ]:
# 분석에 필요한 칼럼들을 이용한 테이블 생성
temp_paid = pre_user_activation_payment_dm[['user_id', 'lesson_complete_prepay_30d', 'content_end_prepay_30d', 'unique_content_prepay_30d', 'days_to_payment', 'paid_30d', 'first_plan_price']]
# 결제 여부와 컨텐츠 완료 수 별 유저수 확인
temp_paid.groupby(['paid_30d', 'days_to_payment']).agg({
    'user_id' : 'count',
    'lesson_complete_prepay_30d' : 'mean',
    'content_end_prepay_30d' : 'mean'
}).reset_index()
temp_paid_content_complete = temp_paid.groupby(['paid_30d', 'content_end_prepay_30d']).agg({
    'user_id' : 'count',
    'days_to_payment' : 'mean',
}).reset_index()
pay_by_cotent_complete = temp_paid_content_complete[temp_paid_content_complete['paid_30d'] == 1]
# 구간 별 유저 수 확인
bin = [-1, 0, 1, 2, 3, 4, 5, 6, float('inf')]
label = ['0', '1', '2', '3', '4', '5', '6', '7+']

pay_by_cotent_complete['count_group'] = pd.cut(
    pay_by_cotent_complete['content_end_prepay_30d'],
    bins=bin,
    labels=label)
pay_by_cotent_complete


In [ ]:
temp_paid_rate_content_complete = temp_paid.groupby(['content_end_prepay_30d']).agg({
    'user_id' : 'count',
    'paid_30d' : 'mean',
    'days_to_payment' : 'mean',
}).reset_index()
temp_paid_rate_content_complete


- 컨텐츠를 수강하지 않았을 때의 결제 기여도가 높은 것으로 보임
- 컨텐츠 완료 수에 따른 결제율 변화가 뚜렷하게 확인되지 않기 때문에 추후 분석 진행X
- 컨텐츠 완료 수가 0일 때 결제 기여도가 높았음을 이유로 무료 체험 서비스를 폐지했을 가능성이 있음


## 결제 30일 전에 경험한 컨텐츠 수
- 컨텐츠 수강완료는 너무 타이트한 조건일 수 있기 때문에 경험한 컨텐츠 수로 추가 확인하였습니다.
- 유의미한 영향을 미치지 않는 것으로 보여 세부적인 분석으로 나아가지 않았습니다


In [ ]:
# 결제 여부와 컨텐츠 경험 수 별 유저수 확인
temp_paid_content = temp_paid.groupby(['paid_30d', 'unique_content_prepay_30d']).agg({
    'user_id' : 'count',
    'days_to_payment' : 'mean',
}).reset_index()
# 결제
pay_by_content = temp_paid_content[temp_paid_content['paid_30d'] == 1]
# 구간 별 유저 수 확인
bin = [-1, 0, 1, 2, 3, 4, 5, 6, float('inf')]
label = ['0', '1', '2', '3', '4', '5', '6', '7+']

pay_by_content['count_group'] = pd.cut(
    pay_by_content['unique_content_prepay_30d'],
    bins=bin,
    labels=label)
pay_by_content


## 결제 전 수강한 레슨 수에 따른 결제율


In [ ]:
# 필요한 칼럼 추출
temp_lesson_paid = pre_user_activation_payment_dm[['user_id', 'lesson_complete_prepay_30d', 'paid_30d']]

# 결제 전 일주일 내 수강한 레슨 수 와 첫 결제 별 유저 수 분포
temp_lesson_paid_table = temp_lesson_paid.groupby(['lesson_complete_prepay_30d', 'paid_30d']).agg({
    'user_id' : 'count'
}).reset_index()

# 반복문을 이용한 범주 나누기(cut 안에 반복문 사용 불가)
# 라벨 반복문으로 생성
label = ['0']

for i in range(1, 100, 2):
    label.append(f'{i}~{i+1}')

label.append('100+')

# 구간 생성 / list 작성한 이력이 있으면 list 삭제하고 다시 만들어 주어야 함
bin = [-1] + list(range(0, 101, 2)) + [float('inf')]

# 구간과 라벨을 이용하여 컷
temp_lesson_paid_table['complete_lesson_group'] = pd.cut(
    temp_lesson_paid_table['lesson_complete_prepay_30d'],
    bins = bin,
    labels = label
  )
# 레슨 수강 그룹별 결제율 테이블
temp5 = temp_lesson_paid_table.groupby(['complete_lesson_group', 'paid_30d'])['user_id'].sum().reset_index()
temp6 = temp5.groupby('complete_lesson_group')['user_id'].sum().reset_index()
temp6 = temp6.rename(columns={'user_id' : 'sum_user_count'})
complete_lesson_before_pay = temp5.merge(temp6, how='left', on = 'complete_lesson_group')
complete_lesson_before_pay['rate'] = round((complete_lesson_before_pay['user_id'] / complete_lesson_before_pay['sum_user_count']), 3) * 100
complete_lesson_before_pay_rate = complete_lesson_before_pay[complete_lesson_before_pay['paid_30d'] == 1]
complete_lesson_before_pay_rate


In [ ]:
# 레슨 수강 그룹별 결제율 시각화(전구간)
plt.figure(figsize=(12, 6))

ax = sns.lineplot(
    data=complete_lesson_before_pay_rate,
    x='complete_lesson_group',
    y='rate',
    marker='o'
)

''' 좌표값 표시
 for x, y in zip(

    complete_lesson_before_pay_rate['complete_lesson_group'],
    complete_lesson_before_pay_rate['rate']
):
    ax.annotate(
        f'{y:.1f}%',      # 퍼센트 표시
        (x, y),
        textcoords="offset points",
        xytext=(0, 8),   # 점에서 위로 8pt
        ha='center'
    ) '''

# 그래프 가독성 좋게 그리기
plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.xlabel('결제 이전 레슨 수강 수')
plt.ylabel('결제율')
plt.title('결제 이전 레슨 수강 수 별 결제율',
          fontsize=20)

plt.tight_layout()
plt.show()


- 결제 이전 수강 완료한 레슨 수가 0개에서 11~12개 그룹까지 상승하는 모습을 보이지만 이후에는 상승과 하락을 반복하는 패턴을 보임
- 그룹 결제 전환율에 편차가 큰 것으로 보이며 이후 tableau를 이용하여 추세선을 그리는 것이 필요하다 판단  


In [ ]:
# 레슨 수강 그룹별 결제율 시각화(초반 구간)
partial_complete_lesson_before_pay_rate = complete_lesson_before_pay_rate.head(11)

plt.figure(figsize=(12, 6))

ax = sns.lineplot(
    data = partial_complete_lesson_before_pay_rate,
    x = 'complete_lesson_group',
    y = 'rate',
    marker = 'o'
)

# 좌표값 표시
for x, y in zip(
    partial_complete_lesson_before_pay_rate['complete_lesson_group'],
    partial_complete_lesson_before_pay_rate['rate']
):
    ax.annotate(
        f'{y:.1f}%',      # 퍼센트 표시
        (x, y),
        textcoords="offset points",
        xytext=(0, 8),   # 점에서 위로 8pt
        ha='center'
    )

# 그래프 가독성 좋게 그리기
plt.grid(axis='y', alpha=0.3)

plt.xlabel('결제 이전 레슨 수강 수')
plt.ylabel('결제율')
plt.title('결제 이전 레슨 수강 수 별 결제율',
          fontsize=20)

plt.ylim(0, 70)
plt.tight_layout()
plt.show()


- 무료 체험 서비스 폐지 이후보다 레슨 수강 완료 후 결제율이 큰 폭으로 상승하는 것을 확인할 수 있음
- 무료 체험 서비스 전 레슨 수강이 결제 전환율에 더 큰 영향을 미쳤지만 무료 체험 서비스가 수강으로의 전환에 유의미한 영향을 못주었을 가능성이 있음


## 수강한 레슨 수 별 유저 행동 비교
- 여기 이상하게 나와서 확인이 필요할 것 같습니다
- 코드가 잘못된 것 같은데 문제 확인이 어렵네요
- 이후 관련 지표는 표의 형태로 작성 해두었습니다


In [ ]:
def binary(x):
  if x == 0:
    return 0
  else:
    return 1

temp = pre_user_activation_payment_dm[['user_id', 'paid_30d', 'unique_lesson_complete_7d', 'question_click_7d']]
temp['question_click_7d'] = temp['question_click_7d'].apply(lambda x: binary(x))

temp['lesson_group'] = pd.cut(
    temp['unique_lesson_complete_7d'],
    bins = [-1, 0, 6, 12, float('inf')],
    labels = ['0', '1~6', '7~12', '12+']
  )

lesson_group_paid_question = temp.groupby('lesson_group').agg(
    가입_유저_수=('user_id', 'nunique'),
    첫_결제율=('paid_30d', 'mean'),
    질문박스_사용율=('question_click_7d', 'mean')
).reset_index()


In [ ]:
# 시각화
tidy_df = pd.melt(lesson_group_paid_question,
                  id_vars=['lesson_group'],
                  value_vars=['첫_결제율', '질문박스_사용율'],
                  var_name='지표종류',
                  value_name='비율')

# 3. 만약 0.08, 0.15 같은 소수점을 8%, 15% 형태로 시각화하고 싶다면 100을 곱해줍니다.
tidy_df['비율'] = tidy_df['비율'] * 100

# 4. 그래프 그리기
plt.figure(figsize=(12, 6))

ax = sns.barplot(data=tidy_df,
            x='lesson_group',  # x축 (0과 1)
            y='비율',               # 세로축 높이 (%)
            hue='지표종류')         # 막대 색깔 분해 (첫_결제율 vs 질문박스_사용율)

plt.title('결제 전 레슨 수강 그룹 별 주요 지표 비교', fontsize=14, pad=15)
plt.ylabel('비율 (%)', fontsize=12)
plt.xlabel('레슨 수강 그룹', fontsize=12)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=11)

plt.xticks([0, 1, 2, 3], ['미수강', '레슨 1~6개 수강', '레슨 7~12개 수강', '레슨 13개 이상 수강'])

plt.ylim(0, 100)
plt.tight_layout()
plt.show()


- 결제율 부분이 이상하게 나와서 확인이 필요할 것 같습니다


## 첫 레슨 완료 여부 별 주요 행동 비율


In [ ]:
temp = pre_user_activation_payment_dm[['user_id', 'paid_30d', 'lesson_complete_prepay_7d', 'question_click_7d']]
# 활성화 여부로 칼럼 변경
temp['lesson_activation'] = temp['lesson_complete_prepay_7d'].apply(lambda x: binary(x))
temp['question_click_7d'] = temp['question_click_7d'].apply(lambda x: binary(x))
# 고활동 저활동 유저 분리
temp['lesson_complete_prepay_7d'] = pd.cut(
    temp['lesson_complete_prepay_7d'],
    bins = [-1, 6, float('inf')],
    labels = ['row', 'high']
)
lesson_activarion_temp = temp.groupby('lesson_activation').agg(
    가입_유저_수=('user_id', 'nunique'),
    첫_결제율=('paid_30d', 'mean'),
    질문박스_사용율=('question_click_7d', 'mean')
).reset_index()


In [ ]:
# 시각화
tidy_df = pd.melt(lesson_activarion_temp,
                  id_vars=['lesson_activation'],
                  value_vars=['첫_결제율', '질문박스_사용율'],
                  var_name='지표종류',
                  value_name='비율')

# 3. 만약 0.08, 0.15 같은 소수점을 8%, 15% 형태로 시각화하고 싶다면 100을 곱해줍니다.
tidy_df['비율'] = tidy_df['비율'] * 100

# 4. 그래프 그리기
plt.figure(figsize=(12, 6))

ax = sns.barplot(data=tidy_df,
            x='lesson_activation',  # x축 (0과 1)
            y='비율',               # 세로축 높이 (%)
            hue='지표종류')         # 막대 색깔 분해 (첫_결제율 vs 질문박스_사용율)

plt.title('결제 전 첫 레슨완료 여부별 주요 지표 비교', fontsize=14, pad=15)
plt.ylabel('비율 (%)', fontsize=12)
plt.xlabel('결제 전 첫 레슨완료 여부', fontsize=12)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=11)

plt.xticks([0, 1], ['레슨 미완료', '레슨 완료'])

plt.ylim(0, 80)
plt.tight_layout()
plt.show()


- 레슨 미완료시 결제율은 약 9%, 질문 박스 사용율은 약 12%로 확인됨
-  레슨 완료시 결제율은 약 40%, 질문 박스 사용율은 약 64%로 확인됨
- 결제 전 레슨을 수강한 그룹과 수강하지 않은 비교했을 때 결제율은 약 31%p, 질문박스 사용율은 약 52%p 상승하는 것을 확인


## 완료 수 구간별 결제 유저 기여도


In [ ]:
temp = pre_user_activation_payment_dm[['user_id', 'paid_30d', 'lesson_complete_prepay_7d']]
temp['lesson_complete_prepay_7d'] = pd.cut(
    temp['lesson_complete_prepay_7d'],
    bins=[-1, 0, 10, 20, 50, 100, 200, float('inf')],
    labels = ['0', '1~10', '11~20', '21~50', '51~100', '100~200', '200+']
    )

user_paid_share = temp.groupby('lesson_complete_prepay_7d').agg(
    총_유저_수=('user_id', 'nunique'),
    결제_유저_수=('paid_30d', 'sum'),
    결제율=('paid_30d', 'mean')
).reset_index()

user_paid_share['결제_기여도'] = user_paid_share['결제_유저_수'] / user_paid_share['결제_유저_수'].sum()

user_paid_share


- 무료 체험 서비스 기간 내 가장 높은 기여도를 보인 곳은 결제 이전 수강하지 않은 그룹으로 약 86%의 결제유저 비중을 차지
- 0개 수강 그룹 외 각 그룹마다의 결제 기여도는 상이한 것으로 보임
- 무료 체험 서비스가 실행 중임에도 불구하고 강의를 체험하지 않은 유저집단에서 높은 결제 기여도를 보이는 것으로 보아 이를 이유로 무료 체험 서비스를 폐지했을 가능성이 있음
- 무료 체험 서비스 폐지 이후 기여도와 다른 모습을 보여주고 있어 추가적인 확인 필요


## 완료수 구간 별 행동패턴 분석 결과(결제율, 평균 레슨 진입횟수, 질문 클릭률)


In [ ]:
temp = pre_user_activation_payment_dm[['user_id', 'paid_30d', 'lesson_complete_prepay_7d', 'lesson_enter_7d', 'question_click_7d']]
temp['lesson_complete_prepay_7d'] = pd.cut(
    temp['lesson_complete_prepay_7d'],
    bins=[-1, 0, 6, 12, float('inf')],
    labels = ['0', '1~6', '7~12', '13+']
    )
temp['question_click_7d'] = temp['question_click_7d'].apply(lambda x: binary(x))
temp.groupby('lesson_complete_prepay_7d').agg(
    결제율=('paid_30d', 'mean'),
    평균_레슨_진입횟수=('lesson_enter_7d', 'mean'),
    질문_클릭률=('question_click_7d', 'mean')
).reset_index()


- 0개 완료/ 1~6개 완료/ 7~12개 완료/ 13개 이상 완료 그룹의 행동 패턴을 비교
- 결제 이전 레슨 수강 횟수 많은 그룹일수록 결제율, 평균 레슨 진입횟수, 질문 클릭률 등 모든 지표가 증가하는 것을 확인
- 특히 13개 이상 수강한 유저 그룹에서는 결제율이 약 43%, 평균 레슨 진입 횟수 161회, 질문 클릭률 77.6%로 고활동 유저로 판단하기에 적합할 것으로 보인다.
- 하지만 1400건, 600건 등 수강한 유저들이 있었기 때문에 이상치는 아니더라고 특이값이 존재한다는 것을 감안해야 한다고 생각합니다.


# 파트 2
- 무료체험 폐지가 정말 회사 입장에서 도움이 되었는가?


## 질문 버튼 클릭 수에 따른 이용자의 레슨 완료 / 컨텐츠 완료 수
- 시간에 관계없이 질문 버튼 클릭 수 / 리뷰 버튼 클릭 수 가 레슨 완료에 미치는 영향을 확인
- 레슨 완료를 재수강을 포함한 수를 확인할 것인가 / 고유 레슨 수를 확인할 것인가에서 큰 차이가 나타나지 않음을 확인
- 추후 정해진다면 하나의 기준으로 통일하는 것으로 결정


In [ ]:
temp_action = pre_user_activation_payment_dm[['user_id', 'cohort_month', 'unique_lesson_complete_7d', 'unique_content_end_7d', 'unique_content_start_7d', 'review_click_7d', 'question_click_7d', 'lesson_complete_prepay_7d', 'content_end_prepay_7d']].reset_index(drop=True)

temp_question_lesson = temp_action.groupby('question_click_7d').agg({
    'user_id':'count' ,
    'unique_lesson_complete_7d' : 'sum',
    'unique_content_end_7d' : 'sum'}).reset_index()
temp_question_lesson = temp_question_lesson.rename(columns={'user_id':'user_count'})
temp_question_lesson['lesson_completion_rate_by_question_click'] = temp_question_lesson['unique_lesson_complete_7d'] / temp_question_lesson['user_count']
temp_question_lesson['content_completion_rate_by_question_click'] = temp_question_lesson['unique_content_end_7d'] / temp_question_lesson['user_count']

temp_question_lesson['click_group'] = pd.cut(
    temp_question_lesson['question_click_7d'],
    bins=[-1, 0, 10, 30, 100, float('inf')],
    labels=['0', '1~10', '11~30', '31~100', '100+']
)

lesson_completion_by_question_df = temp_question_lesson.groupby('click_group').agg({
    'user_count' : 'sum',
    'lesson_completion_rate_by_question_click' : 'mean'
  }).reset_index()

lesson_completion_by_question_df.rename(columns = {
    'click_group' : '질문박스 사용 수 그룹',
    'user_count' : '유저 수',
    'lesson_completion_rate_by_question_click' : '평균 레슨 수강 수'})


- 질문 목록 클릭 수가 많을 수록 평균 레슨 완료 수가 높아지는 것을 확인
- 0번 클릭 그룹의 평균 레슨 완료 수는 약 0.5건,
- 1~10번 클릭 그룹의 평균 레슨 완료수는 약 31건,
- 11~30번 클릭 그룹의 평균 레슨 완료수는 약 73건,
- 31~100번 클릭 그룹의 평균 레슨 완료수는 약 120건,
- 100+의 경우 평균 레슨 완료수는 167건이지만 표본이 24명임에 유의


In [ ]:
content_completion_by_question_df = temp_question_lesson.groupby('click_group').agg({
    'user_count' : 'sum',
    'content_completion_rate_by_question_click' : 'mean'
  }).reset_index()
content_completion_by_question_df.rename(columns = {
    'click_group' : '질문박스 사용 수 그룹',
    'user_count' : '유저 수',
    'content_completion_rate_by_question_click' : '평균 컨텐츠 완료 수'})


- 질문 목록 클릭 수가 많을 수록 평균 레슨 완료 수가 높아지는 것을 확인할 수 있다.
- 컨텐츠 완료수는 높아지지만 타이트한 조건이기 때문에 효과를 확실히 이야기 하기 어려울 수 있다.
- 0번 클릭 그룹의 평균 컨텐츠 완료 수는 약 0.01건
- 1~10번 클릭 그룹의 평균 컨텐츠 완료수는 약 0.7건
- 11~30번 클릭 그룹의 평균 컨텐츠 완료수는 약 1.8건
- 31~100번 클릭 그룹의 평균 컨텐츠 완료수는 약 3건
- 100+의 경우 평균 컨텐츠 완료수는 약 4건이지만 표본이 24명임에 유의


## 리뷰 버튼 클릭한 그룹의 평균 레슨 완료 횟수
- 이진 분류를 통한 비교가 필요하다면 추후 추가
- 한 번 이상 리뷰를 클릭한 그룹의 레슨 평균 수강 횟수가 많아짐을 확인
- 단 표본이 너무 적다는 문제가 있을 수 있음
- 1-5 그룹은 234명, 6-10 그룹은 24명


In [ ]:
temp_review_lesson = temp_action.groupby('review_click_7d').agg({
    'user_id':'count',
    'unique_lesson_complete_7d':'sum',
    'unique_content_end_7d':'sum'
}).reset_index()

temp_review_lesson['lesson_completion_rate_by_review_click'] = temp_review_lesson['unique_lesson_complete_7d'] / temp_review_lesson['user_id']
temp_review_lesson['content_completion_rate_by_review_click'] = temp_review_lesson['unique_content_end_7d'] / temp_review_lesson['user_id']

temp_review_lesson['click_group'] = pd.cut(
    temp_review_lesson['review_click_7d'],
    bins=[-1, 0, 5, 10, 20, float('inf')],
    labels=['0', '1~5', '6~10', '10~20', '20+']
)

lesson_completion_by_review_df = temp_review_lesson.groupby('click_group').agg({
    'user_id' : 'sum',
    'lesson_completion_rate_by_review_click' : 'mean'
  }).reset_index()
lesson_completion_by_review_df.rename(columns = {
    'click_group' : '후기 사용 수 그룹',
    'user_id' : '유저 수',
    'lesson_completion_rate_by_review_click' : '평균 레슨 완료 수'})


- 후기를 확인하는 그룹의 평균 레슨 완료수가 높았으나 확인하지 않은 그룹에 비해 적음
- 활성화 유저 자체가 1% 정도의 비중이고 선택 편향 문제가 생겼을 가능성이 있어 주요 지표로 활용하기 어려울 것으로 보임


## 유저의 월 별 레슨 페이지 진입 당 레슨 완료 횟수


In [ ]:
lesson_completion_rate = pre_user_activation_payment_dm[['user_id', 'signup_time', 'cohort_month', 'lesson_enter_7d', 'unique_lesson_enter_7d', 'lesson_complete_7d', 'unique_lesson_complete_7d']]
lesson_completion_rate_by_cohort = lesson_completion_rate.groupby('cohort_month')[['lesson_enter_7d', 'lesson_complete_7d']].sum().reset_index()
lesson_completion_rate_by_cohort['completion_rate'] = lesson_completion_rate_by_cohort['lesson_complete_7d'] / lesson_completion_rate_by_cohort['lesson_enter_7d']
lesson_completion_rate_by_cohort.rename(columns = {
    'cohort_month' : '년-월(코호트)',
    'lesson_enter_7d' : '가입 후 7일 내 레슨 페이지 진입 수',
    'lesson_complete_7d' : '가입 후 7일 내 레슨 완료 수',
    'completion_rate' : '페이지 진입 대비 완료 비율'
})


- 5월, 6월, 11월의 코호트 그룹에서 레슨 페이지 진입대비 완료 횟수가 저조한 것을 확인
- 이 수치의 차이가 나머지 월들의 lesson_enter 수 자체가 적어서 생겼을 가능 성이 있음
- 11월의 경우 프로모션이 진행 되었을 것으로 예상되는 달로 레슨 페이지 진입 수는 높았지만 비율이 적은 것을 확인
- 프로모션만 진행하는 것은 장기적으로는 위험할 수 있음


### 레슨 페이지 진입을 unique한 조건을 주었을 때


In [ ]:
unique_lesson_completion_rate_by_cohort = lesson_completion_rate.groupby('cohort_month')[['unique_lesson_enter_7d', 'lesson_complete_7d']].sum().reset_index()
unique_lesson_completion_rate_by_cohort['completion_rate'] = unique_lesson_completion_rate_by_cohort['lesson_complete_7d'] / unique_lesson_completion_rate_by_cohort['unique_lesson_enter_7d']
unique_lesson_completion_rate_by_cohort.rename(columns = {
    'cohort_month' : '년-월(코호트)',
    'unique_lesson_enter_7d' : '가입 후 7일 내 레슨 페이지 진입 수',
    'lesson_complete_7d' : '가입 후 7일 내 레슨 완료 수',
    'completion_rate' : '페이지 진입 대비 완료 비율'
})


- 레슨 페이지 진입에 unique 조건을 주었을 때와 주지 않았을 때 큰 폭의 차이가 있는 것을 확인
- 레슨 시작 이벤트가 있으면 더 확실한 지표를 얻을 수 있을 것으로 예상됨
- 11월의 경우도 직전 약 20%의 전환율에서 약 86% 정도로 큰 폭으로 변화한 것을 확인
- unique 조건을 주었을 때가 더 정확한 인사이트를 얻을 수 있을 것으로 보이지만 논의가 필요해보임


## 무료 체험 서비스 폐지 이전 질문/후기 행동 조합별 전환 신호 확인


### 질문 클릭×레슨 완료 조합 별 첫 결제율


In [ ]:
temp = pre_user_activation_payment_dm[['user_id', 'paid_30d', 'unique_content_prepay_7d', 'lesson_complete_prepay_7d', 'review_click_7d', 'question_click_7d']]

temp['review_click_7d'] = temp['review_click_7d'].apply(lambda x : binary(x))
temp['question_click_7d'] = temp['question_click_7d'].apply(lambda x : binary(x))

def make_question_segment(row):
    lesson_count = row['lesson_complete_prepay_7d']
    question_click = row['question_click_7d'] > 0

    if lesson_count == 0 and not question_click:
        return '비학습형'
    elif lesson_count == 0 and question_click:
        return '질문 탐색형'
    elif 1 <= lesson_count <= 6 and not question_click:
        return '약한 학습형'
    elif 1 <= lesson_count <= 6 and question_click:
        return '약한 학습+질문형'
    elif lesson_count >= 7 and not question_click:
        return '핵심 학습형'
    elif lesson_count >= 7 and question_click:
        return '고몰입 질문형'
    else:
        return '기타'

temp['question_lesson_segment'] = temp.apply(make_question_segment, axis=1)
question_order = ['비학습형', '질문 탐색형', '약한 학습형', '약한 학습+질문형', '핵심 학습형', '고몰입 질문형']

question_segment = temp.groupby('question_lesson_segment').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reindex(question_order).reset_index()

question_segment['결제_기여도'] = question_segment['결제자_수'] / question_segment['결제자_수'].sum()
question_segment


- 결제 기여도가 가장 높은 그룹은 질문 탐색형으로 약 49% 비중을 차지하고 있음
- 두 번째로 높았던 그룹은 비학습형으로 결제율은 낮지만 결제 기여도는 약 37%로 나머지 그룹들 대비 높은 기여도를 차지하고 있음
- 해당 분석 자료를 이용하여 결제자 수를 높이고자 한다면 질문박스 사용을 유도하거나 비학습형의 결제 이유는 무엇일지 확인하여 그 패턴으로 유도하거나 질문 탐색형으로 유도하는 것이 유리할 것으로 보임


In [ ]:
# 시각화
from matplotlib.ticker import PercentFormatter

ax = sns.barplot(
    data=question_segment,
    x='첫_결제율',
    y='question_lesson_segment'
)

# x축을 %로 표시
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

# 막대 위에 수치 표시
for p in ax.patches:
    width = p.get_width()

    ax.annotate(
        f'{width:.1%}',
        (width + 0.01, p.get_y() + p.get_height()/2),
        ha='left',
        va='center'
    )

plt.title("질문 클릭 X 레슨 완료 조합별 결제율")
plt.ylabel("질문-레슨완료 조합")
plt.xlim(0, 0.7)
plt.tight_layout()
plt.show()


- 무료체험 폐지 이전 질문×레슨 조합에서 비학습형 결제율은 약 2.74%, 약한 학습형은 약 30.5%, 약한 학습 + 질문형은 약 29.2%, 질문 탐색형의 결제율은 약 42.5% 로 학습 의지를 보였을 때 결제율이 큰 폭으로 상승하는 것을 볼 수 있음
- 앞선 표의 형태로 결제 기여도를 살펴보았을 때 질문박스를 사용하지 않은 그룹보다 질문박스를 사용한 유저의 결제 기여도가 높은 것으로 확인됨
- 따라서 무료체험 폐지 이전에는 질문박스 사용을 유도하는 것이 유리했을 것이라 판단됨


### 후기 클릭×콘텐츠 시작 조합별 전환 신호 확인


In [ ]:
def make_review_segment(row):
    content_count = row['unique_content_prepay_7d']
    review_click = row['review_click_7d'] > 0

    if content_count == 0 and not review_click:
        return '미검토 미시작'
    elif content_count == 0 and review_click:
        return '후기 검토 후 미시작'
    elif 1 <= content_count  and not review_click:
        return '후기 없이 시작'
    elif 1 <= content_count  and review_click:
        return '후기 검토 후 시작'
    else:
        return '기타'

temp['review_content_segment'] = temp.apply(make_review_segment, axis=1)
question_order = ['미검토 미시작', '후기 검토 후 미시작', '후기 없이 시작', '후기 검토 후 시작']

review_segment = temp.groupby('review_content_segment').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reindex(question_order).reset_index()


In [ ]:
review_segment['결제_기여도'] = review_segment['결제자_수'] / review_segment['결제자_수'].sum()
review_segment


In [ ]:
# 시각화
ax = sns.barplot(
    data=review_segment,
    x='첫_결제율',
    y='review_content_segment'
)

# x축을 %로 표시
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

# 막대 위에 수치 표시
for p in ax.patches:
    width = p.get_width()

    ax.annotate(
        f'{width:.1%}',
        (width + 0.01, p.get_y() + p.get_height()/2),
        ha='left',
        va='center'
    )

plt.title("후기 클릭x컨텐츠 시작 조합 별 첫 결제율")
plt.ylabel("후기-컨텐츠 시작 조합")
plt.xlim(0, 1.2)
plt.tight_layout()
plt.show()


- 후기 검토한 그룹의 결제율이 높은 것으로 보이나 결제 기여도 측면으로 봤을 때 후기 검토 후 미시작 그룹은 약 1.4%, 검토 후 시작 그룹은 약 3.5% 로 다른 두 그룹에 비해 기여도는 적은 것으로 보임
- 후기를 더 쉽게 볼 수 있도록 UX를 개선하거나 홈페이지 배너(온보딩) 화면에 후기를 노출하는 방안을 생각해볼 수 있음


 - 컨텐츠 시작 테이블의 경우 무료체험 서비스 폐지 이전 파트의 집계 기간을 모두 포함하는 것이 아니기 때문에 데이터가 오염되었을 수 있음
- 컨텐츠 시작한 유저 자체가 축소되어 집계되었기 때문에 컨텐츠 시작 파트가 감소되고 미시작 파트의 수가 늘어나야 한다고 생각
- 후기 검토 후 미시작의 경우 모수가 39명으로 집계 기간의 영향이 적었을 가능성은 있으나 신중한 판단 필요


## 솔메님이 하신 최종 세그먼트별 성과


In [ ]:
final_temp = pre_user_activation_payment_dm[['user_id', 'paid_30d', 'unique_content_prepay_7d', 'lesson_complete_prepay_7d', 'content_view_7d', 'content_start_7d', 'review_click_7d', 'question_click_7d', 'content_end_prepay_7d']]

def make_final_segment(row):
    lesson_count = row['lesson_complete_prepay_7d']
    content_end = row['content_end_prepay_7d']
    unique_content = row['unique_content_prepay_7d']
    question_click = row['question_click_7d'] > 0
    content_experience = unique_content > 0 or row['content_view_7d'] > 0 or row['content_start_7d'] > 0

    if (lesson_count >= 7 or content_end >= 1) and question_click:
        return '고몰입 질문형'
    elif content_end >= 1:
        return '완주형'
    elif lesson_count >= 21:
        return '고몰입·헤비 학습형'
    elif lesson_count >= 11:
        return '수익 핵심 학습형'
    elif lesson_count >= 7:
        return '핵심 학습형'
    elif lesson_count == 6:
        return '임계 직전형'
    elif 1 <= lesson_count <= 5:
        return '약한 학습형'
    elif content_experience:
        return '탐색형'
    else:
        return '무행동형'

final_order = ['무행동형', '탐색형', '약한 학습형', '임계 직전형', '핵심 학습형', '수익 핵심 학습형', '고몰입·헤비 학습형', '완주형', '고몰입 질문형']
final_temp['최종 유저 세그먼트'] = final_temp.apply(make_final_segment, axis=1)

final_segment = final_temp.groupby('최종 유저 세그먼트').agg(
    유저_수=('user_id', 'nunique'),
    결제자_수=('paid_30d', 'sum'),
    첫_결제율=('paid_30d', 'mean')
).reindex(final_order).fillna(0).reset_index()

final_segment['유저 비중(%)'] = final_segment['유저_수'] / len(final_temp) * 100
final_segment['결제 기여도(%)'] = final_segment['결제자_수'] / final_temp['paid_30d'].sum() * 100
final_segment['첫 결제율(%)'] = final_segment['첫_결제율'] * 100

user_share_median = final_segment['유저 비중(%)'].median()

final_display = final_segment[['최종 유저 세그먼트', '유저_수', '유저 비중(%)', '결제자_수', '첫 결제율(%)', '결제 기여도(%)']].copy()
final_display.columns = ['최종 유저 세그먼트', '유저 수', '유저 비중(%)', '결제자 수', '첫 결제율(%)', '결제 기여도(%)']
for col in ['유저 비중(%)', '첫 결제율(%)', '결제 기여도(%)']:
    final_display[col] = final_display[col].round(2)

display(final_display)


In [ ]:
# 시각화
import plotly.express as px

fig = px.scatter(
    final_segment,
    x='유저 비중(%)', y='첫 결제율(%)', size='결제 기여도(%)', color='최종 유저 세그먼트',
    hover_name='최종 유저 세그먼트',
    hover_data={'유저_수': ':,', '결제자_수': ':,', '유저 비중(%)': ':.2f', '첫 결제율(%)': ':.2f', '결제 기여도(%)': ':.2f'},
    labels={'유저 비중(%)': '전체 유저 비중(%)', '첫 결제율(%)': '첫 결제율(%)'},
    title='최종 유저 세그먼트별 규모, 첫 결제율, 결제 기여도'
)
fig.update_layout(height=640, legend_title_text='세그먼트')
fig.show()


- 탐색형의 결제 기여도가 약 71%로 가장 높았으며 탐색형과 무행동형이 결제 기여도의 약85%를 차지
- 두 유형을 제외하고 가장 기여도가 높았던 그룹은 고몰입 질문형으로 약 8.5% 기여도를 차지하고 있으며 첫 결제율도 상위권으로 나타남
- 다른 패턴들의 경우 첫 결제율은 높으나 기여도 측면에서 매우 작기 때문에 특정 유형의 결제율을 늘리는 방향보단 특정 유형으로 유저를 유도하는 게 효율적일 것이라고 판단됨

- 개인적인 의견
  - 무행동형을 탐색형으로 끌어들이는 게 주 목표였던 것 같은데 앞선 결제 기여도를 확인했을 때 무료체험 서비스를 진행했음에도 미학습 유저의 기여도가 높았고, 폐지 이후 학습율은 늘었으나 결제율이 감소한 것으로 보아 대대적인 개편이 진행되었다고 예상 됨


# 파트 3


# 첫 결제 이후 구독 갱신으로 이어지는 유저의 행동 패턴
- 집계 기간 내 첫 결제자를 기준으로 데이터 마트를 형성
- 때문에 정기 결제부분이 과소평가 되었을 수 있음


# 정기결제 리텐션 지표 파트
- 집계 기간 내 첫 결제자를 기준으로 데이터 마트를 형성
- 때문에 정기 결제부분이 과소평가 되었을 수 있음


In [ ]:
# 첫 결제 -> 정기 결제 데이터 합치기
retention = pd.read_csv("data/private/dm_user_subscription_retention_core.csv")
activation = pd.read_csv("data/private/dm_user_activation_payment.csv")

activation_cols = [
    "user_id",
    "lesson_complete_prepay_7d",
    "content_end_prepay_7d",
    "unique_content_prepay_7d"
]

final_retention = retention.merge(
    activation[activation_cols],
    on="user_id",
    how="left"
)

final_retention.shape


## 비싼 금액의 구독이 장기권인가?
- value_counts를 이용해 확인해봤을 때 아닌 것 같음


### 전체 집계 기간 내 플랜 별 추가 결제로 이어지는지
- 단순한 추가 결제로 이어지는 집계만 확인할 거라면 90일 내 갱신 여부를 이용하여 칼럼을 줄이는 것이 좋아보임


In [ ]:
temp = final_retention
temp2 = temp[['first_plan_price', 'renewed_30d', 'renewed_60d', 'renewed_90d']].value_counts().reset_index().sort_values(by=['first_plan_price', 'renewed_30d', 'renewed_60d', 'renewed_90d'], ascending=[False, False, False, False])
temp3 = temp2.groupby('first_plan_price')['count'].sum().reset_index()
plan_retention_rate = temp2.merge(temp3, how='left', on ='first_plan_price')
plan_retention_rate['group_rate'] = round((plan_retention_rate['count_x'] / plan_retention_rate['count_y']), 3) * 100
plan_retention_rate.rename(columns={
    'first_plan_price' : '결제 플랜',
    'renewed_30d' : '30일 내 갱신여부',
    'renewed_60d' : '60일 내 갱신여부',
    'renewed_90d' : '90일 내 갱신여부',
    'count_x' : '그룹 유저 수',
    'count_y' : '플랜 유저 수',
    'group_rate' : '그룹 비율'
})


- 분석 내용
  - 저가 플랜(만원 대)의 갱신율(추가 결제율)이 약 66%
  - 고가 플랜(4만원 이상)의 갱신율은 약 20% 미만으로 저가 플랜과 고가 플랜의 갱신율 차이가 있음
  - 첫 결제 금액이 적을수록 정기결제로 이어지는 경향이 있는 것으로 보임
- 추론
  - 높은 금액대의 플랜일수록 만족도가 적은 상태일 수 있거나 충분한 경험 후 이탈일 수 있음
  - 4만원이상의 플랜부터 추가적인 결제로 이어지지 않는 것으로 보아 충분한 가치를 느끼지 못하는 것으로 보인다.
- 의문점
  - 30일 내 갱신은 아니어도 60일 내 갱신 가능, 90일 내 갱신 가능인 부분이 사실 만료 후 재구독으로 넘어가야 하는 것 아닌가?


### 무료 체험 서비스 폐지 이후 기간 동안 결제 플랜별 추가 결제


In [ ]:
temp = final_retention[final_retention['period_group'] == 'post_free_trial_removed']
temp2 = temp[['first_plan_price', 'renewed_30d', 'renewed_60d', 'renewed_90d']].value_counts().reset_index().sort_values(by=['first_plan_price', 'renewed_30d', 'renewed_60d', 'renewed_90d'], ascending=[False, False, False, False])
temp3 = temp2.groupby('first_plan_price')['count'].sum().reset_index()
plan_retention_rate = temp2.merge(temp3, how='left', on ='first_plan_price')
plan_retention_rate['group_rate'] = round((plan_retention_rate['count_x'] / plan_retention_rate['count_y']), 3) * 100
plan_retention_rate = plan_retention_rate.rename(columns={
    'first_plan_price' : '결제 플랜',
    'renewed_30d' : '30일 내 갱신여부',
    'renewed_60d' : '60일 내 갱신여부',
    'renewed_90d' : '90일 내 갱신여부',
    'count_x' : '그룹 유저 수',
    'count_y' : '플랜 유저 수',
    'group_rate' : '그룹 비율'
})
plan_retention_rate


In [ ]:
temp_plan_users= plan_retention_rate.groupby('결제 플랜')['그룹 유저 수'].sum().reindex([131600, 118440, 15920, 14328]).reset_index()
temp_plan_users['유저 비중'] = round((temp_plan_users['그룹 유저 수'] / temp_plan_users['그룹 유저 수'].sum()), 3) * 100
temp_plan_users['플랜 별 기대 수익(원)'] = (temp_plan_users['결제 플랜'] * temp_plan_users['그룹 유저 수']).astype(str) + '원'
temp_plan_users


In [ ]:
temp_plan_users2= plan_retention_rate.groupby('결제 플랜')['그룹 유저 수'].sum().reindex([131600, 118440, 15920, 14328]).reset_index()
temp_plan_users2['유저 비중'] = round((temp_plan_users2['그룹 유저 수'] / temp_plan_users2['그룹 유저 수'].sum()), 3) * 100
temp_plan_users2['플랜 별 기대 수익(원)'] = temp_plan_users2['결제 플랜'] * temp_plan_users2['그룹 유저 수']


In [ ]:
# 시각화
plt.figure(figsize=(7, 7))

colors = sns.color_palette('pastel')

plt.pie(
    temp_plan_users['유저 비중'],
    labels=temp_plan_users['결제 플랜'],
    autopct='%1.1f%%',
    colors = colors,
    startangle=90,           # 12시 방향부터 시작
    counterclock=False,      # 시계 방향
    pctdistance=0.75,
    labeldistance=1.1,
  wedgeprops={
        'width': 0.45,
        'edgecolor': 'white',
        'linewidth': 2
    },
    textprops={
        'fontsize': 11
    },
)

plt.title(
    '결제 플랜 분포',
    fontsize=14,
    pad=20
)

plt.tight_layout()
plt.show()


In [ ]:
# 시각화
plt.figure(figsize=(7, 7))

colors = sns.color_palette('pastel')

plt.pie(
    temp_plan_users2['플랜 별 기대 수익(원)'],
    labels=temp_plan_users2['결제 플랜'],
    autopct='%1.1f%%',
    colors = colors,
    startangle=90,           # 12시 방향부터 시작
    counterclock=False,      # 시계 방향
    pctdistance=0.75,
    labeldistance=1.1,
  wedgeprops={
        'width': 0.45,
        'edgecolor': 'white',
        'linewidth': 2
    },
    textprops={
        'fontsize': 11
    },
)

plt.title(
    '결제 플랜 수익 분포',
    fontsize=14,
    pad=20
)

plt.tight_layout()
plt.show()


- 무료 체험 서비스 폐지 이후 집계기간 내 분석
- 분석 내용
  - 가장 비싼 131600 플랜의 경우 추가 결제로 이어지는 비율이 약 1.3%, 118440 플랜의 경우 0% 로 전체 기간 내 추가 결제로 이어지는 비율과 차이가 있는 것으로 보임
  - 저가 플랜으로 예상되는 15920 플랜의 경우 추가 결제로 이어지는 비율이 약 70%, 14328 플랜의 경우 약 71.4% 로 무료체험 서비스 페지 이전과 비슷한 것으로 보임


## 무료 체험 서비스 폐지 이후 기간 할인율에 따른 추가 결제율


In [ ]:
temp = final_retention[final_retention['period_group'] == 'post_free_trial_removed']
temp2 = temp[['first_discount_rate', 'renewed_30d', 'renewed_60d', 'renewed_90d']].value_counts().reset_index().sort_values(by=['first_discount_rate', 'renewed_30d', 'renewed_60d', 'renewed_90d'], ascending=[False, False, False, False])
temp3 = temp2.groupby('first_discount_rate')['count'].sum().reset_index()
plan_retention_rate = temp2.merge(temp3, how='left', on ='first_discount_rate')
plan_retention_rate['group_rate'] = round((plan_retention_rate['count_x'] / plan_retention_rate['count_y']), 3) * 100
plan_retention_rate.rename(columns={
    'first_discount_rate' : '첫 결제 할인율',
    'renewed_30d' : '30일 내 갱신여부',
    'renewed_60d' : '60일 내 갱신여부',
    'renewed_90d' : '90일 내 갱신여부',
    'count_x' : '그룹 유저 수',
    'count_y' : '플랜 유저 수',
    'group_rate' : '그룹 비율'
})


- 분석 내용
  - 9% , 7% 할인의 경우 모수가 너무 적어 제외하는 것이 좋아보임
  - 할인이 없을 때 첫 결제를 진행한 유저가 추가 결제로 이어지는 비율은 약 60%
  - 할인이 10%일 때 첫 결제를 진행한 유저가 추가 결제로 이어지는 비율은 약 52%
  - 할인이 15%일 때 첫 결제를 진행한 유저가 추가 결제로 이어지는 비율은 약 48%
  - 할인이 20% 이상인 경우도 모수가 적어 제외하는 것이 좋아보임
- 결론
  - 할인을 받은 유저가 추가 결제로 이어질 것으로 예상했으나 오히려 할인이 없을 때 진입한 유저가 추가 결제로 이어지는 것으로 확인됨
  - 따라서 할인 프로모션의 진행보다 신규 유저가 유입할 수 있는 광고 전략을 세우는 것이 유리할 것으로 보임


## 활동 여부 그룹 간 구독 갱신율 비교 (전체 기간)


In [ ]:
def grouping(x):
  if x == 0:
    return '미활동 유저'
  elif 1 <= x < 7:
    return '저활동 유저'
  elif x > 7:
    return('고활동 유저')

temp = final_retention[final_retention['period_group'] == 'post_free_trial_removed']
final_retention['lesson_activation_group'] = final_retention['lesson_complete_postpay_30d'].apply(lambda x: grouping(x))

renewed_by_activation = final_retention.groupby('lesson_activation_group').agg(
    유저_수 = ('user_id', 'count'),
    갱신유저_수_30일내 = ('renewed_30d', 'sum'),
    갱신유저_수_60일내 = ('renewed_60d', 'sum'),
    갱신유저_수_90일내 = ('renewed_90d', 'sum')
).reindex(['고활동 유저', '저활동 유저', '미활동 유저']).reset_index()

renewed_by_activation['30일 내 갱신 유저 비율'] = round((renewed_by_activation['갱신유저_수_30일내'] / renewed_by_activation['유저_수']), 3) * 100
renewed_by_activation['60일 내 갱신 유저 비율'] = round((renewed_by_activation['갱신유저_수_60일내'] / renewed_by_activation['유저_수']), 3) * 100
renewed_by_activation['90일 내 갱신 유저 비율'] = round((renewed_by_activation['갱신유저_수_90일내'] / renewed_by_activation['유저_수']), 3) * 100
renewed_by_activation


- 분석 내용
  - 고활동 유저가 너무 많이 집계되는 것으로 보아 결제 이후 7개 이상의 강의 수강이 아닌 다른 기준으로 고활동 유저를 선별해야 할 필요가 있어 보임
  - 결제 이후 30일 내 수강한 레슨 수에 따른 유저 수 분포 설정할까


In [ ]:
# 시각화
temp = renewed_by_activation.melt(
    id_vars='lesson_activation_group',
    value_vars=['30일 내 갱신 유저 비율', '60일 내 갱신 유저 비율', '90일 내 갱신 유저 비율'],
    var_name='갱신 기간',
    value_name='갱신 비율'
)
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=temp,
    x='lesson_activation_group',
    y='갱신 비율',
    hue='갱신 기간'
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        labels=[f'{v:.1f}%' for v in container.datavalues],
        padding=3
    )

plt.xlabel('유저 그룹')
plt.ylabel('갱신 유저 비율')
plt.title('활동도별 갱신율')
plt.ylim(0, 80)
plt.show()


- 분석 내용
  - 전체 기간 내에서 0회 / 7회 미만 / 7회 이상을 기준으로 활동 그룹을 세그먼트 했을 때 갱신 유저(추가 결제) 비율은 다음과 같음
  - 고활동 유저의 경우 30일 내 갱신율이 23%, 60일 내 갱신율이 58%, 90일 내 갱신율이 59%
  - 저활동 유저의 경우 30일 내 갱신율이 26%, 60일 내 갱신율이 49%, 90일 내 갱신율이 49%
  - 미활동 유저의 경우 30일 내 갱신율이 25%, 60일 내 갱신율이 39%, 90일 내 갱신율이 39%
- 세그먼트 조건 설정 방법
  - 레슨 수강수 분포를 봤을 때 평균 및 중앙값이 매우 크기 때문에 적절한 기준을 이용하여 세그먼트 할 필요가 있음(사분위수 확인하는 것은 어떨까?)
- 추가적으로 확인할만한 부분
  - 특이한 부분은 미활동 유저의 갱신 유지율이 조금 높았기 때문에 추가적으로 확인할 부분이 있을까? 라고 생각됨


## 무료 체험 폐지 이후 활동도 별 갱신 유저 비율


In [ ]:
def grouping(x):
  if x == 0:
    return '미활동 유저'
  elif 1 <= x < 7:
    return '저활동 유저'
  elif x > 7:
    return('고활동 유저')

temp = final_retention[final_retention['period_group'] == 'post_free_trial_removed']
temp['lesson_activation_group'] = temp['lesson_complete_postpay_30d'].apply(lambda x: grouping(x))

renewed_by_activation_post = temp.groupby('lesson_activation_group').agg(
    유저_수 = ('user_id', 'count'),
    갱신유저_수_30일내 = ('renewed_30d', 'sum'),
    갱신유저_수_60일내 = ('renewed_60d', 'sum'),
    갱신유저_수_90일내 = ('renewed_90d', 'sum')
).reindex(['고활동 유저', '저활동 유저', '미활동 유저']).reset_index()

renewed_by_activation_post['30일 내 갱신 유저 비율'] = round((renewed_by_activation_post['갱신유저_수_30일내'] / renewed_by_activation_post['유저_수']), 3) * 100
renewed_by_activation_post['60일 내 갱신 유저 비율'] = round((renewed_by_activation_post['갱신유저_수_60일내'] / renewed_by_activation_post['유저_수']), 3) * 100
renewed_by_activation_post['90일 내 갱신 유저 비율'] = round((renewed_by_activation_post['갱신유저_수_90일내'] / renewed_by_activation_post['유저_수']), 3) * 100
renewed_by_activation_post


In [ ]:
# 시각화
temp = renewed_by_activation_post.melt(
    id_vars='lesson_activation_group',
    value_vars=['30일 내 갱신 유저 비율', '60일 내 갱신 유저 비율', '90일 내 갱신 유저 비율'],
    var_name='갱신 기간',
    value_name='갱신 비율'
)
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=temp,
    x='lesson_activation_group',
    y='갱신 비율',
    hue='갱신 기간'
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        labels=[f'{v:.1f}%' for v in container.datavalues],
        padding=3
    )

plt.xlabel('유저 그룹')
plt.ylabel('갱신 유저 비율')
plt.title('활동도별 갱신율')
plt.ylim(0, 80)
plt.show()


### 레슨완료 수를 이용하여 활동유저를 세그먼트 했을 때


In [ ]:
def grouping2(x):
  if 1 <= x <= 50 :
    return '저활동 유저'
  elif 51 <= x <= 100:
    return '활동 유저'
  elif 101 <= x <= 400:
    return '고활동 유저'

temp2 = final_retention[final_retention['period_group'] == 'post_free_trial_removed']
temp2['lesson_activation_group'] = temp2['lesson_complete_postpay_30d'].apply(lambda x: grouping2(x))

renewed_by_activation_post2 = temp2.groupby('lesson_activation_group').agg(
    유저_수 = ('user_id', 'count'),
    갱신유저_수_30일내 = ('renewed_30d', 'sum'),
    갱신유저_수_60일내 = ('renewed_60d', 'sum'),
    갱신유저_수_90일내 = ('renewed_90d', 'sum')
).reindex(['저활동 유저', '활동 유저', '고활동 유저']).reset_index()

renewed_by_activation_post2['30일 내 갱신 유저 비율'] = round((renewed_by_activation_post2['갱신유저_수_30일내'] / renewed_by_activation_post2['유저_수']), 3) * 100
renewed_by_activation_post2['60일 내 갱신 유저 비율'] = round((renewed_by_activation_post2['갱신유저_수_60일내'] / renewed_by_activation_post2['유저_수']), 3) * 100
renewed_by_activation_post2['90일 내 갱신 유저 비율'] = round((renewed_by_activation_post2['갱신유저_수_90일내'] / renewed_by_activation_post2['유저_수']), 3) * 100
renewed_by_activation_post2


In [ ]:
# 시각화
temp = renewed_by_activation_post2.melt(
    id_vars='lesson_activation_group',
    value_vars=['30일 내 갱신 유저 비율', '60일 내 갱신 유저 비율', '90일 내 갱신 유저 비율'],
    var_name='갱신 기간',
    value_name='갱신 비율'
)
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=temp,
    x='lesson_activation_group',
    y='갱신 비율',
    hue='갱신 기간'
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        labels=[f'{v:.1f}%' for v in container.datavalues],
        padding=3
    )

plt.xlabel('유저 그룹')
plt.ylabel('갱신 유저 비율')
plt.title('활동도별 갱신율')
plt.ylim(0, 80)
plt.show()


- 무료 체험 폐지 이후 기간 내 0회 / 7회 미만 / 7회 이상을 기준으로 활동 그룹을 세그먼트 했을 때 갱신 유저(추가 결제) 비율은 다음과 같음
- 고활동 유저의 경우 30일 내 갱신율이 18%, 60일 내 갱신율이 60%, 90일 내 갱신율이 61%
- 저활동 유저의 경우 30일 내 갱신율이 17%, 60일 내 갱신율이 49.5%, 90일 내 갱신율이 50%
- 미활동 유저의 경우 30일 내 갱신율이 10%, 60일 내 갱신율이 31.6%, 90일 내 갱신율이 31.9%
- 구독하고 60일 후에 갱신을 하면 만료 후 재구독이 아니었을 지 의문


## 전체 기간 첫 결제 이후 30일 내 레슨 수강 수 분포


In [ ]:
def attibute_group(x):
  if x == 0:
    return '미활동'
  elif 0 < x <= 20:
    return '1~20'
  elif 21 < x <= 50:
    return '21~50'
  elif 51 < x <= 100:
    return '51~100'
  elif 101 < x <= 200:
    return '101~200'
  elif 201 < x <= 400:
    return '201~400'
  else:
    return '초고활동 유저'


user_attribution = final_retention
user_attribution['lesson_attribute_group'] = user_attribution['lesson_complete_postpay_30d'].apply(lambda x : attibute_group(x))
user_attribution = user_attribution.groupby('lesson_attribute_group')['user_id'].count().reset_index()
user_attribution['rate'] = user_attribution['user_id'] / user_attribution['user_id'].sum()
user_attribution_by_lesson = user_attribution.groupby('lesson_attribute_group')['rate'].sum().reset_index()

order = [
    '미활동',
    '1~20',
    '21~50',
    '51~100',
    '101~200',
    '201~400',
    '초고활동 유저'
]

user_attribution_by_lesson = user_attribution_by_lesson.set_index('lesson_attribute_group').reindex(order).reset_index()


In [ ]:
# 시각
plt.figure(figsize=(6,6))

plt.pie(
    user_attribution_by_lesson['rate'],
    labels=user_attribution_by_lesson['lesson_attribute_group'],
    autopct='%.1f%%',
    labeldistance=1.15,   # 범주 위치
    pctdistance=0.7       # 퍼센트 위치
)

plt.title('유저 활동 그룹 분포')
plt.show()


- 결제 이후 30일 이내 미활동한 유저는 11.6%
- 1~20회 이하 수강한 그룹이 약 17.4%
- 21~50회 이하 수강한 그룹이 약 18.1%
- 51~100회 이하 수강한 그룹이 약 21.4%로 가장 많음
- 101~200회 이하 수강한 그룹이 약 17.3%
- 201~400회  이하 수강한 그룹이 약 10%
- 400회 이상 초고활동 유저로 집계된 인원이 약 4.2%
- 가장 많이 수강한 인원은 1686건


# 무료 체험 폐지 이후 결제 이후 학습 참여일수 별 추가 결제율


In [ ]:
final_retention.groupby('active_learning_days_postpay_30d')['user_id'].count()


- 집계 기준에 의해 결제 직후 ~ 다음달(일수가 31일까지 있는 때)까지 기준이기 때문에 32일 학습 이벤트 진행한 유저가 있음
- 30일 내 7일 / 15일 / 16일 이상 학습한 유저로 세그먼트


- 학습 이벤트가 발생한 고유 일수 → 자주 접속하는 것이 영향을 주는 가
    - 고유 일수에 따른 학습양 , 정기결제로 이어지는 가
    - enter_mainpage 테이블의 경우 user_id가 공란인 부분이 매우 많았기 때문에 푸시 알람의 영향이 학습 이벤트가 발생한 고유 일수를 늘리는 데 유의미 할 것이라는 가정을 할 수 있음


In [ ]:
def segmentation_days(x):
  if 0 <= x <= 7:
    return '0 ~ 7회'
  elif 7 < x <= 15:
    return '8 ~ 15회'
  elif 15 < x <= 32:
    return '16회 이상'
  else:
    return '오류'
temp = final_retention
temp['learning_segment'] = temp['active_learning_days_postpay_30d'].apply(lambda x: segmentation_days(x))
renew_rate_by_activation_days = temp.groupby('learning_segment').agg(
    유저_수 = ('user_id', 'count'),
    갱신유저_수_30일내 = ('renewed_30d', 'sum'),
    갱신유저_수_60일내 = ('renewed_60d', 'sum'),
    갱신유저_수_90일내 = ('renewed_90d', 'sum')
).reindex(['0 ~ 7회', '8 ~ 15회', '16회 이상']).reset_index()
renew_rate_by_activation_days['갱신유저_비율_30일내'] = round((renew_rate_by_activation_days['갱신유저_수_30일내'] / renew_rate_by_activation_days['유저_수']), 3) * 100
renew_rate_by_activation_days['갱신유저_비율_60일내'] = round((renew_rate_by_activation_days['갱신유저_수_60일내'] / renew_rate_by_activation_days['유저_수']), 3) * 100
renew_rate_by_activation_days['갱신유저_비율_90일내'] = round((renew_rate_by_activation_days['갱신유저_수_90일내'] / renew_rate_by_activation_days['유저_수']), 3) * 100
renew_rate_by_activation_days


In [ ]:
renew_rate_by_activation_days_melt = renew_rate_by_activation_days.melt(
    id_vars='learning_segment',
    value_vars=['갱신유저_비율_30일내', '갱신유저_비율_60일내', '갱신유저_비율_90일내'],
    var_name='갱신 기간',
    value_name='갱신 비율'
)
renew_rate_by_activation_days_melt


In [ ]:
# 시각화
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=renew_rate_by_activation_days_melt,
    x='learning_segment',
    y='갱신 비율',
    hue='갱신 기간'
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        labels=[f'{v:.1f}%' for v in container.datavalues],
        padding=3
    )

plt.xlabel('유저 그룹')
plt.ylabel('갱신 유저 비율')
plt.title('활동일수 별 갱신율')
plt.ylim(0, 80)
plt.show()


- 결제 이후 30일 내 0~7회 학습 활동을 한 유저의 갱신 비율은 30일 내 25%, 60일 내 49%, 90일 내 50%
-  결제 이후 30일 내 8~15회 학습 활동을 한 유저의 갱신 비율은 30일 내 25%, 60일 내 60%, 90일 내 61%
- 결제 이후 30일 내 16회 이상 학습 활동을 한 유저의 갱신 비율은 30일 내 19%, 60일 내 62%, 90일 내 64%
- 생각이 필요한 부분
  - 학습 활동을 16회 이상 한 그룹에서 30일 내 갱신 비율이 낮은 것을 보아 한 달 기간을 모두 채우고 추가 결제를 하는 것이 일반적이라고 볼 수 있나?
  - 만료 이전 결제(30일 이내 결제) 그룹을 만족도가 높은 그룹이라고 할 수 있을까?
  - 하지만 16회 이상 학습하면서 30일 이후 60일 이내 결제율이 높은 것으로 보아 만족도와는 연관짓기 힘들어 보이기도 하고


### 최종 세그먼트 분류


In [ ]:
def seg(x):
  if 0 <= x <= 4:
    return '0 ~ 4회'
  elif 4 < x <= 13:
    return '5 ~ 13회'
  elif 13 < x <= 32:
    return '14회 이상'
  else:
    return '오류'


temp3 = final_retention
temp3['activation_days_seg']= temp3['active_learning_days_postpay_30d'].apply(lambda x : seg(x))
temp4 = temp3.groupby('activation_days_seg').agg(
    유저_수 = ('user_id', 'count'),
    갱신유저_수_30일내 = ('renewed_30d', 'sum'),
    갱신유저_수_60일내 = ('renewed_60d', 'sum'),
    갱신유저_수_90일내 = ('renewed_90d', 'sum')
).reindex(['0 ~ 4회', '5 ~ 13회', '14회 이상']).reset_index()
temp4['갱신유저_비율_30일내'] = round((temp4['갱신유저_수_30일내'] / temp4['유저_수']), 3) * 100
temp4['갱신유저_비율_60일내'] = round((temp4['갱신유저_수_60일내'] / temp4['유저_수']), 3) * 100
temp4['갱신유저_비율_90일내'] = round((temp4['갱신유저_수_90일내'] / temp4['유저_수']), 3) * 100
temp4


In [ ]:
activation_days_seg_melt = temp4.melt(
    id_vars='activation_days_seg',
    value_vars=['갱신유저_비율_30일내', '갱신유저_비율_60일내', '갱신유저_비율_90일내'],
    var_name='갱신 기간',
    value_name='갱신 비율'
)
activation_days_seg_melt


In [ ]:
# 시각화
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=activation_days_seg_melt,
    x='activation_days_seg',
    y='갱신 비율',
    hue='갱신 기간'
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        labels=[f'{v:.1f}%' for v in container.datavalues],
        padding=3
    )

plt.xlabel('유저 그룹')
plt.ylabel('갱신 유저 비율')
plt.title('활동일수 별 갱신율')
plt.ylim(0, 80)
plt.show()


### 활동 일수를 이용한 활동 그룹별 갱신율


In [ ]:
def grouping3(x):
  if 1 <= x <= 14 :
    return '저활동 유저'
  elif 15 <= x <= 46:
    return '활동 유저'
  elif 47 <= x <= 400:
    return '고활동 유저'

temp2 = final_retention[final_retention['period_group'] == 'post_free_trial_removed']
temp2['lesson_activation_group'] = temp2['lesson_complete_postpay_30d'].apply(lambda x: grouping2(x))

renewed_by_activation_post2 = temp2.groupby('lesson_activation_group').agg(
    유저_수 = ('user_id', 'count'),
    갱신유저_수_30일내 = ('renewed_30d', 'sum'),
    갱신유저_수_60일내 = ('renewed_60d', 'sum'),
    갱신유저_수_90일내 = ('renewed_90d', 'sum')
).reindex(['저활동 유저', '활동 유저', '고활동 유저']).reset_index()

renewed_by_activation_post2['30일 내 갱신 유저 비율'] = round((renewed_by_activation_post2['갱신유저_수_30일내'] / renewed_by_activation_post2['유저_수']), 3) * 100
renewed_by_activation_post2['60일 내 갱신 유저 비율'] = round((renewed_by_activation_post2['갱신유저_수_60일내'] / renewed_by_activation_post2['유저_수']), 3) * 100
renewed_by_activation_post2['90일 내 갱신 유저 비율'] = round((renewed_by_activation_post2['갱신유저_수_90일내'] / renewed_by_activation_post2['유저_수']), 3) * 100
renewed_by_activation_post2


### 후속 분석
- 결제 후 학습 활동 일수 별 60일 내 갱신율


In [ ]:
renew_rate_by_activation_days_temp = temp.groupby('active_learning_days_postpay_30d').agg(
    유저_수 = ('user_id', 'count'),
    갱신유저_수_30일내 = ('renewed_30d', 'sum'),
    갱신유저_수_60일내 = ('renewed_60d', 'sum'),
    갱신유저_수_90일내 = ('renewed_90d', 'sum')
).reset_index()

renew_rate_by_activation_days_temp['갱신유저_비율_30일내'] = round((renew_rate_by_activation_days_temp['갱신유저_수_30일내'] / renew_rate_by_activation_days_temp['유저_수']), 3) * 100
renew_rate_by_activation_days_temp['갱신유저_비율_60일내'] = round((renew_rate_by_activation_days_temp['갱신유저_수_60일내'] / renew_rate_by_activation_days_temp['유저_수']), 3) * 100
renew_rate_by_activation_days_temp['갱신유저_비율_90일내'] = round((renew_rate_by_activation_days_temp['갱신유저_수_90일내'] / renew_rate_by_activation_days_temp['유저_수']), 3) * 100
renew_rate_by_activation_days_temp


- segmentation 기준이 다른 두 개


In [ ]:
def segment(x):
  if 0 <= x < 5:
    return 'seg1'
  elif 5 <= x < 13:
    return 'seg2'
  elif 13 <= x <= 32:
    return 'seg3'
  else:
    return '오류'

renew_rate_by_activation_days_temp['segment_group'] = renew_rate_by_activation_days_temp['active_learning_days_postpay_30d'].apply(lambda x : segment(x))
seg1_df = renew_rate_by_activation_days_temp.groupby('segment_group').agg(
    유저_수 = ('유저_수', 'sum'),
    갱신_유저_수_60일내 = ('갱신유저_수_60일내', 'sum')
).reset_index()
seg1_df['갱신 비율'] = round((seg1_df['갱신_유저_수_60일내'] / seg1_df['유저_수']), 3) * 100
seg1_df


In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=seg1_df,
    x='segment_group',
    y='갱신 비율',
    hue='segment_group',
    legend=False
)

# 막대 위에 수치 표시
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',      # 소수점 1자리 %
        padding=3,
        fontsize=11
    )

plt.title('세그먼트별 구독 갱신율', fontsize=16, pad=15)
plt.xlabel('Segment Group', fontsize=12)
plt.ylabel('Renewal Rate (%)', fontsize=12)

plt.xticks(rotation=20)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# 숫자가 잘리지 않도록 여유 공간
plt.ylim(0, seg1_df['갱신 비율'].max() * 1.15)

plt.tight_layout()
plt.show()


In [ ]:
def segment2 (x):
  if 0 <= x < 5:
    return '저활동 유저'
  elif 5 <= x <= 13:
    return '활동 유저'
  elif 13 < x <= 32:
    return '고활동 유저'
  else:
    return '오류'

renew_rate_by_activation_days_temp['학습 활동일 수 그룹'] = renew_rate_by_activation_days_temp['active_learning_days_postpay_30d'].apply(lambda x : segment2(x))
seg2_df = renew_rate_by_activation_days_temp.groupby('학습 활동일 수 그룹').agg(
    유저_수 = ('유저_수', 'sum'),
    갱신_유저_수_60일내 = ('갱신유저_수_60일내', 'sum')
).reindex(['저활동 유저', '활동 유저', '고활동 유저']).reset_index()
seg2_df['갱신 비율'] = round((seg2_df['갱신_유저_수_60일내'] / seg2_df['유저_수']), 3) * 100
seg2_df


In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=seg2_df,
    x='학습 활동일 수 그룹',
    y='갱신 비율',
    hue='학습 활동일 수 그룹',
    legend=False
)

# 막대 위에 수치 표시
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',      # 소수점 1자리 %
        padding=3,
        fontsize=11
    )

plt.title('학습 활동일 수 그룹 별 갱신율', fontsize=16, pad=15)
plt.xlabel('학습 활동일 수 그룹', fontsize=12)
plt.ylabel('갱신율', fontsize=12)


plt.grid(axis='y', linestyle='--', alpha=0.3)

# 숫자가 잘리지 않도록 여유 공간
plt.ylim(0, seg1_df['갱신 비율'].max() * 1.3)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.lineplot(
    renew_rate_by_activation_days_temp,
    x='active_learning_days_postpay_30d',
    y='갱신유저_비율_60일내',
    marker='o',
    linewidth=2.5
)

plt.title(
    '결제 후 학습 활동일수와 60일 내 갱신율',
    fontsize=16,
    pad=15
)

plt.xlabel('결제 후 학습 활동일수', fontsize=12)
plt.ylabel('60일 내 갱신율 (%)', fontsize=12)

# y축이 31.5, 42.8 형태일 경우
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0f}%')

# 데이터 포인트 위에 값 표시
for x, y in zip(
    renew_rate_by_activation_days_temp['active_learning_days_postpay_30d'],
    renew_rate_by_activation_days_temp['갱신유저_비율_60일내']
):
    ax.annotate(
        f'{y:.1f}%',
        (x, y),
        textcoords='offset points',
        xytext=(0, 8),
        ha='center',
        fontsize=9
    )

plt.grid(
    axis='y',
    linestyle='--',
    alpha=0.3
)

sns.despine()
plt.tight_layout()
plt.ylim(30, 80)
plt.show()


- 분석 내용
  - 각 그룹 별 튀는 구간을 확인하기 위해 전체 활동일수를 기준으로 갱신율을 확인
  - 학습 활동일수 5일이 갱신율 상승의 첫 번째 임계점
    - 5일 ~ 15일 구간 간 50~60 % 사이의 비중이 많음
  - 15일 이상 학습했을 때를 두 번째 임계점
    - 15일 이상 그룹에서는 60~70% 사이의 비중이 많음
- 제안 방향
  - 특정 임계점까지 학습을 유도하기 위한 장치를 마련하는 것이 유리해보임
    - 알람 / 출석 체크 서비스
